# 🌿 JungleSurvivor — Kaggle Gradio Demo

**叢林求生離線 AI 助手 — 互動式 Demo**

上傳植物/動物照片，AI 辨識危險物種、可食植物、毒蛇，並提供求生建議。

**環境需求：** Kaggle Notebook + GPU T4 x2 + Gemma 4 E2B

**功能：**
1. 兩階段辨識系統（Stage 1 危險物種 → Stage 2 可食/藥用物種 → 合併取 Top 3）
2. 混淆物種自動偵測 + 互動式測試引導
3. 絕對信心度評估（特徵吻合越多 → 分數越高）
4. 進一步觀察指引（多角度拍攝、野外可食性測試）
5. 歷史紀錄追蹤
6. 多照片支援

**知識庫：** 有毒植物 20 種 | 可食植物 20 種 | 危險動物 13 種 | 混淆對 12 組 | 急救 SOP 5 份



In [ ]:
!pip install -q --upgrade transformers accelerate gradio

import json, re, os, torch, requests
from io import BytesIO
from PIL import Image
from dataclasses import dataclass, field
from typing import Optional
from datetime import datetime
from transformers import AutoProcessor, AutoModelForImageTextToText

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"         VRAM: {mem:.1f} GB")



In [ ]:
# === 模型設定 ===
MODEL_ID = "google/gemma-4-E2B-it"
MAX_NEW_TOKENS = 4096
DEFAULT_DTYPE = "bfloat16"

# === 辨識閾值 ===
THRESHOLDS = {
    "danger_screening": 60,
    "confusion_pairs": 80,
    "useful_resources": 70,
}

DEFAULT_REGION = "east_asia_subtropical"

AVAILABLE_REGIONS = {
    "east_asia_subtropical": {
        "name_zh": "東亞亞熱帶（台灣、華南、日本南部）",
        "climate_zone": "亞熱帶",
        "default_altitude": 500,
        "default_vegetation": "低海拔闊葉林",
    },
}

JSON_START_MARKER = "<JSON_START>"
JSON_END_MARKER = "<JSON_END>"

WARNING_LEVELS = {
    "RED": {"color": "#FF0000", "label_zh": "🔴 危險", "action": "立即遠離"},
    "YELLOW": {"color": "#FFD700", "label_zh": "🟡 注意", "action": "謹慎處理"},
    "GREEN": {"color": "#00AA00", "label_zh": "🟢 安全", "action": "可安全使用"},
    "GRAY": {"color": "#888888", "label_zh": "⚪ 不確定", "action": "無法判斷，建議不碰"},
}



## 📚 知識庫嵌入

以下 Cell 包含完整的離線知識庫資料（20 種有毒植物、20 種可食植物、13 種危險動物、12 組混淆對、5 份急救 SOP）。



In [ ]:
import json

_KB_TOXIC_PLANTS = json.loads("[{\"id\": \"alocasia_odora\", \"scientific_name\": \"Alocasia odora\", \"common_names\": {\"zh-TW\": \"\u59d1\u5a46\u828b\", \"en\": \"Giant Elephant Ear\"}, \"danger_level\": \"high\", \"toxicity\": \"\u5168\u682a\u6709\u6bd2\uff0c\u6c41\u6db2\u542b\u8349\u9178\u9223\u91dd\u6676\uff0c\u8aa4\u98df\u81f4\u53e3\u8154\u707c\u50b7\u3001\u8179\u7009\u3001\u5589\u56a8\u816b\u8139\", \"morphology\": {\"leaf_shape\": \"\u5927\u578b\u5fc3\u5f62\u6216\u7bad\u5f62\uff0c\u9577 50-90cm\uff0c\u5bec 40-60cm\", \"leaf_surface\": \"\u5149\u6ed1\u767c\u4eae\uff0c\u6df1\u7da0\u8272\uff0c\u6c34\u73e0\u4e0d\u6210\u73e0\u72c0\uff08\u6703\u6524\u5e73\uff09\", \"leaf_venation\": \"\u660e\u986f\u7684\u7fbd\u72c0\u8108\uff0c\u4e3b\u8108\u7c97\u58ef\u7a81\u51fa\", \"leaf_tip\": \"\u671d\u4e0a\u6216\u6c34\u5e73\u6307\u5411\", \"petiole\": \"\u7c97\u58ef\uff0c\u9577 50-100cm\uff0c\u7da0\u8272\u5e38\u5e36\u7d2b\u6688\uff0c\u63a5\u5728\u8449\u7247\u908a\u7de3\", \"stem\": \"\u7c97\u77ed\u76f4\u7acb\u8396\uff0c\u591a\u6c41\", \"flower\": \"\u4f5b\u7130\u82de\u82b1\u5e8f\uff0c\u9ec3\u7da0\u8272\", \"fruit\": \"\u7d05\u8272\u6f3f\u679c\u4e32\"}, \"habitat\": \"\u6797\u4e0b\u3001\u6eaa\u908a\u3001\u6f6e\u6fd5\u9670\u6697\u8655\uff0c\u6d77\u62d4 0-1500m\", \"confusion_with\": [\"colocasia_esculenta\"], \"affected_parts\": [\"\u5168\u682a\", \"\u6c41\u6db2\"], \"symptoms\": [\"\u53e3\u8154\u707c\u50b7\", \"\u820c\u982d\u9ebb\u75fa\", \"\u8179\u7009\", \"\u5589\u56a8\u816b\u8139\", \"\u5614\u5410\"], \"first_aid\": \"\u7acb\u5373\u7528\u5927\u91cf\u6e05\u6c34\u6f31\u53e3\uff0c\u52ff\u50ac\u5410\uff0c\u5118\u901f\u5c31\u91ab\u3002\u76ae\u819a\u63a5\u89f8\u6c41\u6db2\u6642\u4ee5\u5927\u91cf\u6e05\u6c34\u6c96\u6d17\u3002\", \"distribution\": {\"altitude_range\": [0, 1500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"cerbera_manghas\", \"scientific_name\": \"Cerbera manghas\", \"common_names\": {\"zh-TW\": \"\u6d77\u6aac\u679c\", \"en\": \"Sea Mango\"}, \"danger_level\": \"critical\", \"toxicity\": \"\u5168\u682a\u6709\u6bd2\uff0c\u7a2e\u5b50\u5287\u6bd2\uff0c\u542b\u6d77\u6aac\u679c\u6bd2\u7d20\uff08cerberin\uff09\uff0c\u8aa4\u98df\u53ef\u81f4\u5fc3\u81df\u9ebb\u75fa\u6b7b\u4ea1\", \"morphology\": {\"leaf_shape\": \"\u5012\u62ab\u91dd\u5f62\u6216\u9577\u6a62\u5713\u5f62\uff0c\u9577 15-30cm\uff0c\u5bec 3-5cm\uff0c\u9769\u8cea\", \"leaf_surface\": \"\u5149\u6ed1\uff0c\u6df1\u7da0\u8272\uff0c\u6709\u5149\u6fa4\", \"leaf_venation\": \"\u5074\u8108\u660e\u986f\uff0c\u5e73\u884c\u6392\u5217\", \"leaf_tip\": \"\u6f38\u5c16\", \"petiole\": \"\u77ed\uff0c\u9577 1-3cm\", \"stem\": \"\u76f4\u7acb\u55ac\u6728\uff0c\u9ad8\u53ef\u9054 8-15m\uff0c\u65b7\u9762\u6709\u767d\u8272\u4e73\u6c41\", \"flower\": \"\u767d\u8272\u4e94\u74e3\u82b1\uff0c\u4e2d\u5fc3\u7c89\u7d05\u8272\uff0c\u6709\u9999\u5473\uff0c\u805a\u7e56\u82b1\u5e8f\", \"fruit\": \"\u5375\u5713\u5f62\u6838\u679c\uff0c\u76f4\u5f91 5-7cm\uff0c\u9752\u7da0\u8f49\u6697\u7d05\uff0c\u5167\u542b\u5287\u6bd2\u7a2e\u5b50\"}, \"habitat\": \"\u6d77\u5cb8\u3001\u6cb3\u53e3\u3001\u7d05\u6a39\u6797\u908a\u7de3\u3001\u516c\u5712\uff08\u5e38\u4f5c\u666f\u89c0\u6a39\uff09\", \"confusion_with\": [], \"affected_parts\": [\"\u5168\u682a\", \"\u7a2e\u5b50\uff08\u5287\u6bd2\uff09\", \"\u4e73\u6c41\"], \"symptoms\": [\"\u5614\u5410\", \"\u8179\u75db\", \"\u5fc3\u5f8b\u4e0d\u6574\", \"\u5fc3\u81df\u9ebb\u75fa\", \"\u53ef\u81f4\u6b7b\"], \"first_aid\": \"\u7acb\u5373\u50ac\u5410\uff08\u82e5\u610f\u8b58\u6e05\u9192\uff09\uff0c\u5118\u901f\u9001\u91ab\uff0c\u544a\u77e5\u91ab\u5e2b\u7591\u4f3c\u6d77\u6aac\u679c\u4e2d\u6bd2\u3002\u6b64\u70ba\u5fc3\u81df\u6bd2\u6027\uff0c\u9700\u7dca\u6025\u8655\u7406\u3002\", \"distribution\": {\"altitude_range\": [0, 100], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"datura_stramonium\", \"scientific_name\": \"Datura stramonium\", \"common_names\": {\"zh-TW\": \"\u66fc\u9640\u7f85\", \"en\": \"Jimsonweed\"}, \"danger_level\": \"critical\", \"toxicity\": \"\u5168\u682a\u5287\u6bd2\uff0c\u542b\u83a8\u83ea\u9e7c\uff08scopolamine\uff09\u3001\u6771\u83a8\u83ea\u9e7c\uff08atropine\uff09\uff0c\u8aa4\u98df\u81f4\u8b6b\u5984\u3001\u5e7b\u89ba\u3001\u6b7b\u4ea1\", \"morphology\": {\"leaf_shape\": \"\u5375\u5f62\uff0c\u9577 8-20cm\uff0c\u908a\u7de3\u6709\u7c97\u92f8\u9f52\u6216\u4e0d\u898f\u5247\u88c2\u7247\", \"leaf_surface\": \"\u7565\u7c97\u7cd9\uff0c\u6697\u7da0\u8272\uff0c\u6413\u63c9\u5f8c\u6709\u81ed\u5473\", \"leaf_venation\": \"\u7db2\u72c0\u8108\", \"leaf_tip\": \"\u92b3\u5c16\", \"petiole\": \"\u9577 3-6cm\", \"stem\": \"\u76f4\u7acb\u8349\u672c\u6216\u4e9e\u704c\u6728\uff0c\u9ad8 50-150cm\uff0c\u8396\u7da0\u8272\u6216\u7d2b\u8272\uff0c\u5149\u6ed1\", \"flower\": \"\u767d\u8272\u6216\u6de1\u7d2b\u8272\u5587\u53ed\u5f62\u5927\u82b1\uff0c\u9577 6-10cm\uff0c\u55ae\u751f\u65bc\u679d\u814b\", \"fruit\": \"\u7403\u5f62\u84b4\u679c\uff0c\u76f4\u5f91 3-5cm\uff0c\u5916\u88ab\u786c\u523a\uff08\u95dc\u9375\u7279\u5fb5\uff09\"}, \"habitat\": \"\u8352\u5730\u3001\u8def\u65c1\u3001\u5ee2\u68c4\u5730\u3001\u8fb2\u7530\u908a\u7de3\", \"confusion_with\": [], \"affected_parts\": [\"\u5168\u682a\", \"\u7a2e\u5b50\uff08\u6fc3\u5ea6\u6700\u9ad8\uff09\", \"\u8449\"], \"symptoms\": [\"\u77b3\u5b54\u653e\u5927\", \"\u53e3\u4e7e\", \"\u5fc3\u8df3\u52a0\u901f\", \"\u5e7b\u89ba\", \"\u8b6b\u5984\", \"\u62bd\u6410\", \"\u660f\u8ff7\", \"\u53ef\u81f4\u6b7b\"], \"first_aid\": \"\u7acb\u5373\u50ac\u5410\uff08\u82e5\u610f\u8b58\u6e05\u9192\u4e14\u5728 1 \u5c0f\u6642\u5167\uff09\uff0c\u5118\u901f\u9001\u91ab\u3002\u6ce8\u610f\u7dad\u6301\u547c\u5438\u9053\u66a2\u901a\u3002\", \"distribution\": {\"altitude_range\": [0, 1000], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u6eab\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"urtica_thunbergiana\", \"scientific_name\": \"Urtica thunbergiana\", \"common_names\": {\"zh-TW\": \"\u54ac\u4eba\u8c93\", \"en\": \"Stinging Nettle (Taiwan)\"}, \"danger_level\": \"medium\", \"toxicity\": \"\u8396\u8449\u6709\u900f\u660e\u712e\u523a\u6bdb\uff0c\u89f8\u78b0\u6ce8\u5165\u87fb\u9178\u7b49\u5316\u5b78\u7269\u8cea\uff0c\u5f15\u8d77\u5287\u70c8\u707c\u75db\uff0c\u6301\u7e8c\u6578\u5c0f\u6642\", \"morphology\": {\"leaf_shape\": \"\u95ca\u5375\u5f62\u81f3\u5fc3\u5f62\uff0c\u9577 6-13cm\uff0c\u908a\u7de3\u6709\u7c97\u92f8\u9f52\", \"leaf_surface\": \"\u5169\u9762\u5747\u6709\u712e\u523a\u6bdb\uff08\u900f\u660e\u91dd\u72c0\u7a81\u8d77\uff0c\u8089\u773c\u53ef\u898b\uff09\u2014\u2014\u95dc\u9375\u7279\u5fb5\", \"leaf_venation\": \"\u7db2\u72c0\u8108\uff0c3-5 \u689d\u57fa\u51fa\u8108\", \"leaf_tip\": \"\u92b3\u5c16\", \"petiole\": \"\u9577 3-8cm\uff0c\u6709\u523a\u6bdb\", \"stem\": \"\u76f4\u7acb\u8349\u672c\uff0c\u9ad8 50-120cm\uff0c\u8396\u4e0a\u4e5f\u6709\u523a\u6bdb\uff0c\u65b9\u5f62\u8396\", \"flower\": \"\u7a57\u72c0\u82b1\u5e8f\uff0c\u5c0f\u578b\u7da0\u8272\u82b1\", \"fruit\": \"\u7626\u679c\uff0c\u5fae\u5c0f\"}, \"habitat\": \"\u4e2d\u6d77\u62d4\u5c71\u5340\u6797\u4e0b\uff0c\u6f6e\u6fd5\u9670\u6697\u8655\uff0c\u6d77\u62d4 500-3000m\", \"confusion_with\": [], \"affected_parts\": [\"\u8396\", \"\u8449\", \"\u712e\u523a\u6bdb\"], \"symptoms\": [\"\u5287\u70c8\u707c\u75db\", \"\u76ae\u819a\u7d05\u816b\", \"\u4e18\u75b9\", \"\u75bc\u75db\u6301\u7e8c\u6578\u5c0f\u6642\u81f3\u4e00\u5929\"], \"first_aid\": \"\u7528\u81a0\u5e36\u9ecf\u9664\u76ae\u819a\u4e0a\u7684\u523a\u6bdb\u3002\u51b7\u6c34\u6c96\u6d17\u60a3\u8655\u3002\u53ef\u5857\u62b9\u542b\u985e\u56fa\u9187\u7684\u6b62\u7662\u85e5\u818f\u3002\u907f\u514d\u6414\u6293\u3002\", \"distribution\": {\"altitude_range\": [500, 3000], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u6eab\u5e36\"]}}, {\"id\": \"dendrocnide_meyeniana\", \"scientific_name\": \"Dendrocnide meyeniana\", \"common_names\": {\"zh-TW\": \"\u54ac\u4eba\u72d7\", \"en\": \"Stinging Tree (Taiwan)\"}, \"danger_level\": \"high\", \"toxicity\": \"\u712e\u523a\u6bdb\u542b\u87fb\u9178\u7b49\u6bd2\u7d20\uff0c\u89f8\u78b0\u5f15\u8d77\u5287\u70c8\u707c\u75db\uff0c\u75bc\u75db\u53ef\u6301\u7e8c\u6578\u9031\u81f3\u6578\u6708\", \"morphology\": {\"leaf_shape\": \"\u5927\u578b\u5375\u5f62\u81f3\u5713\u5f62\uff0c\u9577 15-30cm\uff0c\u5bec 10-20cm\uff0c\u5168\u7de3\u6216\u5fae\u6ce2\u72c0\", \"leaf_surface\": \"\u6b63\u9762\u5149\u6ed1\u6216\u758f\u88ab\u77ed\u6bdb\uff0c\u80cc\u9762\u5bc6\u751f\u523a\u6bdb\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"leaf_venation\": \"\u660e\u986f\u7684\u4e09\u51fa\u8108\", \"leaf_tip\": \"\u920d\u5c16\u6216\u92b3\u5c16\", \"petiole\": \"\u9577 5-15cm\", \"stem\": \"\u704c\u6728\u81f3\u5c0f\u55ac\u6728\uff0c\u9ad8 2-5m\uff0c\u8396\u4e0a\u6709\u523a\u6bdb\", \"flower\": \"\u805a\u7e56\u82b1\u5e8f\uff0c\u5c0f\u578b\u6de1\u7da0\u8272\u82b1\", \"fruit\": \"\u5c0f\u578b\u6838\u679c\"}, \"habitat\": \"\u4f4e\u6d77\u62d4\u95ca\u8449\u6797\uff0c\u6797\u7de3\u6216\u958b\u95ca\u5730\uff0c\u6d77\u62d4 0-800m\", \"confusion_with\": [], \"affected_parts\": [\"\u5168\u682a\u8868\u9762\uff08\u523a\u6bdb\uff09\"], \"symptoms\": [\"\u5287\u70c8\u707c\u75db\", \"\u6301\u7e8c\u75bc\u75db\u6578\u9031\u81f3\u6578\u6708\", \"\u76ae\u819a\u7d05\u816b\", \"\u56b4\u91cd\u8005\u53ef\u80fd\u904e\u654f\u6027\u4f11\u514b\"], \"first_aid\": \"\u7528\u81a0\u5e36\u53cd\u8986\u9ecf\u9664\u523a\u6bdb\u3002\u51b7\u6c34\u6c96\u6d17\u3002\u52ff\u7528\u624b\u89f8\u6478\u60a3\u8655\u3002\u75bc\u75db\u6301\u7e8c\u8d85\u904e\u4e00\u5929\u61c9\u5c31\u91ab\u3002\", \"distribution\": {\"altitude_range\": [0, 800], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"chlorophyllum_molybdites\", \"scientific_name\": \"Chlorophyllum molybdites\", \"common_names\": {\"zh-TW\": \"\u7da0\u8936\u83c7\", \"en\": \"Green-spored Parasol\"}, \"danger_level\": \"high\", \"toxicity\": \"\u542b\u591a\u7a2e\u6bd2\u7d20\uff0c\u8aa4\u98df\u81f4\u56b4\u91cd\u8178\u80c3\u708e\uff0c\u70ba\u53f0\u7063\u6700\u5e38\u898b\u7684\u8aa4\u98df\u6bd2\u83c7\", \"morphology\": {\"cap\": \"\u5098\u84cb\u76f4\u5f91 10-30cm\uff0c\u521d\u7403\u5f62\u5f8c\u5c55\u5e73\uff0c\u767d\u8272\u81f3\u6de1\u8910\u8272\uff0c\u4e2d\u592e\u6709\u8910\u8272\u9c57\u7247\", \"gills\": \"\u521d\u767d\u8272\uff0c\u6210\u719f\u5f8c\u8f49\u70ba\u7da0\u8272\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"stipe\": \"\u83cc\u67c4\u9577 10-25cm\uff0c\u767d\u8272\uff0c\u57fa\u90e8\u81a8\u5927\uff0c\u6709\u53ef\u79fb\u52d5\u7684\u83cc\u74b0\", \"spore_print\": \"\u7da0\u8272\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"flesh\": \"\u767d\u8272\uff0c\u53d7\u50b7\u5f8c\u4e0d\u8b8a\u8272\u6216\u5fae\u8b8a\u8910\u8272\"}, \"habitat\": \"\u8349\u5730\u3001\u516c\u5712\u3001\u8349\u576a\u4e0a\u7fa4\u751f\uff0c\u96e8\u5f8c\u5927\u91cf\u51fa\u73fe\uff0c\u6d77\u62d4 0-500m\", \"confusion_with\": [\"macrolepiota_procera\"], \"affected_parts\": [\"\u5b50\u5be6\u9ad4\uff08\u5168\u682a\uff09\"], \"symptoms\": [\"\u56b4\u91cd\u5614\u5410\", \"\u8179\u7009\", \"\u8179\u75db\", \"\u812b\u6c34\", \"\u75c7\u72c0\u5728\u98df\u5f8c 1-3 \u5c0f\u6642\u51fa\u73fe\"], \"first_aid\": \"\u50ac\u5410\uff08\u82e5\u5728\u98df\u5f8c 1 \u5c0f\u6642\u5167\uff09\uff0c\u5927\u91cf\u88dc\u5145\u6c34\u5206\u9632\u6b62\u812b\u6c34\uff0c\u5118\u901f\u5c31\u91ab\u3002\u4fdd\u7559\u6b98\u9918\u83c7\u9ad4\u4f9b\u91ab\u5e2b\u8fa8\u8b58\u3002\", \"distribution\": {\"altitude_range\": [0, 500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"amanita_phalloides\", \"scientific_name\": \"Amanita phalloides\", \"common_names\": {\"zh-TW\": \"\u9c57\u67c4\u767d\u6bd2\u9d5d\u818f\uff08\u6b7b\u5e3d\u8548\u8fd1\u4f3c\u7a2e\uff09\", \"en\": \"Death Cap (related species)\"}, \"danger_level\": \"critical\", \"toxicity\": \"\u542b\u9d5d\u818f\u6bd2\u7d20\uff08amatoxin\uff09\uff0c\u81f4\u6b7b\u91cf\u6975\u4f4e\uff0c\u8aa4\u98df\u81f4\u809d\u814e\u8870\u7aed\uff0c\u6b7b\u4ea1\u7387\u6975\u9ad8\", \"morphology\": {\"cap\": \"\u5098\u84cb\u76f4\u5f91 5-15cm\uff0c\u767d\u8272\u81f3\u6de1\u7da0\u8272\u6216\u6de1\u9ec3\u8272\uff0c\u5149\u6ed1\uff0c\u5e7c\u6642\u534a\u7403\u5f62\u5f8c\u5c55\u5e73\", \"gills\": \"\u767d\u8272\uff0c\u96e2\u751f\uff0c\u5bc6\u96c6\u6392\u5217\", \"stipe\": \"\u83cc\u67c4\u9577 8-15cm\uff0c\u767d\u8272\uff0c\u57fa\u90e8\u6709\u660e\u986f\u7684\u83cc\u6258\uff08\u676f\u72c0\uff0c\u95dc\u9375\u7279\u5fb5\uff09\", \"ring\": \"\u83cc\u67c4\u4e0a\u90e8\u6709\u767d\u8272\u8584\u819c\u72c0\u83cc\u74b0\", \"spore_print\": \"\u767d\u8272\", \"flesh\": \"\u767d\u8272\uff0c\u7121\u7279\u6b8a\u6c23\u5473\uff08\u5371\u96aa\uff01\u6613\u88ab\u8a8d\u70ba\u7121\u6bd2\uff09\"}, \"habitat\": \"\u95ca\u8449\u6797\u6216\u91dd\u95ca\u6df7\u5408\u6797\u4e0b\uff0c\u5e38\u5728\u6a39\u6839\u9644\u8fd1\uff0c\u6d77\u62d4 500-2500m\", \"confusion_with\": [], \"affected_parts\": [\"\u5b50\u5be6\u9ad4\uff08\u5168\u682a\uff09\"], \"symptoms\": [\"\u6f5b\u4f0f\u671f 6-24 \u5c0f\u6642\uff08\u5371\u96aa\uff01\u521d\u671f\u7121\u75c7\u72c0\uff09\", \"\u5614\u5410\u8179\u7009\uff08\u5047\u6027\u6062\u5fa9\u671f\u5f8c\u52a0\u91cd\uff09\", \"\u809d\u814e\u8870\u7aed\", \"\u9ec3\u75b8\", \"\u591a\u5668\u5b98\u8870\u7aed\", \"\u6b7b\u4ea1\"], \"first_aid\": \"\u7acb\u5373\u9001\u91ab\uff01\u544a\u77e5\u7591\u4f3c\u9d5d\u818f\u6bd2\u7d20\u4e2d\u6bd2\u3002\u4fdd\u7559\u6b98\u9918\u83c7\u9ad4\u3002\u6b64\u6bd2\u7d20\u7121\u7279\u6548\u89e3\u6bd2\u5291\uff0c\u9700\u7dca\u6025\u652f\u6301\u6027\u6cbb\u7642\u3002\u6642\u9593\u662f\u95dc\u9375\u3002\", \"distribution\": {\"altitude_range\": [500, 2500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u6eab\u5e36\"]}}, {\"id\": \"toxicodendron_vernicifluum\", \"scientific_name\": \"Toxicodendron vernicifluum\", \"common_names\": {\"zh-TW\": \"\u6f06\u6a39\", \"en\": \"Chinese Lacquer Tree\"}, \"danger_level\": \"medium\", \"toxicity\": \"\u6a39\u6db2\u542b\u6f06\u915a\uff08urushiol\uff09\uff0c\u63a5\u89f8\u81f4\u56b4\u91cd\u904e\u654f\u6027\u63a5\u89f8\u6027\u76ae\u819a\u708e\", \"morphology\": {\"leaf_shape\": \"\u5947\u6578\u7fbd\u72c0\u8907\u8449\uff0c\u5c0f\u8449 7-13 \u679a\uff0c\u6bcf\u679a\u9577 7-15cm\uff0c\u5375\u5f62\u81f3\u6a62\u5713\u5f62\", \"leaf_surface\": \"\u5e7c\u8449\u5fae\u88ab\u6bdb\uff0c\u6210\u719f\u8449\u8fd1\u5149\u6ed1\uff0c\u79cb\u5b63\u8f49\u7d05\uff08\u6f02\u4eae\u4f46\u5371\u96aa\uff09\", \"leaf_venation\": \"\u7fbd\u72c0\u8108\", \"leaf_tip\": \"\u6f38\u5c16\", \"petiole\": \"\u8449\u8ef8\u9577 20-40cm\", \"stem\": \"\u843d\u8449\u55ac\u6728\uff0c\u9ad8 10-20m\uff0c\u6a39\u76ae\u7070\u767d\u8272\uff0c\u5207\u5272\u5f8c\u6d41\u51fa\u767d\u8272\u6f06\u6db2\uff08\u63a5\u89f8\u5373\u904e\u654f\uff09\", \"flower\": \"\u5713\u9310\u82b1\u5e8f\uff0c\u5c0f\u578b\u9ec3\u7da0\u8272\u82b1\", \"fruit\": \"\u6241\u7403\u5f62\u6838\u679c\uff0c\u6de1\u9ec3\u8272\"}, \"habitat\": \"\u4e2d\u4f4e\u6d77\u62d4\u95ca\u8449\u6797\uff0c\u5c71\u5761\u5730\uff0c\u6d77\u62d4 200-2500m\", \"confusion_with\": [], \"affected_parts\": [\"\u6a39\u6db2\uff08\u6f06\u6db2\uff09\", \"\u679d\u8449\"], \"symptoms\": [\"\u56b4\u91cd\u76ae\u819a\u7d05\u816b\", \"\u6c34\u6ce1\", \"\u5287\u70c8\u6414\u7662\", \"\u904e\u654f\u8005\u53ef\u80fd\u5168\u8eab\u6027\u53cd\u61c9\"], \"first_aid\": \"\u7acb\u5373\u7528\u5927\u91cf\u80a5\u7682\u6c34\u6e05\u6d17\u63a5\u89f8\u90e8\u4f4d\uff0815\u5206\u9418\u5167\u6700\u6709\u6548\uff09\u3002\u52ff\u6414\u6293\u6c34\u6ce1\u3002\u5857\u62b9\u542b\u985e\u56fa\u9187\u85e5\u818f\u3002\u56b4\u91cd\u8005\u5c31\u91ab\u3002\", \"distribution\": {\"altitude_range\": [200, 2500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u6eab\u5e36\"]}}, {\"id\": \"nerium_oleander\", \"scientific_name\": \"Nerium oleander\", \"common_names\": {\"zh-TW\": \"\u593e\u7af9\u6843\", \"en\": \"Oleander\"}, \"danger_level\": \"critical\", \"toxicity\": \"\u5168\u682a\u5287\u6bd2\uff0c\u542b\u5f37\u5fc3\u82f7\uff08oleandrin\uff09\uff0c\u8aa4\u98df\u6975\u5c11\u91cf\u5373\u53ef\u81f4\u5fc3\u81df\u505c\u8df3\u6b7b\u4ea1\", \"morphology\": {\"leaf_shape\": \"\u7dda\u72c0\u62ab\u91dd\u5f62\uff0c\u9577 10-20cm\uff0c\u5bec 1-2.5cm\uff0c\u9769\u8cea\uff0c\u8449\u7de3\u5168\u7de3\u7565\u53cd\u6372\", \"leaf_surface\": \"\u6df1\u7da0\u8272\uff0c\u5149\u6ed1\uff0c\u8449\u80cc\u6709\u660e\u986f\u4e2d\u8108\", \"leaf_venation\": \"\u7fbd\u72c0\u5074\u8108\u5bc6\u96c6\uff0c\u8fd1\u5e73\u884c\", \"leaf_arrangement\": \"\u4e09\u8449\u8f2a\u751f\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"petiole\": \"\u6975\u77ed\", \"stem\": \"\u5e38\u7da0\u704c\u6728\uff0c\u9ad8 2-5m\uff0c\u5206\u679d\u591a\uff0c\u679d\u689d\u7070\u7da0\u8272\", \"flower\": \"\u7c89\u7d05\u3001\u767d\u6216\u7d05\u8272\uff0c\u4e94\u74e3\uff0c\u6f0f\u6597\u5f62\uff0c\u805a\u7e56\u82b1\u5e8f\uff0c\u6709\u82b3\u9999\", \"fruit\": \"\u7d30\u9577\u84c7\u8456\u679c\uff0c\u9577 10-20cm\uff0c\u6210\u719f\u958b\u88c2\u9732\u51fa\u5e36\u6bdb\u7a2e\u5b50\"}, \"habitat\": \"\u516c\u5712\u3001\u884c\u9053\u6a39\u3001\u5ead\u5712\uff08\u5e38\u898b\u89c0\u8cde\u690d\u7269\uff09\uff0c\u6cb3\u5cb8\uff0c\u6d77\u62d4 0-500m\", \"confusion_with\": [], \"affected_parts\": [\"\u5168\u682a\uff08\u5305\u62ec\u4e7e\u67af\u7684\u8449\u548c\u679d\uff09\", \"\u82b1\u871c\", \"\u71c3\u71d2\u7159\u9727\u4e5f\u6709\u6bd2\"], \"symptoms\": [\"\u5641\u5fc3\u5614\u5410\", \"\u8179\u75db\", \"\u5fc3\u5f8b\u4e0d\u6574\", \"\u5fc3\u8df3\u904e\u6162\", \"\u8996\u89ba\u7570\u5e38\", \"\u5fc3\u81df\u505c\u8df3\", \"\u53ef\u81f4\u6b7b\"], \"first_aid\": \"\u7acb\u5373\u50ac\u5410\uff08\u82e5\u610f\u8b58\u6e05\u9192\uff09\uff0c\u5118\u901f\u9001\u91ab\u3002\u6b64\u70ba\u5fc3\u81df\u6bd2\u6027\uff0c\u9700\u6301\u7e8c\u5fc3\u96fb\u5716\u76e3\u6e2c\u3002\u52ff\u7528\u593e\u7af9\u6843\u679d\u689d\u70e4\u98df\u7269\u3002\", \"distribution\": {\"altitude_range\": [0, 500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\", \"\u6eab\u5e36\"]}}, {\"id\": \"cycas_revoluta\", \"scientific_name\": \"Cycas revoluta\", \"common_names\": {\"zh-TW\": \"\u8607\u9435\uff08\u9435\u6a39\uff09\", \"en\": \"Sago Palm\"}, \"danger_level\": \"high\", \"toxicity\": \"\u7a2e\u5b50\u548c\u5ae9\u8449\u542b\u8607\u9435\u82f7\uff08cycasin\uff09\uff0c\u8aa4\u98df\u81f4\u809d\u81df\u640d\u5bb3\uff0c\u6709\u81f4\u764c\u6027\", \"morphology\": {\"leaf_shape\": \"\u5927\u578b\u7fbd\u72c0\u8907\u8449\uff0c\u9577 50-150cm\uff0c\u5c0f\u8449\u7dda\u5f62\uff0c\u786c\u633a\uff0c\u908a\u7de3\u53cd\u6372\", \"leaf_surface\": \"\u6df1\u7da0\u8272\uff0c\u9769\u8cea\uff0c\u6709\u5149\u6fa4\", \"leaf_venation\": \"\u5c0f\u8449\u50c5\u4e00\u689d\u4e2d\u8108\", \"leaf_arrangement\": \"\u53e2\u751f\u65bc\u8396\u9802\uff0c\u5411\u5916\u8f3b\u5c04\u5c55\u958b\uff08\u68d5\u6ada\u72c0\uff09\", \"petiole\": \"\u8449\u67c4\u57fa\u90e8\u6709\u523a\", \"stem\": \"\u7c97\u58ef\u5713\u67f1\u5f62\u6a39\u5e79\uff0c\u8868\u9762\u6709\u8449\u75d5\uff0c\u9ad8 1-3m\uff08\u751f\u9577\u6975\u6162\uff09\", \"flower\": \"\u96cc\u96c4\u7570\u682a\uff0c\u96c4\u82b1\u5e8f\u5713\u67f1\u5f62\u76f4\u7acb\uff0c\u96cc\u82b1\u5e8f\u6241\u7403\u5f62\", \"fruit\": \"\u7a2e\u5b50\u7403\u5f62\uff0c\u76f4\u5f91 3-4cm\uff0c\u6210\u719f\u6642\u6a59\u7d05\u8272\"}, \"habitat\": \"\u516c\u5712\u3001\u5ead\u5712\uff08\u89c0\u8cde\u690d\u7269\uff09\uff0c\u5c71\u5761\u5730\uff0c\u6d77\u62d4 0-500m\", \"confusion_with\": [], \"affected_parts\": [\"\u7a2e\u5b50\uff08\u6bd2\u6027\u6700\u9ad8\uff09\", \"\u5ae9\u8449\", \"\u6839\"], \"symptoms\": [\"\u5614\u5410\", \"\u8179\u7009\", \"\u809d\u81df\u640d\u5bb3\uff08\u5ef6\u9072\u6027\uff09\", \"\u9ec3\u75b8\", \"\u9577\u671f\u63a5\u89f8\u6709\u81f4\u764c\u98a8\u96aa\"], \"first_aid\": \"\u50ac\u5410\uff08\u82e5\u5728\u98df\u5f8c 1 \u5c0f\u6642\u5167\uff09\uff0c\u5118\u901f\u9001\u91ab\u3002\u9700\u8ffd\u8e64\u809d\u529f\u80fd\u6307\u6a19\u3002\", \"distribution\": {\"altitude_range\": [0, 500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"lantana_camara\", \"scientific_name\": \"Lantana camara\", \"common_names\": {\"zh-TW\": \"\u99ac\u7e93\u4e39\", \"en\": \"Lantana\"}, \"danger_level\": \"high\", \"toxicity\": \"\u5168\u682a\u5177\u6bd2\u6027\uff0c\u4ee5\u672a\u6210\u719f\u7da0\u8272\u6f3f\u679c\u6bd2\u6027\u6700\u986f\u8457\uff0c\u542b\u4e09\u841c\u985e\u6bd2\u7d20 lantadene\uff1b\u8aa4\u98df\u53ef\u5f15\u8d77\u5614\u5410\u3001\u8179\u7009\u3001\u809d\u640d\u50b7\uff0c\u56b4\u91cd\u53ef\u5371\u53ca\u751f\u547d\u3002\u6210\u719f\u7d2b\u9ed1\u8272\u679c\u5be6\u4ecd\u5b58\u5728\u98a8\u96aa\uff0c\u61c9\u4e00\u5f8b\u907f\u514d\u63a1\u98df\u3002\", \"morphology\": {\"leaf_arrangement\": \"\u5c0d\u751f\uff0c\u5169\u5169\u76f8\u5c0d\u65bc\u8396\u7bc0\u4e0a\", \"leaf_shape\": \"\u5375\u5f62\u81f3\u95ca\u5375\u5f62\uff0c\u9577 3-8cm\uff0c\u5148\u7aef\u920d\u6216\u92b3\u5c16\uff0c\u8449\u7de3\u5177\u7c97\u92f8\u9f52\uff0c\u92f8\u9f52\u660e\u986f\u6613\u898b\", \"leaf_surface\": \"\u8868\u9762\u7565\u7c97\u7cd9\uff0c\u6df1\u7da0\u8272\uff0c\u5e38\u5e36\u77ed\u67d4\u6bdb\uff1b\u6413\u63c9\u8449\u7247\u6703\u6563\u51fa\u7279\u6b8a\u523a\u9f3b\u81ed\u5473\uff08\u8fa8\u8b58\u7dda\u7d22\uff09\", \"leaf_venation\": \"\u7fbd\u72c0\u7db2\u72c0\u8108\uff0c\u5074\u8108\u660e\u986f\u5411\u8449\u7de3\u5ef6\u4f38\", \"flower\": \"\u5c0f\u578b\u82b1\u5bc6\u96c6\u7d44\u6210\u982d\u72c0\u6216\u534a\u7403\u5f62\u7e56\u623f\u72c0\u82b1\u5e8f\uff1b\u82b1\u8272\u96a8\u958b\u82b1\u9032\u7a0b\u8b8a\u5316\uff0c\u5e38\u898b\u7531\u9ec3\u8f49\u6a59\u518d\u8f49\u7d05\uff0c\u4ea6\u6709\u7c89\u3001\u6a59\u7d05\u7b49\u8272\uff0c\u82b1\u671f\u9577\", \"fruit\": \"\u7403\u5f62\u6f3f\u679c\uff0c\u672a\u719f\u6642\u9bae\u7da0\u8272\u3001\u8cea\u5730\u786c\uff08\u6bd2\u6027\u98a8\u96aa\u9ad8\uff09\uff0c\u6210\u719f\u8f49\u6df1\u7d2b\u9ed1\u8272\uff1b\u591a\u9846\u805a\u751f\u65bc\u82b1\u5e8f\u8ef8\u4e0a\", \"stem\": \"\u6728\u8cea\u5316\u704c\u6728\u72c0\u5206\u679d\uff0c\u8396\u65b9\u6216\u7565\u5713\uff0c\u5e7c\u8396\u5e38\u88ab\u77ed\u6bdb\uff0c\u6613\u5f62\u6210\u5bc6\u751f\u704c\u53e2\"}, \"habitat\": \"\u53f0\u5317\u4f4e\u6d77\u62d4\u6975\u5e38\u898b\uff1a\u8352\u5730\u3001\u8def\u65c1\u3001\u516c\u5712\u7da0\u5e36\u3001\u5c71\u5761\u958b\u95ca\u5730\uff1b\u8010\u65f1\u8010\u8ca7\u7620\uff0c\u6d77\u62d4\u7d04 0-500m \u7686\u53ef\u80fd\u5927\u91cf\u51fa\u73fe\u3002\", \"confusion_with\": [], \"affected_parts\": [\"\u5168\u682a\", \"\u672a\u719f\u7da0\u8272\u6f3f\u679c\uff08\u9ad8\u98a8\u96aa\uff09\", \"\u6210\u719f\u6f3f\u679c\", \"\u8449\"], \"symptoms\": [\"\u5614\u5410\", \"\u8179\u7009\", \"\u8179\u75db\", \"\u865b\u5f31\", \"\u809d\u529f\u80fd\u7570\u5e38\uff08\u56b4\u91cd\u6642\uff09\", \"\u547c\u5438\u56f0\u96e3\uff08\u5c11\u898b\u4f46\u56b4\u91cd\uff09\"], \"first_aid\": \"\u52ff\u518d\u9032\u98df\uff1b\u82e5\u525b\u8aa4\u98df\u53ef\u65bc\u610f\u8b58\u6e05\u9192\u4e14\u91ab\u8b77\u6307\u793a\u4e0b\u8655\u7406\u3002\u5118\u901f\u5c31\u91ab\u4e26\u544a\u77e5\u8aa4\u98df\u99ac\u7e93\u4e39\u6f3f\u679c\u3002\u4fdd\u7559\u6a23\u672c\u7167\u7247\u6216\u679d\u689d\u4ee5\u5229\u8fa8\u8b58\u3002\u5927\u91cf\u6e05\u6c34\u6f31\u53e3\u3002\", \"distribution\": {\"altitude_range\": [0, 500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"parthenium_hysterophorus\", \"scientific_name\": \"Parthenium hysterophorus\", \"common_names\": {\"zh-TW\": \"\u9280\u81a0\u83ca\", \"en\": \"Parthenium Weed\"}, \"danger_level\": \"medium\", \"toxicity\": \"\u5165\u4fb5\u7a2e\u3002\u82b1\u7c89\u3001\u788e\u88c2\u690d\u682a\u8207\u6c41\u6db2\u53ef\u8a98\u767c\u56b4\u91cd\u904e\u654f\u6027\u76ae\u819a\u708e\uff08\u767c\u7d05\u3001\u816b\u3001\u6c34\u6ce1\u3001\u5287\u7662\uff09\uff0c\u4ea6\u53ef\u52a0\u91cd\u6216\u8a98\u767c\u6c23\u5598\u8207\u547c\u5438\u9053\u904e\u654f\u3002\u9577\u671f\u6216\u91cd\u8907\u66b4\u9732\u98a8\u96aa\u9ad8\u3002\", \"morphology\": {\"habit\": \"\u4e00\u5e74\u751f\u76f4\u7acb\u8349\u672c\uff0c\u682a\u9ad8\u901a\u5e38 30-100cm\uff0c\u591a\u5206\u679d\uff0c\u6574\u682a\u5448\u758f\u9b06\u704c\u53e2\u72c0\", \"leaf_arrangement\": \"\u8449\u4e92\u751f\uff0c\u4e0b\u4f4d\u8449\u67c4\u8f03\u9577\uff0c\u5411\u4e0a\u6f38\u77ed\", \"leaf_shape\": \"\u8449\u7fbd\u72c0\u6df1\u88c2\uff0c\u88c2\u7247\u72f9\u7a84\u3001\u7dda\u5f62\u81f3\u62ab\u91dd\u5f62\uff0c\u6574\u9ad4\u5916\u89c0\u7e41\u8907\u5982\u8439\u82e3\u985e\u6216\u8334\u84bf\u985e\u5207\u8449\", \"leaf_surface\": \"\u8868\u9762\u758f\u88ab\u77ed\u6bdb\u6216\u8fd1\u5149\u6ed1\uff0c\u7070\u7da0\u81f3\u6df1\u7da0\u8272\uff0c\u63c9\u788e\u6709\u8349\u672c\u690d\u7269\u6c23\u5473\", \"inflorescence\": \"\u82b1\u5e8f\u70ba\u591a\u6578\u5c0f\u578b\u767d\u8272\u982d\u72c0\u82b1\u5bc6\u96c6\u6392\u5217\u65bc\u679d\u7aef\u6216\u814b\u751f\u5713\u9310\u72c0\u5206\u679d\u4e0a\uff0c\u9060\u89c0\u5982\u5c0f\u767d\u83ca\u6216\u6eff\u5929\u661f\u72c0\u7684\u767d\u8272\u82b1\u5718\", \"flower_detail\": \"\u55ae\u6735\u982d\u72c0\u82b1\u5e8f\u6975\u5c0f\uff0c\u7e3d\u82de\u7247\u6578\u5c64\uff0c\u82b1\u51a0\u767d\u8272\uff0c\u96cc\u96c4\u540c\u682a\u6216\u7570\u682a\u72c0\u6392\u5217\u65bc\u540c\u4e00\u690d\u682a\u4e0d\u540c\u82b1\u5e8f\uff08\u65cf\u7fa4\u5167\u53ef\u898b\u5dee\u7570\uff09\", \"stem\": \"\u8396\u5713\u67f1\u5f62\uff0c\u5e7c\u6642\u5e38\u5e36\u7d2b\u7d05\u689d\u7d0b\uff0c\u8cea\u5730\u8106\uff0c\u6613\u6298\u65b7\u4e26\u63da\u8d77\u82b1\u7c89\u8207\u788e\u7247\"}, \"habitat\": \"\u8352\u5730\u3001\u8fb2\u7530\u908a\u3001\u8def\u65c1\u3001\u6cb3\u5cb8\u88f8\u9732\u5730\u3001\u5efa\u7bc9\u7a7a\u5730\uff1b\u96a8\u98a8\u50b3\u64ad\uff0c\u5728\u90fd\u6703\u908a\u7de3\u8207\u4f4e\u6d77\u62d4\u958b\u62d3\u5730\u5feb\u901f\u5b9a\u6b96\u3002\", \"confusion_with\": [], \"affected_parts\": [\"\u82b1\u7c89\", \"\u8449\u8207\u8396\u6c41\u6db2\", \"\u63da\u8d77\u7684\u690d\u682a\u788e\u5c51\"], \"symptoms\": [\"\u76ae\u819a\u7d05\u816b\", \"\u4e18\u75b9\u8207\u6c34\u6ce1\", \"\u5287\u70c8\u6414\u7662\", \"\u76ae\u708e\u53ef\u4e45\u6cbb\u4e0d\u6108\", \"\u54b3\u55fd\", \"\u5598\u9cf4\", \"\u9f3b\u7662\u5674\u568f\"], \"first_aid\": \"\u96e2\u958b\u73fe\u5834\u3002\u4ee5\u5927\u91cf\u6e05\u6c34\u6c96\u6d17\u76ae\u819a\uff0c\u66f4\u63db\u8863\u7269\u3002\u76ae\u819a\u708e\u53ef\u51b7\u6577\u4e26\u4f9d\u91ab\u56d1\u4f7f\u7528\u6297\u7d44\u7e54\u80fa\u6216\u985e\u56fa\u9187\u85e5\u818f\uff1b\u547c\u5438\u5598\u4fc3\u6216\u80f8\u60b6\u61c9\u7acb\u5373\u5c31\u91ab\u3002\u5f9e\u4e8b\u6e05\u9664\u5de5\u4f5c\u61c9\u6234\u53e3\u7f69\u3001\u9577\u8896\u8207\u624b\u5957\u3002\", \"distribution\": {\"altitude_range\": [0, 800], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"brugmansia_suaveolens\", \"scientific_name\": \"Brugmansia suaveolens\", \"common_names\": {\"zh-TW\": \"\u5927\u82b1\u66fc\u9640\u7f85\", \"en\": \"Angel's Trumpet\"}, \"danger_level\": \"critical\", \"toxicity\": \"\u5168\u682a\u5287\u6bd2\uff0c\u5bcc\u542b\u83a8\u83ea\u9e7c\u3001\u6771\u83a8\u83ea\u9e7c\u7b49\u83a8\u83ea\u70f7\u985e\u751f\u7269\u9e7c\uff1b\u8aa4\u98df\u6216\u8aa4\u7528\u5165\u85e5\u53ef\u81f4\u8b6b\u5984\u3001\u5e7b\u89ba\u3001\u77b3\u5b54\u653e\u5927\u3001\u5fc3\u8df3\u8207\u9ad4\u6eab\u7570\u5e38\u3001\u62bd\u6410\u3001\u660f\u8ff7\u53ca\u547c\u5438\u6291\u5236\uff0c\u81f4\u6b7b\u98a8\u96aa\u9ad8\u3002\", \"morphology\": {\"habit\": \"\u704c\u6728\u81f3\u5c0f\u55ac\u6728\uff0c\u9ad8\u5e38\u898b 2-5m\uff0c\u5ead\u5712\u8207\u516c\u5712\u5e38\u4fee\u526a\u6210\u6a39\u5f62\", \"leaf_shape\": \"\u55ae\u8449\u5927\u578b\uff0c\u5bec\u6a62\u5713\u5f62\u81f3\u5375\u5f62\uff0c\u9577\u53ef\u9054 15-30cm\uff0c\u8449\u57fa\u5e38\u6954\u5f62\u6216\u5713\u920d\uff0c\u8449\u5c16\u9577\u6f38\u5c16\uff0c\u5168\u7de3\u6216\u5fae\u6ce2\u72c0\u7de3\", \"leaf_surface\": \"\u8449\u9762\u6df1\u7da0\u3001\u8cea\u5730\u8584\u9769\u8cea\u81f3\u8349\u8cea\uff0c\u8449\u80cc\u8272\u8f03\u6de1\uff1b\u4e3b\u8108\u7c97\uff0c\u5074\u8108\u5f27\u5f62\u4e0a\u8209\", \"petiole\": \"\u8449\u67c4\u9577\uff0c\u7565\u7c97\uff0c\u652f\u6490\u4e0b\u5782\u7684\u5927\u8449\", \"flower\": \"\u82b1\u6975\u5927\uff0c\u9577\u7b52\u72c0\u5587\u53ed\u5f62\uff0c\u9577\u7d04 20-30cm\uff0c\u82b1\u51a0\u5148\u7aef\u958b\u5c55\uff1b\u95dc\u9375\u7279\u5fb5\u70ba\u82b1\u6735\u660e\u986f\u4e0b\u5782\uff08\u671d\u4e0b\u61f8\u639b\uff09\uff0c\u6709\u6fc3\u90c1\u751c\u9999\uff0c\u82b1\u8272\u4ee5\u767d\u6216\u6de1\u9ec3\u6700\u70ba\u5e38\u898b\", \"fruit\": \"\u84b4\u679c\u985e\uff0c\u8868\u9762\u5149\u6ed1\u7121\u660e\u986f\u786c\u523a\uff08\u8207\u66fc\u9640\u7f85 Datura \u5177\u523a\u84b4\u679c\u4e0d\u540c\uff0c\u70ba\u91cd\u8981\u5340\u5225\uff09\", \"comparison_note\": \"\u8207\u4e00\u5e74\u751f\u66fc\u9640\u7f85\u76f8\u8f03\uff1a\u672c\u7a2e\u55ac\u6728\u5316\u3001\u82b1\u4e0b\u5782\u4e14\u66f4\u5927\u3001\u679c\u5be6\u8868\u9762\u4e0d\u5177\u66fc\u9640\u7f85\u4e4b\u7403\u5f62\u523a\u84b4\u679c\u7279\u5fb5\"}, \"habitat\": \"\u6eab\u6696\u5730\u5340\u5ead\u5712\u3001\u516c\u5712\u3001\u5bfa\u5edf\u5468\u908a\u8207\u79c1\u4eba\u7da0\u5730\u5e38\u4f5c\u89c0\u8cde\u683d\u57f9\uff1b\u53f0\u5317\u4f4e\u6d77\u62d4\u90fd\u5e02\u7da0\u5730\u53ef\u898b\u3002\u91ce\u751f\u9038\u51fa\u500b\u9ad4\u5076\u898b\u65bc\u6eab\u6696\u907f\u98a8\u8655\u3002\", \"confusion_with\": [\"datura_stramonium\"], \"affected_parts\": [\"\u5168\u682a\", \"\u82b1\", \"\u8449\", \"\u7a2e\u5b50\"], \"symptoms\": [\"\u53e3\u4e7e\", \"\u77b3\u5b54\u653e\u5927\", \"\u8996\u529b\u6a21\u7cca\", \"\u5fc3\u60b8\", \"\u8b6b\u5984\", \"\u5e7b\u89ba\", \"\u8e81\u52d5\u6216\u55dc\u7761\", \"\u62bd\u6410\", \"\u660f\u8ff7\", \"\u547c\u5438\u8870\u7aed\"], \"first_aid\": \"\u7acb\u5373\u547c\u53eb\u6025\u6551\u3002\u52ff\u81ea\u884c\u50ac\u5410\u9664\u975e\u91ab\u8b77\u6307\u793a\u3002\u4fdd\u6301\u547c\u5438\u9053\u66a2\u901a\uff0c\u5074\u81e5\u9632\u55c6\u54b3\u3002\u5118\u901f\u9001\u91ab\u4e26\u651c\u5e36\u690d\u682a\u7167\u7247\u3002\u91ab\u9662\u4ee5\u652f\u6301\u7642\u6cd5\u70ba\u4e3b\uff0c\u53ef\u80fd\u9700\u93ae\u975c\u8207\u76e3\u6e2c\u3002\", \"distribution\": {\"altitude_range\": [0, 800], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"phytolacca_americana\", \"scientific_name\": \"Phytolacca americana\", \"common_names\": {\"zh-TW\": \"\u7f8e\u6d32\u5546\u9678\", \"en\": \"American Pokeweed\"}, \"danger_level\": \"high\", \"toxicity\": \"\u5168\u682a\u6709\u6bd2\uff0c\u4ee5\u6839\u90e8\u6bd2\u6027\u6700\u5f37\uff1b\u542b\u5546\u9678\u7682\u82f7\u3001\u5546\u9678\u6bd2\u7d20\u7b49\u3002\u8aa4\u98df\u53ef\u81f4\u56b4\u91cd\u5614\u5410\u3001\u8179\u7009\u3001\u96fb\u89e3\u8cea\u5931\u8861\uff0c\u4e43\u81f3\u4e2d\u6a1e\u6291\u5236\u8207\u5faa\u74b0\u8870\u7aed\u3002\u63a1\u91ce\u83dc\u8005\u6700\u6613\u8aa4\u8a8d\u3002\", \"morphology\": {\"habit\": \"\u591a\u5e74\u751f\u76f4\u7acb\u8349\u672c\uff0c\u9ad8\u53ef\u9054 1-3m\uff0c\u8396\u7c97\u58ef\uff0c\u5168\u682a\u591a\u6c41\", \"stem\": \"\u6210\u719f\u8396\u5e38\u5448\u7d05\u7d2b\u6216\u7d2b\u7d05\u8272\uff0c\u5e7c\u8396\u7da0\u8272\u5e36\u7d2b\u6591\uff0c\u6613\u8207\u67d0\u4e9b\u53ef\u98df\u8396\u8449\u690d\u7269\u6df7\u6dc6\uff08\u5371\u96aa\uff09\", \"leaf_shape\": \"\u5927\u578b\u4e92\u751f\u8449\uff0c\u6a62\u5713\u5f62\u81f3\u5375\u72c0\u62ab\u91dd\u5f62\uff0c\u9577 10-30cm\uff0c\u5bec 5-15cm\uff0c\u5148\u7aef\u6f38\u5c16\uff0c\u5168\u7de3\", \"leaf_surface\": \"\u8449\u8cea\u8584\u81f3\u7565\u539a\uff0c\u8868\u9762\u6df1\u7da0\uff0c\u8449\u80cc\u8f03\u6de1\uff1b\u8449\u67c4\u9577\uff0c\u8457\u751f\u65bc\u8396\u4e0a\u5448\u87ba\u65cb\u4e92\u751f\", \"flower\": \"\u7e3d\u72c0\u82b1\u5e8f\u814b\u751f\u6216\u9802\u751f\uff0c\u5c0f\u82b1\u767d\u8272\uff0c\u82b1\u6897\u8207\u82b1\u8ef8\u96a8\u767c\u80b2\u4f38\u9577\uff1b\u82b1\u671f\u9577\uff0c\u82b1\u5e8f\u76f4\u7acb\u6216\u7565\u4e0b\u5782\", \"fruit\": \"\u7403\u5f62\u6f3f\u679c\uff0c\u672a\u719f\u7da0\u8272\uff0c\u6210\u719f\u8f49\u7d2b\u9ed1\u4eae\u8272\uff1b\u591a\u679c\u5bc6\u96c6\u6392\u5217\u65bc\u5ef6\u9577\u4e4b\u82b1\u5e8f\u8ef8\u4e0a\u5448\u4e0b\u5782\u9577\u7a57\u72c0\uff08\u95dc\u9375\u8fa8\u8b58\u7d44\u5408\uff1a\u7d2b\u8396+\u9577\u7a57\u7d2b\u9ed1\u6f3f\u679c\uff09\", \"root\": \"\u4e3b\u6839\u7c97\u5927\u6210\u7d21\u9318\u72c0\u6216\u80a5\u539a\uff0c\u5916\u76ae\u6de1\u9ec3\u81f3\u7c89\u7d05\u8272\uff0c\u5207\u7247\u660e\u986f\u540c\u5fc3\u74b0\u7d0b\uff08\u91ce\u5916\u52ff\u8a66\u5690\uff09\"}, \"habitat\": \"\u8352\u5730\u3001\u6797\u7de3\u3001\u6eaa\u7554\u3001\u958b\u58be\u5730\u8207\u704c\u53e2\u908a\u7de3\uff1b\u4f4e\u6d77\u62d4\u81f3\u4e2d\u6d77\u62d4\u7686\u53ef\u5076\u9047\uff0c\u6f6e\u6fd5\u80a5\u6c83\u5730\u751f\u9577\u5c24\u65fa\u3002\", \"confusion_with\": [\"amaranthus_viridis\"], \"affected_parts\": [\"\u6839\uff08\u6700\u9ad8\u98a8\u96aa\uff09\", \"\u8396\", \"\u8449\", \"\u6f3f\u679c\"], \"symptoms\": [\"\u5641\u5fc3\u5614\u5410\", \"\u8179\u75db\u8179\u7009\", \"\u5927\u91cf\u8179\u7009\u81f4\u812b\u6c34\", \"\u982d\u6688\", \"\u5fc3\u8df3\u8b8a\u5316\", \"\u62bd\u6410\uff08\u56b4\u91cd\uff09\", \"\u547c\u5438\u6291\u5236\"], \"first_aid\": \"\u7981\u6b62\u518d\u9032\u98df\u3002\u610f\u8b58\u6e05\u695a\u6642\u5118\u901f\u5c31\u91ab\uff0c\u52ff\u81ea\u884c\u50ac\u5410\u9664\u975e\u91ab\u8b77\u5efa\u8b70\u3002\u4fdd\u7559\u6b98\u9918\u690d\u7269\u4f9b\u8fa8\u8b58\u3002\u6ce8\u610f\u88dc\u5145\u6c34\u5206\u8207\u96fb\u89e3\u8cea\u9700\u7531\u91ab\u7642\u4eba\u54e1\u8a55\u4f30\u3002\", \"distribution\": {\"altitude_range\": [0, 1500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u6eab\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"zantedeschia_aethiopica\", \"scientific_name\": \"Zantedeschia aethiopica\", \"common_names\": {\"zh-TW\": \"\u6d77\u828b\", \"en\": \"Calla Lily\"}, \"danger_level\": \"medium\", \"toxicity\": \"\u542b\u8349\u9178\u9223\u91dd\u6676\u53ca\u6eb6\u86cb\u767d\u9175\u7d20\u7b49\u523a\u6fc0\u6027\u6210\u5206\uff1b\u8aa4\u98df\u81f4\u53e3\u8154\u3001\u820c\u982d\u3001\u54bd\u5589\u707c\u75db\u3001\u6d41\u6d8e\u3001\u541e\u56a5\u56f0\u96e3\uff0c\u53ef\u4f34\u5614\u5410\u3002\u6c41\u6db2\u63a5\u89f8\u9ecf\u819c\u6216\u654f\u611f\u76ae\u819a\u4ea6\u53ef\u523a\u6fc0\u3002\", \"morphology\": {\"habit\": \"\u591a\u5e74\u751f\u8349\u672c\uff0c\u9ad8\u7d04 40-100cm\uff0c\u5177\u5730\u4e0b\u8396\uff0c\u5e38\u6210\u7fa4\u843d\u683d\u57f9\", \"leaf_shape\": \"\u8449\u5927\u578b\u7bad\u5f62\u81f3\u621f\u5f62\uff0c\u9577 15-45cm\uff0c\u8449\u57fa\u6df1\u5fc3\u5f62\u88c2\u7247\uff0c\u5148\u7aef\u6f38\u5c16\uff0c\u5168\u7de3\u5149\u6ed1\", \"leaf_surface\": \"\u6df1\u7da0\u8272\uff0c\u5177\u881f\u8cea\u5149\u6fa4\uff0c\u8cea\u5730\u539a\uff1b\u4e3b\u8108\u65bc\u80cc\u9762\u9686\u8d77\u660e\u986f\", \"petiole\": \"\u8449\u67c4\u9577\u800c\u7c97\uff0c\u6dfa\u7da0\u8272\uff0c\u5177\u7e31\u7a1c\uff0c\u652f\u6490\u8449\u7247\u633a\u7acb\u6216\u7565\u5916\u5c55\", \"flower\": \"\u95dc\u9375\u7279\u5fb5\u70ba\u7d14\u767d\u8272\uff08\u6216\u683d\u57f9\u8b8a\u7a2e\u5176\u4ed6\u8272\uff09\u5927\u578b\u4f5b\u7130\u82de\uff0c\u5448\u6f0f\u6597\u72c0\u6216\u5587\u53ed\u72c0\u5305\u88ab\u4e2d\u592e\u4e4b\u8089\u7a57\u82b1\u5e8f\uff1b\u8089\u7a57\u82b1\u5e8f\u9ec3\u8272\uff0c\u7c97\u77ed\u5713\u67f1\u5f62\uff0c\u6574\u9ad4\u5916\u89c0\u512a\u96c5\u4f3c\u300c\u767d\u82b1\u7248\u672c\u4e4b\u828b\u985e\u82b1\u5e8f\u300d\", \"seasonality\": \"\u683d\u57f9\u65bc\u7af9\u5b50\u6e56\u3001\u5ead\u5712\u6642\u958b\u82b1\u671f\u96c6\u4e2d\uff0c\u8352\u91ce\u9038\u51fa\u500b\u9ad4\u4ecd\u5177\u76f8\u540c\u5178\u578b\u767d\u82b1\u4f5b\u7130\u82de\", \"underground\": \"\u5730\u4e0b\u5177\u584a\u8396\u72c0\u6839\u8396\uff0c\u8089\u773c\u4e0d\u53ef\u898b\u6642\u4ecd\u8207\u53ef\u98df\u828b\u985e\u6982\u5ff5\u6df7\u6dc6\uff0c\u552f\u5730\u4e0a\u82b1\u90e8\u5f62\u614b\u5dee\u7570\u5927\"}, \"habitat\": \"\u967d\u660e\u5c71\u7af9\u5b50\u6e56\u53ca\u5317\u53f0\u7063\u9ad8\u51b7\u83dc\u5712\u5340\u5927\u91cf\u4f5c\u666f\u89c0\u8207\u5207\u82b1\u751f\u7522\uff1b\u6eaa\u908a\u6fd5\u5730\u3001\u5ead\u5712\u3001\u516c\u5712\u82b1\u53f0\u4ea6\u5e38\u898b\u683d\u57f9\u3002\u559c\u6fd5\u6f64\u6392\u6c34\u826f\u597d\u571f\u58e4\u3002\", \"confusion_with\": [\"colocasia_esculenta\", \"alocasia_odora\"], \"affected_parts\": [\"\u5168\u682a\", \"\u6c41\u6db2\", \"\u5730\u4e0b\u8396\", \"\u4f5b\u7130\u82de\u8207\u8089\u7a57\"], \"symptoms\": [\"\u53e3\u8154\u707c\u75db\", \"\u820c\u982d\u9ebb\u6728\", \"\u6d41\u6d8e\", \"\u5641\u5fc3\u5614\u5410\", \"\u541e\u56a5\u56f0\u96e3\", \"\u76ae\u819a\u7d05\u7662\uff08\u63a5\u89f8\u6027\uff09\"], \"first_aid\": \"\u6e05\u6c34\u6f31\u53e3\u4f46\u4e0d\u8981\u541e\u56a5\u5927\u91cf\u751f\u6c34\u9020\u6210\u55c6\u54b3\uff1b\u79fb\u9664\u53e3\u4e2d\u6b98\u6e23\u3002\u76ae\u819a\u4ee5\u5927\u91cf\u6e05\u6c34\u6c96\u6d17\u3002\u54bd\u5589\u816b\u8139\u6216\u547c\u5438\u4e0d\u9806\u7acb\u5373\u5c31\u91ab\u3002\u907f\u514d\u8207\u98df\u7528\u6cb9\u9e7c\u504f\u65b9\u81ea\u884c\u8655\u7406\u3002\", \"distribution\": {\"altitude_range\": [200, 1200], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u6eab\u5e36\"]}}, {\"id\": \"abrus_precatorius\", \"scientific_name\": \"Abrus precatorius\", \"common_names\": {\"zh-TW\": \"\u96de\u6bcd\u73e0\uff08\u76f8\u601d\u5b50\uff09\", \"en\": \"Rosary Pea\"}, \"danger_level\": \"critical\", \"toxicity\": \"\u7a2e\u5b50\u5287\u6bd2\uff0c\u542b\u96de\u6bcd\u73e0\u6bd2\u86cb\u767d abrin\uff0c\u6bd2\u6027\u6975\u5f37\uff0c\u54ac\u788e\u6216\u7a7f\u523a\u7a2e\u76ae\u5f8c\u91cb\u653e\uff1b\u6587\u737b\u8a18\u8f09\u6975\u5c11\u91cf\u7a2e\u5b50\u5373\u53ef\u81f4\u547d\u3002\u5168\u682a\u5176\u4ed6\u90e8\u4f4d\u4ea6\u4e0d\u5b9c\u98df\u7528\u3002\", \"morphology\": {\"habit\": \"\u6728\u8cea\u6500\u7de3\u85e4\u672c\uff0c\u6372\u9b1a\u6216\u7e8f\u7e5e\u4ed6\u7269\u4e0a\u5347\uff0c\u53ef\u8986\u84cb\u704c\u53e2\u8207\u7c6c\u7b06\", \"leaf_type\": \"\u5076\u6578\u7fbd\u72c0\u8907\u8449\uff0c\u5c0f\u8449 8-15 \u5c0d\uff0c\u5c0d\u751f\u6392\u5217\u65bc\u8449\u8ef8\", \"leaflet_shape\": \"\u5c0f\u8449\u9577\u6a62\u5713\u5f62\uff0c\u9577\u7d04 1-2cm\uff0c\u5bec 0.5-1cm\uff0c\u5148\u7aef\u5713\u6216\u5fae\u51f9\uff0c\u5168\u7de3\uff0c\u8cea\u5730\u8584\u9769\u8cea\uff0c\u8868\u9762\u6df1\u7da0\u5149\u6fa4\", \"flower\": \"\u8776\u5f62\u82b1\u51a0\uff0c\u82b1\u51a0\u6de1\u7c89\u81f3\u7d2b\u7d05\u8272\uff0c\u814b\u751f\u7e3d\u72c0\u82b1\u5e8f\uff0c\u5916\u5f62\u4f3c\u5c0f\u578b\u8c47\u8c46\u985e\u91ce\u82b1\", \"fruit\": \"\u83a2\u679c\u9577\u5713\u5f62\uff0c\u6210\u719f\u4e7e\u88c2\u53ef\u898b\u7a2e\u5b50\", \"seed\": \"\u7a2e\u5b50\u6a62\u5713\u6216\u9577\u5375\u5f62\uff0c\u95dc\u9375\u7279\u5fb5\u70ba\u9bae\u660e\u96d9\u8272\uff1a\u5178\u578b\u70ba\u4e0a\u534a\u9bae\u7d05\u3001\u4e0b\u534a\u4eae\u9ed1\uff08\u4f3c\u74e2\u87f2\u914d\u8272\uff09\uff0c\u8868\u9762\u5149\u6ed1\u6709\u5149\u6fa4\uff0c\u5e38\u7528\u65bc\u6c11\u4fd7\u98fe\u54c1\uff08\u6975\u5371\u96aa\uff0c\u52ff\u88fd\u4f5c\u523a\u7834\u7a2e\u76ae\u7684\u98fe\u54c1\uff09\"}, \"habitat\": \"\u53f0\u7063\u4f4e\u6d77\u62d4\u6797\u7de3\u3001\u704c\u53e2\u3001\u6cb3\u5cb8\u908a\u5761\u3001\u8352\u5730\uff1b\u5e38\u8207\u5176\u4ed6\u6500\u85e4\u6df7\u751f\u3002\u53f0\u5317\u5e02\u90ca\u5c71\u5340\u8207\u6cb3\u8c37\u958b\u95ca\u5730\u53ef\u80fd\u96f6\u661f\u81f3\u5c40\u90e8\u5e38\u898b\u3002\", \"confusion_with\": [], \"affected_parts\": [\"\u7a2e\u5b50\uff08\u5287\u6bd2\uff09\", \"\u83a2\u679c\", \"\u5176\u4ed6\u90e8\u4f4d\u4e0d\u5b9c\u98df\u7528\"], \"symptoms\": [\"\u5ef6\u9072\u6027\u8178\u80c3\u75c7\u72c0\", \"\u56b4\u91cd\u812b\u6c34\", \"\u809d\u814e\u640d\u50b7\", \"\u62bd\u6410\", \"\u4f11\u514b\", \"\u53ef\u81f4\u6b7b\"], \"first_aid\": \"\u61f7\u7591\u541e\u98df\u7a2e\u5b50\u8996\u70ba\u6025\u75c7\uff0c\u7acb\u5373\u9001\u91ab\u3002\u52ff\u7b49\u5f85\u75c7\u72c0\u3002\u544a\u77e5\u91ab\u5e2b\u6642\u9593\u8207\u53ef\u80fd\u6578\u91cf\u3002\u907f\u514d\u50ac\u5410\u9020\u6210\u4e8c\u6b21\u50b7\u5bb3\uff1b\u7531\u91ab\u7642\u7aef\u8655\u7f6e\u3002\u5207\u52ff\u628a\u73a9\u54ac\u7834\u7a2e\u5b50\u3002\", \"distribution\": {\"altitude_range\": [0, 800], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"ricinus_communis\", \"scientific_name\": \"Ricinus communis\", \"common_names\": {\"zh-TW\": \"\u84d6\u9ebb\", \"en\": \"Castor Bean\"}, \"danger_level\": \"critical\", \"toxicity\": \"\u7a2e\u5b50\u542b\u84d6\u9ebb\u6bd2\u86cb\u767d ricin \u53ca\u84d6\u9ebb\u9e7c\u7b49\uff1b\u56bc\u788e\u7a2e\u5b50\u53ef\u81f4\u56b4\u91cd\u8178\u80c3\u51fa\u8840\u3001\u5668\u5b98\u8870\u7aed\uff0c\u6975\u5c11\u91cf\u5373\u53ef\u81f4\u547d\u3002\u8449\u7b49\u4ea6\u53ef\u523a\u6fc0\u3002\", \"morphology\": {\"habit\": \"\u76f4\u7acb\u704c\u6728\u6216\u5c0f\u55ac\u6728\uff0c\u9ad8\u53ef\u9054 2-5m\uff0c\u8396\u7c97\uff0c\u9ad3\u758f\u9b06\uff0c\u5e7c\u8396\u5e38\u5e36\u767d\u7c89\u6216\u7d2b\u7d05\u689d\u7d0b\", \"leaf_shape\": \"\u55ae\u8449\u638c\u72c0\u6df1\u88c2 5-11 \u88c2\uff0c\u76f4\u5f91 15-45cm\uff0c\u88c2\u7247\u72f9\u62ab\u91dd\u5f62\u81f3\u5375\u72c0\uff0c\u5148\u7aef\u9577\u5c16\uff0c\u8449\u7de3\u6709\u92f8\u9f52\uff1b\u6574\u9ad4\u5982\u653e\u5c04\u72c0\u5de8\u8449\", \"petiole\": \"\u8449\u67c4\u6975\u9577\uff0c\u9023\u63a5\u65bc\u8449\u7247\u4e2d\u592e\u8fd1\u5713\u5fc3\u8655\uff0c\u652f\u6490\u5927\u578b\u8449\u65bc\u8396\u9802\u8f2a\u751f\u6216\u4e92\u751f\u6392\u5217\", \"leaf_surface\": \"\u8868\u9762\u4eae\u7da0\uff0c\u80cc\u9762\u8f03\u6de1\uff1b\u4e3b\u8108\u638c\u72c0\u81ea\u4e2d\u5fc3\u653e\u5c04\", \"flower\": \"\u9802\u751f\u7e3d\u72c0\u6216\u5713\u9310\u72c0\u82b1\u5e8f\uff0c\u55ae\u6027\u540c\u682a\uff0c\u96c4\u82b1\u5728\u4e0a\u3001\u96cc\u82b1\u5728\u4e0b\uff0c\u82b1\u90e8\u4e0d\u986f\u773c\uff0c\u7da0\u9ec3\u8272\", \"fruit\": \"\u7403\u5f62\u84b4\u679c\uff0c\u76f4\u5f91\u7d04 1.5-2.5cm\uff0c\u5916\u679c\u76ae\u5177\u8edf\u523a\u6216\u523a\u72c0\u7a81\u8d77\uff0c\u6210\u719f\u6642\u53ef\u88c2\u958b\", \"seed\": \"\u7a2e\u5b50\u6a62\u5713\u5f62\uff0c\u9577\u7d04 1-1.5cm\uff0c\u7a2e\u76ae\u5149\u6ed1\u5177\u8910\u8272\u5927\u7406\u77f3\u6591\u7d0b\uff0c\u7a2e\u81cd\u660e\u986f\uff08\u95dc\u9375\u8fa8\u8b58\uff09\"}, \"habitat\": \"\u8352\u5730\u3001\u8fb2\u7530\u65c1\u3001\u6cb3\u5e8a\u6c99\u6d32\u3001\u8def\u65c1\u8207\u516c\u5712\u5076\u898b\u683d\u57f9\u6216\u9038\u51fa\uff1b\u8010\u65f1\uff0c\u5728\u5168\u53f0\u4f4e\u6d77\u62d4\u53ef\u898b\u3002\", \"confusion_with\": [], \"affected_parts\": [\"\u7a2e\u5b50\uff08\u5287\u6bd2\uff09\", \"\u5e7c\u82d7\u8207\u8449\"], \"symptoms\": [\"\u5641\u5fc3\u5614\u5410\", \"\u8179\u75db\u8840\u4fbf\", \"\u812b\u6c34\", \"\u4f4e\u8840\u58d3\", \"\u62bd\u6410\", \"\u5c3f\u91cf\u6e1b\u5c11\", \"\u591a\u5668\u5b98\u8870\u7aed\"], \"first_aid\": \"\u7acb\u5373\u6025\u6551\u9001\u91ab\u3002\u52ff\u5ef6\u9072\u3002\u63d0\u4f9b\u541e\u98df\u6642\u9593\u8207\u662f\u5426\u5480\u56bc\u3002\u907f\u514d\u81ea\u884c\u704c\u8178\u6216\u504f\u65b9\u3002\u91ab\u9662\u4ee5\u652f\u6301\u7642\u6cd5\u8207\u76e3\u6e2c\u70ba\u4e3b\u3002\", \"distribution\": {\"altitude_range\": [0, 1200], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\", \"\u6eab\u5e36\"]}}, {\"id\": \"triadica_sebifera\", \"scientific_name\": \"Triadica sebifera\", \"common_names\": {\"zh-TW\": \"\u70cf\u6855\", \"en\": \"Chinese Tallow Tree\"}, \"danger_level\": \"medium\", \"toxicity\": \"\u6a39\u76ae\u3001\u8449\u67c4\u3001\u4e73\u6c41\u7b49\u542b\u523a\u6fc0\u6027\u6210\u5206\uff1b\u6c41\u6db2\u6216\u4e73\u6c41\u63a5\u89f8\u76ae\u819a\u53ef\u81f4\u7d05\u816b\u3001\u6c34\u6ce1\u53ca\u904e\u654f\u6027\u76ae\u819a\u708e\u3002\u8aa4\u98df\u90e8\u4f4d\u53ef\u81f4\u8178\u80c3\u4e0d\u9069\u3002\", \"morphology\": {\"habit\": \"\u843d\u8449\u55ac\u6728\uff0c\u9ad8 8-15m \u5e38\u898b\uff0c\u51a0\u5e45\u958b\u5c55\uff0c\u8001\u6a39\u6a39\u76ae\u7070\u8910\u7e31\u88c2\", \"leaf_shape\": \"\u55ae\u8449\u4e92\u751f\uff0c\u83f1\u5f62\u81f3\u83f1\u72c0\u5375\u5f62\uff0c\u9577 3-8cm\uff0c\u5bec 3-9cm\uff0c\u5148\u7aef\u5c3e\u72c0\u5c16\uff0c\u5168\u7de3\uff1b\u79cb\u5b63\u8449\u8272\u5e38\u8f49\u7d05\u6216\u6a59\u7d05\uff0c\u70ba\u91cd\u8981\u666f\u89c0\u8fa8\u8b58\u5b63\u76f8\", \"leaf_surface\": \"\u5e7c\u8449\u53ef\u7d05\u8272\uff0c\u6210\u8449\u8584\u9769\u8cea\uff0c\u8868\u9762\u6df1\u7da0\uff0c\u80cc\u9762\u7a0d\u6de1\uff0c\u4e7e\u71e5\u6642\u7565\u5448\u7d19\u8cea\u6613\u900f\u5149\", \"petiole\": \"\u8449\u67c4\u9802\u7aef\u8fd1\u8449\u7247\u57fa\u90e8\u8655\u5e38\u5177\u5169\u500b\u5c0f\u578b\u817a\u9ad4\uff08\u95dc\u9375\u7279\u5fb5\uff09\uff0c\u817a\u9ad4\u9ec3\u7da0\u8272\u5c0f\u7a81\u8d77\uff0c\u89c0\u5bdf\u89d2\u5ea6\u9700\u9006\u5149\u6216\u8fd1\u770b\", \"inflorescence\": \"\u7a57\u72c0\u82b1\u5e8f\uff0c\u96cc\u96c4\u540c\u682a\uff0c\u96c4\u82b1\u5e8f\u5728\u4e0a\uff0c\u96cc\u82b1\u5e8f\u5728\u4e0b\uff0c\u82b1\u9ec3\u7da0\u8272\uff0c\u98a8\u5a92\", \"fruit\": \"\u84b4\u679c\u8fd1\u7403\u5f62\uff0c\u719f\u6642\u9ed1\u8272\uff0c\u4e09\u88c2\uff1b\u7a2e\u5b50\u88ab\u8986\u767d\u8272\u881f\u8cea\u5047\u7a2e\u76ae\uff0c\u6574\u4e32\u639b\u679d\u4e0a\u4f3c\u5c0f\u767d\u9ede\u7db4\uff08\u63a1\u96c6\u767d\u881f\u7528\uff0c\u975e\u98df\u7269\uff09\"}, \"habitat\": \"\u53f0\u5317\u90fd\u5e02\u884c\u9053\u6a39\u3001\u6821\u5712\u3001\u6cb3\u5824\u3001\u4f4e\u6d77\u62d4\u4e18\u9675\u6b21\u751f\u6797\uff1b\u559c\u967d\u5149\uff0c\u5ee3\u6cdb\u683d\u57f9\u8207\u6b78\u5316\u3002\", \"confusion_with\": [], \"affected_parts\": [\"\u4e73\u6c41\uff0f\u6c41\u6db2\", \"\u7a2e\u5b50\u5916\u881f\u5c64\", \"\u8449\u53ca\u6a39\u76ae\"], \"symptoms\": [\"\u76ae\u819a\u7d05\u816b\", \"\u6414\u7662\", \"\u6c34\u6ce1\", \"\u8178\u80c3\u4e0d\u9069\uff08\u8aa4\u98df\uff09\"], \"first_aid\": \"\u76ae\u819a\u63a5\u89f8\u7acb\u5373\u80a5\u7682\u6c34\u5fb9\u5e95\u6e05\u6d17\u3002\u52ff\u6293\u7834\u6c34\u6ce1\u3002\u816b\u75db\u660e\u986f\u6216\u64f4\u5927\u5c31\u91ab\u3002\u8aa4\u98df\u5c11\u91cf\u4ea6\u5efa\u8b70\u8aee\u8a62\u6bd2\u7269\u79d1\u6216\u6025\u8a3a\u3002\", \"distribution\": {\"altitude_range\": [0, 1000], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\", \"\u6eab\u5e36\"]}}, {\"id\": \"epipremnum_aureum\", \"scientific_name\": \"Epipremnum aureum\", \"common_names\": {\"zh-TW\": \"\u9ec3\u91d1\u845b\", \"en\": \"Golden Pothos\"}, \"danger_level\": \"low\", \"toxicity\": \"\u542b\u8349\u9178\u9223\u91dd\u6676\uff1b\u54ac\u98df\u53ef\u81f4\u53e3\u8154\u523a\u75db\u3001\u6d41\u6d8e\u3001\u5614\u5410\uff1b\u6c41\u6db2\u63a5\u89f8\u773c\u775b\u6216\u50b7\u53e3\u6703\u5287\u70c8\u523a\u6fc0\u3002\u5c45\u5bb6\u8207\u516c\u5712\u5e38\u898b\uff0c\u5152\u7ae5\u8207\u5bf5\u7269\u8aa4\u98df\u98a8\u96aa\u9700\u6ce8\u610f\u3002\", \"morphology\": {\"habit\": \"\u5e38\u7da0\u6500\u7de3\u85e4\u672c\uff0c\u5e7c\u682a\u8513\u751f\u6216\u5782\u540a\uff0c\u6210\u682a\u7bc0\u4e0a\u53ef\u898b\u767c\u9054\u6c23\u751f\u6839\u9644\u8457\u6a39\u5e79\u6216\u7246\u9762\", \"leaf_shape\": \"\u5e7c\u8449\u5fc3\u5f62\u5168\u7de3\uff0c\u6210\u682a\u8449\u53ef\u8b8a\u5927\u578b\u88c2\u8449\uff1b\u4e00\u822c\u89c0\u8cde\u578b\u4ee5\u5fc3\u5f62\u8449\u70ba\u4e3b\uff0c\u9577 5-15cm \u8996\u9f61\u671f\u8b8a\u5316\", \"leaf_surface\": \"\u9769\u8cea\u5149\u6fa4\uff0c\u6df1\u7da0\u5e95\u8272\uff0c\u8868\u9762\u6709\u9ec3\u8272\u81f3\u4e73\u9ec3\u8272\u7e31\u5411\u6591\u7d0b\u6216\u584a\u6591\uff08\u5712\u85dd\u54c1\u7cfb\u773e\u591a\uff0c\u6591\u7d0b\u5bc6\u5ea6\u4e0d\u4e00\uff09\", \"leaf_venation\": \"\u5169\u57fa\u51fa\u8108\u81ea\u5fc3\u5f62\u57fa\u90e8\u4f38\u5411\u8449\u5c16\u5169\u5074\uff0c\u5074\u8108\u5f27\u66f2\", \"stem\": \"\u8396\u7bc0\u9593\u660e\u986f\uff0c\u8513\u8396\u5713\u67f1\uff0c\u5e38\u7da0\uff0c\u71df\u990a\u5145\u8db3\u6642\u7bc0\u9593\u7e2e\u77ed\u5448\u5bc6\u8449\", \"aerial_roots\": \"\u6c23\u6839\u68d5\u8272\uff0c\u591a\u689d\u81ea\u7bc0\u4e0a\u4f38\u51fa\uff0c\u53ef\u6500\u9644\u7c97\u7cd9\u8868\u9762\"}, \"habitat\": \"\u5ba4\u5167\u76c6\u683d\u3001\u5927\u6a13\u7da0\u7246\u3001\u516c\u5712\u990a\u8b77\u826f\u597d\u7684\u7da0\u7c6c\u8207\u6a39\u5e79\u8986\u88ab\uff1b\u53f0\u5317\u90fd\u6703\u5340\u6975\u5ea6\u5e38\u898b\u3002\u8010\u9670\uff0c\u4ea6\u898b\u65bc\u534a\u65e5\u7167\u5916\u7246\u3002\", \"confusion_with\": [], \"affected_parts\": [\"\u8449\", \"\u8396\", \"\u6c41\u6db2\"], \"symptoms\": [\"\u53e3\u8154\u707c\u71b1\", \"\u6d41\u6d8e\", \"\u5641\u5fc3\u5614\u5410\", \"\u76ae\u819a\u523a\u6fc0\", \"\u773c\u90e8\u5287\u75db\uff08\u6c41\u6db2\u8aa4\u5165\uff09\"], \"first_aid\": \"\u6e05\u6c34\u6f31\u53e3\uff1b\u773c\u775b\u4ee5\u751f\u7406\u98df\u9e7d\u6c34\u6216\u6e05\u6c34\u6c96\u6d17\u81f3\u5c11 15 \u5206\u9418\u4e26\u5c31\u91ab\u3002\u76ae\u819a\u6c96\u6d17\u4e7e\u6de8\u3002\u591a\u6578\u75c7\u72c0\u53ef\u6f38\u7de9\uff0c\u4f46\u816b\u8139\u5f71\u97ff\u547c\u5438\u9700\u6025\u8a3a\u3002\", \"distribution\": {\"altitude_range\": [0, 500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"digitalis_purpurea\", \"scientific_name\": \"Digitalis purpurea\", \"common_names\": {\"zh-TW\": \"\u6bdb\u5730\u9ec3\", \"en\": \"Common Foxglove\"}, \"danger_level\": \"critical\", \"toxicity\": \"\u5168\u682a\u5287\u6bd2\uff0c\u5bcc\u542b\u5f37\u5fc3\u82f7\uff08cardiac glycosides\uff09\u3002\u8aa4\u98df\u53ef\u81f4\u5641\u5fc3\u5614\u5410\u3001\u8996\u89ba\u7570\u5e38\uff08\u9ec3\u8996\u3001\u7da0\u8996\uff09\u3001\u5fc3\u5f8b\u4e0d\u6574\uff0c\u56b4\u91cd\u5fc3\u5ba4\u983b\u8108\u6216\u7de9\u8108\u7686\u53ef\u81f4\u547d\uff1b\u8207\u9ad4\u8cea\u53ca\u5291\u91cf\u76f8\u95dc\u3002\", \"morphology\": {\"habit\": \"\u4e8c\u5e74\u751f\u8349\u672c\uff0c\u7b2c\u4e00\u5e74\u5e38\u53ea\u751f\u84ee\u5ea7\u8449\uff0c\u7b2c\u4e8c\u5e74\u62bd\u85b9\uff0c\u9ad8 60-150cm\", \"basal_leaves\": \"\u57fa\u751f\u8449\u53e2\u751f\uff0c\u9577\u6a62\u5713\u81f3\u5375\u5f62\uff0c\u8449\u57fa\u6f38\u72f9\u6210\u67c4\uff0c\u8449\u9762\u5bc6\u88ab\u77ed\u7d68\u6bdb\uff0c\u89f8\u611f\u7c97\u7cd9\u7070\u7da0\u8272\", \"stem_leaves\": \"\u8396\u751f\u8449\u4e92\u751f\uff0c\u5411\u4e0a\u6f38\u5c0f\uff0c\u8449\u67c4\u77ed\u6216\u7121\u67c4\uff0c\u8449\u7247\u4ecd\u5bc6\u6bdb\", \"inflorescence\": \"\u9802\u751f\u7e3d\u72c0\u82b1\u5e8f\uff0c\u82b1\u6735\u6578\u91cf\u591a\uff0c\u7531\u4e0b\u65b9\u4f9d\u5e8f\u5411\u4e0a\u958b\u5c55\", \"flower\": \"\u82b1\u51a0\u9418\u5f62\u6216\u7b52\u72c0\u9418\u5f62\uff0c\u7565\u4e0b\u5782\uff1b\u82b1\u8272\u5e38\u7d2b\u7d05\u8272\uff0c\u51a0\u7b52\u5167\u5074\u6709\u660e\u986f\u6df1\u8272\u6591\u9ede\u6216\u7db2\u7d0b\uff08\u95dc\u9375\u5712\u85dd\u8207\u91ce\u751f\u8fa8\u8b58\u7279\u5fb5\uff09\uff1b\u82b1\u843c\u88c2\u7247\u77ed\", \"stem\": \"\u8396\u7c97\u58ef\u76f4\u7acb\uff0c\u56db\u89d2\u6216\u5713\u67f1\uff0c\u5168\u682a\u88ab\u6bdb\uff0c\u8cea\u8106\u6613\u6298\"}, \"habitat\": \"\u51b7\u6dbc\u6fd5\u6f64\u5c71\u5761\u3001\u6797\u7de3\u8349\u5730\uff1b\u967d\u660e\u5c71\u53ca\u4e2d\u9ad8\u6d77\u62d4\u6dbc\u51b7\u5340\u6709\u539f\u751f\u6216\u683d\u57f9\u9038\u51fa\u53ef\u80fd\u3002\u90fd\u5e02\u82b1\u5703\u4ea6\u898b\u5712\u85dd\u7a2e\u690d\u3002\", \"confusion_with\": [], \"affected_parts\": [\"\u5168\u682a\", \"\u82b1\", \"\u8449\", \"\u7a2e\u5b50\"], \"symptoms\": [\"\u5641\u5fc3\u5614\u5410\", \"\u8179\u75db\", \"\u8179\u7009\", \"\u982d\u6688\", \"\u8996\u529b\u7570\u5e38\uff08\u8272\u89ba\u6539\u8b8a\uff09\", \"\u5fc3\u7387\u4e0d\u6574\", \"\u6688\u53a5\"], \"first_aid\": \"\u7acb\u5373\u9001\u91ab\u76e3\u6e2c\u5fc3\u96fb\u5716\uff1b\u52ff\u81ea\u884c\u670d\u7528\u89e3\u85e5\u3002\u4fdd\u7559\u690d\u682a\u3002\u82e5\u610f\u8b58\u6e05\u9192\u53ef\u4f9d\u6307\u793a\u8655\u7406\uff0c\u4f46\u5f37\u5fc3\u82f7\u4e2d\u6bd2\u4ee5\u91ab\u9662\u8655\u7f6e\u70ba\u4e3b\u3002\u907f\u514d\u5496\u5561\u8207\u523a\u6fc0\u7269\u3002\", \"distribution\": {\"altitude_range\": [500, 2500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u6eab\u5e36\"]}}]")

_KB_EDIBLE_PLANTS = json.loads("[{\"id\": \"asplenium_nidus\", \"scientific_name\": \"Asplenium nidus\", \"common_names\": {\"zh-TW\": \"\u5c71\u8607\uff08\u9ce5\u5de2\u8568\uff09\", \"en\": \"Bird's Nest Fern\"}, \"category\": \"edible\", \"edible_parts\": [\"\u5ae9\u6372\u8449\uff08\u4e2d\u5fc3\u5c1a\u672a\u5c55\u958b\u7684\u5e7c\u8449\uff09\"], \"morphology\": {\"leaf_shape\": \"\u5927\u578b\u55ae\u8449\uff0c\u9577\u62ab\u91dd\u5f62\uff0c\u9577 50-120cm\uff0c\u5bec 10-15cm\", \"leaf_arrangement\": \"\u53e2\u751f\uff0c\u5f9e\u4e2d\u5fc3\u5411\u5916\u8f3b\u5c04\u5c55\u958b\uff0c\u5f62\u6210\u9ce5\u5de2\u72c0\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"leaf_surface\": \"\u5149\u6ed1\uff0c\u9769\u8cea\uff0c\u5168\u7de3\uff08\u7121\u92f8\u9f52\uff09\uff0c\u9bae\u7da0\u8272\", \"midrib\": \"\u660e\u986f\u7684\u9ed1\u8910\u8272\u4e2d\u808b\", \"sori\": \"\u8449\u80cc\u6709\u7dda\u72c0\u5b62\u5b50\u56ca\u7fa4\uff0c\u6cbf\u5074\u8108\u6392\u5217\", \"growth_pattern\": \"\u9644\u751f\u5728\u6a39\u5e79\u4e0a\u6216\u5ca9\u77f3\u4e0a\"}, \"habitat\": \"\u4e2d\u4f4e\u6d77\u62d4\u6f6e\u6fd5\u68ee\u6797\uff0c\u9644\u751f\u65bc\u5927\u6a39\u5e79\u6216\u5ca9\u58c1\uff0c\u6d77\u62d4 200-1500m\", \"harvesting\": {\"target\": \"\u53ea\u63a1\u4e2d\u5fc3\u5c1a\u672a\u5b8c\u5168\u5c55\u958b\u7684\u5ae9\u6372\u8449\", \"method\": \"\u7528\u624b\u6216\u5200\u5f9e\u57fa\u90e8\u6298\u65b7\u5ae9\u82bd\uff0c\u4fdd\u7559\u6210\u719f\u8449\u78ba\u4fdd\u690d\u7269\u6301\u7e8c\u751f\u9577\", \"season\": \"\u5168\u5e74\u53ef\u63a1\uff0c\u6625\u590f\u5ae9\u82bd\u6700\u591a\", \"sustainability\": \"\u6bcf\u682a\u6700\u591a\u63a1 1-2 \u652f\u5ae9\u82bd\uff0c\u52ff\u904e\u5ea6\u63a1\u96c6\"}, \"preparation\": {\"method\": \"\u5ddd\u71d9\u5f8c\u5373\u53ef\u98df\u7528\uff0c\u4e5f\u53ef\u5feb\u7092\", \"cooking_time\": \"\u5ddd\u71d9 1-2 \u5206\u9418\", \"taste\": \"\u53e3\u611f\u6ed1\u5ae9\uff0c\u5fae\u5e36\u9ecf\u6027\", \"notes\": \"\u53ef\u52a0\u8591\u7d72\u3001\u849c\u672b\u8abf\u5473\uff0c\u662f\u53f0\u7063\u539f\u4f4f\u6c11\u50b3\u7d71\u91ce\u83dc\"}, \"nutrition\": \"\u5bcc\u542b\u7dad\u751f\u7d20A\u3001\u9223\u3001\u9435\u3001\u81b3\u98df\u7e96\u7dad\", \"caution\": \"\u78ba\u8a8d\u662f\u5c71\u8607\u800c\u975e\u5176\u4ed6\u8568\u985e\u3002\u53ea\u98df\u7528\u5ae9\u6372\u8449\uff0c\u6210\u719f\u8449\u8f03\u786c\u4e0d\u9069\u5408\u98df\u7528\u3002\", \"confusion_risk\": \"low\", \"distribution\": {\"altitude_range\": [200, 1500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"diplazium_esculentum\", \"scientific_name\": \"Diplazium esculentum\", \"common_names\": {\"zh-TW\": \"\u904e\u8c93\uff08\u904e\u6e9d\u83dc\u8568\uff09\", \"en\": \"Vegetable Fern\"}, \"category\": \"edible\", \"edible_parts\": [\"\u5ae9\u8396\", \"\u5e7c\u8449\uff08\u6372\u66f2\u7684\u62f3\u982d\u72c0\u5ae9\u82bd\uff09\"], \"morphology\": {\"leaf_shape\": \"\u4e8c\u56de\u7fbd\u72c0\u8907\u8449\uff0c\u6210\u719f\u682a\u9ad8\u53ef\u9054 1-2m\", \"leaf_arrangement\": \"\u53e2\u751f\", \"young_shoot\": \"\u9802\u7aef\u6372\u66f2\u5448\u62f3\u982d\u72c0\uff08\u8568\u985e\u5178\u578b\u7279\u5fb5\uff09\u2014\u2014\u53ef\u98df\u90e8\u4f4d\", \"leaf_surface\": \"\u5ae9\u8449\u7da0\u8272\u5e36\u5fae\u7d05\uff0c\u6709\u7d30\u6bdb\", \"stem\": \"\u8396\u90e8\u9ed1\u8910\u8272\uff0c\u6709\u9c57\u7247\"}, \"habitat\": \"\u6eaa\u908a\u3001\u6f6e\u6fd5\u5730\u3001\u6c34\u7530\u65c1\uff0c\u6d77\u62d4 0-1000m\", \"harvesting\": {\"target\": \"\u5ae9\u8396\u9802\u7aef\u6372\u66f2\u8655\uff08\u7d04 15-20cm\uff09\", \"method\": \"\u7528\u624b\u5f9e\u67d4\u8edf\u8655\u6298\u65b7\", \"season\": \"\u5168\u5e74\u53ef\u63a1\uff0c\u6625\u590f\u6700\u5ae9\", \"sustainability\": \"\u53ea\u63a1\u5ae9\u82bd\uff0c\u4fdd\u7559\u6839\u90e8\u548c\u6210\u719f\u690d\u682a\"}, \"preparation\": {\"method\": \"\u5ddd\u71d9\u6216\u5feb\u7092\uff0c\u52d9\u5fc5\u716e\u719f\", \"cooking_time\": \"\u5ddd\u71d9 2-3 \u5206\u9418\", \"taste\": \"\u53e3\u611f\u6ed1\u5ae9\uff0c\u5fae\u5e36\u9ecf\u6db2\u611f\", \"notes\": \"\u4e0d\u5efa\u8b70\u751f\u98df\u3002\u53ef\u6dbc\u62cc\uff08\u5ddd\u71d9\u5f8c\uff09\u6216\u7092\u8089\u7d72\u3002\"}, \"nutrition\": \"\u86cb\u767d\u8cea\u3001\u9435\u8cea\u3001\u7dad\u751f\u7d20C\", \"caution\": \"\u907f\u514d\u63a1\u98df\u4e0d\u8a8d\u8b58\u7684\u8568\u985e\u3002\u90e8\u5206\u8568\u985e\u542b\u81f4\u764c\u7269\u8cea\uff0c\u904e\u8c93\u9700\u716e\u719f\u98df\u7528\u3002\", \"confusion_risk\": \"medium\", \"distribution\": {\"altitude_range\": [0, 1000], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"hedychium_coronarium\", \"scientific_name\": \"Hedychium coronarium\", \"common_names\": {\"zh-TW\": \"\u91ce\u8591\u82b1\", \"en\": \"White Ginger Lily\"}, \"category\": \"edible_and_medicinal\", \"edible_parts\": [\"\u82b1\u74e3\", \"\u5ae9\u8396\", \"\u6839\u8396\uff08\u8abf\u5473\u7528\uff09\"], \"morphology\": {\"leaf_shape\": \"\u9577\u6a62\u5713\u5f62\u62ab\u91dd\u72c0\uff0c\u9577 20-40cm\uff0c\u4e92\u751f\u6392\u5217\", \"leaf_surface\": \"\u7da0\u8272\uff0c\u5149\u6ed1\uff0c\u8449\u80cc\u5fae\u88ab\u6bdb\", \"flower\": \"\u767d\u8272\u8774\u8776\u5f62\u82b1\u6735\uff0c\u9999\u6c23\u6fc3\u90c1\uff08\u95dc\u9375\u8fa8\u8b58\u7279\u5fb5\uff09\", \"stem\": \"\u76f4\u7acb\u8349\u672c\uff0c\u9ad8 100-200cm\", \"rhizome\": \"\u6839\u8396\u584a\u72c0\uff0c\u6709\u8591\u5473\"}, \"habitat\": \"\u6eaa\u908a\u3001\u6c34\u7530\u65c1\u3001\u6f6e\u6fd5\u5730\uff0c\u6d77\u62d4 0-800m\", \"harvesting\": {\"target\": \"\u82b1\u6735\uff08\u76db\u958b\u6642\u6700\u4f73\uff09\u3001\u5ae9\u8396\u5fc3\", \"method\": \"\u82b1\u6735\u7528\u624b\u6458\u53d6\u3002\u5ae9\u8396\u5f9e\u57fa\u90e8\u6298\u65b7\u3002\", \"season\": \"\u82b1\u671f 6-10 \u6708\", \"sustainability\": \"\u63a1\u82b1\u4e0d\u5f71\u97ff\u690d\u682a\u751f\u9577\"}, \"preparation\": {\"method\": \"\u82b1\u74e3\u53ef\u76f4\u63a5\u98df\u7528\u6216\u5165\u83dc\u3002\u5ae9\u8396\u53ef\u7092\u98df\u3002\u6839\u8396\u53ef\u7576\u8591\u4f7f\u7528\u3002\", \"cooking_time\": \"\u5ae9\u8396\u5feb\u7092 3-5 \u5206\u9418\", \"taste\": \"\u82b1\u74e3\u6e05\u9999\uff0c\u5ae9\u8396\u6e05\u8106\uff0c\u6839\u8396\u8f9b\u8fa3\u5982\u8591\", \"notes\": \"\u91ce\u8591\u82b1\u7cbd\u662f\u53f0\u7063\u77e5\u540d\u6599\u7406\u3002\u82b1\u6735\u53ef\u6ce1\u8336\u3002\"}, \"nutrition\": \"\u7dad\u751f\u7d20C\u3001\u7926\u7269\u8cea\", \"medicinal_uses\": [\"\u6839\u8396\uff1a\u53bb\u5bd2\u3001\u6d88\u816b\"], \"caution\": \"\u78ba\u8a8d\u82b1\u6735\u70ba\u7d14\u767d\u8774\u8776\u5f62\u4e14\u6709\u6fc3\u90c1\u9999\u6c23\u3002\u52ff\u8207\u5176\u4ed6\u8591\u79d1\u690d\u7269\u6df7\u6dc6\u3002\", \"confusion_risk\": \"medium\", \"distribution\": {\"altitude_range\": [0, 800], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"plantago_major\", \"scientific_name\": \"Plantago major\", \"common_names\": {\"zh-TW\": \"\u8eca\u524d\u8349\", \"en\": \"Broadleaf Plantain\"}, \"category\": \"edible_and_medicinal\", \"edible_parts\": [\"\u5ae9\u8449\"], \"morphology\": {\"leaf_shape\": \"\u6a62\u5713\u5f62\u81f3\u5375\u5f62\uff0c\u9577 5-20cm\uff0c\u5bec 3-10cm\", \"leaf_arrangement\": \"\u57fa\u751f\u84ee\u5ea7\u72c0\uff08\u6240\u6709\u8449\u5f9e\u57fa\u90e8\u9577\u51fa\uff0c\u8cbc\u5730\u5c55\u958b\uff09\", \"leaf_surface\": \"\u7da0\u8272\uff0c\u8449\u9762\u6709\u660e\u986f\u7684 3-7 \u689d\u5e73\u884c\u8108\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"leaf_margin\": \"\u5168\u7de3\u6216\u5fae\u6ce2\u72c0\", \"flower\": \"\u7a57\u72c0\u82b1\u5e8f\uff0c\u76f4\u7acb\u82b1\u8396\uff0c\u9577 5-20cm\uff0c\u5c0f\u82b1\u5bc6\u96c6\", \"stem\": \"\u7121\u660e\u986f\u8396\uff0c\u50c5\u6709\u82b1\u8396\u76f4\u7acb\"}, \"habitat\": \"\u8def\u908a\u3001\u8352\u5730\u3001\u8349\u5730\u3001\u6797\u7de3\u3001\u6b65\u9053\u65c1\uff0c\u6d77\u62d4 0-2500m\uff0c\u6975\u5e38\u898b\", \"harvesting\": {\"target\": \"\u5ae9\u8449\uff08\u6625\u5b63\u6700\u5ae9\uff09\", \"method\": \"\u5f9e\u57fa\u90e8\u6458\u53d6\u5ae9\u8449\", \"season\": \"\u6625\u590f\u6700\u4f73\", \"sustainability\": \"\u4fdd\u7559\u6839\u90e8\uff0c\u6703\u6301\u7e8c\u9577\u51fa\u65b0\u8449\"}, \"preparation\": {\"method\": \"\u53ef\u751f\u98df\u6216\u716e\u98df\u3002\u5ae9\u8449\u5ddd\u71d9\u5f8c\u6dbc\u62cc\u3002\", \"cooking_time\": \"\u5ddd\u71d9 1-2 \u5206\u9418\", \"taste\": \"\u7565\u5e36\u82e6\u5473\uff0c\u5ae9\u8449\u8f03\u751c\", \"notes\": \"\u8001\u8449\u7e96\u7dad\u591a\uff0c\u8f03\u9069\u5408\u716e\u6e6f\u3002\"}, \"nutrition\": \"\u7dad\u751f\u7d20A\u3001C\u3001K\u3001\u9223\u3001\u9435\", \"medicinal_uses\": [\"\u6d88\u708e\", \"\u6297\u83cc\", \"\u6b62\u8840\", \"\u5229\u5c3f\", \"\u6417\u788e\u9bae\u8449\u6577\u50b7\u53e3\u53ef\u6b62\u8840\u6d88\u708e\"], \"caution\": \"\u78ba\u8a8d\u5e73\u884c\u8108\u7279\u5fb5\u3002\u907f\u514d\u63a1\u96c6\u8def\u908a\u53d7\u8fb2\u85e5\u6216\u6c7d\u8eca\u5ee2\u6c23\u6c61\u67d3\u7684\u690d\u682a\u3002\", \"confusion_risk\": \"low\", \"distribution\": {\"altitude_range\": [0, 2500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u6eab\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"solanum_nigrum\", \"scientific_name\": \"Solanum nigrum\", \"common_names\": {\"zh-TW\": \"\u9f8d\u8475\uff08\u9ed1\u7c7d\u4ed4\u83dc\uff09\", \"en\": \"Black Nightshade\"}, \"category\": \"edible\", \"edible_parts\": [\"\u5ae9\u8396\u8449\", \"\u6210\u719f\u679c\u5be6\uff08\u5168\u9ed1\u6642\uff09\"], \"morphology\": {\"leaf_shape\": \"\u5375\u5f62\u81f3\u6a62\u5713\u5f62\uff0c\u9577 3-10cm\uff0c\u8449\u7de3\u5168\u7de3\u6216\u5fae\u6ce2\u72c0\", \"leaf_surface\": \"\u7da0\u8272\uff0c\u8584\u7d19\u8cea\", \"flower\": \"\u767d\u8272\u5c0f\u82b1\uff0c5 \u74e3\uff0c\u82b1\u5fc3\u9ec3\u8272\uff0c2-8 \u6735\u6210\u805a\u7e56\u82b1\u5e8f\", \"fruit\": \"\u7403\u5f62\u6f3f\u679c\uff0c\u76f4\u5f91 6-8mm\uff0c\u672a\u719f\u6642\u7da0\u8272\uff0c\u6210\u719f\u5f8c\u5168\u9ed1\uff08\u95dc\u9375\uff09\", \"stem\": \"\u8349\u672c\uff0c\u9ad8 30-100cm\uff0c\u8396\u591a\u5206\u679d\"}, \"habitat\": \"\u8352\u5730\u3001\u7530\u908a\u3001\u8def\u65c1\u3001\u5ee2\u8015\u5730\uff0c\u6d77\u62d4 0-1500m\uff0c\u6975\u5e38\u898b\", \"harvesting\": {\"target\": \"\u5ae9\u8396\u8449\uff08\u5168\u5e74\uff09\u3002\u679c\u5be6\u50c5\u63a1\u5168\u9ed1\u6210\u719f\u8005\u3002\", \"method\": \"\u6458\u53d6\u5ae9\u679d\u9802\u7aef 10-15cm\", \"season\": \"\u5168\u5e74\u53ef\u63a1\", \"sustainability\": \"\u6458\u53d6\u5ae9\u679d\u6703\u4fc3\u9032\u5206\u679d\u751f\u9577\"}, \"preparation\": {\"method\": \"\u5ae9\u8396\u8449\u5ddd\u71d9\u6216\u7092\u98df\u3002\u6210\u719f\u9ed1\u679c\u53ef\u76f4\u63a5\u98df\u7528\u3002\", \"cooking_time\": \"\u5ddd\u71d9 2-3 \u5206\u9418\", \"taste\": \"\u5ae9\u8449\u7565\u5e36\u82e6\u5473\uff0c\u9ed1\u679c\u5fae\u751c\", \"notes\": \"\u662f\u53f0\u7063\u5e38\u898b\u7684\u91ce\u83dc\uff0c\u5e38\u7528\u849c\u672b\u5feb\u7092\u3002\"}, \"nutrition\": \"\u7dad\u751f\u7d20C\u3001\u86cb\u767d\u8cea\u3001\u9223\", \"caution\": \"\u26a0\ufe0f \u672a\u6210\u719f\u7684\u7da0\u8272\u679c\u5be6\u542b\u9f8d\u8475\u9e7c\uff08solanine\uff09\uff0c\u6709\u6bd2\uff01\u53ea\u80fd\u98df\u7528\u5b8c\u5168\u6210\u719f\u7684\u9ed1\u8272\u679c\u5be6\u3002\u5ae9\u8396\u8449\u9700\u716e\u719f\u3002\", \"confusion_risk\": \"medium\", \"distribution\": {\"altitude_range\": [0, 1500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u6eab\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"crassocephalum_crepidioides\", \"scientific_name\": \"Crassocephalum crepidioides\", \"common_names\": {\"zh-TW\": \"\u662d\u548c\u8349\uff08\u98db\u6a5f\u8349\uff09\", \"en\": \"Redflower Ragleaf\"}, \"category\": \"edible\", \"edible_parts\": [\"\u5ae9\u8396\u8449\"], \"morphology\": {\"leaf_shape\": \"\u9577\u6a62\u5713\u5f62\uff0c\u9577 5-15cm\uff0c\u8449\u7de3\u6709\u4e0d\u898f\u5247\u92f8\u9f52\u6216\u88c2\u7247\", \"leaf_surface\": \"\u7da0\u8272\uff0c\u7565\u5e36\u7d2b\u6688\uff0c\u67d4\u8edf\", \"flower\": \"\u982d\u72c0\u82b1\u5e8f\uff0c\u4e0b\u5782\uff0c\u82b1\u51a0\u6a58\u7d05\u8272\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"stem\": \"\u76f4\u7acb\u8349\u672c\uff0c\u9ad8 20-80cm\uff0c\u8396\u4e2d\u7a7a\uff0c\u65b7\u9762\u6709\u767d\u8272\u4e73\u6c41\", \"fruit\": \"\u7626\u679c\u5e36\u767d\u8272\u51a0\u6bdb\uff08\u4f3c\u84b2\u516c\u82f1\uff09\uff0c\u53ef\u96a8\u98a8\u98c4\u6563\"}, \"habitat\": \"\u8352\u5730\u3001\u8def\u65c1\u3001\u8fb2\u7530\u908a\u3001\u958b\u95ca\u5730\uff0c\u6d77\u62d4 0-2000m\uff0c\u975e\u5e38\u5e38\u898b\", \"harvesting\": {\"target\": \"\u5ae9\u8396\u8449\uff08\u958b\u82b1\u524d\u6700\u5ae9\u6700\u751c\uff09\", \"method\": \"\u6458\u53d6\u9802\u7aef\u5ae9\u8396 10-20cm\", \"season\": \"\u5168\u5e74\u53ef\u63a1\uff0c\u51ac\u6625\u6700\u4f73\", \"sustainability\": \"\u751f\u9577\u5feb\u901f\uff0c\u63a1\u96c6\u4e0d\u5f71\u97ff\u65cf\u7fa4\"}, \"preparation\": {\"method\": \"\u5ddd\u71d9\u5f8c\u6dbc\u62cc\u3001\u7092\u98df\u3001\u716e\u6e6f\u7686\u53ef\u3002\", \"cooking_time\": \"\u5ddd\u71d9 1-2 \u5206\u9418\", \"taste\": \"\u53e3\u611f\u6ed1\u5ae9\uff0c\u6709\u7279\u6b8a\u6e05\u9999\uff0c\u7565\u5e36\u7518\u751c\", \"notes\": \"\u5473\u9053\u985e\u4f3c\u833c\u84bf\uff0c\u662f\u91ce\u5916\u6700\u5bb9\u6613\u53d6\u5f97\u7684\u91ce\u83dc\u4e4b\u4e00\u3002\u53ef\u716e\u5473\u564c\u6e6f\u3002\"}, \"nutrition\": \"\u7dad\u751f\u7d20A\u3001C\u3001\u9223\u3001\u9435\", \"caution\": \"\u8fa8\u8b58\u91cd\u9ede\u662f\u6a58\u7d05\u8272\u4e0b\u5782\u82b1\u5e8f\u548c\u65b7\u8396\u767d\u8272\u4e73\u6c41\u3002\u907f\u514d\u8207\u5176\u4ed6\u83ca\u79d1\u690d\u7269\u6df7\u6dc6\u3002\", \"confusion_risk\": \"low\", \"distribution\": {\"altitude_range\": [0, 2000], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"colocasia_esculenta\", \"scientific_name\": \"Colocasia esculenta\", \"common_names\": {\"zh-TW\": \"\u828b\u982d\", \"en\": \"Taro\"}, \"category\": \"edible\", \"edible_parts\": [\"\u584a\u8396\uff08\u9808\u716e\u719f\uff09\", \"\u5ae9\u8396\uff08\u9808\u716e\u719f\uff09\"], \"morphology\": {\"leaf_shape\": \"\u5fc3\u5f62\uff0c\u9577 20-50cm\", \"leaf_surface\": \"\u6709\u5929\u9d5d\u7d68\u822c\u7684\u5fae\u7d68\u6bdb\u8cea\u611f\uff0c\u4e0d\u592a\u5149\u6ed1\", \"water_droplet_test\": \"\u6c34\u73e0\u5728\u8449\u9762\u5448\u5713\u73e0\u72c0\u6efe\u52d5\uff08\u8377\u8449\u6548\u61c9\uff09\", \"leaf_tip\": \"\u901a\u5e38\u671d\u4e0b\u5782\", \"petiole\": \"\u63a5\u5728\u8449\u7247\u908a\u7de3\u5167\u5074\uff08\u76fe\u72c0\u8457\u751f\uff09\uff0c\u7da0\u8272\uff0c\u8f03\u5c11\u7d2b\u6688\", \"underground\": \"\u6709\u5713\u5f62\u584a\u8396\"}, \"habitat\": \"\u7530\u9593\u683d\u57f9\u3001\u6eaa\u908a\u91ce\u751f\u5316\uff0c\u6d77\u62d4 0-1000m\", \"harvesting\": {\"target\": \"\u584a\u8396\uff08\u5730\u4e0b\u90e8\u5206\uff09\", \"method\": \"\u6316\u6398\u5730\u4e0b\u584a\u8396\", \"season\": \"\u5168\u5e74\u53ef\u63a1\uff0c\u79cb\u51ac\u584a\u8396\u6700\u5927\", \"sustainability\": \"\u4fdd\u7559\u90e8\u5206\u584a\u8396\u53ef\u518d\u751f\"}, \"preparation\": {\"method\": \"\u5fc5\u9808\u5b8c\u5168\u716e\u719f\uff01\u751f\u828b\u982d\u542b\u8349\u9178\u9223\uff0c\u98df\u7528\u6703\u5c0e\u81f4\u53e3\u8154\u9ebb\u75fa\u3002\", \"cooking_time\": \"\u84b8\u716e 20-30 \u5206\u9418\u81f3\u719f\u900f\", \"taste\": \"\u9b06\u8edf\u7dbf\u5bc6\uff0c\u5fae\u751c\", \"notes\": \"\u8655\u7406\u751f\u828b\u982d\u6642\u5efa\u8b70\u6234\u624b\u5957\uff0c\u6c41\u6db2\u53ef\u80fd\u5f15\u8d77\u76ae\u819a\u6414\u7662\u3002\"}, \"nutrition\": \"\u78b3\u6c34\u5316\u5408\u7269\uff08\u80fd\u91cf\u4f86\u6e90\uff09\u3001\u81b3\u98df\u7e96\u7dad\u3001\u9240\", \"caution\": \"\u26a0\ufe0f \u6975\u6613\u8207\u6709\u6bd2\u7684\u59d1\u5a46\u828b\u6df7\u6dc6\uff01\u5fc5\u9808\u505a\u6c34\u73e0\u6e2c\u8a66\u548c\u8449\u67c4\u63a5\u5408\u8655\u78ba\u8a8d\u3002\u82e5\u7121\u6cd5\u78ba\u5b9a\uff0c\u7981\u6b62\u98df\u7528\u3002\u751f\u828b\u982d\u4e0d\u53ef\u98df\u7528\u3002\", \"confusion_risk\": \"critical\", \"distribution\": {\"altitude_range\": [0, 1000], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"auricularia_auricula_judae\", \"scientific_name\": \"Auricularia auricula-judae\", \"common_names\": {\"zh-TW\": \"\u6728\u8033\", \"en\": \"Wood Ear Mushroom\"}, \"category\": \"edible\", \"edible_parts\": [\"\u5b50\u5be6\u9ad4\uff08\u6574\u6735\uff09\"], \"morphology\": {\"shape\": \"\u8033\u72c0\u6216\u7897\u72c0\uff0c\u76f4\u5f91 3-10cm\", \"surface\": \"\u5916\u9762\u6709\u7d30\u5fae\u7d68\u6bdb\uff0c\u7070\u8910\u8272\u3002\u5167\u9762\u5149\u6ed1\uff0c\u6df1\u8910\u8272\u81f3\u9ed1\u8272\u3002\", \"texture\": \"\u65b0\u9bae\u6642\u81a0\u8cea\u72c0\u3001\u534a\u900f\u660e\u3001\u6709\u5f48\u6027\u3002\u4e7e\u71e5\u5f8c\u786c\u7e2e\uff0c\u6ce1\u6c34\u6062\u5fa9\u3002\", \"growth_pattern\": \"\u53e2\u751f\u65bc\u67af\u6728\u6216\u8150\u6728\u8868\u9762\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"color\": \"\u6df1\u8910\u8272\u81f3\u9ed1\u8272\"}, \"habitat\": \"\u67af\u6728\u3001\u8150\u6728\u8868\u9762\uff0c\u6f6e\u6fd5\u68ee\u6797\u4e2d\uff0c\u6d77\u62d4 0-2000m\", \"harvesting\": {\"target\": \"\u65b0\u9bae\u3001\u5b8c\u6574\u3001\u7121\u7570\u5473\u7684\u5b50\u5be6\u9ad4\", \"method\": \"\u7528\u624b\u6216\u5200\u5f9e\u6728\u982d\u4e0a\u6458\u53d6\", \"season\": \"\u96e8\u5f8c\u6700\u591a\uff0c\u6625\u79cb\u6700\u4f73\", \"sustainability\": \"\u4fdd\u7559\u90e8\u5206\u8b93\u5176\u91cb\u653e\u5b62\u5b50\u7e41\u6b96\"}, \"preparation\": {\"method\": \"\u5ddd\u71d9\u6216\u7092\u98df\u3002\u65b0\u9bae\u6728\u8033\u53ef\u76f4\u63a5\u6599\u7406\u3002\", \"cooking_time\": \"\u5ddd\u71d9 2-3 \u5206\u9418\u6216\u5feb\u7092 3-5 \u5206\u9418\", \"taste\": \"\u53e3\u611f\u8106\u5ae9\u6ed1\u5f48\uff0c\u5e7e\u4e4e\u7121\u5473\uff0c\u5438\u9644\u6e6f\u6c41\u4f73\", \"notes\": \"\u6dbc\u62cc\u6642\u5efa\u8b70\u5148\u5ddd\u71d9\u6bba\u83cc\u3002\"}, \"nutrition\": \"\u9435\u8cea\u3001\u81b3\u98df\u7e96\u7dad\u3001\u591a\u91a3\u9ad4\", \"caution\": \"\u78ba\u8a8d\u751f\u9577\u5728\u67af\u6728\u4e0a\uff08\u800c\u975e\u571f\u58e4\uff09\u3002\u907f\u514d\u63a1\u96c6\u5df2\u8150\u721b\u3001\u767c\u81ed\u3001\u6216\u984f\u8272\u7570\u5e38\u7684\u83c7\u9ad4\u3002\u4e0d\u78ba\u5b9a\u6642\u7981\u6b62\u98df\u7528\u3002\", \"confusion_risk\": \"low\", \"distribution\": {\"altitude_range\": [0, 2000], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u6eab\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"nephrolepis_cordifolia\", \"scientific_name\": \"Nephrolepis cordifolia\", \"common_names\": {\"zh-TW\": \"\u814e\u8568\uff08\u5c71\u9cf3\u68a8\uff09\", \"en\": \"Tuberous Sword Fern\"}, \"category\": \"edible_and_water\", \"edible_parts\": [\"\u7403\u5f62\u5132\u6c34\u584a\u8396\uff08\u5730\u4e0b\uff09\"], \"morphology\": {\"leaf_shape\": \"\u4e00\u56de\u7fbd\u72c0\u8907\u8449\uff0c\u9577 30-80cm\uff0c\u5c0f\u8449\u4e92\u751f\uff0c\u9577 2-4cm\", \"leaf_arrangement\": \"\u53e2\u751f\uff0c\u5411\u5916\u5f27\u5f62\u5c55\u958b\", \"leaf_surface\": \"\u7da0\u8272\uff0c\u8584\u8349\u8cea\", \"underground\": \"\u6839\u90e8\u6709\u7403\u5f62\u5132\u6c34\u584a\u8396\uff0c\u76f4\u5f91 1-2cm\uff08\u95dc\u9375\u7279\u5fb5\uff0c\u542b\u6c34\u5206\uff09\", \"growth_pattern\": \"\u5730\u751f\u6216\u9644\u751f\uff0c\u5e38\u898b\u65bc\u8def\u65c1\u5ca9\u58c1\"}, \"habitat\": \"\u4f4e\u4e2d\u6d77\u62d4\u6797\u7de3\u3001\u8def\u65c1\u3001\u5ca9\u58c1\uff0c\u6d77\u62d4 0-1500m\uff0c\u6975\u5e38\u898b\", \"harvesting\": {\"target\": \"\u5730\u4e0b\u7403\u5f62\u5132\u6c34\u584a\u8396\", \"method\": \"\u6316\u6398\u6839\u90e8\uff0c\u6458\u53d6\u7403\u5f62\u584a\u8396\uff0c\u525d\u53bb\u5916\u76ae\", \"season\": \"\u5168\u5e74\u53ef\u63a1\", \"sustainability\": \"\u4fdd\u7559\u6839\u7cfb\uff0c\u6703\u518d\u751f\u584a\u8396\"}, \"preparation\": {\"method\": \"\u525d\u76ae\u5f8c\u53ef\u76f4\u63a5\u751f\u98df\u6216\u716e\u98df\u3002\u584a\u8396\u542b\u6c34\u5206\uff0c\u53ef\u89e3\u6e34\u3002\", \"cooking_time\": \"\u53ef\u751f\u98df\uff0c\u6216\u716e 5-10 \u5206\u9418\", \"taste\": \"\u5fae\u751c\uff0c\u542b\u6c34\u5206\uff0c\u53e3\u611f\u50cf\u8378\u85ba\", \"notes\": \"\u5728\u7f3a\u6c34\u60c5\u6cc1\u4e0b\u662f\u91cd\u8981\u7684\u6c34\u5206\u4f86\u6e90\u3002\u53f0\u7063\u539f\u4f4f\u6c11\u7a31\u70ba\u300c\u5c71\u9cf3\u68a8\u300d\u3002\"}, \"nutrition\": \"\u6c34\u5206\uff08\u4e3b\u8981\u50f9\u503c\uff09\u3001\u78b3\u6c34\u5316\u5408\u7269\", \"caution\": \"\u78ba\u8a8d\u662f\u814e\u8568\uff08\u4e00\u56de\u7fbd\u72c0\u8907\u8449 + \u5730\u4e0b\u7403\u5f62\u584a\u8396\uff09\u3002\u5176\u4ed6\u8568\u985e\u7121\u6b64\u584a\u8396\u3002\", \"confusion_risk\": \"low\", \"distribution\": {\"altitude_range\": [0, 1500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\", \"\u6eab\u5e36\"]}}, {\"id\": \"oenanthe_javanica\", \"scientific_name\": \"Oenanthe javanica\", \"common_names\": {\"zh-TW\": \"\u5c71\u82b9\u83dc\uff08\u6c34\u82b9\u83dc\uff09\", \"en\": \"Water Dropwort\"}, \"category\": \"edible\", \"edible_parts\": [\"\u5ae9\u8396\u8449\"], \"morphology\": {\"leaf_shape\": \"\u4e8c\u81f3\u4e09\u56de\u7fbd\u72c0\u8907\u8449\uff0c\u5c0f\u8449\u5375\u5f62\u81f3\u83f1\u5f62\uff0c\u9577 1-3cm\uff0c\u8449\u7de3\u92f8\u9f52\u72c0\", \"leaf_surface\": \"\u7da0\u8272\uff0c\u5149\u6ed1\", \"flower\": \"\u8907\u7e56\u5f62\u82b1\u5e8f\uff0c\u767d\u8272\u5c0f\u82b1\", \"stem\": \"\u4e2d\u7a7a\uff0c\u5149\u6ed1\uff0c\u6709\u7bc0\uff0c\u530d\u5310\u8396\u53ef\u6c34\u751f\", \"smell\": \"\u6709\u7368\u7279\u7684\u82b9\u83dc\u9999\u5473\uff08\u95dc\u9375\u8fa8\u8b58\u7279\u5fb5\uff09\"}, \"habitat\": \"\u6eaa\u908a\u3001\u6c34\u7530\u3001\u6f6e\u6fd5\u5730\u3001\u6cbc\u6fa4\u908a\u7de3\uff0c\u6d77\u62d4 0-1500m\", \"harvesting\": {\"target\": \"\u5ae9\u8396\u8449\uff08\u9802\u7aef 10-15cm\uff09\", \"method\": \"\u5f9e\u7bc0\u4e0a\u65b9\u6298\u53d6\u5ae9\u8396\", \"season\": \"\u5168\u5e74\u53ef\u63a1\uff0c\u6625\u5b63\u6700\u5ae9\", \"sustainability\": \"\u530d\u5310\u8396\u6703\u6301\u7e8c\u751f\u9577\"}, \"preparation\": {\"method\": \"\u5ddd\u71d9\u6216\u5feb\u7092\u3002\u9999\u5473\u6fc3\u90c1\uff0c\u9069\u5408\u7092\u8089\u7d72\u6216\u716e\u6e6f\u3002\", \"cooking_time\": \"\u5feb\u7092 2-3 \u5206\u9418\", \"taste\": \"\u82b9\u83dc\u9999\u5473\u6fc3\u90c1\uff0c\u53e3\u611f\u8106\u5ae9\", \"notes\": \"\u5473\u9053\u6bd4\u4e00\u822c\u82b9\u83dc\u66f4\u6fc3\u90c1\u3002\"}, \"nutrition\": \"\u7dad\u751f\u7d20A\u3001C\u3001\u9223\u3001\u9435\", \"caution\": \"\u26a0\ufe0f \u6ce8\u610f\uff01\u540c\u5c6c\u7684\u6bd2\u6c34\u82b9\uff08Oenanthe crocata\uff09\u5287\u6bd2\u3002\u78ba\u8a8d\u6709\u82b9\u83dc\u9999\u5473\uff0c\u4e14\u751f\u9577\u5728\u4e7e\u6de8\u6c34\u6e90\u65c1\u3002\u6c34\u8cea\u4e0d\u4f73\u7684\u5730\u65b9\u4e0d\u5efa\u8b70\u63a1\u96c6\uff08\u5bc4\u751f\u87f2\u98a8\u96aa\uff09\u3002\", \"confusion_risk\": \"high\", \"distribution\": {\"altitude_range\": [0, 1500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u6eab\u5e36\"]}}, {\"id\": \"bidens_pilosa_var_radiata\", \"scientific_name\": \"Bidens pilosa var. radiata\", \"common_names\": {\"zh-TW\": \"\u5927\u82b1\u54b8\u8c50\u8349\uff08\u9b3c\u91dd\u8349\uff09\", \"en\": \"Spanish Needle (Radiate Variety)\"}, \"category\": \"edible\", \"edible_parts\": [\"\u5ae9\u8396\u8449\", \"\u82b1\uff08\u6ce1\u8336\uff09\"], \"morphology\": {\"plant_habit\": \"\u4e00\u5e74\u751f\u8349\u672c\uff0c\u682a\u9ad8 30-100cm\uff0c\u5206\u679d\u591a\u800c\u8513\u5ef6\u6216\u76f4\u7acb\", \"stem\": \"\u8396\u5177\u56db\u7a1c\u6216\u8fd1\u4f3c\u65b9\u5f62\u3001\u6709\u660e\u986f\u7a1c\u7dda\uff08\u4fbf\u65bc\u624b\u89f8\u8fa8\u8b58\uff09\", \"leaves\": \"\u8449\u5c0d\u751f\uff0c\u4e09\u51fa\u8907\u8449\u6216\u7fbd\u72c0\u8907\u8449\uff0c\u5c0f\u8449\u5375\u5f62\u3001\u8449\u7de3\u5177\u92f8\u9f52\uff0c\u8cea\u5730\u8584\u800c\u7a0d\u7c97\u7cd9\", \"inflorescence\": \"\u982d\u72c0\u82b1\u5e8f\u9802\u751f\u6216\u814b\u751f\uff1b\u4e2d\u592e\u70ba\u9ec3\u8272\u7ba1\u72c0\u82b1\uff0c\u5468\u570d\u5177 5-8 \u679a\u767d\u8272\u820c\u72c0\u82b1\uff08\u95dc\u9375\u7279\u5fb5\uff0c\u8207\u7121\u820c\u72c0\u82b1\u8b8a\u7a2e\u5340\u5225\uff09\", \"fruit\": \"\u7626\u679c\u5177 2-3 \u679a\u5012\u9264\u523a\u72c0\u8292\uff0c\u6975\u6613\u9ecf\u9644\u8863\u7269\u8207\u52d5\u7269\u6bdb\u9aee\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"overall\": \"\u53f0\u7063\u4f4e\u6d77\u62d4\u81f3\u4e2d\u6d77\u62d4\u6700\u5e38\u898b\u7684\u53ef\u98df\u91ce\u8349\u4e4b\u4e00\uff0c\u5168\u5e74\u53ef\u898b\u958b\u82b1\u7d50\u679c\"}, \"habitat\": \"\u8def\u65c1\u3001\u8352\u5730\u3001\u7530\u908a\u3001\u7a7a\u5c4b\u65c1\u8207\u8349\u576a\u908a\u7de3\uff0c\u6d77\u62d4 0-2500m\uff0c\u5317\u81fa\u8207\u90fd\u6703\u5340\u6975\u5e38\u898b\", \"harvesting\": {\"target\": \"\u672a\u6728\u8cea\u5316\u4e4b\u5ae9\u8396\u8207\u5e7c\u8449\uff1b\u76db\u958b\u6216\u534a\u958b\u82b1\u6735\u63a1\u5c11\u91cf\u6ce1\u8336\", \"method\": \"\u6298\u53d6\u9802\u82bd\u6216\u5074\u679d\u5ae9\u6bb5\uff0c\u907f\u514d\u63a1\u96c6\u958b\u904e\u591a\u8001\u8396\uff1b\u82b1\u4ee5\u6307\u5c16\u6458\u982d\u72c0\u82b1\u5e8f\", \"season\": \"\u5168\u5e74\u53ef\u63a1\uff0c\u6625\u590f\u79cb\u5ae9\u8449\u54c1\u8cea\u6700\u4f73\", \"sustainability\": \"\u65cf\u7fa4\u91cf\u5927\uff0c\u63a1\u5ae9\u68a2\u4e0d\u50b7\u6839\u5373\u53ef\u6301\u7e8c\u840c\u8617\uff1b\u52ff\u9023\u6839\u62d4\u5149\u540c\u4e00\u7247\u7fa4\u805a\"}, \"preparation\": {\"method\": \"\u5ae9\u8396\u8449\u5ddd\u71d9\u5f8c\u6dbc\u62cc\u3001\u5feb\u7092\u6216\u716e\u6e6f\uff1b\u4e7e\u71e5\u82b1\u6735\u53ef\u6c96\u6ce1\u82b1\u8349\u8336\", \"cooking_time\": \"\u5ddd\u71d9 1-2 \u5206\u9418\uff1b\u82b1\u8336\u4ee5\u71b1\u6c34\u60b6\u6ce1 3-5 \u5206\u9418\", \"taste\": \"\u5ae9\u8449\u7565\u5e36\u8349\u672c\u6e05\u9999\uff0c\u5fae\u82e6\u56de\u7518\uff1b\u82b1\u8336\u6e05\u6de1\", \"notes\": \"\u8001\u8396\u7e96\u7dad\u7c97\u786c\uff1b\u6458\u63a1\u5f8c\u82e5\u8863\u7269\u6cbe\u7626\u679c\uff0c\u61c9\u5373\u6642\u6e05\u9664\u4ee5\u514d\u523a\u75db\u76ae\u819a\u3002\"}, \"nutrition\": \"\u7dad\u751f\u7d20C\u3001\u985e\u9ec3\u916e\u3001\u81b3\u98df\u7e96\u7dad\", \"caution\": \"\u8fa8\u8b58\u820c\u72c0\u82b1\u8207\u9264\u523a\u7626\u679c\u5373\u53ef\u8207\u591a\u6578\u83ca\u79d1\u91ce\u83dc\u5340\u5206\u3002\u8178\u80c3\u654f\u611f\u8005\u4e0d\u5b9c\u5927\u91cf\u751f\u98df\u3002\", \"confusion_risk\": \"low\", \"distribution\": {\"altitude_range\": [0, 2500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"alpinia_zerumbet\", \"scientific_name\": \"Alpinia zerumbet\", \"common_names\": {\"zh-TW\": \"\u6708\u6843\", \"en\": \"Shell Ginger\"}, \"category\": \"edible\", \"edible_parts\": [\"\u5ae9\u8449\uff08\u5305\u7cbd\u3001\u88f9\u98df\uff09\", \"\u82b1\", \"\u7a2e\u5b50\uff08\u50b3\u7d71\u52a0\u5de5\u539f\u6599\uff09\"], \"morphology\": {\"plant_habit\": \"\u591a\u5e74\u751f\u5927\u578b\u8349\u672c\uff0c\u682a\u9ad8 1-3m\uff0c\u5177\u5730\u4e0b\u530d\u5310\u8396\u8207\u660e\u986f\u5730\u4e0a\u5047\u8396\", \"leaves\": \"\u8449\u4e92\u751f\uff0c\u9577\u6a62\u5713\u72c0\u62ab\u91dd\u5f62\uff0c\u9577 40-70cm\u3001\u5bec 8-15cm\uff0c\u5177\u5e73\u884c\u8108\u3001\u8449\u8cea\u539a\u800c\u4eae\u7da0\", \"aroma\": \"\u8449\u7247\u6413\u63c9\u6216\u6495\u88c2\u5f8c\u6563\u767c\u6fc3\u90c1\u82b3\u9999\uff08\u95dc\u9375\u8fa8\u8b58\u7279\u5fb5\uff09\uff0c\u8fd1\u4f3c\u8591\u82b1\u8207\u8089\u6842\u6df7\u5408\u8abf\u6027\", \"inflorescence\": \"\u5713\u9310\u82b1\u5e8f\u81ea\u8449\u814b\u62bd\u51fa\u4e26\u4e0b\u5782\uff0c\u62ab\u8986\u881f\u8cea\u82de\u7247\uff1b\u82b1\u767d\u8272\u81f3\u6de1\u7c89\u7d05\uff0c\u5507\u74e3\u8fd1\u9ec3\u5e95\u8272\u4e26\u5e38\u898b\u7d05\u8272\u6216\u7d2b\u7d05\u689d\u7d0b\", \"fruit\": \"\u7403\u5f62\u84b4\u679c\uff0c\u5177\u7e31\u7a1c\uff0c\u6210\u719f\u6642\u679c\u76ae\u7e96\u7dad\u8cea\uff1b\u5167\u542b\u7a2e\u5b50\u53ef\u4f9b\u50b3\u7d71\u300c\u4ec1\u4e39\u300d\u985e\u88fd\u54c1\u539f\u6599\", \"habit_note\": \"\u5e38\u6210\u7247\u751f\u9577\u65bc\u6797\u7de3\u8207\u6eaa\u8c37\uff0c\u8449\u53e2\u5bec\u5927\u5448\u300c\u82ad\u8549\u985e\u300d\u5916\u89c0\u4f46\u5177\u8591\u79d1\u5178\u578b\u9798\u72c0\u8449\u67c4\"}, \"habitat\": \"\u4f4e\u6d77\u62d4\u68ee\u6797\u908a\u7de3\u3001\u6eaa\u908a\u8207\u8c37\u5730\u704c\u53e2\uff0c\u6f6e\u6fd5\u534a\u906e\u852d\u8655\uff0c\u6d77\u62d4 0-800m\", \"harvesting\": {\"target\": \"\u5c55\u958b\u4e2d\u5e7c\u5ae9\u5927\u8449\uff08\u5305\u7cbd\uff09\u3001\u76db\u958b\u524d\u5f8c\u82b1\u6735\u3001\u6210\u719f\u84b4\u679c\u5167\u7a2e\u5b50\uff08\u52a0\u5de5\u7528\u9808\u77e5\u60c5\u8207\u6cd5\u898f\uff09\", \"method\": \"\u8449\u81ea\u8449\u9798\u4e0a\u65b9\u6574\u7247\u622a\u53d6\uff1b\u82b1\u9023\u77ed\u6897\u526a\u4e0b\uff1b\u679c\u5be6\u5f85\u7a0d\u4e7e\u88c2\u524d\u5f8c\u63a1\u6536\", \"season\": \"\u8449\u5168\u5e74\u53ef\u7528\uff0c\u76db\u82b1\u671f\u591a\u96c6\u4e2d\u5728\u6625\u590f\uff0c\u679c\u719f\u671f\u8996\u6d77\u62d4\u7565\u7570\", \"sustainability\": \"\u540c\u4e00\u53e2\u8f2a\u6d41\u63a1\u5916\u570d\u8449\u7247\uff0c\u4fdd\u7559\u4e2d\u592e\u751f\u9577\u9ede\uff1b\u52ff\u780d\u9664\u6574\u682a\"}, \"preparation\": {\"method\": \"\u5ae9\u8449\u716e\u8edf\u5f8c\u53ef\u4f5c\u300c\u6708\u6843\u7cbd\u300d\u5916\u76ae\uff1b\u82b1\u53ef\u6c46\u71d9\u5165\u83dc\u6216\u751c\u6e6f\uff1b\u7a2e\u5b50\u50c5\u5efa\u8b70\u4f7f\u7528\u5e02\u552e\u5408\u6cd5\u52a0\u5de5\u54c1\", \"cooking_time\": \"\u7cbd\u8449\u9700\u8207\u7c73\u7cc9\u540c\u84b8\u716e 1h \u4ee5\u4e0a\u81f3\u8edf\uff1b\u82b1\u6c46\u71d9 30 \u79d2\u81f3 1 \u5206\u9418\", \"taste\": \"\u8449\u9999\u6fc3\u90c1\u7565\u5e36\u8f9b\u8fa3\u82b3\u9999\uff1b\u82b1\u6e05\u751c\u4e26\u5e36\u8591\u79d1\u6e05\u9999\", \"notes\": \"\u81ea\u884c\u63a1\u7a2e\u5b50\u5236\u50c5\u4f9b\u6587\u5316\u8a8d\u8b58\uff0c\u98df\u7528\u524d\u5b9c\u78ba\u8a8d\u5b89\u5168\u8207\u6cd5\u6e90\uff1b\u5916\u89c0\u8fd1\u4f3c\u5c71\u8591\u5c6c\u4ed6\u7a2e\u6642\u4ee5\u9999\u6c23\u6bd4\u5c0d\u3002\"}, \"nutrition\": \"\u82b3\u9999\u7cbe\u6cb9\u6210\u5206\u3001\u81b3\u98df\u7e96\u7dad\uff08\u8449\uff09\u3001\u5c11\u91cf\u7926\u7269\u8cea\", \"caution\": \"\u8207\u89c0\u8cde\u6731\u8549\u7b49\u4e0d\u540c\u79d1\u5c6c\u690d\u7269\u8449\u5f62\u5076\u4f3c\uff0c\u52d9\u5fc5\u4ee5\u6413\u8449\u9999\u6c23\u8207\u82b1\u5e8f\u7d50\u69cb\u8907\u6838\u3002\", \"confusion_risk\": \"low\", \"distribution\": {\"altitude_range\": [0, 800], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"broussonetia_papyrifera\", \"scientific_name\": \"Broussonetia papyrifera\", \"common_names\": {\"zh-TW\": \"\u69cb\u6a39\uff08\u69cb\u6843\uff09\", \"en\": \"Paper Mulberry\"}, \"category\": \"edible\", \"edible_parts\": [\"\u6210\u719f\u805a\u5408\u5047\u679c\uff08\u4fd7\u7a31\u69cb\u6a39\u679c\uff09\", \"\u5ae9\u8449\u8207\u5ae9\u82bd\uff08\u9700\u716e\u719f\uff09\"], \"morphology\": {\"plant_habit\": \"\u843d\u8449\u55ac\u6728\uff0c\u9ad8 10\u201320m\uff0c\u6a39\u76ae\u7070\u8910\uff0c\u6a39\u51a0\u958b\u5c55\uff0c\u70ba\u5148\u9a45\u967d\u6027\u6a39\u7a2e\", \"leaves\": \"\u8449\u4e92\u751f\uff0c\u5375\u5f62\u81f3\u5fc3\u5f62\uff0c\u5e38\u898b 3\u20135 \u88c2\u4e4b\u638c\u72c0\u88c2\u8449\uff0c\u9577 6\u201318cm\uff1b\u5e7c\u8449\u5bc6\u88ab\u6bdb\", \"leaf_texture\": \"\u8449\u9762\u6975\u7c97\u7cd9\u3001\u5982\u7802\u7d19\u89f8\u611f\uff08\u95dc\u9375\u7279\u5fb5\uff09\uff0c\u8449\u80cc\u5bc6\u88ab\u67d4\u6bdb\u5448\u6de1\u7070\u7da0\", \"sexes\": \"\u96cc\u96c4\u7570\u682a\uff1b\u96c4\u82b1\u5e8f\u67d4\u8351\u72c0\uff0c\u96cc\u682a\u7d50\u7403\u5f62\u805a\u5408\u679c\", \"fruit\": \"\u96cc\u682a\u679c\u5e8f\u5713\u7403\u5f62\uff0c\u7531\u591a\u6578\u5c0f\u6838\u679c\u96c6\u751f\uff1b\u6210\u719f\u6642\u6a59\u7d05\u8272\uff0c\u8868\u9762\u5177\u8089\u8cea\u5c0f\u7a81\u8d77\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"twigs\": \"\u5c0f\u679d\u5bc6\u6bdb\uff0c\u5ae9\u68a2\u53ef\u98df\u6642\u9078\u672a\u6728\u8cea\u5316\u8005\", \"ecology\": \"\u90fd\u5e02\u7a7a\u5ee2\u5730\u3001\u6cb3\u5cb8\u3001\u9632\u706b\u5df7\u8207\u5761\u5730\u7686\u6975\u5e38\u898b\"}, \"habitat\": \"\u967d\u6027\u958b\u95ca\u5730\u3001\u6cb3\u5cb8\u3001\u8def\u65c1\u3001\u5ee2\u8015\u5730\u8207\u90fd\u5e02\u7da0\u5730\u908a\u7de3\uff0c\u6d77\u62d4 0-1500m\", \"harvesting\": {\"target\": \"\u5168\u719f\u6a59\u7d05\u4e4b\u805a\u5408\u679c\uff1b\u6625\u5b63\u5e7c\u5ae9\u8449\u82bd\", \"method\": \"\u679c\u5be6\u7528\u624b\u6458\u53d6\u679c\u5e8f\uff1b\u5ae9\u68a2\u6298\u53d6 10-15cm \u9802\u7aef\", \"season\": \"\u679c\u719f\u591a\u590f\u79cb\uff0c\u5ae9\u8449\u4ee5\u6625\u5b63\u6700\u4f73\", \"sustainability\": \"\u96cc\u96c4\u682a\u5206\u96e2\uff0c\u63a1\u679c\u4e0d\u9808\u780d\u6a39\uff1b\u907f\u514d\u525d\u76ae\u5f0f\u63a1\u8449\u50b7\u6a39\u5e79\"}, \"preparation\": {\"method\": \"\u679c\u5be6\u53ef\u751f\u98df\u6216\u6253\u6210\u679c\u6c41\uff1b\u5ae9\u8449\u52d9\u5fc5\u5ddd\u71d9\u6216\u4e45\u716e\u53bb\u8349\u9178\u8207\u55ae\u5be7\u6f80\u5473\", \"cooking_time\": \"\u5ae9\u8449\u5ddd\u71d9 2-3 \u5206\u9418\u6216\u6efe\u716e 5 \u5206\u4ee5\u4e0a\", \"taste\": \"\u719f\u679c\u591a\u6c41\u5fae\u751c\u5e36\u5730\u74dc\u679c\u9999\uff1b\u5ae9\u8449\u716e\u719f\u5f8c\u6f80\u5473\u4e0b\u964d\uff0c\u7565\u4f3c\u6851\u79d1\u8449\u83dc\", \"notes\": \"\u672a\u719f\u7da0\u679c\u9178\u6f80\u5f37\uff1b\u904e\u654f\u9ad4\u8cea\u9996\u6b21\u5c11\u91cf\u8a66\u98df\u3002\"}, \"nutrition\": \"\u7cd6\u985e\u3001\u6c34\u5206\u3001\u7dad\u751f\u7d20C\uff08\u679c\u5be6\uff09\u3001\u81b3\u98df\u7e96\u7dad\", \"caution\": \"\u52d9\u5fc5\u63a1\u5b8c\u5168\u8457\u8272\u4e4b\u719f\u679c\uff1b\u5ae9\u76ae\u5c64\u53ef\u80fd\u523a\u6fc0\u53e3\u8154\uff0c\u5b9c\u716e\u719f\u3002\u52ff\u8207\u6851\u79d1\u672a\u78ba\u8a8d\u7a2e\u985e\u4e4b\u4e73\u6c41\u690d\u7269\u6df7\u6dc6\u3002\", \"confusion_risk\": \"low\", \"distribution\": {\"altitude_range\": [0, 1500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\", \"\u6eab\u5e36\"]}}, {\"id\": \"morus_australis\", \"scientific_name\": \"Morus australis\", \"common_names\": {\"zh-TW\": \"\u5c0f\u8449\u6851\uff08\u91ce\u751f\u6851\uff09\", \"en\": \"Chinese Mulberry / Small-leaf Mulberry\"}, \"category\": \"edible\", \"edible_parts\": [\"\u6210\u719f\u6851\u6939\uff08\u805a\u5408\u679c\uff09\", \"\u5ae9\u8449\uff08\u716e\u6e6f\uff09\"], \"morphology\": {\"plant_habit\": \"\u843d\u8449\u704c\u6728\u81f3\u5c0f\u55ac\u6728\uff0c\u9ad8 3-10m\uff0c\u679d\u689d\u67d4\u97cc\uff0c\u76ae\u5b54\u660e\u986f\", \"leaves\": \"\u8449\u4e92\u751f\uff0c\u5375\u5f62\u81f3\u5ee3\u5375\u5f62\uff0c\u6216\u4e0d\u898f\u5247\u7f3a\u523b\u8207\u6dfa\u88c2\uff0c\u9577\u5bec\u8b8a\u7570\u5927\uff0c\u8449\u7de3\u5177\u920d\u92f8\u9f52\", \"leaf_surface\": \"\u8868\u9762\u6df1\u7da0\u3001\u8cea\u5730\u7d19\u8cea\u81f3\u8584\u9769\u8cea\uff1b\u5ae9\u8449\u6709\u6bdb\uff0c\u8001\u8449\u8f03\u5149\u6ed1\u7121\u5149\u6fa4\", \"fruit\": \"\u96cc\u82b1\u7d50\u9577\u5713\u7b52\u5f62\u81f3\u7565\u67f1\u72c0\u4e4b\u805a\u5408\u679c\uff08\u4fd7\u7a31\u6851\u6939\uff09\uff1b\u672a\u719f\u7d05\u8272\uff0c\u5b8c\u5168\u6210\u719f\u6df1\u7d2b\u81f3\u8fd1\u9ed1\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"latex\": \"\u5e7c\u679d\u65b7\u88c2\u6216\u8449\u67c4\u6495\u88c2\u53ef\u898b\u767d\u8272\u4e73\u6c41\uff08\u6851\u79d1\u5e38\u898b\uff0c\u8f14\u52a9\u8fa8\u8b58\uff09\", \"seasonal\": \"\u6625\u5b63\u767c\u8449\uff0c\u590f\u79cb\u679c\u719f\uff0c\u51ac\u5b63\u843d\u8449\", \"habitat_note\": \"\u4f4e\u6d77\u62d4\u9053\u8def\u908a\u5761\u3001\u7530\u57c2\u8207\u6cb3\u5e8a\u704c\u53e2\u5e38\u898b\"}, \"habitat\": \"\u4f4e\u6d77\u62d4\u5c71\u5340\u6797\u7de3\u3001\u6eaa\u908a\u3001\u8def\u80a9\u8207\u8fb2\u7530\u704c\u53e2\uff0c\u6d77\u62d4 0-1500m\", \"harvesting\": {\"target\": \"\u6613\u812b\u843d\u3001\u8272\u6df1\u7d2b\u9ed1\u4e4b\u5b8c\u719f\u6851\u6939\uff1b\u672a\u5c55\u958b\u4e4b\u5e7c\u5ae9\u8449\", \"method\": \"\u679c\u5be6\u4ee5\u5bb9\u5668\u627f\u63a5\u6296\u843d\u6216\u624b\u6458\uff1b\u5ae9\u8449\u6298\u68a2\u7aef\", \"season\": \"\u6851\u6939\u591a\u6625\u590f\u81f3\u521d\u79cb\uff0c\u5ae9\u8449\u6625\u5b63\u6700\u4f73\", \"sustainability\": \"\u7559\u6a39\u4e0a\u4e00\u90e8\u5206\u679c\u5be6\u4f9b\u9ce5\u985e\u8207\u64ad\u7a2e\uff1b\u63a1\u8449\u52ff\u53bb\u5149\u55ae\u682a\u6240\u6709\u8449\u7247\"}, \"preparation\": {\"method\": \"\u6851\u6939\u53ef\u9bae\u98df\u3001\u505a\u679c\u91ac\u6216\u716e\u751c\u6e6f\uff1b\u5ae9\u8449\u5ddd\u71d9\u716e\u86cb\u82b1\u6e6f\u6216\u8089\u6e6f\", \"cooking_time\": \"\u5ae9\u8449\u5ddd\u71d9 1-2 \u5206\u9418\u5f8c\u716e\u6e6f 5-10 \u5206\u9418\", \"taste\": \"\u719f\u6939\u871c\u751c\u5fae\u9178\uff1b\u5ae9\u8449\u6e6f\u6e05\u65b0\u7565\u6f80\", \"notes\": \"\u767d\u6939\u6216\u7d05\u6939\u672a\u8f49\u9ed1\u524d\u55ae\u5be7\u8207\u9178\u5ea6\u8f03\u9ad8\uff1b\u8863\u7269\u6cbe\u67d3\u7d2b\u6c41\u96e3\u6d17\u3002\"}, \"nutrition\": \"\u82b1\u9752\u7d20\u3001\u7dad\u751f\u7d20C\u3001\u9435\u3001\u81b3\u98df\u7e96\u7dad\", \"caution\": \"\u8207\u5176\u4ed6\u805a\u5408\u6f3f\u679c\u6df7\u6dc6\u6642\u4ee5\u6a39\u5f62\u3001\u8449\u5f62\u8207\u4e73\u6c41\u78ba\u8a8d\u3002\u80c3\u9178\u904e\u591a\u8005\u5c11\u98df\u5927\u91cf\u9178\u6939\u3002\", \"confusion_risk\": \"medium\", \"distribution\": {\"altitude_range\": [0, 1500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u6eab\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"amaranthus_viridis\", \"scientific_name\": \"Amaranthus viridis\", \"common_names\": {\"zh-TW\": \"\u91ce\u83a7\u83dc\uff08\u7da0\u83a7\uff09\", \"en\": \"Green Amaranth\"}, \"category\": \"edible\", \"edible_parts\": [\"\u5ae9\u8396\u8449\"], \"morphology\": {\"plant_habit\": \"\u4e00\u5e74\u751f\u8349\u672c\uff0c\u9ad8 20-80cm\uff0c\u5177\u591a\u6578\u5206\u679d\uff0c\u5168\u682a\u9bae\u7da0\u6216\u8396\u5e36\u6de1\u7d05\", \"stem\": \"\u8396\u76f4\u7acb\u6216\u659c\u4e0a\uff0c\u8868\u9762\u6e9d\u7a1c\u660e\u986f\uff1b\u7bc0\u9593\u9577\u5ea6\u4e2d\u7b49\", \"leaves\": \"\u8449\u4e92\u751f\uff0c\u5375\u5f62\u81f3\u83f1\u5f62\uff0c\u9577 3-8cm\uff0c\u5148\u7aef\u6f38\u5c16\uff0c\u8449\u7de3\u5168\u7de3\uff0c\u8449\u67c4\u9577\", \"inflorescence\": \"\u7a57\u72c0\u82b1\u5e8f\u9802\u751f\u8207\u814b\u751f\uff0c\u5c0f\u82b1\u5bc6\u96c6\u5448\u7d30\u9577\u6216\u5713\u9310\u72c0\uff0c\u82b1\u88ab\u6de1\u7da0\u4e0d\u986f\u773c\", \"seeds\": \"\u80de\u679c\u958b\u88c2\u91cb\u51fa\u5927\u91cf\u9ed1\u8272\u3001\u5fae\u5c0f\u800c\u5149\u4eae\u7a2e\u5b50\uff08\u95dc\u9375\u65bc\u6210\u719f\u690d\u682a\u8fa8\u8b58\uff09\", \"overall\": \"\u5e38\u898b\u65bc\u8015\u4f5c\u5730\u65c1\u8207\u6c34\u6ce5\u7e2b\u9699\uff0c\u8cea\u5730\u67d4\u5ae9\uff0c\u6613\u8207\u67d0\u4e9b\u6709\u6bd2\u53cc\u5b50\u8449\u5e7c\u682a\u6df7\u6dc6\"}, \"habitat\": \"\u8352\u5730\u3001\u7530\u908a\u3001\u83dc\u5712\u7566\u6e9d\u3001\u516c\u5bd3\u82b1\u53f0\u571f\uff0c\u6d77\u62d4 0-2000m\uff0c\u5168\u5e74\u53ef\u898b\", \"harvesting\": {\"target\": \"\u682a\u9ad8\u672a\u62bd\u9577\u82b1\u5e8f\u524d\u4e4b\u5ae9\u68a2\u8207\u672a\u8001\u786c\u8449\u7247\", \"method\": \"\u6298\u53d6\u9802\u82bd 10-20cm\uff1b\u907f\u514d\u9023\u6839\u62d4\u8d77\u4ee5\u5229\u571f\u58e4\u4fdd\u6301\", \"season\": \"\u5168\u5e74\u53ef\u63a1\uff0c\u51ac\u6625\u6eab\u6696\u6642\u4ecd\u898b\u5e7c\u682a\", \"sustainability\": \"\u591a\u5229\u7528\u66ab\u6642\u88f8\u5730\uff0c\u63a1\u5f8c\u5e38\u7531\u81ea\u64ad\u5c0f\u82d7\u88dc\u4f4d\"}, \"preparation\": {\"method\": \"\u5ae9\u8396\u8449\u5ddd\u71d9\u5f8c\u7092\u98df\u3001\u716e\u7ca5\u6216\u716e\u7fb9\uff1b\u4ea6\u53ef\u5207\u788e\u714e\u86cb\", \"cooking_time\": \"\u5ddd\u71d9 1 \u5206\u9418\u5f8c\u5feb\u7092 2-3 \u5206\u9418\", \"taste\": \"\u6e05\u6de1\u7565\u5e36\u571f\u9999\uff0c\u53e3\u611f\u67d4\u6ed1\u4f3c\u683d\u57f9\u83a7\u83dc\", \"notes\": \"\u8001\u8396\u6613\u6728\u8cea\u5316\uff1b\u6c34\u7530\u908a\u63a1\u96c6\u61c9\u7559\u610f\u8fb2\u85e5\u8207\u5bc4\u751f\u87f2\u3002\"}, \"nutrition\": \"\u9223\u3001\u9435\u3001\u03b2-\u80e1\u863f\u8514\u7d20\u3001\u81b3\u98df\u7e96\u7dad\", \"caution\": \"\u26a0\ufe0f \u5e7c\u682a\u6613\u8207\u7f8e\u6d32\u5546\u9678\uff08Phytolacca americana\uff09\u7b49\u6709\u6bd2\u690d\u7269\u6df7\u6dc6\uff01\u5546\u9678\u5e38\u5177\u7d2b\u7d05\u8396\u3001\u4e92\u751f\u5927\u8449\u8207\u4e0b\u5782\u7e3d\u72c0\u7d2b\u9ed1\u6f3f\u679c\uff1b\u7121\u628a\u63e1\u5207\u52ff\u63a1\u98df\u3002\", \"confusion_risk\": \"medium\", \"distribution\": {\"altitude_range\": [0, 2000], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\", \"\u6eab\u5e36\"]}}, {\"id\": \"houttuynia_cordata\", \"scientific_name\": \"Houttuynia cordata\", \"common_names\": {\"zh-TW\": \"\u9b5a\u8165\u8349\uff08\u6298\u8033\u6839\uff09\", \"en\": \"Fish Mint / Chameleon Plant\"}, \"category\": \"edible_and_medicinal\", \"edible_parts\": [\"\u5ae9\u8396\u8449\", \"\u5730\u4e0b\u530d\u5310\u8396\uff08\u7bc0\u4e0a\u751f\u6839\u8005\uff09\"], \"morphology\": {\"plant_habit\": \"\u591a\u5e74\u751f\u8349\u672c\uff0c\u9ad8 15-40cm\uff0c\u5177\u530d\u5310\u5730\u4e0b\u8396\uff0c\u5e38\u6210\u7247\u8513\u751f\", \"stem\": \"\u5730\u4e0a\u8396\u76f4\u7acb\u6216\u659c\u5347\uff0c\u7d05\u7d2b\u81f3\u7da0\u7d2b\u8272\u7e31\u5411\u689d\u7d0b\u660e\u986f\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"leaves\": \"\u8449\u4e92\u751f\uff0c\u5fc3\u5f62\u81f3\u5375\u72c0\u5fc3\u5f62\uff0c\u9577 4-8cm\uff0c\u57fa\u90e8\u5fc3\u5f62\u62b1\u8396\u6216\u8fd1\u622a\u5f62\uff0c\u80cc\u9762\u5e38\u5e36\u7d2b\u7d05\", \"inflorescence\": \"\u7a57\u72c0\u82b1\u5e8f\u9802\u751f\uff0c\u82b1\u5e8f\u4e0b\u65b9\u5177 4 \u679a\u767d\u8272\u82b1\u74e3\u72c0\u7e3d\u82de\u7247\uff0c\u5c55\u6392\u5448\u5341\u5b57\uff08\u8996\u89ba\u4e0a\u4f3c\u56db\u51fa\u767d\u82b1\uff09\", \"smell\": \"\u5168\u682a\u6413\u63c9\u5177\u5f37\u70c8\u9b5a\u8165\u8207\u7279\u6b8a\u8349\u8165\u5473\uff08\u95dc\u9375\u8fa8\u8b58\u7279\u5fb5\uff09\uff0c\u559c\u60e1\u611f\u53d7\u56e0\u4eba\u800c\u7570\", \"habitat_hint\": \"\u8010\u6fd5\u8010\u9670\uff0c\u5e38\u898b\u65bc\u6392\u6c34\u4e0d\u826f\u4e4b\u6eaa\u6e9d\u3001\u6e9d\u908a\u6c34\u6ce5\u65c1\u8207\u5927\u6a39\u4e0b\u88f8\u5730\"}, \"habitat\": \"\u6eaa\u6d41\u908a\u7de3\u3001\u6f6e\u6fd5\u6797\u7de3\u3001\u7530\u57c2\u6e9d\u6e20\u8207\u90fd\u5e02\u9670\u6fd5\u89d2\u843d\u5730\u88ab\uff0c\u6d77\u62d4 0-2000m\", \"harvesting\": {\"target\": \"\u672a\u5f00\u82b1\u6216\u521d\u958b\u82b1\u4e4b\u5e7c\u5ae9\u8449\u8207\u8fd1\u5730\u8868\u5ae9\u5310\u8396\", \"method\": \"\u4ee5\u526a\u5200\u526a\u53d6\u6216\u624b\u6307\u6390\u5c16\uff1b\u5730\u4e0b\u8396\u6dfa\u6398\u6311\u7bc0\u9593\u5ae9\u6bb5\", \"season\": \"\u5168\u5e74\u53ef\u63a1\uff0c\u6625\u590f\u6700\u8106\u5ae9\", \"sustainability\": \"\u4fdd\u7559\u530d\u5310\u8396\u7bc0\u9ede\u8207\u6839\uff0c\u5207\u6bb5\u5f0f\u63a1\u6536\u52ff\u638f\u7a7a\u6574\u7247\u7fa4\u843d\"}, \"preparation\": {\"method\": \"\u6dbc\u62cc\uff08\u91cd\u8fa3\u918b\u849c\uff09\u3001\u5ddd\u71d9\u6cbe\u91ac\u6216\u716e\u6e6f\uff1b\u719f\u98df\u8165\u5473\u7565\u964d\", \"cooking_time\": \"\u5ddd\u71d9 30 \u79d2\u81f3 1 \u5206\u9418\u53ef\u4fdd\u8106\uff1b\u4e45\u716e\u5473\u6563\u5165\u6e6f\", \"taste\": \"\u751f\u98df\u8165\u9999\u6fc3\u70c8\u3001\u7565\u5e36\u8f9b\u8fa3\u8207\u91d1\u5c6c\u611f\u5c3e\u97fb\uff1b\u719f\u5f8c\u8f49\u6e05\u751c\u8349\u672c\", \"notes\": \"\u521d\u6b21\u54c1\u5690\u5efa\u8b70\u5c11\u91cf\uff1b\u8207\u8584\u8377\u3001\u6c34\u82b9\u83dc\u7b49\u6c23\u5473\u690d\u7269\u52ff\u50c5\u6191\u8449\u5f62\u8fa8\u8b58\u3002\"}, \"nutrition\": \"\u591a\u91a3\u9ad4\u3001\u9240\u3001\u985e\u9ec3\u916e\", \"medicinal_uses\": [\"\u6e05\u71b1\u89e3\u6bd2\", \"\u6297\u83cc\u3001\u6297\u767c\u708e\uff08\u6c11\u4fd7\u8207\u90e8\u5206\u7814\u7a76\uff09\", \"\u5229\u5c3f\u3001\u6d88\u816b\uff08\u50b3\u7d71\u914d\u65b9\u7684\u5ba3\u7a31\uff0c\u9700\u9075\u91ab\u56d1\uff09\"], \"caution\": \"\u5b55\u5a66\u3001\u814e\u529f\u80fd\u7570\u5e38\u8207\u9577\u671f\u670d\u85e5\u8005\u5927\u91cf\u836f\u98df\u61c9\u8aee\u8a62\u91ab\u5e2b\u3002\u81ed\u693f\u5e7c\u8449\u7b49\u4e0d\u5177\u5341\u5b57\u82de\u7247\u3002\", \"confusion_risk\": \"low\", \"distribution\": {\"altitude_range\": [0, 2000], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u6eab\u5e36\"]}}, {\"id\": \"miscanthus_floridulus\", \"scientific_name\": \"Miscanthus floridulus\", \"common_names\": {\"zh-TW\": \"\u4e94\u7bc0\u8292\uff08\u5ae9\u7b4d\uff09\", \"en\": \"Giant Chinese Silver Grass\"}, \"category\": \"edible\", \"edible_parts\": [\"\u6625\u5b63\u5e7c\u5ae9\u7b4d\u72c0\u840c\u8617\uff08\u525d\u9664\u5916\u8449\u9798\u5f8c\u4e4b\u5ae9\u82bd\uff09\"], \"morphology\": {\"plant_habit\": \"\u591a\u5e74\u751f\u5927\u578b\u79be\u8349\uff0c\u6210\u682a\u9ad8 2-4m\uff0c\u5e38\u5f62\u6210\u5ee3\u95ca\u53e2\u805a\", \"culm\": \"\u7a08\u7c97\u76f4\uff0c\u7bc0\u90e8\u5e38\u88ab\u767d\u7c89\uff0c\u8cea\u786c\uff1b\u57fa\u90e8\u5bbf\u5b58\u8449\u9798\u5305\u88ab\", \"leaves\": \"\u8449\u7dda\u5f62\uff0c\u9577 40-80cm\uff0c\u8449\u8eab\u8207\u8449\u820c\u908a\u7de3\u92d2\u5229\uff0c\u6613\u5272\u50b7\u76ae\u819a\uff08\u64cd\u4f5c\u9808\u6234\u624b\u5957\uff09\", \"inflorescence\": \"\u79cb\u5b63\u62bd\u51fa\u5927\u578b\u7fbd\u72c0\u9280\u767d\u8272\u81f3\u6de1\u7d2b\u8272\u9aee\u72c0\u82b1\u5e8f\uff0c\u958b\u5c55\u5982\u7fbd\u6247\", \"young_shoot\": \"\u6625\u5b63\u81ea\u5730\u9762\u6216\u820a\u7a08\u814b\u62bd\u51fa\uff0c\u5916\u5f62\u4f3c\u5c0f\u7af9\u7b4d\uff0c\u5916\u88ab\u5c64\u5c64\u9769\u8cea\u8449\u9798\uff0c\u5167\u5c64\u8272\u6de1\u53ef\u98df\uff08\u95dc\u9375\u53ef\u98df\u671f\u6307\u6a19\uff09\", \"rhizome\": \"\u5730\u4e0b\u6839\u8396\u6a6b\u8d70\uff0c\u53e2\u682a\u53ef\u9010\u5e74\u64f4\u5f35\", \"ecology\": \"\u967d\u6027\u5761\u5730\u3001\u8def\u5824\u3001\u6cb3\u5e8a\u8207\u5ee2\u8015\u5730\u5148\u92d2\uff0c\u5317\u81fa\u4f4e\u6d77\u62d4\u6975\u5e38\u898b\"}, \"habitat\": \"\u5c71\u5761\u8349\u539f\u3001\u7522\u696d\u9053\u908a\u5761\u3001\u6cb3\u5cb8\u9ad8\u7058\u8207\u958b\u95ca\u8655\uff0c\u6d77\u62d4 0-1500m\", \"harvesting\": {\"target\": \"\u5c1a\u672a\u9ad8\u65bc\u819d\u84cb\u3001\u9798\u7dca\u5305\u4e14\u6307\u58d3\u4ecd\u8106\u4e4b\u5e7c\u7b4d\", \"method\": \"\u659c\u5200\u5207\u96e2\u57fa\u90e8\u5f8c\u5c64\u5c64\u525d\u9664\u5916\u9798\u81f3\u4e0d\u518d\u73fe\u7e96\u7dad\u786c\u9aef\u8655\u622a\u65b7\", \"season\": \"\u6625\u5b63\u70ba\u4e3b\uff0c\u51b7\u6696\u5e74\u7bc0\u6c23\u53ef\u7565\u524d\u5f8c\u79fb\", \"sustainability\": \"\u540c\u4e00\u53e2\u8f2a\u63a1\u5916\u570d\u840c\u8617\uff0c\u4fdd\u7559\u5167\u5074\u6bcd\u682a\uff1b\u52ff\u6574\u53e2\u6316\u9664\"}, \"preparation\": {\"method\": \"\u5982\u7af9\u7b4d\u8655\u7406\uff1a\u51b7\u6c34\u6216\u7c73\u6cd4\u6c34\u716e\u53bb\u82e6\u6f80\uff0c\u518d\u6dbc\u62cc\u3001\u6ef7\u716e\u6216\u7092\u8089\u7d72\", \"cooking_time\": \"\u53bb\u6f80\u716e 15-25 \u5206\u9418\u8996\u7c97\u7d30\uff1b\u518d\u6599\u7406 3-8 \u5206\u9418\", \"taste\": \"\u8cea\u5730\u4ecb\u65bc\u7af9\u7b4d\u8207\u8606\u7b4d\u4e4b\u9593\uff0c\u6e05\u751c\u5e36\u6de1\u6de1\u8349\u9752\u5473\", \"notes\": \"\u8001\u7b4d\u6728\u8cea\u5316\u6975\u5feb\uff0c\u63a1\u5f8c\u61c9\u5118\u901f\u8655\u7406\uff1b\u8aa4\u63a1\u5176\u4ed6\u4e0d\u78ba\u8a8d\u79be\u672c\u79d1\u5e7c\u82bd\u6709\u98a8\u96aa\u3002\"}, \"nutrition\": \"\u6c34\u5206\u3001\u5c11\u91cf\u86cb\u767d\u8cea\u3001\u81b3\u98df\u7e96\u7dad\u3001\u77fd\u8cea\", \"caution\": \"\u8449\u7de3\u5272\u624b\uff1b\u9808\u78ba\u8a8d\u70ba\u8292\u5c6c\u5927\u578b\u53e2\u751f\u578b\u614b\u8207\u6625\u5b63\u7b4d\u671f\uff0c\u52ff\u8207\u5e7c\u7af9\u6216\u6709\u6bd2\u8349\u672c\u82bd\u6df7\u6dc6\u3002\", \"confusion_risk\": \"medium\", \"distribution\": {\"altitude_range\": [0, 1500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\", \"\u6eab\u5e36\"]}}, {\"id\": \"zanthoxylum_ailanthoides\", \"scientific_name\": \"Zanthoxylum ailanthoides\", \"common_names\": {\"zh-TW\": \"\u98df\u8331\u8438\uff08\u7d05\u523a\u6912\uff09\", \"en\": \"Ailanthus-leaf Prickly Ash\"}, \"category\": \"edible\", \"edible_parts\": [\"\u6625\u5b63\u5ae9\u82bd\u8207\u5e7c\u5ae9\u7fbd\u72c0\u8907\u8449\uff08\u8abf\u5473\u3001\u62cc\u98df\u5c11\u91cf\uff09\"], \"morphology\": {\"plant_habit\": \"\u843d\u8449\u55ac\u6728\uff0c\u9ad8 5-15m\uff0c\u6a39\u51a0\u5ee3\u5375\u5f62\uff0c\u6a39\u76ae\u7c97\u7cd9\u7e31\u88c2\u81f3\u7247\u72c0\u525d\u843d\", \"trunk\": \"\u6a39\u5e79\u5e38\u5177\u7624\u72c0\u7a81\u8d77\u8207\u6728\u8cea\u5316\u7624\u523a\uff0c\u679d\u689d\u5177\u76f4\u523a\u6216\u5f4e\u523a\uff08\u8b66\u6212\uff1a\u907f\u514d\u5f92\u624b\u6500\u6298\uff09\", \"leaves\": \"\u5947\u6578\u7fbd\u72c0\u8907\u8449\u4e92\u751f\uff0c\u5c0f\u8449 7-15 \u679a\uff0c\u5375\u72c0\u62ab\u91dd\u5f62\uff0c\u9577 6-12cm\uff0c\u8449\u7de3\u7d30\u92f8\u9f52\", \"aroma\": \"\u5c0f\u8449\u6417\u788e\u6216\u6413\u63c9\u91cb\u653e\u5f37\u70c8\u82b1\u6912\u3001\u6ab8\u6aac\u70ef\u985e\u8f9b\u9999\uff08\u95dc\u9375\u7279\u5fb5\uff09\uff0c\u56de\u5473\u9ebb\u820c\", \"fruit\": \"\u84c7\u8456\u679c\u5c0f\uff0c\u6210\u719f\u6642\u7d05\u8272\uff0c\u679c\u76ae\u5177\u817a\u9ede\u8207\u6cb9\u80de\uff0c\u6563\u767c\u9999\u8f9b\u6599\u6c23\u606f\", \"seasonal\": \"\u6625\u5b63\u840c\u82bd\u6642\u8449\u5e8f\u7dca\u5bc6\u82bd\u9c57\u5305\u88ab\uff0c\u63a1\u300c\u6912\u82bd\u300d\u70ba\u5730\u65b9\u98a8\u5473\u98df\u6750\"}, \"habitat\": \"\u4e2d\u4f4e\u6d77\u62d4\u5411\u967d\u6797\u7de3\u3001\u6b21\u751f\u6797\u8207\u4e18\u9675\u5761\u5730\uff0c\u6d77\u62d4 300-1800m\", \"harvesting\": {\"target\": \"\u672a\u5b8c\u5168\u5c55\u958b\u4e4b\u5ae9\u82bd\u6216\u5c55\u958b\u5f8c\u4ecd\u67d4\u8edf\u7684\u7b2c\u4e00\u8f2a\u5c0f\u8449\", \"method\": \"\u4f7f\u7528\u526a\u5200\u907f\u958b\u679d\u523a\u526a\u68a2\uff1b\u6234\u624b\u5957\u9632\u523a\", \"season\": \"\u6625\u5b63 3-5 \u6708\u70ba\u4e3b\uff0c\u8996\u6d77\u62d4\u63d0\u524d\u6216\u5ef6\u5f8c\", \"sustainability\": \"\u6bcf\u682a\u63a1\u5c11\u91cf\u9802\u82bd\uff0c\u52ff\u91cd\u8907\u63a1\u5149\u540c\u4e00\u4e3b\u679d\"}, \"preparation\": {\"method\": \"\u5ae9\u82bd\u6c46\u71d9\u6578\u79d2\u53bb\u9752\u5f8c\u6dbc\u62cc\u3001\u714e\u86cb\u6216\u5207\u788e\u914d\u8089\uff1b\u4ea6\u4e7e\u71e5\u7814\u78e8\u4f5c\u8f9b\u9999\u6599\", \"cooking_time\": \"\u6c46\u71d9 10-20 \u79d2\u5373\u53ef\uff1b\u4e45\u716e\u9ebb\u9999\u6613\u6563\u5931\", \"taste\": \"\u8f9b\u9999\u9ebb\u820c\u3001\u67d1\u6a58\u985e\u6e05\u82b3\uff0c\u8fd1\u4f3c\u82b1\u6912\u800c\u8449\u9999\u66f4\u8349\u7da0\", \"notes\": \"\u7528\u91cf\u5b9c\u5c11\uff1b\u4e0d\u8010\u9ebb\u5473\u8005\u53ef\u8207\u91ac\u6cb9\u3001\u7cd6\u5e73\u8861\u3002\u907f\u514d\u8aa4\u63a1\u81ed\u693f\u7b49\u7fbd\u72c0\u8907\u8449\u4f46\u7121\u82b1\u6912\u55c5\u5473\u6a39\u7a2e\u3002\"}, \"nutrition\": \"\u63ee\u767c\u6cb9\uff08\u4ee5\u82b3\u6a1f\u9187\u3001\u6ab8\u70ef\u985e\u70ba\u4e3b\uff09\u3001\u5c11\u91cf\u7dad\u751f\u7d20C\", \"caution\": \"\u82b1\u6912\u79d1\u6709\u523a\uff1b\u904e\u91cf\u751f\u98df\u523a\u6fc0\u8178\u80c3\uff1b\u5c0d\u8f9b\u9999\u6599\u904e\u654f\u8005\u614e\u7528\u3002\u8207\u82e6\u695d\u3001\u81ed\u693f\u7b49\u5340\u5206\u4e4b\u95dc\u9375\u5728\u65bc\u6413\u8449\u6c23\u5473\u8207\u9ebb\u611f\u3002\", \"confusion_risk\": \"medium\", \"distribution\": {\"altitude_range\": [300, 1800], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u6eab\u5e36\"]}}, {\"id\": \"trema_orientalis\", \"scientific_name\": \"Trema orientalis\", \"common_names\": {\"zh-TW\": \"\u5c71\u9ec3\u9ebb\", \"en\": \"Indian Charcoal Tree\"}, \"category\": \"edible\", \"edible_parts\": [\"\u5ae9\u8449\uff08\u716e\u719f\uff09\", \"\u6210\u719f\u5c0f\u578b\u6f3f\u679c\u72c0\u6838\u679c\"], \"morphology\": {\"plant_habit\": \"\u5e38\u7da0\u5c0f\u55ac\u6728\u81f3\u4e2d\u55ac\u6728\uff0c\u9ad8 5-15m\uff0c\u679d\u689d\u7d30\u9577\uff0c\u6a39\u76ae\u7070\u8910\u800c\u7a0d\u5149\u6ed1\", \"leaves\": \"\u8449\u4e92\u751f\uff0c\u5169\u5217\u8fd1\u65bc\u6c34\u5e73\u4f38\u5c55\uff0c\u9577\u5375\u5f62\u81f3\u5375\u72c0\u62ab\u91dd\u5f62\uff0c\u9577 6-14cm\uff0c\u5148\u7aef\u6f38\u5c16\uff0c\u8449\u7de3\u7d30\u92f8\u9f52\", \"leaf_surface\": \"\u8449\u9762\u7c97\u7cd9\u4f3c\u7802\u5e03\u4f46\u8f03\u69cb\u6a39\u7d30\u7dfb\uff0c\u80cc\u9762\u8272\u6de1\u7db2\u8108\u660e\u986f\uff1b\u812b\u843d\u524d\u53ef\u8f49\u9ec3\", \"fruit\": \"\u814b\u751f\u5c0f\u578b\u6838\u679c\uff0c\u76f4\u5f91\u7d04 3-4mm\uff0c\u719f\u6642\u7531\u6a59\u7d05\u8b8a\u6df1\u7d2b\u81f3\u8fd1\u9ed1\uff08\u95dc\u9375\u7279\u5fb5\uff09\uff0c\u591a\u8089\u8cea\u8584\u5c64\", \"habitat_ecology\": \"\u5148\u9a45\u967d\u6027\u6a39\u7a2e\uff0c\u90fd\u5e02\u8352\u5ee2\u5730\u3001\u5761\u574e\u3001\u9632\u706b\u5df7\u8207\u6cb3\u5cb8\u88f8\u5730\u6700\u5e38\u898b\u4e4b\u4e00\", \"bark\": \"\u5e7c\u6a39\u76ae\u5e73\u6ed1\u8001\u6a39\u7565\u7e31\u88c2\uff0c\u6613\u8207\u5176\u4ed6\u6986\u79d1\u8fd1\u7de3\u7a2e\u6df7\u6dc6\uff0c\u9808\u9760\u679c\u8207\u8449\u8cea\u5408\u5224\"}, \"habitat\": \"\u4f4e\u6d77\u62d4\u958b\u95ca\u5730\u3001\u6cb3\u5cb8\u51b2\u7a4d\u5730\u3001\u9053\u8def\u908a\u5761\u8207\u6b21\u751f\u6797\u7a97\u53e3\uff0c\u6d77\u62d4 0-1500m\", \"harvesting\": {\"target\": \"\u5ae9\u68a2\u8449\u6216\u5df2\u8f49\u8272\u4e4b\u6210\u719f\u5c0f\u679c\", \"method\": \"\u6458\u53d6\u672a\u786c\u5316\u4e4b\u9802\u7aef 2-4 \u679a\u8449\uff1b\u679c\u5be6\u4ee5\u5bb9\u5668\u627f\u63a5\u6416\u843d\", \"season\": \"\u5ae9\u8449\u6625\u590f\uff1b\u679c\u719f\u591a\u590f\u79cb\uff0c\u56e0\u5730\u57df\u7565\u7570\", \"sustainability\": \"\u8f15\u5ea6\u6458\u8449\u4e0d\u5f71\u97ff\u5148\u9a45\u529f\u80fd\uff1b\u7559\u6a39\u90e8\u5206\u679c\u5be6\u7dad\u6301\u9ce5\u985e\u98df\u6e90\"}, \"preparation\": {\"method\": \"\u5ae9\u8449\u5ddd\u71d9\u5f8c\u7092\u98df\u6216\u716e\u6e6f\uff1b\u719f\u679c\u5c11\u91cf\u9bae\u98df\u6216\u716e\u751c\u6e6f\", \"cooking_time\": \"\u5ae9\u8449\u5ddd\u71d9 1-2 \u5206\u9418\u5f8c\u7092 2-3 \u5206\u9418\", \"taste\": \"\u8449\u716e\u719f\u5f8c\u7565\u6709\u8349\u672c\u6f80\u5473\u8207\u8c46\u8165\u8abf\u6027\uff1b\u679c\u5473\u6de1\u751c\u5fae\u6f80\u3001\u6c41\u6db2\u5c11\", \"notes\": \"\u539f\u4f4f\u6c11\u8207\u5c71\u5340\u805a\u843d\u50b3\u7d71\u5076\u4f5c\u6551\u8352\u91ce\u83dc\uff1b\u9996\u6b21\u5c11\u91cf\u8a66\u98df\u3002\u8207\u6734\u6a39\u7b49\u8fd1\u4f3c\u6642\u4ee5\u679c\u5f91\u8207\u8449\u8cea\u78ba\u8a8d\u3002\"}, \"nutrition\": \"\u7926\u7269\u8cea\u3001\u5c11\u91cf\u7dad\u751f\u7d20\u3001\u81b3\u98df\u7e96\u7dad\", \"caution\": \"\u679c\u5be6\u6975\u5c0f\uff0c\u8aa4\u63a1\u5c1a\u672a\u7531\u7d05\u8f49\u9ed1\u8005\u6f80\u5473\u8207\u55ae\u5be7\u8f03\u9ad8\u3002\u6c61\u67d3\u74b0\u5883\u690d\u682a\u52ff\u63a1\u3002\", \"confusion_risk\": \"medium\", \"distribution\": {\"altitude_range\": [0, 1500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\"]}}, {\"id\": \"pterocypsela_indica\", \"scientific_name\": \"Pterocypsela indica\", \"common_names\": {\"zh-TW\": \"\u9d5d\u4ed4\u8349\uff08\u9d5d\u83dc\u3001\u5c71\u8435\u82e3\uff09\", \"en\": \"Indian Lettuce\"}, \"category\": \"edible\", \"edible_parts\": [\"\u5ae9\u8396\u8449\"], \"morphology\": {\"plant_habit\": \"\u4e00\u5e74\u751f\u8349\u672c\uff0c\u9ad8 30-100cm\uff0c\u5168\u682a\u542b\u4e73\u6c41\uff0c\u65b7\u9762\u660e\u986f\u4e2d\u7a7a\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"stem\": \"\u8396\u76f4\u7acb\u5177\u7e31\u7a1c\uff0c\u8868\u9762\u5149\u6ed1\u6216\u7565\u88ab\u7c89\uff1b\u7bc0\u9593\u9577\u5ea6\u4e2d\u81f3\u9577\", \"basal_leaves\": \"\u57fa\u751f\u8449\u5927\u578b\uff0c\u7434\u5f62\u7fbd\u88c2\u81f3\u6df1\u7fbd\u72c0\u5206\u88c2\uff0c\u9577 10-25cm\uff0c\u88c2\u7247\u4e0d\u7b49\u5927\", \"cauline_leaves\": \"\u8396\u751f\u8449\u6f38\u5c0f\uff0c\u57fa\u90e8\u5e38\u62b1\u8396\u6216\u534a\u62b1\u8396\uff0c\u5148\u7aef\u88c2\u7247\u8b8a\u5316\u5927\", \"inflorescence\": \"\u982d\u72c0\u82b1\u5e8f\u5c0f\u578b\uff0c\u591a\u82b1\u6392\u5217\u6210\u9b06\u6563\u7e56\u623f\u6216\u5713\u9310\u72c0\uff1b\u82b1\u6de1\u9ec3\u8272\u820c\u72c0\u82b1\uff0c\u7121\u660e\u986f\u5927\u578b\u820c\u74e3\u5c55\u793a\", \"pappus\": \"\u7626\u679c\u5177\u767d\u8272\u51a0\u6bdb\uff0c\u6210\u719f\u6642\u96a8\u98a8\u98c4\u6563\uff08\u4f3c\u84b2\u516c\u82f1\uff0c\u8f14\u52a9\u8207\u90e8\u5206\u83ca\u79d1\u5340\u5206\uff09\", \"latex\": \"\u4efb\u4f55\u5275\u50b7\u8655\u6ef2\u51fa\u767d\u8272\u4e73\u6c41\uff0c\u8207\u662d\u548c\u8349\u53ef\u6bd4\u5c0d\u4f46\u8449\u88c2\u8207\u82b1\u8272\u4e0d\u540c\"}, \"habitat\": \"\u8352\u5ee2\u83dc\u5712\u3001\u8def\u80a9\u3001\u6cb3\u5cb8\u788e\u77f3\u5730\u8207\u90fd\u5e02\u7a7a\u5730\uff0c\u6d77\u62d4 0-2500m\uff0c\u5168\u5e74\u53ef\u898b\", \"harvesting\": {\"target\": \"\u62bd\u85b9\u524d\u4e4b\u84ee\u5ea7\u671f\u5ae9\u8449\u6216\u62bd\u85b9\u521d\u671f\u4e4b\u5ae9\u8396\u68a2\", \"method\": \"\u6298\u53d6\u672a\u6728\u8cea\u5316\u68a2\u6bb5 15-25cm\uff1b\u6234\u624b\u5957\u9632\u4e73\u6c41\u904e\u654f\", \"season\": \"\u5168\u5e74\u53ef\u63a1\uff0c\u51ac\u6625\u57fa\u751f\u8449\u80a5\u539a\", \"sustainability\": \"\u65cf\u7fa4\u6062\u5fa9\u5feb\uff0c\u63a1\u96c6\u4ee5\u526a\u4ee3\u62d4\uff0c\u4fdd\u7559\u6700\u4f4e\u4e00\u682a\u4ee5\u7dad\u6301\u81ea\u64ad\"}, \"preparation\": {\"method\": \"\u5ddd\u71d9\u53bb\u82e6\u5473\u5f8c\u6dbc\u62cc\uff08\u9ebb\u6cb9\u3001\u849c\u6ce5\uff09\u3001\u5feb\u7092\u6216\u716e\u7ca5\", \"cooking_time\": \"\u5ddd\u71d9 1-2 \u5206\u9418\uff1b\u5feb\u7092 2-3 \u5206\u9418\", \"taste\": \"\u7565\u82e6\u56de\u7518\uff0c\u53e3\u611f\u6e05\u8106\u591a\u6c41\uff0c\u8fd1\u4f3c\u8435\u82e3\u91ce\u83dc\", \"notes\": \"\u4e73\u6c41\u53ef\u80fd\u5f15\u8d77\u76ae\u819a\u523a\u7662\uff1b\u6613\u8207\u82e6\u82e3\u83dc\u5c6c\u6df7\u6dc6\uff0c\u61c9\u4ee5\u982d\u72c0\u82b1\u8207\u8449\u88c2\u6a21\u5f0f\u4ea4\u53c9\u6bd4\u5c0d\u3002\"}, \"nutrition\": \"\u7dad\u751f\u7d20A\u3001C\u3001\u4e73\u6c41\u4e2d\u542b\u841c\u985e\u82e6\u5473\u7269\u8cea\u3001\u81b3\u98df\u7e96\u7dad\", \"caution\": \"\u4e73\u6c41\u904e\u654f\u8005\u6234\u624b\u5957\uff1b\u516c\u8def\u908a\u690d\u682a\u6ce8\u610f\u91cd\u91d1\u5c6c\u8207\u7070\u5875\u3002\u8207\u6709\u6bd2\u4e73\u8349\u985e\u52ff\u50c5\u9760\u8449\u5f62\u5224\u65b7\u3002\", \"confusion_risk\": \"medium\", \"distribution\": {\"altitude_range\": [0, 2500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u6eab\u5e36\"]}}]")

_KB_DANGEROUS_ANIMALS = json.loads("[{\"id\": \"trimeresurus_stejnegeri\", \"scientific_name\": \"Trimeresurus stejnegeri\", \"common_names\": {\"zh-TW\": \"\u8d64\u5c3e\u9752\u7af9\u7d72\", \"en\": \"Stejneger's Green Bamboo Viper\"}, \"category\": \"venomous_snake\", \"danger_level\": \"high\", \"venom_type\": \"\u51fa\u8840\u6027\u6bd2\", \"morphology\": {\"body_color\": \"\u5168\u8eab\u9bae\u7da0\u8272\uff0c\u9ad4\u5074\u6709\u767d\u8272\u6216\u6de1\u9ec3\u8272\u7e31\u7dda\", \"tail\": \"\u5c3e\u5df4\u672b\u7aef\u70ba\u78da\u7d05\u8272\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"head_shape\": \"\u4e09\u89d2\u5f62\uff0c\u660e\u986f\u6bd4\u9838\u90e8\u5bec\uff08\u876e\u86c7\u79d1\u7279\u5fb5\uff09\", \"pupil\": \"\u7e31\u88c2\u77b3\u5b54\uff08\u8c93\u773c\u72c0\uff09\", \"pit_organ\": \"\u773c\u8207\u9f3b\u5b54\u4e4b\u9593\u6709\u660e\u986f\u7684\u9830\u7aa9\uff08\u7d05\u5916\u7dda\u611f\u61c9\u5668\uff09\", \"body_size\": \"\u4e2d\u5c0f\u578b\uff0c\u5168\u9577 60-90cm\uff0c\u9ad4\u578b\u7d30\u9577\", \"scales\": \"\u80cc\u9c57\u6709\u7a1c\u810a\"}, \"habitat\": \"\u4f4e\u4e2d\u6d77\u62d4\u68ee\u6797\u3001\u7af9\u6797\u3001\u704c\u6728\u53e2\uff0c\u5e38\u7e8f\u7e5e\u5728\u6a39\u679d\u4e0a\uff0c\u6d77\u62d4 0-2000m\", \"behavior\": \"\u591c\u884c\u6027\uff0c\u653b\u64ca\u6027\u4e2d\u7b49\u3002\u5e38\u975c\u6b62\u4e0d\u52d5\u507d\u88dd\u5728\u679d\u8449\u9593\u3002\u53d7\u9a5a\u6642\u6703\u5feb\u901f\u54ac\u64ca\u3002\", \"bite_symptoms\": [\"\u50b7\u53e3\u5287\u70c8\u75bc\u75db\", \"\u5c40\u90e8\u8fc5\u901f\u816b\u8139\", \"\u51fa\u8840\u4e0d\u6b62\uff08\u51dd\u8840\u529f\u80fd\u969c\u7919\uff09\", \"\u50b7\u53e3\u5468\u570d\u7600\u9752\", \"\u56b4\u91cd\u8005\u53ef\u80fd\u7d44\u7e54\u58de\u6b7b\"], \"first_aid\": {\"do\": [\"\u4fdd\u6301\u51b7\u975c\uff0c\u6e1b\u5c11\u6d3b\u52d5\", \"\u8a18\u4f4f\u86c7\u7684\u5916\u89c0\u7279\u5fb5\uff08\u62cd\u7167\u66f4\u4f73\uff09\", \"\u79fb\u9664\u50b7\u80a2\u4e0a\u7684\u6212\u6307\u3001\u624b\u9336\u7b49\u675f\u7e1b\u7269\", \"\u4fdd\u6301\u50b7\u80a2\u4f4e\u65bc\u5fc3\u81df\u4f4d\u7f6e\", \"\u5118\u901f\u9001\u91ab\uff0c\u9700\u65bd\u6253\u6297\u86c7\u6bd2\u8840\u6e05\"], \"do_not\": [\"\u52ff\u5207\u958b\u50b7\u53e3\", \"\u52ff\u7528\u5634\u5438\u6bd2\", \"\u52ff\u7d81\u6b62\u8840\u5e36\uff08\u52a0\u58d3\u7e43\u5e36\u4f8b\u5916\uff09\", \"\u52ff\u51b0\u6577\", \"\u52ff\u98f2\u9152\", \"\u52ff\u5617\u8a66\u6355\u86c7\"], \"antivenom\": \"\u51fa\u8840\u6027\u86c7\u6bd2\u8840\u6e05\uff08\u53f0\u7063\u5404\u5927\u91ab\u9662\u6025\u8a3a\u5099\u6709\uff09\"}, \"activity_pattern\": \"\u591c\u884c\u6027\u70ba\u4e3b\uff0c\u9670\u5929\u6216\u96e8\u5f8c\u767d\u5929\u4e5f\u6703\u6d3b\u52d5\", \"distribution\": {\"altitude_range\": [0, 2000], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\", \"\u71b1\u5e36\"]}, \"encounter_frequency\": \"\u53f0\u7063\u6700\u5e38\u898b\u7684\u6bd2\u86c7\uff0c\u54ac\u50b7\u6848\u4f8b\u6578\u6700\u591a\"}, {\"id\": \"protobothrops_mucrosquamatus\", \"scientific_name\": \"Protobothrops mucrosquamatus\", \"common_names\": {\"zh-TW\": \"\u9f9c\u6bbc\u82b1\", \"en\": \"Taiwan Habu\"}, \"category\": \"venomous_snake\", \"danger_level\": \"high\", \"venom_type\": \"\u51fa\u8840\u6027\u6bd2\", \"morphology\": {\"body_color\": \"\u7070\u8910\u8272\u5e95\uff0c\u80cc\u90e8\u6709\u6df1\u8910\u8272\u5927\u578b\u9f9c\u7532\u72c0\u6591\u7d0b\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"head_shape\": \"\u4e09\u89d2\u5f62\uff0c\u5bec\u5927\uff0c\u8207\u9838\u90e8\u5206\u754c\u660e\u986f\", \"pupil\": \"\u7e31\u88c2\u77b3\u5b54\", \"pit_organ\": \"\u773c\u524d\u65b9\u6709\u9830\u7aa9\", \"body_size\": \"\u4e2d\u5927\u578b\uff0c\u5168\u9577 80-130cm\uff0c\u9ad4\u578b\u7c97\u58ef\", \"scales\": \"\u80cc\u9c57\u6709\u5f37\u7a1c\u810a\uff0c\u9ad4\u8868\u7c97\u7cd9\u611f\"}, \"habitat\": \"\u4f4e\u6d77\u62d4\u68ee\u6797\u3001\u8fb2\u7530\u9644\u8fd1\u3001\u4f4f\u5bb6\u5468\u570d\u3001\u77f3\u5806\u3001\u67f4\u5806\uff0c\u6d77\u62d4 0-1500m\", \"behavior\": \"\u591c\u884c\u6027\uff0c\u653b\u64ca\u6027\u8f03\u9ad8\u3002\u53d7\u9a5a\u6642\u6703\u5feb\u901f\u53cd\u64ca\u3002\u5e38\u51fa\u73fe\u5728\u4eba\u985e\u6d3b\u52d5\u5340\u57df\u9644\u8fd1\u3002\", \"bite_symptoms\": [\"\u50b7\u53e3\u5287\u70c8\u816b\u75db\", \"\u8fc5\u901f\u816b\u8139\u64f4\u6563\", \"\u51fa\u8840\u4e0d\u6b62\", \"\u7d44\u7e54\u58de\u6b7b\uff08\u56b4\u91cd\u8005\uff09\", \"\u53ef\u80fd\u9700\u8981\u622a\u80a2\uff08\u6975\u56b4\u91cd\u8005\uff09\"], \"first_aid\": {\"do\": [\"\u4fdd\u6301\u51b7\u975c\uff0c\u6e1b\u5c11\u6d3b\u52d5\", \"\u8a18\u4f4f\u86c7\u7684\u5916\u89c0\u7279\u5fb5\", \"\u79fb\u9664\u50b7\u80a2\u675f\u7e1b\u7269\", \"\u4fdd\u6301\u50b7\u80a2\u4f4e\u65bc\u5fc3\u81df\u4f4d\u7f6e\", \"\u5118\u901f\u9001\u91ab\"], \"do_not\": [\"\u52ff\u5207\u958b\u50b7\u53e3\", \"\u52ff\u7528\u5634\u5438\u6bd2\", \"\u52ff\u7d81\u6b62\u8840\u5e36\", \"\u52ff\u51b0\u6577\", \"\u52ff\u5617\u8a66\u6355\u86c7\"], \"antivenom\": \"\u51fa\u8840\u6027\u86c7\u6bd2\u8840\u6e05\"}, \"activity_pattern\": \"\u591c\u884c\u6027\uff0c\u9ec3\u660f\u81f3\u591c\u9593\u6700\u6d3b\u8e8d\", \"distribution\": {\"altitude_range\": [0, 1500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\"]}, \"encounter_frequency\": \"\u53f0\u7063\u7b2c\u4e8c\u5e38\u898b\u7684\u6bd2\u86c7\u54ac\u50b7\u4f86\u6e90\"}, {\"id\": \"bungarus_multicinctus\", \"scientific_name\": \"Bungarus multicinctus\", \"common_names\": {\"zh-TW\": \"\u96e8\u5098\u7bc0\uff08\u9280\u74b0\u86c7\uff09\", \"en\": \"Many-banded Krait\"}, \"category\": \"venomous_snake\", \"danger_level\": \"critical\", \"venom_type\": \"\u795e\u7d93\u6027\u6bd2\", \"morphology\": {\"body_color\": \"\u5168\u8eab\u9ed1\u767d\u76f8\u9593\u7684\u74b0\u72c0\u6a6b\u5e36\uff08\u95dc\u9375\u7279\u5fb5\uff09\uff0c\u9ed1\u767d\u5e36\u7b49\u5bec\u6216\u9ed1\u5e36\u7565\u5bec\", \"head_shape\": \"\u6a62\u5713\u5f62\uff0c\u8207\u9838\u90e8\u5340\u5206\u4e0d\u660e\u986f\uff08\u975e\u4e09\u89d2\u5f62\uff09\", \"pupil\": \"\u5713\u5f62\u77b3\u5b54\", \"body_size\": \"\u4e2d\u578b\uff0c\u5168\u9577 100-150cm\", \"dorsal_scales\": \"\u80cc\u90e8\u4e2d\u592e\u4e00\u5217\u9c57\u7247\u660e\u986f\u64f4\u5927\u5448\u516d\u89d2\u5f62\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"tail\": \"\u5c3e\u90e8\u77ed\uff0c\u672b\u7aef\u920d\u5713\"}, \"habitat\": \"\u4f4e\u6d77\u62d4\u8fb2\u7530\u3001\u6c34\u908a\u3001\u4f4f\u5bb6\u9644\u8fd1\u3001\u6392\u6c34\u6e9d\uff0c\u6d77\u62d4 0-500m\", \"behavior\": \"\u591c\u884c\u6027\uff0c\u52d5\u4f5c\u7de9\u6162\uff0c\u767d\u5929\u6eab\u99b4\u3002\u4f46\u6bd2\u6027\u6975\u5f37\uff01\u591c\u9593\u6703\u4e3b\u52d5\u8993\u98df\u3002\", \"bite_symptoms\": [\"\u26a0\ufe0f \u54ac\u50b7\u521d\u671f\u5e7e\u4e4e\u4e0d\u75db\uff08\u6975\u5ea6\u5371\u96aa\uff01\u5bb9\u6613\u88ab\u5ffd\u8996\uff09\", \"\u6578\u5c0f\u6642\u5f8c\u51fa\u73fe\u773c\u77bc\u4e0b\u5782\", \"\u541e\u56a5\u56f0\u96e3\", \"\u5168\u8eab\u808c\u8089\u7121\u529b\", \"\u547c\u5438\u56f0\u96e3\", \"\u547c\u5438\u8870\u7aed\uff08\u53ef\u81f4\u6b7b\uff09\"], \"first_aid\": {\"do\": [\"\u26a0\ufe0f \u5373\u4f7f\u4e0d\u75db\u4e5f\u5fc5\u9808\u7acb\u5373\u5c31\u91ab\uff01\u9019\u662f\u6700\u5371\u96aa\u7684\u90e8\u5206\", \"\u4fdd\u6301\u51b7\u975c\", \"\u8a18\u4f4f\u86c7\u7684\u5916\u89c0\uff08\u9ed1\u767d\u76f8\u9593\uff09\", \"\u53ef\u4f7f\u7528\u5f48\u6027\u7e43\u5e36\u52a0\u58d3\u5305\u7d2e\uff08\u58d3\u529b\u56fa\u5b9a\u6cd5\uff09\uff0c\u6e1b\u7de9\u6bd2\u6db2\u64f4\u6563\", \"\u5118\u901f\u9001\u91ab\uff0c\u9700\u65bd\u6253\u795e\u7d93\u6027\u86c7\u6bd2\u8840\u6e05\", \"\u6ce8\u610f\u547c\u5438\u72c0\u6cc1\uff0c\u6e96\u5099\u4eba\u5de5\u547c\u5438\"], \"do_not\": [\"\u52ff\u5ffd\u8996\uff01\u5373\u4f7f\u50b7\u53e3\u4e0d\u75db\u4e5f\u8981\u5c31\u91ab\", \"\u52ff\u5207\u958b\u50b7\u53e3\", \"\u52ff\u7528\u5634\u5438\u6bd2\", \"\u52ff\u7d81\u6b62\u8840\u5e36\"], \"antivenom\": \"\u795e\u7d93\u6027\u86c7\u6bd2\u8840\u6e05\uff08\u6642\u9593\u662f\u95dc\u9375\uff0c\u8d8a\u65e9\u65bd\u6253\u8d8a\u597d\uff09\"}, \"activity_pattern\": \"\u56b4\u683c\u591c\u884c\u6027\", \"distribution\": {\"altitude_range\": [0, 500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\"]}, \"encounter_frequency\": \"\u4e0d\u5e38\u898b\uff0c\u4f46\u81f4\u6b7b\u7387\u6700\u9ad8\u7684\u53f0\u7063\u6bd2\u86c7\"}, {\"id\": \"naja_atra\", \"scientific_name\": \"Naja atra\", \"common_names\": {\"zh-TW\": \"\u773c\u93e1\u86c7\uff08\u98ef\u5319\u5029\uff09\", \"en\": \"Chinese Cobra\"}, \"category\": \"venomous_snake\", \"danger_level\": \"critical\", \"venom_type\": \"\u6df7\u5408\u6bd2\uff08\u795e\u7d93\u6bd2 + \u7d30\u80de\u6bd2\uff09\", \"morphology\": {\"body_color\": \"\u5168\u8eab\u9ed1\u8272\u6216\u6df1\u8910\u8272\uff0c\u8179\u9762\u6de1\u8272\", \"hood\": \"\u53d7\u5a01\u8105\u6642\u9838\u90e8\u5f35\u958b\u5448\u6241\u5e73\u72c0\uff08\u95dc\u9375\u7279\u5fb5\uff09\uff0c\u53ef\u898b\u767d\u8272\u773c\u93e1\u72c0\u6591\u7d0b\", \"head_shape\": \"\u6a62\u5713\u5f62\uff0c\u53d7\u5a01\u8105\u6642\u9838\u90e8\u660e\u986f\u64f4\u5c55\", \"pupil\": \"\u5713\u5f62\u77b3\u5b54\", \"body_size\": \"\u4e2d\u5927\u578b\uff0c\u5168\u9577 120-150cm\", \"scales\": \"\u80cc\u9c57\u5149\u6ed1\"}, \"habitat\": \"\u4f4e\u6d77\u62d4\u8fb2\u7530\u3001\u6eaa\u908a\u3001\u7af9\u6797\u3001\u4f4f\u5bb6\u9644\u8fd1\uff0c\u6d77\u62d4 0-800m\", \"behavior\": \"\u65e5\u884c\u6027\u70ba\u4e3b\u3002\u53d7\u9a5a\u6642\u6703\u62ac\u8d77\u524d\u8eab\u3001\u5f35\u958b\u9838\u90e8\u3001\u767c\u51fa\u5636\u5636\u8072\u8b66\u544a\u3002\u53ef\u5674\u6bd2\uff08\u5c04\u7a0b\u9054 2m\uff09\u3002\", \"bite_symptoms\": [\"\u50b7\u53e3\u75bc\u75db\", \"\u5c40\u90e8\u816b\u8139\", \"\u7d44\u7e54\u58de\u6b7b\", \"\u773c\u77bc\u4e0b\u5782\uff08\u795e\u7d93\u6bd2\u75c7\u72c0\uff09\", \"\u547c\u5438\u56f0\u96e3\", \"\u88ab\u5674\u6bd2\u6db2\u5165\u773c\uff1a\u5287\u70c8\u75bc\u75db\u3001\u8996\u529b\u53d7\u640d\"], \"first_aid\": {\"do\": [\"\u7acb\u5373\u9060\u96e2\uff08\u6b64\u86c7\u653b\u64ca\u7bc4\u570d\u5927\uff09\", \"\u5118\u901f\u9001\u91ab\", \"\u5982\u6bd2\u6db2\u5674\u5165\u773c\u775b\uff1a\u7acb\u5373\u7528\u5927\u91cf\u6e05\u6c34\u6c96\u6d17\u81f3\u5c11 15 \u5206\u9418\", \"\u8a18\u4f4f\u86c7\u7684\u5916\u89c0\u7279\u5fb5\"], \"do_not\": [\"\u52ff\u5617\u8a66\u8207\u5176\u5c0d\u5cd9\", \"\u52ff\u5207\u958b\u50b7\u53e3\", \"\u52ff\u7528\u5634\u5438\u6bd2\"], \"antivenom\": \"\u795e\u7d93\u6027 + \u51fa\u8840\u6027\u6df7\u5408\u86c7\u6bd2\u8840\u6e05\"}, \"activity_pattern\": \"\u65e5\u884c\u6027\u70ba\u4e3b\uff0c\u5076\u723e\u591c\u9593\u6d3b\u52d5\", \"distribution\": {\"altitude_range\": [0, 800], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\"]}, \"encounter_frequency\": \"\u4e2d\u7b49\uff0c\u4f46\u653b\u64ca\u6027\u5f37\uff0c\u54ac\u50b7\u5f8c\u679c\u56b4\u91cd\"}, {\"id\": \"deinagkistrodon_acutus\", \"scientific_name\": \"Deinagkistrodon acutus\", \"common_names\": {\"zh-TW\": \"\u767e\u6b65\u86c7\uff08\u5c16\u543b\u876e\uff09\", \"en\": \"Hundred-pace Pit Viper\"}, \"category\": \"venomous_snake\", \"danger_level\": \"critical\", \"venom_type\": \"\u51fa\u8840\u6027\u6bd2 + \u7d30\u80de\u6bd2\", \"morphology\": {\"body_color\": \"\u7070\u8910\u8272\u5e95\uff0c\u80cc\u90e8\u6709\u5927\u578b\u6df1\u8272\u4e09\u89d2\u5f62\u6591\u7d0b\uff0c\u6392\u5217\u6210\u93c8\u72c0\", \"head_shape\": \"\u5927\u4e09\u89d2\u5f62\uff0c\u543b\u7aef\u660e\u986f\u4e0a\u7ff9\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"pupil\": \"\u7e31\u88c2\u77b3\u5b54\", \"pit_organ\": \"\u773c\u524d\u65b9\u6709\u660e\u986f\u9830\u7aa9\", \"body_size\": \"\u5927\u578b\uff0c\u5168\u9577 120-150cm\uff0c\u9ad4\u578b\u6975\u7c97\u58ef\", \"tail\": \"\u5c3e\u7aef\u6709\u89d2\u8cea\u5316\u5c16\u523a\"}, \"habitat\": \"\u4e2d\u4f4e\u6d77\u62d4\u539f\u59cb\u68ee\u6797\u3001\u5c71\u5340\uff0c\u5ca9\u77f3\u5806\u3001\u843d\u8449\u5806\u4e2d\uff0c\u6d77\u62d4 300-1500m\", \"behavior\": \"\u52d5\u4f5c\u9072\u7de9\uff0c\u4f46\u653b\u64ca\u901f\u5ea6\u6975\u5feb\u3002\u4fdd\u8b77\u8272\u6975\u4f73\uff0c\u5e38\u96b1\u85cf\u5728\u843d\u8449\u4e2d\u4e0d\u6613\u767c\u73fe\u3002\u53f0\u7063\u539f\u4f4f\u6c11\u65cf\u7684\u6587\u5316\u5716\u9a30\u3002\", \"bite_symptoms\": [\"\u50b7\u53e3\u5287\u70c8\u75bc\u75db\", \"\u8fc5\u901f\u5927\u9762\u7a4d\u816b\u8139\", \"\u5927\u91cf\u51fa\u8840\", \"\u56b4\u91cd\u7d44\u7e54\u58de\u6b7b\", \"\u5168\u8eab\u6027\u51dd\u8840\u529f\u80fd\u969c\u7919\", \"\u53ef\u81f4\u6b7b\u6216\u622a\u80a2\"], \"first_aid\": {\"do\": [\"\u4fdd\u6301\u51b7\u975c\uff0c\u7acb\u5373\u9060\u96e2\", \"\u6e1b\u5c11\u6d3b\u52d5\uff0c\u52ff\u5954\u8dd1\", \"\u8a18\u4f4f\u86c7\u7684\u7279\u5fb5\uff08\u4e09\u89d2\u5f62\u982d\u3001\u4e0a\u7ff9\u543b\u7aef\uff09\", \"\u5118\u901f\u9001\u91ab\uff0c\u6b64\u70ba\u6025\u91cd\u75c7\", \"\u53ef\u7528\u5f48\u6027\u7e43\u5e36\u52a0\u58d3\u5305\u7d2e\"], \"do_not\": [\"\u52ff\u5207\u958b\u50b7\u53e3\", \"\u52ff\u7528\u5634\u5438\u6bd2\", \"\u52ff\u7d81\u6b62\u8840\u5e36\", \"\u52ff\u5617\u8a66\u6355\u86c7\uff08\u6b64\u86c7\u70ba\u4fdd\u80b2\u985e\uff09\"], \"antivenom\": \"\u51fa\u8840\u6027\u86c7\u6bd2\u8840\u6e05\uff08\u5927\u5291\u91cf\uff09\"}, \"activity_pattern\": \"\u591c\u884c\u6027\u70ba\u4e3b\uff0c\u767d\u5929\u5728\u843d\u8449\u4e2d\u975c\u81e5\", \"distribution\": {\"altitude_range\": [300, 1500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\"]}, \"encounter_frequency\": \"\u5c11\u898b\uff08\u4fdd\u80b2\u985e\uff09\uff0c\u4f46\u54ac\u50b7\u5f8c\u679c\u6975\u56b4\u91cd\"}, {\"id\": \"vespa_velutina\", \"scientific_name\": \"Vespa velutina\", \"common_names\": {\"zh-TW\": \"\u9ec3\u8170\u864e\u982d\u8702\", \"en\": \"Asian Hornet\"}, \"category\": \"venomous_insect\", \"danger_level\": \"high\", \"venom_type\": \"\u8702\u6bd2\uff08\u86cb\u767d\u8cea\u985e\u6bd2\u7d20\uff09\", \"morphology\": {\"body_color\": \"\u80f8\u90e8\u9ed1\u8272\uff0c\u8179\u90e8\u524d\u534a\u9ed1\u8272\u3001\u5f8c\u534a\u9ec3\u6a59\u8272\uff08\u95dc\u9375\u7279\u5fb5\uff1a\u8179\u90e8\u6709\u9ec3\u8272\u74b0\u5e36\uff09\uff0c\u6574\u9ad4\u9ed1\u9ec3\u76f8\u9593\", \"body_size\": \"\u5de5\u8702\u7d04 2-2.5cm\", \"head_shape\": \"\u9ed1\u8272\uff0c\u89f8\u89d2\u6df1\u8910\u8272\", \"wings\": \"\u6df1\u8910\u8272\", \"nest\": \"\u7403\u5f62\u7d19\u8cea\u5de2\uff0c\u5e38\u5728\u6a39\u4e0a\u3001\u4f4f\u5bb6\u5c4b\u7c37\uff1b\u5916\u89c0\u5448\u725b\u76ae\u7d19\u8272\u591a\u5c64\u7d19\u6bbc\"}, \"habitat\": \"\u4f4e\u4e2d\u6d77\u62d4\u68ee\u6797\u3001\u516c\u5712\u3001\u4f4f\u5bb6\u5c4b\u7c37\u7bc9\u5de2\", \"behavior\": \"\u7fa4\u9ad4\u653b\u64ca\uff0c\u6703\u8ffd\u9010\u5165\u4fb5\u8005\u6578\u5341\u516c\u5c3a\u3002\u653b\u64ca\u6027\u4e2d\u7b49\uff0c\u4f46\u5728\u5de2\u908a\u6216\u632f\u52d5\u5e72\u64fe\u6642\u6613\u6fc0\u6012\u3002\", \"sting_symptoms\": [\"\u5287\u70c8\u75bc\u75db\u3001\u7acb\u5373\u7d05\u816b\", \"\u591a\u8655\u87ab\u50b7\u53ef\u81f4\u5168\u8eab\u904e\u654f\u53cd\u61c9\", \"\u56b4\u91cd\u8005\u904e\u654f\u6027\u4f11\u514b\u3001\u547c\u5438\u56f0\u96e3\", \"\u6975\u5927\u91cf\u87ab\u50b7\u53ef\u5371\u53ca\u751f\u547d\"], \"first_aid\": {\"do\": [\"\u8fc5\u901f\u9060\u96e2\u73fe\u5834\uff0c\u4ee5\u8863\u7269\u8b77\u4f4f\u982d\u9838\", \"\u4ee5\u786c\u908a\u7de3\u522e\u9664\u87ab\u91dd\uff08\u52ff\u4ee5\u9477\u5b50\u64e0\u58d3\u6bd2\u56ca\uff09\", \"\u51b0\u6577\u6e1b\u8f15\u816b\u75db\", \"\u89c0\u5bdf\u662f\u5426\u51fa\u73fe\u5168\u8eab\u8541\u9ebb\u75b9\u3001\u80f8\u60b6\u3001\u547c\u5438\u56f0\u96e3\uff0f\u6688\u53a5\", \"\u591a\u8655\u87ab\u50b7\u3001\u547c\u5438\u56f0\u96e3\u6216\u610f\u8b58\u6539\u8b8a\uff1a\u7acb\u5373\u5c31\u91ab\u6216\u547c\u53eb\u6551\u8b77\"], \"do_not\": [\"\u52ff\u5f92\u624b\u64e0\u58d3\u87ab\u91dd\u6216\u6bd2\u56ca\", \"\u52ff\u5728\u8702\u7fa4\u672a\u96e2\u53bb\u6642\u505c\u7559\u6572\u6253\u5de2\u7a74\", \"\u904e\u654f\u9ad4\u8cea\u8005\u52ff\u8f15\u5ffd\u55ae\u4e00\u87ab\u50b7\u75c7\u72c0\"]}, \"activity_pattern\": \"\u65e5\u9593\u6d3b\u8e8d\uff0c\u6eab\u6696\u5b63\u7bc0\u6700\u983b\u7e41\", \"distribution\": {\"altitude_range\": [0, 1500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\"]}, \"encounter_frequency\": \"\u53f0\u5317\u5c71\u5340\u6700\u5e38\u9047\u5230\u7684\u864e\u982d\u8702\u7a2e\u985e\u4e4b\u4e00\"}, {\"id\": \"vespa_basalis\", \"scientific_name\": \"Vespa basalis\", \"common_names\": {\"zh-TW\": \"\u9ed1\u8179\u864e\u982d\u8702\", \"en\": \"Black-bellied Hornet\"}, \"category\": \"venomous_insect\", \"danger_level\": \"critical\", \"venom_type\": \"\u8702\u6bd2\uff08\u86cb\u767d\u8cea\u985e\u6bd2\u7d20\uff0c\u6bd2\u91cf\u8207\u653b\u64ca\u6027\u6975\u9ad8\uff09\", \"morphology\": {\"body_color\": \"\u5168\u8eab\u8fd1\u4e4e\u9ed1\u8272\uff0c\u8179\u90e8\u672b\u7aef\u5169\u7bc0\u70ba\u9ec3\u6a59\u8272\uff08\u95dc\u9375\u7279\u5fb5\uff1a\u5e7e\u4e4e\u5168\u9ed1\u7684\u5927\u578b\u864e\u982d\u8702\uff0c\u50c5\u672b\u7aef\u660e\u986f\u6a59\u9ec3\uff09\", \"body_size\": \"\u5de5\u8702\u7d04 2.5-3cm\uff0c\u8702\u5f8c\u53ef\u9054\u7d04 4cm\uff08\u53f0\u7063\u9ad4\u578b\u6700\u5927\u864e\u982d\u8702\u4e4b\u4e00\uff09\", \"head_shape\": \"\u5bec\u5927\u9ed1\u8272\u982d\u90e8\uff0c\u5927\u984e\u767c\u9054\", \"wings\": \"\u6df1\u8910\u8272\u8fd1\u9ed1\uff0c\u98db\u884c\u8072\u6c89\u539a\", \"nest\": \"\u5de2\u5e38\u7bc9\u65bc\u6a39\u6d1e\u3001\u5730\u7a74\u6216\u96b1\u853d\u8655\uff0c\u4e0d\u6613\u4e8b\u5148\u5bdf\u89ba\"}, \"habitat\": \"\u4e2d\u4f4e\u6d77\u62d4\u68ee\u6797\uff1b\u6a39\u6d1e\u3001\u5730\u4e0b\u6216\u96b1\u853d\u8655\u7bc9\u5de2\", \"behavior\": \"\u653b\u64ca\u6027\u6975\u5f37\uff0c\u53ef\u8ffd\u9010\u5165\u4fb5\u8005\u6578\u767e\u516c\u5c3a\uff1b\u7fa4\u9ad4\u653b\u64ca\u6642\u6975\u5ea6\u5371\u96aa\u3002\u88ab\u87ab\u8d85\u904e\u5341\u8655\u4ee5\u4e0a\u81f4\u6b7b\u98a8\u96aa\u986f\u8457\u4e0a\u5347\u3002\", \"sting_symptoms\": [\"\u77ac\u9593\u5287\u75db\u8207\u5feb\u901f\u816b\u8139\", \"\u5927\u91cf\u87ab\u50b7\uff1a\u6eb6\u8840\u3001\u6025\u6027\u814e\u640d\u50b7\u3001\u904e\u654f\u6027\u4f11\u514b\", \"\u547c\u5438\u56f0\u96e3\u3001\u8840\u58d3\u4e0b\u964d\", \"\u9ad8\u5291\u91cf\u6bd2\u6db2\u53ef\u81f4\u591a\u91cd\u5668\u5b98\u8870\u7aed\"], \"first_aid\": {\"do\": [\"\u7acb\u5373\u4ee5\u6700\u5feb\u901f\u5ea6\u96e2\u958b\u8a72\u5340\u57df\u4e26\u906e\u63a9\u982d\u90e8\", \"\u522e\u9664\u53ef\u898b\u87ab\u91dd\uff0c\u5927\u91cf\u87ab\u50b7\u8996\u70ba\u6025\u8a3a\", \"\u51b0\u6577\u4e26\u76e3\u6e2c\u610f\u8b58\u3001\u547c\u5438\u3001\u8840\u58d3\", \"\u4efb\u4f55\u5168\u8eab\u75c7\u72c0\u6216\u5341\u8655\u4ee5\u4e0a\u87ab\u50b7\uff1a\u7acb\u523b\u9001\u91ab\"], \"do_not\": [\"\u52ff\u8fd4\u56de\u5de2\u5340\u6216\u5617\u8a66\u6417\u6bc0\u8702\u5de2\", \"\u52ff\u4ee5\u9152\u7cbe\u6216\u5287\u70c8\u904b\u52d5\u52a0\u901f\u6bd2\u6db2\u5faa\u74b0\", \"\u52ff\u5ef6\u8aa4\u5c31\u91ab\u8a55\u4f30\uff08\u6b64\u7269\u7a2e\u81f4\u6b7b\u7387\u70ba\u53f0\u7063\u864e\u982d\u8702\u4e2d\u6700\u9ad8\u4e4b\u4e00\uff09\"]}, \"activity_pattern\": \"\u767d\u5929\u70ba\u4e3b\uff0c\u5b88\u5de2\u53cd\u61c9\u5f37\u70c8\", \"distribution\": {\"altitude_range\": [0, 2000], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\"]}, \"encounter_frequency\": \"\u5c71\u5340\u6b65\u9053\u9130\u8fd1\u6797\u7f18\u6642\u6709\u906d\u9047\u7d00\u9304\uff1b\u5de2\u4f4d\u96b1\u853d\u4f7f\u8aa4\u95d6\u98a8\u96aa\u9ad8\"}, {\"id\": \"vespa_mandarinia\", \"scientific_name\": \"Vespa mandarinia\", \"common_names\": {\"zh-TW\": \"\u4e2d\u83ef\u5927\u864e\u982d\u8702\", \"en\": \"Asian Giant Hornet\"}, \"category\": \"venomous_insect\", \"danger_level\": \"critical\", \"venom_type\": \"\u8702\u6bd2\uff08\u86cb\u767d\u8cea\u985e\u6bd2\u7d20\uff0c\u55ae\u6b21\u6ce8\u6bd2\u91cf\u6975\u5927\uff09\", \"morphology\": {\"body_color\": \"\u982d\u90e8\u6a59\u9ec3\u8272\uff0c\u80f8\u90e8\u9ed1\u8272\uff0c\u8179\u90e8\u9ed1\u9ec3\u76f8\u9593\u5bec\u6a6b\u5e36\uff08\u5c0d\u6bd4\u5f37\u70c8\u3001\u9ad4\u578b\u5de8\u5927\u6613\u8fa8\uff09\", \"body_size\": \"\u5de5\u8702\u7d04 3.5-4cm\uff0c\u8702\u5f8c\u53ef\u9054\u7d04 5cm\uff08\u4e16\u754c\u6700\u5927\u578b\u8702\u985e\u4e4b\u4e00\uff09\", \"head_shape\": \"\u5bec\u5927\u9ec3\u6a59\u8272\u982d\u90e8\uff0c\u8907\u773c\u660e\u986f\", \"wings\": \"\u5f37\u97cc\u3001\u5bec\u5927\uff0c\u6df1\u8910\u8272\", \"nest\": \"\u591a\u65bc\u571f\u4e2d\u3001\u6797\u7de3\u5730\u4e0b\u7bc9\u5de2\uff0c\u5165\u53e3\u5e38\u70ba\u55ae\u4e00\u6d1e\u5b54\"}, \"habitat\": \"\u4e2d\u6d77\u62d4\u68ee\u6797\u3001\u6797\u7f18\uff1b\u571f\u4e2d\u7bc9\u5de2\u70ba\u4e3b\", \"behavior\": \"\u653b\u64ca\u6027\u5f37\uff0c\u7fa4\u9ad4\u9632\u885b\u6642\u53ef\u6301\u4e45\u8ffd\u64ca\uff1b\u55ae\u8702\u5373\u53ef\u9020\u6210\u56b4\u91cd\u5c40\u90e8\u53cd\u61c9\uff0c\u6bd2\u6db2\u7e3d\u91cf\u5927\u3002\u5728\u81fa\u5317\u8fd1\u90ca\u5c71\u5340\u76f8\u5c0d\u5c11\u898b\u4f46\u4e0d\u53ef\u8f15\u5ffd\u3002\", \"sting_symptoms\": [\"\u6975\u5ea6\u75bc\u75db\u8207\u5927\u7bc4\u570d\u816b\u8139\", \"\u591a\u8655\u87ab\u50b7\uff1a\u4f11\u514b\u8207\u5668\u5b98\u53d7\u640d\u98a8\u96aa\", \"\u904e\u654f\u53cd\u61c9\u8207\u795e\u7d93\u8840\u7ba1\u6027\u6c34\u816b\"], \"first_aid\": {\"do\": [\"\u8fc5\u901f\u64a4\u96e2\u4e26\u9632\u8b77\u982d\u9838\", \"\u522e\u9664\u87ab\u91dd\uff0c\u51b0\u6577\u4e26\u76f4\u5954\u6025\u8a3a\uff08\u7279\u5225\u662f\u591a\u8655\u87ab\u50b7\uff09\", \"\u8a18\u9304\u87ab\u50b7\u6642\u9593\u8207\u90e8\u4f4d\u6578\u91cf\u4f9b\u91ab\u5e2b\u8a55\u4f30\"], \"do_not\": [\"\u52ff\u62cd\u6253\u6216\u5f92\u624b\u6293\u8702\uff08\u6613\u5f15\u767c\u66f4\u591a\u653b\u64ca\uff09\", \"\u52ff\u8f15\u5ffd\u55ae\u4e00\u6df1\u90e8\u87ab\u50b7\u7684\u75bc\u75db\u8207\u816b\u8139\u9032\u5c55\"]}, \"activity_pattern\": \"\u65e5\u9593\u6d3b\u52d5\uff0c\u79cb\u5b63\u65cf\u7fa4\u91cf\u5927\u6642\u9632\u885b\u6027\u63d0\u9ad8\", \"distribution\": {\"altitude_range\": [300, 2000], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\"]}, \"encounter_frequency\": \"\u4e16\u754c\u6700\u5927\u864e\u982d\u8702\uff0c\u81fa\u5317\u5c71\u5340\u8f03\u5c11\u898b\u4f46\u82e5\u9047\u898b\u98a8\u96aa\u6975\u9ad8\"}, {\"id\": \"scolopendra_subspinipes_mutilans\", \"scientific_name\": \"Scolopendra subspinipes mutilans\", \"common_names\": {\"zh-TW\": \"\u8708\u86a3\", \"en\": \"Chinese Red-headed Centipede\"}, \"category\": \"venomous_arthropod\", \"danger_level\": \"medium\", \"venom_type\": \"\u8708\u86a3\u6bd2\uff08\u86cb\u767d\u8cea\u985e\uff09\", \"morphology\": {\"body_color\": \"\u7d05\u8910\u8272\u81f3\u6697\u8910\u8272\u9ad4\u7bc0\uff0c\u8179\u9762\u5e38\u5448\u9ec3\u8272\uff1b\u6574\u9ad4\u5149\u6fa4\u504f\u6697\", \"body_size\": \"\u9ad4\u9577\u7d04 8-20cm\uff08\u5927\u578b\u500b\u9ad4\u9192\u76ee\uff09\", \"head_appendages\": \"\u982d\u90e8\u4e0b\u65b9\u4e00\u5c0d\u7c97\u58ef\u6bd2\u984e\uff08\u95dc\u9375\u7279\u5fb5\uff09\uff0c\u984e\u7aef\u6df1\u8272\", \"legs\": \"21 \u5c0d\u6b65\u8db3\uff0c\u5074\u7de3\u6392\u5e03\u7dca\u5bc6\uff0c\u79fb\u52d5\u8fc5\u901f\", \"segmentation\": \"\u9ad4\u7bc0\u6241\u5bec\uff0c\u80cc\u9762\u4e2d\u592e\u7e31\u7dda\u53ef\u80fd\u8f03\u6dfa\"}, \"habitat\": \"\u4f4e\u6d77\u62d4\u6f6e\u6fd5\u74b0\u5883\u3001\u77f3\u584a\u4e0b\u3001\u843d\u8449\u5806\u3001\u6b65\u9053\u65c1\u7e2b\u9699\u8207\u4f4f\u5bb6\u5468\u908a\u9670\u6fd5\u8655\", \"behavior\": \"\u591c\u884c\u6027\uff0c\u53d7\u9a5a\u6642\u5feb\u901f\u9003\u907f\u6216\u4ee5\u6bd2\u984e\u53cd\u54ac\uff1b\u7ffb\u77f3\u3001\u6574\u7406\u96dc\u7269\u6642\u6613\u88ab\u8aa4\u50b7\u3002\", \"bite_symptoms\": [\"\u54ac\u8655\u5287\u70c8\u523a\u75db\u8207\u707c\u71b1\u611f\", \"\u5c40\u90e8\u7d05\u816b\u3001\u53ef\u80fd\u7600\u9752\", \"\u767c\u71d2\u611f\u3001\u9130\u8fd1\u6dcb\u5df4\u7d50\u816b\u75db\", \"\u5c11\u6578\u500b\u9ad4\u904e\u654f\u6216\u5927\u7247\u816b\u8139\u9700\u5c31\u91ab\"], \"first_aid\": {\"do\": [\"\u6e05\u6c34\u6216\u751f\u7406\u98df\u9e7d\u6c34\u6c96\u6d17\u50b7\u53e3\", \"\u51b0\u6577\u6b62\u75db\u6d88\u816b\", \"\u53ef\u670d\u7528\u91ab\u5e2b\u6307\u793a\u4e4b\u6b62\u75db\u85e5\", \"\u7d05\u816b\u8fc5\u901f\u64f4\u6563\u3001\u767c\u71d2\u6216\u5168\u8eab\u904e\u654f\uff1a\u5c31\u91ab\"], \"do_not\": [\"\u52ff\u4ee5\u53e3\u5438\u542e\u50b7\u53e3\", \"\u52ff\u5207\u958b\u6216\u706b\u71d2\u50b7\u53e3\", \"\u52ff\u5ef6\u8aa4\u51fa\u73fe\u5168\u8eab\u75c7\u72c0\u6642\u7684\u8a55\u4f30\"]}, \"activity_pattern\": \"\u591c\u9593\u8207\u6668\u660f\u70ba\u4e3b\uff0c\u65e5\u9593\u533f\u65bc\u7e2b\u9699\", \"distribution\": {\"altitude_range\": [0, 1500], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\"]}, \"encounter_frequency\": \"\u81fa\u7063\u5e38\u898b\u5927\u578b\u8708\u86a3\uff0c\u90ca\u5c71\u8207\u516c\u5712\u65c1\u6f6e\u6fd5\u8655\u5e38\u6709\u8a18\u9304\"}, {\"id\": \"solenopsis_invicta\", \"scientific_name\": \"Solenopsis invicta\", \"common_names\": {\"zh-TW\": \"\u7d05\u706b\u87fb\", \"en\": \"Red Imported Fire Ant\"}, \"category\": \"venomous_insect\", \"danger_level\": \"high\", \"venom_type\": \"\u87fb\u6bd2\uff08\u542b\u6eb6\u8840\u8207\u904e\u654f\u539f\u6027\u86cb\u767d\uff09\", \"morphology\": {\"body_color\": \"\u5de5\u87fb\u5448\u7d05\u8910\u8272\u81f3\u6df1\u8910\u8272\uff0c\u8179\u7aef\u8272\u5e38\u8f03\u6df1\", \"body_size\": \"\u5de5\u87fb\u9ad4\u9577\u7d04 2-6mm\uff0c\u8089\u773c\u4ecd\u6e05\u6670\u53ef\u898b\", \"nest_mound\": \"\u5730\u8868\u9686\u8d77\u87fb\u4e18\u9ad8\u7d04 10-30cm \u4e4b\u9b06\u571f\u5806\uff08\u95dc\u9375\u91ce\u5916\u8fa8\u8b58\u7279\u5fb5\uff09\uff0c\u4e0b\u96e8\u5f8c\u66f4\u660e\u986f\", \"caste\": \"\u5177\u591a\u578b\u614b\u5de5\u87fb\uff0c\u5927\u984e\u53ef\u593e\u4f4f\u76ae\u819a\u5f8c\u4ee5\u91dd\u523a\u87ab\", \"antenna\": \"\u819d\u72c0\u89f8\u89d2\uff0c\u982d\u80f8\u8179\u4e09\u6bb5\u660e\u986f\"}, \"habitat\": \"\u81fa\u5317\u90fd\u6703\u516c\u5712\u8349\u5730\u3001\u958b\u95ca\u5730\u3001\u6b65\u9053\u908a\u5761\u8207\u8fb2\u7530\u9130\u8fd1\u8352\u5730\", \"behavior\": \"\u7fa4\u9ad4\u9632\u885b\u6027\u5f37\uff0c\u53ef\u540c\u6642\u5927\u91cf\u87ab\u54ac\uff1b\u8aa4\u8e0f\u87fb\u4e18\u6216\u632f\u52d5\u5de2\u9ad4\u6642\u8702\u64c1\u800c\u4e0a\u3002\", \"sting_symptoms\": [\"\u707c\u71b1\u523a\u75db\u8207\u5287\u7662\", \"24 \u5c0f\u6642\u5167\u5f62\u6210\u6e05\u6f88\u6c34\u6ce1\uff0c\u53ef\u80fd\u5316\u81bf\u6210\u81bf\u5305\", \"\u56b4\u91cd\u904e\u654f\uff1a\u5168\u8eab\u8541\u9ebb\u75b9\u3001\u54bd\u5589\u816b\u3001\u4f4e\u8840\u58d3\u3001\u904e\u654f\u6027\u4f11\u514b\", \"\u5152\u7ae5\u6216\u904e\u654f\u9ad4\u8cea\u8005\u906d\u591a\u8655\u87ab\u54ac\u6642\u98a8\u96aa\u66f4\u9ad8\"], \"first_aid\": {\"do\": [\"\u64a5\u96e2\u8eab\u4e0a\u87fb\u96bb\u3001\u9060\u96e2\u87fb\u4e18\", \"\u51b7\u6577\u6e1b\u75db\u6b62\u7662\", \"\u6e05\u6f54\u76ae\u819a\uff0c\u89c0\u5bdf\u904e\u654f\u8207\u547c\u5438\u9053\u75c7\u72c0\", \"\u51fa\u73fe\u80f8\u60b6\u3001\u5507\u816b\u3001\u6688\u53a5\uff1a\u7acb\u5373\u5c31\u91ab\"], \"do_not\": [\"\u52ff\u64e0\u7834\u521d\u671f\u6c34\u6ce1\u4ee5\u514d\u6b21\u767c\u611f\u67d3\", \"\u52ff\u4ee5\u6307\u7532\u6414\u6293\u7834\u76ae\u5927\u9762\u7a4d\u50b7\u53e3\", \"\u52ff\u8f15\u5ffd\u9996\u6b21\u5927\u9762\u7a4d\u87ab\u50b7\"]}, \"activity_pattern\": \"\u5168\u65e5\u7686\u53ef\u6d3b\u8e8d\uff0c\u6eab\u6696\u5b63\u7bc0\u6700\u76db\", \"distribution\": {\"altitude_range\": [0, 800], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\"]}, \"encounter_frequency\": \"\u5165\u4fb5\u7a2e\uff0c\u81fa\u5317\u516c\u5712\u8207\u8349\u576a\u5df2\u6709\u591a\u8d77\u5206\u5e03\u8207\u87ab\u50b7\u901a\u5831\"}, {\"id\": \"leptotrombidium_deliense\", \"scientific_name\": \"Leptotrombidium deliense\", \"common_names\": {\"zh-TW\": \"\u6059\u87f2\uff08\u6059\u87ce\u5e7c\u87f2\uff09\", \"en\": \"Chigger Mite (scrub typhus vector)\"}, \"category\": \"disease_vector\", \"danger_level\": \"high\", \"venom_type\": \"\u4e0d\u9069\u7528\uff08\u75c5\u539f\u70ba\u6059\u87f2\u75c5\u7acb\u514b\u6b21\u9ad4\uff0c\u7d93\u5e7c\u87f2\u553e\u6db2\u50b3\u64ad\uff09\", \"morphology\": {\"body_color\": \"\u5e7c\u87f2\u6a59\u7d05\u8272\uff0c\u5438\u8840\u524d\u5f8c\u984f\u8272\u53ef\u7565\u8b8a\u6df1\", \"body_size\": \"\u5e7c\u87f2\u9ad4\u9577\u50c5\u7d04 0.2-0.3mm\uff08\u8089\u773c\u5e7e\u4e4e\u4e0d\u53ef\u8fa8\uff0c\u591a\u9760\u7126\u75c2\u8207\u75c5\u53f2\u63a8\u65b7\uff09\", \"feeding_site\": \"\u504f\u597d\u76ae\u819a\u67d4\u8edf\u6eab\u6696\u8655\uff1a\u814b\u4e0b\u3001\u8170\u969b\u3001\u9f20\u8e4a\u3001\u9670\u56ca\u3001\u9aee\u969b\u8fd1\u8033\u5f8c\u7b49\", \"seasonality\": \"\u6eab\u6696\u6f6e\u6fd5\u5b63\u7bc0\u6d3b\u52d5\u5347\u9ad8\uff0c\u8349\u53e2\u66b4\u9732\u98a8\u96aa\u6700\u5927\", \"key_sign\": \"\u53ee\u54ac\u8655\u53ef\u80fd\u7559\u4e0b\u9ed1\u8272\u7126\u75c2\uff08eschar\uff0c\u5713\u5f62\u6f70\u760d\u6a23\uff0c\u5177\u9ad8\u5ea6\u8a3a\u65b7\u50f9\u503c\uff09\"}, \"habitat\": \"\u8349\u53e2\u3001\u704c\u6728\u53e2\u3001\u96dc\u8349\u5730\u3001\u90ca\u5c71\u6b65\u9053\u5169\u5074\u9577\u8349\u5340\", \"behavior\": \"\u5e7c\u87f2\u722c\u81f3\u5bbf\u4e3b\u76ae\u819a\u5f8c\u4ee5\u53e3\u5668\u56fa\u8457\u5438\u8840\u6578\u5c0f\u6642\u81f3\u6578\u65e5\uff0c\u591a\u534a\u4e0d\u81ea\u89ba\u5b8c\u6210\u53ee\u54ac\u96e2\u9ad4\u3002\", \"bite_symptoms\": [\"\u53ee\u54ac\u5f8c\u53ef\u80fd\u7121\u611f\u6216\u5fae\u7662\", \"\u7126\u75c2\u5f62\u6210\uff08\u9ed1\u8272\u5713\u5f62\u6f70\u760d\uff0c\u95dc\u9375\u7279\u5fb5\uff09\", \"\u6f5b\u4f0f\u7d04 7-10 \u65e5\u5f8c\uff1a\u7a81\u7136\u9ad8\u71d2\u3001\u5287\u70c8\u982d\u75db\u3001\u7d50\u819c\u5145\u8840\", \"\u6dcb\u5df4\u7d50\u816b\u5927\u3001\u8179\u75db\u6216\u76ae\u75b9\uff1b\u672a\u6cbb\u7642\u53ef\u81f4\u591a\u91cd\u5668\u5b98\u8870\u7aed\uff0c\u6b77\u53f2\u8cc7\u6599\u986f\u793a\u6b7b\u4ea1\u7387\u53ef\u6975\u9ad8\"], \"first_aid\": {\"do\": [\"\u6236\u5916\u6d3b\u52d5\u5f8c\u76e1\u5feb\u6c96\u6fa1\u4e26\u66f4\u63db\u8863\u7269\uff0c\u71b1\u6c34\u6e05\u6d17\u6216\u9ad8\u6eab\u70d8\u4e7e\", \"\u82e5\u51fa\u73fe\u7126\u75c2\u4e26\u5408\u4f75\u767c\u71d2\uff1a\u8996\u70ba\u6025\u75c7\uff0c\u7acb\u5373\u5c31\u91ab\u4e26\u4e3b\u52d5\u544a\u77e5\u767b\u5c71\uff0f\u8349\u53e2\u66b4\u9732\u53f2\", \"\u9700\u6297\u751f\u7d20\u6cbb\u7642\uff08\u91ab\u5e2b\u8655\u65b9\uff0c\u52ff\u81ea\u884c\u505c\u85e5\uff09\uff1b\u53ca\u65e9\u6cbb\u7642\u53ef\u5927\u5e45\u964d\u4f4e\u6b7b\u4ea1\u7387\", \"\u540c\u884c\u8005\u4ea6\u61c9\u63d0\u9ad8\u8b66\u89ba\uff08\u7fa4\u805a\u66b4\u9732\u98a8\u96aa\uff09\"], \"do_not\": [\"\u52ff\u56e0\u7121\u75db\u7662\u800c\u5ffd\u8996\u7126\u75c2\uff0b\u767c\u71d2\u7684\u7d44\u5408\", \"\u52ff\u81ea\u884c\u9577\u671f\u670d\u7528\u9000\u71d2\u85e5\u63a9\u84cb\u75c5\u60c5\u800c\u5ef6\u8aa4\u8a3a\u65b7\", \"\u52ff\u4ee5\u6c11\u4fd7\u7642\u6cd5\u8655\u7406\u7126\u75c2\u9020\u6210\u6b21\u767c\u611f\u67d3\"], \"medical_urgency\": \"\u6059\u87f2\u75c5\u53ef\u9032\u5c55\u8fc5\u901f\uff1b\u672a\u9069\u7576\u6cbb\u7642\u6b7b\u4ea1\u7387\u53ef\u9054\u7d04\u516d\u6210\uff08\u6587\u737b\u8207\u516c\u5171\u885b\u6559\u5e38\u5f15\u8ff0\u4e4b\u56b4\u91cd\u5ea6\uff09\uff0c\u6709\u7591\u4f3c\u75c7\u72c0\u9808\u6578\u5c0f\u6642\u81f3\u7576\u65e5\u5167\u5b8c\u6210\u91ab\u7642\u8a55\u4f30\u3002\"}, \"activity_pattern\": \"\u5b63\u7bc0\u6027\u9ad8\u5cf0\u591a\u5728\u96e8\u5f8c\u8349\u9577\u3001\u6c23\u6eab\u56de\u5347\u6642\u6bb5\", \"distribution\": {\"altitude_range\": [0, 2000], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\"]}, \"encounter_frequency\": \"\u81fa\u7063\u90ca\u5c71\u8207\u7af9\u6797\u8349\u5761\u5e38\u898b\u66b4\u9732\u98a8\u96aa\uff0c\u81fa\u5317\u8fd1\u90ca\u6b65\u9053\u4ea6\u6709\u75c5\u4f8b\u5831\u544a\"}, {\"id\": \"ixodes_granulatus\", \"scientific_name\": \"Ixodes granulatus\", \"common_names\": {\"zh-TW\": \"\u786c\u8731\", \"en\": \"Granular Tick\"}, \"category\": \"disease_vector\", \"danger_level\": \"medium\", \"venom_type\": \"\u4e0d\u9069\u7528\uff08\u53ef\u80fd\u50b3\u64ad\u4f2f\u6c0f\u758f\u87ba\u65cb\u9ad4\u7b49\u75c5\u539f\uff0c\u8207\u840a\u59c6\u75c5\u7b49\u76f8\u95dc\uff1b\u5be6\u969b\u98a8\u96aa\u4f9d\u5730\u57df\u8207\u500b\u6848\uff09\", \"morphology\": {\"body_color\": \"\u672a\u5438\u8840\u6642\u6df1\u8910\u8272\u81f3\u9ed1\u8272\uff0c\u9ad4\u80cc\u53ef\u898b\u9846\u7c92\u72c0\u523b\u9ede\uff08\u7a2e\u5c0f\u540d granulatus \u53cd\u6620\u4e4b\u7279\u5fb5\uff09\", \"body_size\": \"\u672a\u5438\u8840\u82e5\u87f2\uff0f\u6210\u8731\u7d04 2-3mm\uff1b\u98fd\u8840\u5f8c\u96cc\u8731\u81a8\u5927\u5448\u7070\u81f3\u9752\u7070\u8272\uff0c\u53ef\u9054\u7d04 1cm\", \"capitulum\": \"\u982d\u90e8\u53e3\u5668\u986f\u8457\u5411\u524d\u4f38\u51fa\uff0c\u87af\u80a2\u57cb\u5165\u76ae\u819a\u5f8c\u4e0d\u6613\u62d4\u51fa\", \"legs\": \"\u5e7c\u8731\u4e09\u5c0d\u3001\u82e5\u87f2\u8207\u6210\u8731\u56db\u5c0d\uff0c\u8db3\u672b\u5177\u9644\u8457\u722a\", \"engorgement\": \"\u5438\u8840\u6578\u65e5\u81f3\u4e00\u9031\u4ee5\u4e0a\u7686\u53ef\u80fd\uff0c\u8d8a\u5927\u8868\u793a\u5438\u8840\u8d8a\u4e45\"}, \"habitat\": \"\u95ca\u8449\u6797\u3001\u8349\u53e2\u8207\u704c\u6728\u908a\u7de3\uff0c\u6b65\u9053\u5169\u5074\u53ca\u7378\u5f91\u65c1\", \"behavior\": \"\u5728\u8349\u5c16\u6216\u704c\u6728\u8449\u7aef\u300c\u5750\u7b49\u300d\u7d93\u904e\u7684\u5bbf\u4e3b\uff0c\u63a5\u89f8\u5f8c\u722c\u884c\u81f3\u76ba\u8936\u3001\u982d\u76ae\u3001\u819d\u5f8c\u3001\u8170\u969b\u7b49\u8655\u53ee\u54ac\u3002\", \"bite_symptoms\": [\"\u53ee\u54ac\u8655\u53ef\u80fd\u7d05\u7662\u6216\u7121\u660e\u986f\u611f\u89ba\", \"\u840a\u59c6\u75c5\u76f8\u95dc\uff1a\u611f\u67d3\u5f8c\u6578\u65e5\uff0f\u9031\u53ef\u51fa\u73fe\u904a\u8d70\u6027\u7d05\u6591\uff08\u74b0\u72c0\u64f4\u5f35\u7d05\u6591\uff0c\u5177\u53c3\u8003\u50f9\u503c\u4f46\u975e\u6bcf\u4eba\u7686\u6709\uff09\", \"\u767c\u71d2\u3001\u95dc\u7bc0\u75db\u3001\u795e\u7d93\u5b78\u75c7\u72c0\u5c6c\u5f8c\u671f\u53ef\u80fd\u8868\u73fe\uff0c\u9700\u91ab\u5e2b\u5224\u8b80\", \"\u8731\u505c\u7559\u6642\u9593\u8d8a\u4e45\uff0c\u90e8\u5206\u75c5\u539f\u50b3\u64ad\u98a8\u96aa\u53ef\u80fd\u4e0a\u5347\"], \"first_aid\": {\"do\": [\"\u4ee5\u5c16\u982d\u9477\u5b50\u7dca\u8cbc\u76ae\u819a\u593e\u4f4f\u8731\u4e4b\u53e3\u5668\u57fa\u8fd1\u76ae\u819a\u8655\", \"\u7a69\u5b9a\u5782\u76f4\u5411\u4e0a\u62d4\u51fa\uff0c\u907f\u514d\u5de6\u53f3\u6416\u6643\u64e0\u58d3\u87f2\u9ad4\", \"\u4ee5\u9152\u7cbe\u6216\u7898\u6db2\u6d88\u6bd2\u50b7\u53e3\uff0c\u4fdd\u5b58\u8731\u87f2\u65bc\u5bb9\u5668\u4f9b\u5fc5\u8981\u6642\u9451\u5b9a\", \"\u62d4\u9664\u5f8c\u6578\u9031\u89c0\u5bdf\u7d05\u6591\u8207\u6d41\u611f\u6a23\u75c7\u72c0\uff0c\u82e5\u6709\u904a\u8d70\u6027\u7d05\u6591\u6216\u767c\u71d2\uff1a\u5c31\u91ab\u4e26\u544a\u77e5\u8731\u54ac\u53f2\"], \"do_not\": [\"\u52ff\u5857\u6307\u7532\u6cb9\uff0f\u9ede\u706b\uff0f\u7c97\u9b6f\u626f\u65b7\u9020\u6210\u53e3\u5668\u6b98\u7559\", \"\u52ff\u5f92\u624b\u64e0\u58d3\u98fd\u8840\u8731\u7684\u8179\u90e8\uff08\u53ef\u80fd\u589e\u52a0\u75c5\u539f\u64e0\u5165\u98a8\u96aa\uff09\", \"\u52ff\u5ffd\u8996\u6301\u7e8c\u64f4\u5927\u7684\u74b0\u72c0\u7d05\u6591\"], \"medical_urgency\": \"\u591a\u6578\u8731\u54ac\u50c5\u9700\u6b63\u78ba\u79fb\u9664\u8207\u89c0\u5bdf\uff1b\u51fa\u73fe\u5168\u8eab\u75c7\u72c0\u6216\u5178\u578b\u76ae\u819a\u7d05\u6591\u61c9\u76e1\u901f\u7531\u91ab\u5e2b\u8a55\u4f30\u6297\u751f\u7d20\u8207\u8ffd\u8e64\u3002\"}, \"activity_pattern\": \"\u6eab\u5b63\u6d3b\u52d5\u70ba\u4e3b\uff0c\u96e8\u5f8c\u8349\u9577\u6642\u4eba\u8731\u63a5\u89f8\u589e\u52a0\", \"distribution\": {\"altitude_range\": [0, 2000], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\"]}, \"encounter_frequency\": \"\u81fa\u7063\u90ca\u5c71\u8207\u68ee\u6797\u6b65\u9053\u6709\u7d00\u9304\uff0c\u767b\u5c71\u8932\u7ba1\u7d2e\u7dca\u8207\u9a45\u907f\u5291\u53ef\u964d\u4f4e\u9644\u8457\"}, {\"id\": \"haemadipsa_zeylanica\", \"scientific_name\": \"Haemadipsa zeylanica\", \"common_names\": {\"zh-TW\": \"\u6c34\u86ed\", \"en\": \"Tiger Leech\"}, \"category\": \"parasite\", \"danger_level\": \"low\", \"venom_type\": \"\u6297\u51dd\u8840\u8207\u6297\u8840\u5c0f\u677f\u5206\u6ccc\u7269\uff08\u975e\u5178\u578b\u300c\u6bd2\u6db2\u300d\uff0c\u81f4\u6301\u7e8c\u6ef2\u8840\uff09\", \"morphology\": {\"body_color\": \"\u80cc\u9762\u6697\u8910\u81f3\u9ed1\u8272\uff0c\u9ad4\u5074\u53ef\u898b\u9ec3\u8272\u7e31\u5e36\u6216\u6591\u584a\uff08\u500b\u9ad4\u8b8a\u7570\u5b58\u5728\uff09\", \"body_size\": \"\u5145\u5206\u4f38\u5c55\u6642\u9ad4\u9577\u7d04 2-5cm\uff0c\u5438\u8840\u5f8c\u53ef\u660e\u986f\u81a8\u5927\u6210\u5713\u7b52\u72c0\", \"anterior_posterior\": \"\u524d\u5f8c\u5404\u6709\u4e00\u5438\u76e4\uff0c\u524d\u5438\u76e4\u5177\u4e09\u984e\u7247\u53ef\u5207\u958b\u76ae\u819a\", \"texture\": \"\u9ad4\u8868\u5e38\u5177\u660e\u986f\u74b0\u7d0b\uff0c\u722c\u884c\u6642\u53ef\u4f5c\u5c3a\u8816\u5f0f\u4f38\u5c55\", \"habitat_clue\": \"\u96e8\u5f8c\u6b65\u9053\u3001\u6eaa\u6e9d\u65c1\u77f3\u677f\u8207\u6fd5\u82d4\u861a\u4e0a\u5e38\u898b\u722c\u884c\u75d5\u8de1\"}, \"habitat\": \"\u6f6e\u6fd5\u95ca\u8449\u6797\u3001\u6eaa\u6f97\u65c1\u3001\u96e8\u5f8c\u6797\u9053\u8207\u82d4\u861a\u77f3\u9762\", \"behavior\": \"\u611f\u77e5\u6eab\u8840\u52d5\u7269\u632f\u52d5\u8207\u5316\u5b78\u7dda\u7d22\u5f8c\u4e3b\u52d5\u722c\u884c\u9644\u8457\uff1b\u5438\u9644\u5f8c\u5206\u6ccc\u6297\u51dd\u8840\u7269\u8cea\u6301\u7e8c\u5438\u8840\u81f3\u9971\u6ee1\u812b\u843d\u3002\", \"bite_symptoms\": [\"\u5438\u9644\u7576\u4e0b\u53ef\u80fd\u7121\u75db\u6216\u8f15\u75d2\", \"\u812b\u9644\u5f8c\u50b7\u53e3\u6613\u6301\u7e8c\u6e17\u8840 30 \u5206\u9418\u81f3\u6578\u5c0f\u6642\", \"\u5c40\u90e8\u8f15\u5fae\u7d05\u816b\u6216\u904e\u654f\u6027\u4e18\u75b9\u5c11\u898b\u4f46\u53ef\u80fd\", \"\u4e8c\u6b21\u611f\u67d3\u591a\u8207\u4e0d\u6f54\u8655\u7406\u6709\u95dc\"], \"first_aid\": {\"do\": [\"\u5728\u9644\u7740\u9ede\u6492\u9e7d\u6216\u4ee5\u5b89\u5168\u65b9\u5f0f\u6ef4\u5c11\u91cf\u9152\u7cbe\u4fc3\u4f7f\u9b06\u53e3\", \"\u4ea6\u53ef\u4f7f\u7528\u7d93\u77ed\u66ab\u52a0\u71b1\u7684\u91d1\u5c6c\u5319\u80cc\u9760\u8fd1\u5176\u9ad4\u5074\uff08\u9632\u71d9\u50b7\uff09\u4fc3\u5176\u9b06\u5438\uff0c\u5207\u52ff\u786c\u626f\", \"\u812b\u843d\u540e\u52a0\u58d3\u5305\u624e\u81f3\u8840\u6b62\uff0c\u518d\u4ee5\u6e05\u6c34\u6216\u6d88\u6bd2\u6db2\u6e05\u6f54\", \"\u82e5\u8840\u6d41\u8d85\u904e\u9810\u671f\u6216\u6577\u6599\u53cd\u8986\u6fd5\u900f\uff1a\u5c31\u91ab\u8655\u7406\"], \"do_not\": [\"\u52ff\u5f92\u624b\u66b4\u529b\u62c9\u626f\u9020\u6210\u53e3\u5668\u6b98\u7559\u6216\u6495\u88c2\u50b7\", \"\u52ff\u4ee5\u660e\u706b\u76f4\u63a5\u71d2\u707c\u76ae\u819a\", \"\u52ff\u5ffd\u7565\u7cd6\u5c3f\u75c5\u6216\u51dd\u8840\u7570\u5e38\u8005\u4e4b\u6301\u7e8c\u51fa\u8840\"]}, \"activity_pattern\": \"\u6fd5\u5b63\u8207\u964d\u96e8\u5f8c\u6700\u6d3b\u8e8d\uff0c\u6668\u9593\u8349\u9732\u672a\u4e7e\u6642\u4ea6\u5e38\u898b\", \"distribution\": {\"altitude_range\": [0, 2000], \"climate_zones\": [\"\u4e9e\u71b1\u5e36\"]}, \"encounter_frequency\": \"\u81fa\u5317\u8fd1\u90ca\u6eaa\u6d41\u90ca\u5c71\u6b65\u9053\u96e8\u5f8c\u6975\u5e38\u898b\uff0c\u70ba\u4f4e\u6d77\u62d4\u90ca\u5c71\u5065\u884c\u300c\u5e38\u5ba2\u300d\"}]")

_KB_CONFUSION_PAIRS = json.loads("[{\"id\": \"taro_vs_alocasia\", \"safe_species\": {\"id\": \"colocasia_esculenta\", \"scientific_name\": \"Colocasia esculenta\", \"common_name_zh\": \"\u828b\u982d\", \"common_name_en\": \"Taro\", \"key_features\": {\"leaf_surface\": \"\u6709\u5929\u9d5d\u7d68\u822c\u7684\u5fae\u7d68\u6bdb\u8cea\u611f\uff0c\u4e0d\u592a\u5149\u6ed1\", \"water_droplet_test\": \"\u6c34\u73e0\u5728\u8449\u9762\u5448\u5713\u73e0\u72c0\u6efe\u52d5\uff08\u8377\u8449\u6548\u61c9\uff09\", \"leaf_tip\": \"\u901a\u5e38\u671d\u4e0b\u5782\", \"petiole_attachment\": \"\u63a5\u5728\u8449\u7247\u908a\u7de3\u5167\u5074\uff08\u76fe\u72c0\u8457\u751f\uff09\", \"petiole_color\": \"\u7da0\u8272\uff0c\u8f03\u5c11\u7d2b\u6688\", \"leaf_size\": \"\u8f03\u5c0f\uff0c\u8449\u9577 20-50cm\", \"underground\": \"\u6709\u5713\u5f62\u584a\u8396\uff0c\u53ef\u98df\u7528\"}}, \"dangerous_species\": {\"id\": \"alocasia_odora\", \"scientific_name\": \"Alocasia odora\", \"common_name_zh\": \"\u59d1\u5a46\u828b\", \"common_name_en\": \"Giant Elephant Ear\", \"danger_level\": \"high\", \"key_features\": {\"leaf_surface\": \"\u5149\u6ed1\u767c\u4eae\uff0c\u8cea\u5730\u50cf\u6253\u4e86\u881f\", \"water_droplet_test\": \"\u6c34\u73e0\u6524\u5e73\u4e0d\u6210\u73e0\u72c0\uff08\u4e0d\u5177\u8377\u8449\u6548\u61c9\uff09\", \"leaf_tip\": \"\u671d\u4e0a\u6216\u6c34\u5e73\u6307\u5411\", \"petiole_attachment\": \"\u63a5\u5728\u8449\u7247\u908a\u7de3\uff08\u975e\u76fe\u72c0\uff09\", \"petiole_color\": \"\u7da0\u8272\u5e38\u5e36\u7d2b\u6688\", \"leaf_size\": \"\u8f03\u5927\uff0c\u8449\u9577 50-90cm\", \"underground\": \"\u6839\u8396\u7c97\u77ed\uff0c\u6709\u6bd2\"}}, \"lethality\": \"high\", \"comparison_table\": [{\"feature\": \"\u8449\u9762\u8cea\u611f\", \"safe\": \"\u5fae\u7d68\u6bdb\uff0c\u4e0d\u5149\u6ed1\", \"danger\": \"\u5149\u6ed1\u767c\u4eae\uff0c\u50cf\u6253\u881f\"}, {\"feature\": \"\u6c34\u73e0\u6e2c\u8a66\", \"safe\": \"\u6c34\u73e0\u6210\u5713\u73e0\u72c0\u6efe\u52d5\", \"danger\": \"\u6c34\u73e0\u6524\u5e73\"}, {\"feature\": \"\u8449\u5c16\u65b9\u5411\", \"safe\": \"\u671d\u4e0b\u5782\", \"danger\": \"\u671d\u4e0a\u6216\u6c34\u5e73\"}, {\"feature\": \"\u8449\u67c4\u63a5\u5408\", \"safe\": \"\u8449\u7247\u5167\u5074\uff08\u76fe\u72c0\uff09\", \"danger\": \"\u8449\u7247\u908a\u7de3\"}, {\"feature\": \"\u8449\u7247\u5927\u5c0f\", \"safe\": \"\u8f03\u5c0f\uff0820-50cm\uff09\", \"danger\": \"\u8f03\u5927\uff0850-90cm\uff09\"}, {\"feature\": \"\u8449\u67c4\u7d2b\u6688\", \"safe\": \"\u5c11\", \"danger\": \"\u5e38\u898b\"}], \"interactive_tests\": [{\"test_name\": \"\u6c34\u73e0\u6e2c\u8a66\", \"priority\": 1, \"is_critical\": true, \"instruction_zh\": \"\u5728\u8449\u9762\u6ef4\u4e00\u6ef4\u6c34\uff0c\u89c0\u5bdf\u6c34\u73e0\u7684\u5f62\u72c0\", \"safe_result\": \"\u6c34\u73e0\u5448\u5713\u73e0\u72c0\uff0c\u53ef\u4ee5\u6efe\u52d5 \u2192 \u8f03\u53ef\u80fd\u662f\u828b\u982d\", \"danger_result\": \"\u6c34\u73e0\u6524\u5e73\uff0c\u4e0d\u6210\u73e0\u72c0 \u2192 \u8f03\u53ef\u80fd\u662f\u59d1\u5a46\u828b\"}, {\"test_name\": \"\u8449\u67c4\u63a5\u5408\u8655\u7279\u5beb\", \"priority\": 2, \"is_critical\": true, \"instruction_zh\": \"\u62cd\u651d\u8449\u67c4\u8207\u8449\u7247\u7684\u63a5\u5408\u8655\u7279\u5beb\", \"safe_result\": \"\u8449\u67c4\u63a5\u5728\u8449\u7247\u908a\u7de3\u5167\u5074\uff08\u76fe\u72c0\u8457\u751f\uff09\", \"danger_result\": \"\u8449\u67c4\u63a5\u5728\u8449\u7247\u908a\u7de3\"}, {\"test_name\": \"\u8449\u9762\u8cea\u611f\u7279\u5beb\", \"priority\": 3, \"is_critical\": false, \"instruction_zh\": \"\u62cd\u651d\u8449\u9762\u7279\u5beb\uff0c\u89c0\u5bdf\u662f\u5426\u5149\u6ed1\u5982\u881f\uff0c\u9084\u662f\u6709\u5fae\u7d68\u6bdb\uff1f\", \"safe_result\": \"\u6709\u5fae\u7d68\u6bdb\uff0c\u4e0d\u5149\u6ed1\", \"danger_result\": \"\u5149\u6ed1\u767c\u4eae\uff0c\u50cf\u6253\u4e86\u881f\"}]}, {\"id\": \"parasol_vs_green_spored\", \"safe_species\": {\"id\": \"macrolepiota_procera\", \"scientific_name\": \"Macrolepiota procera\", \"common_name_zh\": \"\u9ad8\u5927\u74b0\u67c4\u83c7\", \"common_name_en\": \"Parasol Mushroom\", \"key_features\": {\"cap\": \"\u5098\u84cb\u76f4\u5f91 10-30cm\uff0c\u8868\u9762\u6709\u8910\u8272\u9c57\u7247\uff0c\u4e2d\u592e\u8f03\u6df1\", \"gills\": \"\u767d\u8272\uff0c\u6210\u719f\u5f8c\u4ecd\u4fdd\u6301\u767d\u8272\u6216\u6de1\u5976\u6cb9\u8272\", \"stipe\": \"\u83cc\u67c4\u7d30\u9577\uff0c\u6709\u86c7\u76ae\u72c0\u6591\u7d0b\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"ring\": \"\u96d9\u5c64\u83cc\u74b0\uff0c\u53ef\u4e0a\u4e0b\u6ed1\u52d5\", \"spore_print\": \"\u767d\u8272\"}}, \"dangerous_species\": {\"id\": \"chlorophyllum_molybdites\", \"scientific_name\": \"Chlorophyllum molybdites\", \"common_name_zh\": \"\u7da0\u8936\u83c7\", \"common_name_en\": \"Green-spored Parasol\", \"danger_level\": \"high\", \"key_features\": {\"cap\": \"\u5098\u84cb\u76f4\u5f91 10-30cm\uff0c\u5916\u89c0\u8207\u9ad8\u5927\u74b0\u67c4\u83c7\u6975\u4f3c\", \"gills\": \"\u521d\u767d\u8272\uff0c\u6210\u719f\u5f8c\u8f49\u70ba\u7da0\u8272\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"stipe\": \"\u83cc\u67c4\u8f03\u7c97\uff0c\u7f3a\u4e4f\u86c7\u76ae\u72c0\u6591\u7d0b\", \"ring\": \"\u83cc\u74b0\u8f03\u8584\", \"spore_print\": \"\u7da0\u8272\uff08\u95dc\u9375\u7279\u5fb5\uff09\"}}, \"lethality\": \"medium\", \"comparison_table\": [{\"feature\": \"\u83cc\u8936\u984f\u8272\", \"safe\": \"\u59cb\u7d42\u767d\u8272\u6216\u5976\u6cb9\u8272\", \"danger\": \"\u6210\u719f\u5f8c\u8f49\u7da0\u8272\"}, {\"feature\": \"\u5b62\u5b50\u5370\u984f\u8272\", \"safe\": \"\u767d\u8272\", \"danger\": \"\u7da0\u8272\"}, {\"feature\": \"\u83cc\u67c4\u7d0b\u8def\", \"safe\": \"\u6709\u86c7\u76ae\u72c0\u6591\u7d0b\", \"danger\": \"\u7121\u86c7\u76ae\u6591\u7d0b\"}, {\"feature\": \"\u83cc\u74b0\", \"safe\": \"\u96d9\u5c64\uff0c\u53ef\u79fb\u52d5\", \"danger\": \"\u8f03\u8584\uff0c\u55ae\u5c64\"}], \"interactive_tests\": [{\"test_name\": \"\u83cc\u8936\u984f\u8272\u6aa2\u67e5\", \"priority\": 1, \"is_critical\": true, \"instruction_zh\": \"\u5c0f\u5fc3\u7ffb\u958b\u5098\u84cb\uff0c\u89c0\u5bdf\u83cc\u8936\uff08\u5098\u84cb\u4e0b\u65b9\u7684\u653e\u5c04\u72c0\u8584\u7247\uff09\u984f\u8272\", \"safe_result\": \"\u83cc\u8936\u70ba\u767d\u8272\u6216\u6de1\u5976\u6cb9\u8272\", \"danger_result\": \"\u83cc\u8936\u5e36\u6709\u7da0\u8272\u6216\u7070\u7da0\u8272\"}, {\"test_name\": \"\u5b62\u5b50\u5370\u6e2c\u8a66\", \"priority\": 2, \"is_critical\": true, \"instruction_zh\": \"\u5c07\u5098\u84cb\u653e\u5728\u767d\u7d19\u4e0a\u975c\u7f6e 2-4 \u5c0f\u6642\uff0c\u89c0\u5bdf\u6389\u843d\u7684\u5b62\u5b50\u7c89\u984f\u8272\", \"safe_result\": \"\u767d\u8272\u5b62\u5b50\u7c89\", \"danger_result\": \"\u7da0\u8272\u6216\u7070\u7da0\u8272\u5b62\u5b50\u7c89\"}]}, {\"id\": \"birds_nest_fern_vs_toxic_fern\", \"safe_species\": {\"id\": \"asplenium_nidus\", \"scientific_name\": \"Asplenium nidus\", \"common_name_zh\": \"\u5c71\u8607\uff08\u9ce5\u5de2\u8568\uff09\", \"common_name_en\": \"Bird's Nest Fern\", \"key_features\": {\"leaf_shape\": \"\u5927\u578b\u55ae\u8449\uff0c\u9577\u62ab\u91dd\u5f62\uff0c\u9577 50-120cm\uff0c\u5bec 10-15cm\", \"leaf_arrangement\": \"\u53e2\u751f\uff0c\u5f9e\u4e2d\u5fc3\u5411\u5916\u8f3b\u5c04\u5c55\u958b\uff0c\u5f62\u6210\u9ce5\u5de2\u72c0\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"leaf_surface\": \"\u5149\u6ed1\uff0c\u9769\u8cea\uff0c\u5168\u7de3\uff08\u7121\u92f8\u9f52\uff09\", \"midrib\": \"\u660e\u986f\u7684\u9ed1\u8910\u8272\u4e2d\u808b\", \"sori\": \"\u8449\u80cc\u6709\u7dda\u72c0\u5b62\u5b50\u56ca\u7fa4\uff0c\u6cbf\u5074\u8108\u6392\u5217\", \"growth\": \"\u9644\u751f\u5728\u6a39\u5e79\u4e0a\u6216\u5ca9\u77f3\u4e0a\"}}, \"dangerous_species\": {\"id\": \"toxic_fern_general\", \"scientific_name\": \"Various\", \"common_name_zh\": \"\u6709\u6bd2\u8568\u985e\uff08\u6cdb\u6307\uff09\", \"common_name_en\": \"Toxic Ferns\", \"danger_level\": \"low\", \"key_features\": {\"leaf_shape\": \"\u591a\u70ba\u7fbd\u72c0\u8907\u8449\uff08\u8207\u5c71\u8607\u7684\u55ae\u8449\u660e\u986f\u4e0d\u540c\uff09\", \"leaf_arrangement\": \"\u901a\u5e38\u4e0d\u5f62\u6210\u9ce5\u5de2\u72c0\", \"leaf_surface\": \"\u53ef\u80fd\u6709\u6bdb\u8338\u6216\u4e0d\u540c\u8cea\u611f\", \"growth\": \"\u591a\u70ba\u5730\u751f\u578b\"}}, \"lethality\": \"low\", \"comparison_table\": [{\"feature\": \"\u8449\u5f62\", \"safe\": \"\u55ae\u8449\uff0c\u9577\u62ab\u91dd\u5f62\", \"danger\": \"\u591a\u70ba\u7fbd\u72c0\u8907\u8449\"}, {\"feature\": \"\u6392\u5217\u65b9\u5f0f\", \"safe\": \"\u9ce5\u5de2\u72c0\u8f3b\u5c04\u5c55\u958b\", \"danger\": \"\u7121\u7279\u5b9a\u6392\u5217\"}, {\"feature\": \"\u751f\u9577\u65b9\u5f0f\", \"safe\": \"\u9644\u751f\uff08\u6a39\u5e79/\u5ca9\u77f3\uff09\", \"danger\": \"\u591a\u5730\u751f\"}, {\"feature\": \"\u4e2d\u808b\u984f\u8272\", \"safe\": \"\u9ed1\u8910\u8272\", \"danger\": \"\u591a\u70ba\u7da0\u8272\"}], \"interactive_tests\": [{\"test_name\": \"\u8449\u80cc\u5b62\u5b50\u56ca\u89c0\u5bdf\", \"priority\": 1, \"is_critical\": false, \"instruction_zh\": \"\u7ffb\u958b\u8449\u7247\u80cc\u9762\uff0c\u89c0\u5bdf\u662f\u5426\u6709\u6cbf\u8457\u5074\u8108\u6392\u5217\u7684\u7dda\u72c0\u8910\u8272\u689d\u7d0b\uff08\u5b62\u5b50\u56ca\u7fa4\uff09\", \"safe_result\": \"\u6709\u6574\u9f4a\u7684\u7dda\u72c0\u5b62\u5b50\u56ca\u7fa4\u6cbf\u5074\u8108\u6392\u5217\", \"danger_result\": \"\u5b62\u5b50\u56ca\u5206\u4f48\u4e0d\u540c\u6216\u7121\u5b62\u5b50\u56ca\"}]}, {\"id\": \"wild_ginger_vs_toxic_ginger\", \"safe_species\": {\"id\": \"hedychium_coronarium\", \"scientific_name\": \"Hedychium coronarium\", \"common_name_zh\": \"\u91ce\u8591\u82b1\", \"common_name_en\": \"White Ginger Lily\", \"key_features\": {\"flower\": \"\u767d\u8272\u8774\u8776\u5f62\u82b1\u6735\uff0c\u9999\u6c23\u6fc3\u90c1\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"leaf_shape\": \"\u9577\u6a62\u5713\u5f62\u62ab\u91dd\u72c0\uff0c\u9577 20-40cm\uff0c\u4e92\u751f\u6392\u5217\", \"rhizome\": \"\u6839\u8396\u6709\u8591\u5473\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"stem\": \"\u76f4\u7acb\u8349\u672c\uff0c\u9ad8 100-200cm\"}}, \"dangerous_species\": {\"id\": \"toxic_zingiberaceae\", \"scientific_name\": \"Various Zingiberaceae\", \"common_name_zh\": \"\u6709\u6bd2\u8591\u79d1\u690d\u7269\", \"common_name_en\": \"Toxic Ginger Family Plants\", \"danger_level\": \"medium\", \"key_features\": {\"flower\": \"\u82b1\u8272\u53ef\u80fd\u4e0d\u540c\uff08\u975e\u7d14\u767d\u8774\u8776\u5f62\uff09\", \"rhizome\": \"\u6839\u8396\u7121\u8591\u5473\u6216\u6709\u523a\u6fc0\u6027\u6c23\u5473\", \"leaf_shape\": \"\u8449\u5f62\u53ef\u80fd\u76f8\u4f3c\u4f46\u7d30\u7bc0\u4e0d\u540c\"}}, \"lethality\": \"medium\", \"comparison_table\": [{\"feature\": \"\u82b1\u7684\u5f62\u614b\", \"safe\": \"\u767d\u8272\u8774\u8776\u5f62\uff0c\u5f37\u70c8\u82b3\u9999\", \"danger\": \"\u82b1\u8272\u6216\u82b1\u5f62\u4e0d\u540c\"}, {\"feature\": \"\u6839\u8396\u6c23\u5473\", \"safe\": \"\u6709\u660e\u986f\u8591\u5473\", \"danger\": \"\u7121\u8591\u5473\u6216\u523a\u6fc0\u6027\u6c23\u5473\"}, {\"feature\": \"\u82b1\u5e8f\u4f4d\u7f6e\", \"safe\": \"\u9802\u751f\u7a57\u72c0\u82b1\u5e8f\", \"danger\": \"\u53ef\u80fd\u4e0d\u540c\"}], \"interactive_tests\": [{\"test_name\": \"\u6839\u8396\u6c23\u5473\u6e2c\u8a66\", \"priority\": 1, \"is_critical\": true, \"instruction_zh\": \"\u5c0f\u5fc3\u522e\u958b\u5c11\u91cf\u6839\u8396\u8868\u76ae\uff0c\u805e\u662f\u5426\u6709\u660e\u986f\u7684\u8591\u5473\", \"safe_result\": \"\u6709\u6e05\u65b0\u7684\u8591\u5473\", \"danger_result\": \"\u7121\u8591\u5473\u3001\u6709\u523a\u6fc0\u6027\u6c23\u5473\u3001\u6216\u6709\u5316\u5b78\u85e5\u5473\"}]}, {\"id\": \"termite_mushroom_vs_white_toxic\", \"safe_species\": {\"id\": \"termitomyces_sp\", \"scientific_name\": \"Termitomyces sp.\", \"common_name_zh\": \"\u96de\u8089\u7d72\u83c7\", \"common_name_en\": \"Termite Mushroom\", \"key_features\": {\"cap\": \"\u5098\u84cb\u7070\u767d\u8272\u81f3\u8910\u8272\uff0c\u4e2d\u592e\u6709\u5c16\u7a81\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"gills\": \"\u767d\u8272\u81f3\u6de1\u7c89\u8272\", \"stipe\": \"\u83cc\u67c4\u4e0b\u65b9\u5ef6\u4f38\u5165\u571f\u4e2d\uff0c\u9023\u63a5\u767d\u87fb\u5de2\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"habitat\": \"\u53ea\u751f\u9577\u5728\u767d\u87fb\u5de2\u4e0a\u65b9\", \"texture\": \"\u6495\u958b\u5f8c\u5448\u7d72\u72c0\uff08\u50cf\u96de\u8089\u7d72\uff0c\u6545\u540d\uff09\"}}, \"dangerous_species\": {\"id\": \"white_toxic_mushroom\", \"scientific_name\": \"Amanita sp. / Leucoagaricus sp.\", \"common_name_zh\": \"\u767d\u8272\u6bd2\u83c7\u985e\", \"common_name_en\": \"White Toxic Mushrooms\", \"danger_level\": \"critical\", \"key_features\": {\"cap\": \"\u767d\u8272\uff0c\u53ef\u80fd\u7121\u4e2d\u592e\u5c16\u7a81\", \"gills\": \"\u767d\u8272\", \"stipe\": \"\u83cc\u67c4\u57fa\u90e8\u53ef\u80fd\u6709\u83cc\u6258\uff08\u676f\u72c0\uff09\u6216\u81a8\u5927\", \"habitat\": \"\u975e\u767d\u87fb\u5de2\u76f8\u95dc\", \"ring\": \"\u53ef\u80fd\u6709\u8584\u819c\u72c0\u83cc\u74b0\"}}, \"lethality\": \"critical\", \"comparison_table\": [{\"feature\": \"\u5098\u84cb\u4e2d\u592e\", \"safe\": \"\u6709\u5c16\u7a81\", \"danger\": \"\u7121\u5c16\u7a81\u6216\u5e73\u6ed1\"}, {\"feature\": \"\u83cc\u67c4\u57fa\u90e8\", \"safe\": \"\u6df1\u5165\u571f\u4e2d\u9023\u63a5\u767d\u87fb\u5de2\", \"danger\": \"\u6709\u83cc\u6258\uff08\u676f\u72c0\uff09\"}, {\"feature\": \"\u751f\u9577\u57fa\u8cea\", \"safe\": \"\u767d\u87fb\u5de2\u4e0a\u65b9\uff08\u627e\u87fb\u4e18\uff09\", \"danger\": \"\u6797\u5730\u3001\u8349\u5730\"}, {\"feature\": \"\u83cc\u8089\u8cea\u5730\", \"safe\": \"\u6495\u958b\u5448\u7d72\u72c0\", \"danger\": \"\u6495\u958b\u5448\u584a\u72c0\"}], \"interactive_tests\": [{\"test_name\": \"\u751f\u9577\u57fa\u8cea\u78ba\u8a8d\", \"priority\": 1, \"is_critical\": true, \"instruction_zh\": \"\u89c0\u5bdf\u83c7\u985e\u751f\u9577\u7684\u5730\u9762\uff0c\u662f\u5426\u6709\u767d\u87fb\u5de2\uff08\u571f\u5806\u72c0\u9686\u8d77\uff09\uff1f\u5c0f\u5fc3\u6316\u958b\u83cc\u67c4\u57fa\u90e8\uff0c\u662f\u5426\u6df1\u5165\u571f\u4e2d\uff1f\", \"safe_result\": \"\u83cc\u67c4\u6df1\u5165\u571f\u4e2d\uff0c\u4e0b\u65b9\u6709\u767d\u87fb\u5de2\u7d50\u69cb\", \"danger_result\": \"\u83cc\u67c4\u57fa\u90e8\u6709\u676f\u72c0\u83cc\u6258\uff0c\u6216\u76f4\u63a5\u9577\u5728\u843d\u8449/\u571f\u58e4\u4e0a\"}, {\"test_name\": \"\u83cc\u8089\u6495\u88c2\u6e2c\u8a66\", \"priority\": 2, \"is_critical\": false, \"instruction_zh\": \"\u5c0f\u5fc3\u6495\u958b\u4e00\u5c0f\u584a\u83cc\u8089\uff0c\u89c0\u5bdf\u8cea\u5730\", \"safe_result\": \"\u5448\u7d72\u72c0\u6495\u958b\uff08\u50cf\u6495\u96de\u8089\u7d72\uff09\", \"danger_result\": \"\u5448\u584a\u72c0\u6216\u7c89\u72c0\u65b7\u88c2\"}]}, {\"id\": \"green_snake_vs_bamboo_viper\", \"safe_species\": {\"id\": \"cyclophiops_major\", \"scientific_name\": \"Cyclophiops major\", \"common_name_zh\": \"\u9752\u86c7\", \"common_name_en\": \"Greater Green Snake\", \"key_features\": {\"body_color\": \"\u5168\u8eab\u7fe0\u7da0\u8272\uff0c\u7121\u767d\u8272\u5074\u7dda\", \"head_shape\": \"\u6a62\u5713\u5f62\uff0c\u8207\u9838\u90e8\u5206\u754c\u4e0d\u660e\u986f\", \"pupil\": \"\u5713\u5f62\u77b3\u5b54\", \"tail\": \"\u5c3e\u90e8\u7da0\u8272\uff0c\u7121\u7d05\u8272\uff08\u95dc\u9375\u5340\u5225\uff09\", \"body_size\": \"\u7d30\u9577\uff0c\u5168\u9577 60-100cm\", \"scales\": \"\u80cc\u9c57\u5149\u6ed1\", \"note\": \"\u7121\u6bd2\uff0c\u53f0\u5317\u5c71\u5340\u5e38\u898b\"}}, \"dangerous_species\": {\"id\": \"trimeresurus_stejnegeri\", \"scientific_name\": \"Trimeresurus stejnegeri\", \"common_name_zh\": \"\u8d64\u5c3e\u9752\u7af9\u7d72\", \"common_name_en\": \"Bamboo Viper\", \"danger_level\": \"high\", \"key_features\": {\"body_color\": \"\u5168\u8eab\u9bae\u7da0\u8272\uff0c\u9ad4\u5074\u6709\u767d\u8272\u6216\u6de1\u9ec3\u8272\u7e31\u7dda\", \"head_shape\": \"\u4e09\u89d2\u5f62\uff0c\u660e\u986f\u6bd4\u9838\u90e8\u5bec\", \"pupil\": \"\u7e31\u88c2\u77b3\u5b54\uff08\u8c93\u773c\u72c0\uff09\", \"tail\": \"\u5c3e\u5df4\u672b\u7aef\u78da\u7d05\u8272\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"body_size\": \"60-90cm\", \"pit_organ\": \"\u773c\u8207\u9f3b\u5b54\u4e4b\u9593\u6709\u9830\u7aa9\", \"note\": \"\u51fa\u8840\u6027\u6bd2\"}}, \"lethality\": \"high\", \"comparison_table\": [{\"feature\": \"\u5c3e\u5df4\u984f\u8272\", \"safe\": \"\u7da0\u8272\uff08\u540c\u9ad4\u8272\uff09\", \"danger\": \"\u78da\u7d05\u8272\"}, {\"feature\": \"\u982d\u5f62\", \"safe\": \"\u6a62\u5713\u5f62\", \"danger\": \"\u4e09\u89d2\u5f62\"}, {\"feature\": \"\u77b3\u5b54\", \"safe\": \"\u5713\u5f62\", \"danger\": \"\u7e31\u88c2\uff08\u8c93\u773c\uff09\"}, {\"feature\": \"\u9ad4\u5074\u7dda\", \"safe\": \"\u7121\", \"danger\": \"\u6709\u767d\u8272\u6216\u9ec3\u8272\u7e31\u7dda\"}, {\"feature\": \"\u9830\u7aa9\", \"safe\": \"\u7121\", \"danger\": \"\u6709\uff08\u773c\u524d\u65b9\u51f9\u9677\uff09\"}], \"interactive_tests\": [{\"test_name\": \"\u5c3e\u90e8\u984f\u8272\u89c0\u5bdf\", \"priority\": 1, \"is_critical\": true, \"instruction_zh\": \"\u5f9e\u5b89\u5168\u8ddd\u96e2\u89c0\u5bdf\u86c7\u5c3e\u672b\u7aef\u984f\u8272\u3002\", \"safe_result\": \"\u5c3e\u5df4\u8207\u8eab\u9ad4\u540c\u8272\uff08\u7da0\u8272\uff09\u3002\", \"danger_result\": \"\u5c3e\u5df4\u672b\u7aef\u660e\u986f\u78da\u7d05\u8272\u3002\"}, {\"test_name\": \"\u982d\u5f62\u89c0\u5bdf\", \"priority\": 2, \"is_critical\": true, \"instruction_zh\": \"\u5f9e\u5b89\u5168\u8ddd\u96e2\u89c0\u5bdf\u982d\u90e8\u5f62\u72c0\u3002\", \"safe_result\": \"\u982d\u90e8\u6a62\u5713\u5f62\uff0c\u8207\u9838\u90e8\u904e\u6e21\u5e73\u6ed1\u3002\", \"danger_result\": \"\u982d\u90e8\u660e\u986f\u4e09\u89d2\u5f62\uff0c\u6bd4\u9838\u90e8\u5bec\u3002\"}]}, {\"id\": \"red_banded_vs_krait\", \"safe_species\": {\"id\": \"lycodon_rufozonatus\", \"scientific_name\": \"Lycodon rufozonatus\", \"common_name_zh\": \"\u7d05\u6591\u86c7\", \"common_name_en\": \"Red-banded Snake\", \"key_features\": {\"body_color\": \"\u7d05\u8910\u8272\u5e95\uff0c\u6709\u4e0d\u898f\u5247\u6df1\u8272\u6a6b\u5e36\u548c\u7d05\u8272\u6591\u7d0b\", \"head_shape\": \"\u7565\u6241\u5e73\uff0c\u4f46\u8207\u9838\u90e8\u5206\u754c\u4e0d\u660e\u986f\", \"pupil\": \"\u5713\u5f62\u77b3\u5b54\", \"body_size\": \"\u5168\u9577 60-100cm\", \"scales\": \"\u80cc\u9c57\u5149\u6ed1\", \"behavior\": \"\u53d7\u9a5a\u6642\u6703\u5f13\u8d77\u8eab\u9ad4\u6a21\u4eff\u6bd2\u86c7\u59ff\u614b\", \"note\": \"\u7121\u6bd2\uff0c\u5e38\u898b\u65bc\u4f4f\u5bb6\u9644\u8fd1\"}}, \"dangerous_species\": {\"id\": \"bungarus_multicinctus\", \"scientific_name\": \"Bungarus multicinctus\", \"common_name_zh\": \"\u96e8\u5098\u7bc0\", \"common_name_en\": \"Many-banded Krait\", \"danger_level\": \"critical\", \"key_features\": {\"body_color\": \"\u9ed1\u767d\u76f8\u9593\u7b49\u5bec\u74b0\u72c0\u6a6b\u5e36\uff08\u95dc\u9375\u7279\u5fb5\uff1a\u9ed1\u767d\u5206\u660e\uff09\", \"head_shape\": \"\u6a62\u5713\u5f62\", \"pupil\": \"\u5713\u5f62\u77b3\u5b54\", \"body_size\": \"\u5168\u9577 100-150cm\", \"dorsal_scales\": \"\u80cc\u90e8\u4e2d\u592e\u4e00\u5217\u9c57\u7247\u64f4\u5927\u5448\u516d\u89d2\u5f62\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"note\": \"\u795e\u7d93\u6027\u6bd2\uff0c\u81f4\u6b7b\u7387\u6700\u9ad8\"}}, \"lethality\": \"critical\", \"comparison_table\": [{\"feature\": \"\u9ad4\u8272\u6a21\u5f0f\", \"safe\": \"\u7d05\u8910\u8272\u5e95+\u4e0d\u898f\u5247\u6591\u7d0b\", \"danger\": \"\u9ed1\u767d\u7b49\u5bec\u74b0\u5e36\uff08\u975e\u5e38\u898f\u5247\uff09\"}, {\"feature\": \"\u6a6b\u5e36\u984f\u8272\", \"safe\": \"\u6df1\u8910\u8272+\u7d05\u8272\", \"danger\": \"\u7d14\u9ed1\u8272+\u7d14\u767d\u8272\"}, {\"feature\": \"\u6a6b\u5e36\u898f\u5247\u6027\", \"safe\": \"\u4e0d\u898f\u5247\", \"danger\": \"\u975e\u5e38\u898f\u5247\u7b49\u5bec\"}, {\"feature\": \"\u80cc\u4e2d\u9c57\", \"safe\": \"\u4e00\u822c\u5927\u5c0f\", \"danger\": \"\u4e2d\u592e\u5217\u660e\u986f\u64f4\u5927\uff08\u516d\u89d2\u5f62\uff09\"}], \"interactive_tests\": [{\"test_name\": \"\u74b0\u5e36\u898f\u5247\u6027\u89c0\u5bdf\", \"priority\": 1, \"is_critical\": true, \"instruction_zh\": \"\u5f9e\u5b89\u5168\u8ddd\u96e2\u89c0\u5bdf\u6a6b\u5e36\u662f\u5426\u70ba\u898f\u5247\u7684\u9ed1\u767d\u7b49\u5bec\u74b0\u5e36\u3002\", \"safe_result\": \"\u6591\u7d0b\u4e0d\u898f\u5247\uff0c\u6709\u7d05\u8272\u3002\", \"danger_result\": \"\u9ed1\u767d\u5e36\u975e\u5e38\u898f\u5247\u3001\u7b49\u5bec\u3002\"}]}, {\"id\": \"bidens_vs_parthenium\", \"safe_species\": {\"id\": \"bidens_pilosa_radiata\", \"scientific_name\": \"Bidens pilosa var. radiata\", \"common_name_zh\": \"\u5927\u82b1\u54b8\u8c50\u8349\", \"common_name_en\": \"Spanish Needles (radiate variety)\", \"key_features\": {\"leaf_shape\": \"\u4e09\u51fa\u8907\u8449\u6216\u7fbd\u72c0\u8907\u8449\uff0c\u5c0f\u8449\u5375\u5f62\u6709\u92f8\u9f52\", \"flower\": \"\u982d\u72c0\u82b1\u5e8f\uff0c\u4e2d\u5fc3\u9ec3\u8272\uff0c\u5468\u570d5-8\u679a\u767d\u8272\u820c\u72c0\u82b1\u74e3\uff08\u82b1\u74e3\u660e\u986f\u5927\u4e14\u6578\u91cf\u5c11\uff09\", \"stem\": \"\u65b9\u5f62\u8396\u6709\u7a1c\", \"fruit\": \"\u7626\u679c\u6709\u5012\u9264\u523a\uff08\u6703\u9ecf\u8863\u670d\uff09\", \"smell\": \"\u7121\u7279\u6b8a\u523a\u6fc0\u6c23\u5473\", \"note\": \"\u53ef\u98df\u7528\u91ce\u8349\"}}, \"dangerous_species\": {\"id\": \"parthenium_hysterophorus\", \"scientific_name\": \"Parthenium hysterophorus\", \"common_name_zh\": \"\u9280\u81a0\u83ca\", \"common_name_en\": \"Parthenium Weed\", \"danger_level\": \"high\", \"key_features\": {\"leaf_shape\": \"\u7fbd\u72c0\u6df1\u88c2\uff0c\u88c2\u7247\u8f03\u7d30\uff08\u95dc\u9375\u5340\u5225\uff1a\u8449\u88c2\u66f4\u6df1\u66f4\u7d30\uff09\", \"flower\": \"\u982d\u72c0\u82b1\u5e8f\u975e\u5e38\u5c0f\uff08\u76f4\u5f91\u50c53-5mm\uff09\uff0c\u767d\u8272\u82b1\u6975\u5c0f\u6975\u5bc6\u96c6\uff08\u4e0d\u50cf\u54b8\u8c50\u8349\u6709\u660e\u986f\u82b1\u74e3\uff09\", \"stem\": \"\u5713\u5f62\u8396\u6709\u77ed\u6bdb\", \"smell\": \"\u6413\u63c9\u6709\u523a\u6fc0\u6027\u6c23\u5473\", \"pollen\": \"\u82b1\u7c89\u91cf\u6975\u5927\uff0c\u6703\u98c4\u6563\", \"note\": \"\u56b4\u91cd\u904e\u654f\u6e90\"}}, \"lethality\": \"low\", \"comparison_table\": [{\"feature\": \"\u8449\u5f62\", \"safe\": \"\u4e09\u51fa\u8907\u8449\uff0c\u5c0f\u8449\u8f03\u5927\", \"danger\": \"\u7fbd\u72c0\u6df1\u88c2\uff0c\u88c2\u7247\u7d30\u788e\"}, {\"feature\": \"\u82b1\u7684\u5927\u5c0f\", \"safe\": \"\u82b1\u5e8f\u76f4\u5f91 1-2cm\uff0c\u82b1\u74e3\u660e\u986f\", \"danger\": \"\u82b1\u5e8f\u76f4\u5f91 3-5mm\uff0c\u6975\u5c0f\"}, {\"feature\": \"\u767d\u8272\u82b1\u74e3\", \"safe\": \"5-8\u679a\u5927\u82b1\u74e3\", \"danger\": \"5\u679a\u6975\u5c0f\u82b1\u74e3\uff08\u8089\u773c\u5e7e\u4e4e\u770b\u4e0d\u5230\uff09\"}, {\"feature\": \"\u8396\u7684\u5f62\u72c0\", \"safe\": \"\u65b9\u5f62\u6709\u7a1c\", \"danger\": \"\u5713\u5f62\u6709\u6bdb\"}, {\"feature\": \"\u6c23\u5473\", \"safe\": \"\u7121\u523a\u6fc0\u5473\", \"danger\": \"\u6413\u63c9\u6709\u523a\u6fc0\u5473\"}], \"interactive_tests\": [{\"test_name\": \"\u82b1\u6735\u5927\u5c0f\u6bd4\u8f03\", \"priority\": 1, \"is_critical\": true, \"instruction_zh\": \"\u89c0\u5bdf\u767d\u8272\u82b1\u6735\u5927\u5c0f\u3002\", \"safe_result\": \"\u82b1\u5e8f\u76f4\u5f91\u7d041-2cm\uff0c\u767d\u8272\u82b1\u74e3\u660e\u986f5-8\u679a\u3002\", \"danger_result\": \"\u82b1\u5e8f\u975e\u5e38\u5c0f\u50c53-5mm\uff0c\u767d\u8272\u5e7e\u4e4e\u770b\u4e0d\u5230\u3002\"}, {\"test_name\": \"\u6c23\u5473\u6e2c\u8a66\", \"priority\": 2, \"is_critical\": false, \"instruction_zh\": \"\u5c0f\u5fc3\u6413\u63c9\u4e00\u7247\u8449\u5b50\u805e\u6c23\u5473\uff08\u52ff\u89f8\u78b0\u773c\u775b\uff09\u3002\", \"safe_result\": \"\u7121\u7279\u6b8a\u523a\u6fc0\u5473\u3002\", \"danger_result\": \"\u6709\u523a\u6fc0\u6027\u6c23\u5473\u3002\"}]}, {\"id\": \"pokeweed_vs_amaranth\", \"safe_species\": {\"id\": \"amaranthus_viridis\", \"scientific_name\": \"Amaranthus viridis\", \"common_name_zh\": \"\u91ce\u83a7\u83dc\", \"common_name_en\": \"Green Amaranth\", \"key_features\": {\"stem_color\": \"\u7da0\u8272\u6216\u5fae\u5e36\u7d05\u8272\uff0c\u9ad820-80cm\", \"leaf_shape\": \"\u5375\u5f62\u81f3\u83f1\u5f62\uff0c\u8f03\u5c0f\uff083-8cm\uff09\", \"stem_size\": \"\u8396\u7d30\uff08\u76f4\u5f91 0.5-1cm\uff09\", \"flower\": \"\u7a57\u72c0\u82b1\u5e8f\uff0c\u7da0\u8272\u5c0f\u82b1\", \"fruit\": \"\u7a2e\u5b50\u9ed1\u8272\u7d30\u5c0f\", \"note\": \"\u53ef\u98df\u7528\u91ce\u83dc\"}}, \"dangerous_species\": {\"id\": \"phytolacca_americana\", \"scientific_name\": \"Phytolacca americana\", \"common_name_zh\": \"\u7f8e\u6d32\u5546\u9678\", \"common_name_en\": \"American Pokeweed\", \"danger_level\": \"high\", \"key_features\": {\"stem_color\": \"\u8396\u7c97\u58ef\u3001\u5e38\u5448\u7d05\u7d2b\u8272\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"leaf_shape\": \"\u5927\u578b\u6a62\u5713\u5f62\uff0c\u660e\u986f\u66f4\u5927\uff0810-30cm\uff09\", \"stem_size\": \"\u8396\u975e\u5e38\u7c97\uff08\u76f4\u5f91 2-5cm\uff09\", \"flower\": \"\u7e3d\u72c0\u82b1\u5e8f\uff08\u8207\u83a7\u83dc\u7a57\u72c0\u4e0d\u540c\uff09\uff0c\u767d\u8272\u81f3\u7c89\u8272\u5c0f\u82b1\", \"fruit\": \"\u7403\u5f62\u6f3f\u679c\u6210\u719f\u7d2b\u9ed1\u8272\u6392\u6210\u7a57\u72c0\uff08\u95dc\u9375\u7279\u5fb5\uff1a\u50cf\u7d2b\u8272\u8461\u8404\u4e32\uff09\", \"note\": \"\u5168\u682a\u6709\u6bd2\"}}, \"lethality\": \"high\", \"comparison_table\": [{\"feature\": \"\u690d\u682a\u5927\u5c0f\", \"safe\": \"\u77ee\u5c0f 20-80cm\", \"danger\": \"\u9ad8\u5927 100-300cm\"}, {\"feature\": \"\u8396\u7684\u7c97\u7d30\", \"safe\": \"\u7d30\uff080.5-1cm\uff09\", \"danger\": \"\u7c97\u58ef\uff082-5cm\uff09\u4e14\u7d2b\u7d05\u8272\"}, {\"feature\": \"\u8449\u7247\u5927\u5c0f\", \"safe\": \"\u5c0f\uff083-8cm\uff09\", \"danger\": \"\u5927\uff0810-30cm\uff09\"}, {\"feature\": \"\u679c\u5be6\", \"safe\": \"\u7121\u660e\u986f\u679c\u5be6\", \"danger\": \"\u7d2b\u9ed1\u8272\u6f3f\u679c\u4e32\uff08\u50cf\u5c0f\u8461\u8404\uff09\"}], \"interactive_tests\": [{\"test_name\": \"\u8396\u90e8\u984f\u8272\u548c\u7c97\u7d30\", \"priority\": 1, \"is_critical\": true, \"instruction_zh\": \"\u89c0\u5bdf\u8396\u90e8\u3002\", \"safe_result\": \"\u8396\u7d30\u3001\u7da0\u8272\u3002\", \"danger_result\": \"\u8396\u7c97\u58ef\u3001\u7d2b\u7d05\u8272\u3002\"}, {\"test_name\": \"\u679c\u5be6\u89c0\u5bdf\", \"priority\": 2, \"is_critical\": true, \"instruction_zh\": \"\u662f\u5426\u6709\u7d2b\u9ed1\u8272\u6f3f\u679c\u4e32\u3002\", \"safe_result\": \"\u7121\u6f3f\u679c\u3002\", \"danger_result\": \"\u6709\u7d2b\u9ed1\u8272\u6f3f\u679c\u6392\u6210\u7a57\u72c0\u3002\"}]}, {\"id\": \"calla_vs_alocasia_vs_taro\", \"safe_species\": {\"id\": \"colocasia_esculenta_calla_pair\", \"scientific_name\": \"Colocasia esculenta\", \"common_name_zh\": \"\u828b\u982d\", \"common_name_en\": \"Taro\", \"key_features\": {\"leaf_surface\": \"\u6709\u7d68\u6bdb\u8cea\u611f\uff0c\u4e0d\u5149\u6ed1\", \"water_test\": \"\u6c34\u73e0\u6210\u5713\u73e0\u72c0\u6efe\u52d5\", \"leaf_size\": \"\u8f03\u5c0f\uff0820-50cm\uff09\", \"flower\": \"\u4e0d\u5e38\u958b\u82b1\", \"underground\": \"\u6709\u5713\u5f62\u584a\u8396\uff08\u53ef\u98df\u7528\uff09\", \"petiole\": \"\u63a5\u5728\u8449\u7247\u908a\u7de3\u5167\u5074\uff08\u76fe\u72c0\u8457\u751f\uff09\", \"context_note\": \"\u7af9\u5b50\u6e56\u7b49\u5730\u6d77\u828b\u5927\u91cf\u5206\u5e03\uff0c\u6613\u8207\u828b\u982d\u6df7\u6dc6\uff1b\u59d1\u5a46\u828b\u8acb\u53e6\u53c3\u8003\u828b\u982d\u8207\u59d1\u5a46\u828b\u689d\u76ee\"}}, \"dangerous_species\": {\"id\": \"zantedeschia_aethiopica\", \"scientific_name\": \"Zantedeschia aethiopica\", \"common_name_zh\": \"\u6d77\u828b\", \"common_name_en\": \"Calla Lily\", \"danger_level\": \"medium\", \"key_features\": {\"leaf_surface\": \"\u5149\u6ed1\u6709\u5149\u6fa4\", \"flower\": \"\u767d\u8272\u4f5b\u7130\u82de\uff08\u95dc\u9375\u7279\u5fb5\uff1a\u6f0f\u6597\u5f62\u7d14\u767d\u82b1\uff0c\u4e2d\u592e\u9ec3\u8272\u82b1\u5e8f\uff09\", \"leaf_shape\": \"\u7bad\u5f62\u6216\u4e09\u89d2\u5f62\uff08\u6bd4\u828b\u982d\u66f4\u5c16\uff09\", \"leaf_size\": \"\u4e2d\u7b49\uff0815-45cm\uff09\", \"underground\": \"\u6839\u8396\u6709\u6bd2\", \"height\": \"\u690d\u682a\u8f03\u5c0f\uff0840-100cm\uff09\"}}, \"lethality\": \"medium\", \"comparison_table\": [{\"feature\": \"\u82b1\", \"safe\": \"\u4e0d\u5e38\u898b\", \"danger\": \"\u767d\u8272\u4f5b\u7130\u82de\uff08\u6f0f\u6597\u5f62\uff09\u975e\u5e38\u986f\u773c\"}, {\"feature\": \"\u8449\u5f62\", \"safe\": \"\u5fc3\u5f62\u5713\u6f64\", \"danger\": \"\u7bad\u5f62\u8f03\u5c16\"}, {\"feature\": \"\u8449\u9762\", \"safe\": \"\u6709\u7d68\u6bdb\", \"danger\": \"\u5149\u6ed1\"}, {\"feature\": \"\u690d\u682a\", \"safe\": \"\u8449\u7247\u8f03\u5927\", \"danger\": \"\u690d\u682a\u6574\u9ad4\u8f03\u5c0f\"}], \"interactive_tests\": [{\"test_name\": \"\u82b1\u6735\u89c0\u5bdf\", \"priority\": 1, \"is_critical\": true, \"instruction_zh\": \"\u662f\u5426\u6709\u767d\u8272\u6f0f\u6597\u5f62\u4f5b\u7130\u82de\u82b1\u3002\", \"safe_result\": \"\u7121\u6b64\u82b1\u6216\u4e0d\u5e38\u958b\u82b1\u3002\", \"danger_result\": \"\u6709\u767d\u8272\u4f5b\u7130\u82de\u3002\"}, {\"test_name\": \"\u6c34\u73e0\u6e2c\u8a66\", \"priority\": 2, \"is_critical\": true, \"instruction_zh\": \"\u540c\u828b\u982d\u8207\u59d1\u5a46\u828b\u4e4b\u6c34\u73e0\u6e2c\u8a66\uff1a\u5728\u8449\u9762\u6ef4\u4e00\u6ef4\u6c34\uff0c\u89c0\u5bdf\u6c34\u73e0\u662f\u5426\u6210\u5713\u73e0\u6efe\u52d5\u3002\", \"safe_result\": \"\u6c34\u73e0\u6210\u5713\u73e0\u72c0\u6efe\u52d5\uff0c\u8f03\u53ef\u80fd\u70ba\u828b\u982d\u3002\", \"danger_result\": \"\u6c34\u73e0\u6613\u6524\u5e73\u3001\u8449\u9762\u5149\u6ed1\uff0c\u8f03\u4e0d\u50cf\u828b\u982d\uff08\u9700\u642d\u914d\u5176\u4ed6\u7279\u5fb5\u5224\u65b7\u6d77\u828b\uff09\u3002\"}]}, {\"id\": \"lantana_vs_mulberry\", \"safe_species\": {\"id\": \"morus_australis\", \"scientific_name\": \"Morus australis\", \"common_name_zh\": \"\u5c0f\u8449\u6851\uff08\u6851\u6939\uff09\", \"common_name_en\": \"Australian Mulberry\", \"key_features\": {\"fruit_shape\": \"\u805a\u5408\u679c\uff0c\u5713\u67f1\u5f62\u95771-2cm\uff08\u50cf\u8ff7\u4f60\u7389\u7c73\u5f62\u72c0\uff09\", \"fruit_color\": \"\u672a\u719f\u7d05\u8272\uff0c\u6210\u719f\u6df1\u7d2b\u9ed1\u8272\", \"leaf\": \"\u5375\u5f62\uff0c\u908a\u7de3\u6709\u920d\u92f8\u9f52\uff0c\u4e92\u751f\", \"plant_type\": \"\u704c\u6728\u6216\u5c0f\u55ac\u6728\", \"taste\": \"\u6210\u719f\u679c\u751c\"}}, \"dangerous_species\": {\"id\": \"lantana_camara\", \"scientific_name\": \"Lantana camara\", \"common_name_zh\": \"\u99ac\u7e93\u4e39\uff08\u672a\u719f\u679c\uff09\", \"common_name_en\": \"Lantana\", \"danger_level\": \"medium\", \"key_features\": {\"fruit_shape\": \"\u7403\u5f62\u6f3f\u679c\uff0c\u76f4\u5f915-7mm\uff0c\u6392\u6210\u7403\u5f62\u805a\u5408\", \"fruit_color\": \"\u672a\u719f\u7da0\u8272\u2192\u85cd\u9ed1\u8272\uff08\u95dc\u9375\u7279\u5fb5\uff1a\u672a\u719f\u7da0\u8272\u6709\u6bd2\uff09\", \"leaf\": \"\u5375\u5f62\uff0c\u5c0d\u751f\uff08\u4e0d\u662f\u4e92\u751f\uff01\uff09\uff0c\u6413\u63c9\u6709\u81ed\u5473\", \"plant_type\": \"\u704c\u6728\", \"flower\": \"\u82b1\u8272\u591a\u8b8a\uff08\u9ec3\u2192\u6a59\u2192\u7d05\uff0c\u540c\u4e00\u82b1\u5e8f\u591a\u8272\uff09\"}}, \"lethality\": \"medium\", \"comparison_table\": [{\"feature\": \"\u679c\u5be6\u5f62\u72c0\", \"safe\": \"\u5713\u67f1\u5f62\u805a\u5408\u679c\", \"danger\": \"\u7403\u5f62\u5c0f\u6f3f\u679c\"}, {\"feature\": \"\u679c\u5be6\u6392\u5217\", \"safe\": \"\u6cbf\u8ef8\u7dda\u6392\u5217\", \"danger\": \"\u7403\u5f62\u805a\u96c6\"}, {\"feature\": \"\u8449\u7684\u6392\u5217\", \"safe\": \"\u4e92\u751f\", \"danger\": \"\u5c0d\u751f\"}, {\"feature\": \"\u82b1\", \"safe\": \"\u4e0d\u986f\u773c\", \"danger\": \"\u591a\u8272\u982d\u72c0\u82b1\u5e8f\uff08\u9ec3\u6a59\u7d05\uff09\"}, {\"feature\": \"\u6c23\u5473\", \"safe\": \"\u7121\u7279\u6b8a\u5473\", \"danger\": \"\u8449\u6413\u63c9\u6709\u81ed\u5473\"}], \"interactive_tests\": [{\"test_name\": \"\u679c\u5be6\u5f62\u72c0\u89c0\u5bdf\", \"priority\": 1, \"is_critical\": true, \"instruction_zh\": \"\u89c0\u5bdf\u679c\u5be6\u5f62\u72c0\u3002\", \"safe_result\": \"\u5713\u67f1\u5f62\uff08\u50cf\u8ff7\u4f60\u7389\u7c73\uff09\u3002\", \"danger_result\": \"\u7403\u5f62\u5c0f\u9846\u7c92\u805a\u96c6\u3002\"}, {\"test_name\": \"\u8449\u7247\u6c23\u5473\", \"priority\": 2, \"is_critical\": false, \"instruction_zh\": \"\u6413\u63c9\u8449\u7247\u3002\", \"safe_result\": \"\u7121\u7279\u6b8a\u5473\u3002\", \"danger_result\": \"\u6709\u523a\u9f3b\u81ed\u5473\u3002\"}]}, {\"id\": \"foxglove_vs_plantain\", \"safe_species\": {\"id\": \"plantago_major\", \"scientific_name\": \"Plantago major\", \"common_name_zh\": \"\u8eca\u524d\u8349\", \"common_name_en\": \"Broadleaf Plantain\", \"key_features\": {\"leaf_arrangement\": \"\u57fa\u751f\u84ee\u5ea7\u72c0\uff08\u6240\u6709\u8449\u5f9e\u57fa\u90e8\u8cbc\u5730\u5c55\u958b\uff09\", \"leaf_surface\": \"\u5149\u6ed1\uff0c\u6709\u660e\u986f3-7\u689d\u5e73\u884c\u8108\uff08\u95dc\u9375\u7279\u5fb5\uff09\", \"leaf_shape\": \"\u6a62\u5713\u5f62\uff0c\u5168\u7de3\", \"flower\": \"\u7a57\u72c0\u82b1\u5e8f\uff0c\u76f4\u7acb\u7d30\u9577\u82b1\u8396\"}}, \"dangerous_species\": {\"id\": \"digitalis_purpurea\", \"scientific_name\": \"Digitalis purpurea\", \"common_name_zh\": \"\u6bdb\u5730\u9ec3\", \"common_name_en\": \"Foxglove\", \"danger_level\": \"critical\", \"key_features\": {\"leaf_arrangement\": \"\u57fa\u751f\u84ee\u5ea7\u72c0\uff08\u4e00\u5e74\u751f\u690d\u682a\uff09\u6216\u4e92\u751f\uff08\u4e8c\u5e74\u751f\uff09\", \"leaf_surface\": \"\u5bc6\u88ab\u7070\u8272\u7d68\u6bdb\uff08\u95dc\u9375\u5340\u5225\uff1a\u8449\u9762\u6709\u7d68\u6bdb\uff09\", \"leaf_shape\": \"\u9577\u6a62\u5713\u5f62\uff0c\u908a\u7de3\u6709\u920d\u92f8\u9f52\", \"flower\": \"\u7e3d\u72c0\u82b1\u5e8f\uff0c\u9418\u5f62\u4e0b\u5782\u82b1\uff08\u7d2b\u7d05\u8272\u6709\u6591\u9ede\uff09\"}}, \"lethality\": \"critical\", \"comparison_table\": [{\"feature\": \"\u8449\u9762\u8cea\u611f\", \"safe\": \"\u5149\u6ed1\", \"danger\": \"\u5bc6\u88ab\u7070\u8272\u7d68\u6bdb\uff08\u6478\u8d77\u4f86\u50cf\u7d68\u5e03\uff09\"}, {\"feature\": \"\u8449\u8108\", \"safe\": \"3-7\u689d\u660e\u986f\u5e73\u884c\u8108\", \"danger\": \"\u7db2\u72c0\u8108\uff08\u4e0d\u5e73\u884c\uff09\"}, {\"feature\": \"\u8449\u7de3\", \"safe\": \"\u5168\u7de3\", \"danger\": \"\u6709\u920d\u92f8\u9f52\"}, {\"feature\": \"\u82b1\", \"safe\": \"\u76f4\u7acb\u7d30\u7a57\", \"danger\": \"\u9418\u5f62\u4e0b\u5782\u82b1\uff08\u7d2b\u7d05\u8272\uff09\"}], \"interactive_tests\": [{\"test_name\": \"\u8449\u9762\u89f8\u611f\", \"priority\": 1, \"is_critical\": true, \"instruction_zh\": \"\u8f15\u6478\u8449\u9762\u3002\", \"safe_result\": \"\u5149\u6ed1\u3002\", \"danger_result\": \"\u6709\u67d4\u8edf\u7d68\u6bdb\u8986\u84cb\u3002\"}, {\"test_name\": \"\u8449\u8108\u89c0\u5bdf\", \"priority\": 2, \"is_critical\": false, \"instruction_zh\": \"\u89c0\u5bdf\u8449\u8108\u3002\", \"safe_result\": \"3-7\u689d\u660e\u986f\u5e73\u884c\u8108\u3002\", \"danger_result\": \"\u7db2\u72c0\u8108\u3002\"}]}]")

_KB_SNAKEBITE_FIRST_AID = json.loads("{\"scenario\": \"snakebite\", \"title_zh\": \"\u86c7\u54ac\u50b7\u7dca\u6025\u8655\u7406 SOP\", \"steps\": [{\"step\": 1, \"action_zh\": \"\u4fdd\u6301\u51b7\u975c\uff0c\u7acb\u5373\u9060\u96e2\u86c7\u7684\u653b\u64ca\u7bc4\u570d\uff08\u81f3\u5c11 2 \u516c\u5c3a\u4ee5\u4e0a\uff09\", \"reason_zh\": \"\u6050\u614c\u6703\u52a0\u901f\u5fc3\u8df3\uff0c\u4f7f\u6bd2\u6db2\u64f4\u6563\u66f4\u5feb\"}, {\"step\": 2, \"action_zh\": \"\u8a18\u4f4f\u86c7\u7684\u5916\u89c0\u7279\u5fb5\uff1a\u9ad4\u8272\u3001\u6591\u7d0b\u3001\u982d\u5f62\u3001\u5927\u5c0f\u3002\u62cd\u7167\u66f4\u4f73\u3002\", \"reason_zh\": \"\u91ab\u9662\u9700\u8981\u5224\u65b7\u86c7\u7a2e\u4ee5\u9078\u64c7\u6b63\u78ba\u7684\u6297\u86c7\u6bd2\u8840\u6e05\"}, {\"step\": 3, \"action_zh\": \"\u79fb\u9664\u50b7\u80a2\u4e0a\u7684\u6212\u6307\u3001\u624b\u9336\u3001\u624b\u74b0\u7b49\u675f\u7e1b\u7269\", \"reason_zh\": \"\u816b\u8139\u6703\u4f7f\u9019\u4e9b\u7269\u54c1\u5361\u4f4f\uff0c\u9020\u6210\u8840\u6d41\u963b\u65b7\"}, {\"step\": 4, \"action_zh\": \"\u4fdd\u6301\u50b7\u80a2\u4f4e\u65bc\u5fc3\u81df\u4f4d\u7f6e\uff0c\u6e1b\u5c11\u6d3b\u52d5\", \"reason_zh\": \"\u6e1b\u7de9\u6bd2\u6db2\u7d93\u7531\u6dcb\u5df4\u548c\u8840\u6db2\u7cfb\u7d71\u64f4\u6563\"}, {\"step\": 5, \"action_zh\": \"\u5982\u6709\u5f48\u6027\u7e43\u5e36\uff0c\u53ef\u5f9e\u50b7\u53e3\u4e0a\u65b9\uff08\u9760\u8fd1\u5fc3\u81df\u65b9\u5411\uff09\u958b\u59cb\u5305\u7d2e\uff0c\u58d3\u529b\u8207\u5305\u7d2e\u626d\u50b7\u7684\u529b\u5ea6\u76f8\u540c\", \"reason_zh\": \"\u58d3\u529b\u56fa\u5b9a\u6cd5\uff08Pressure Immobilization\uff09\u53ef\u6e1b\u7de9\u795e\u7d93\u6027\u6bd2\u6db2\u64f4\u6563\uff0c\u4f46\u5c0d\u51fa\u8840\u6027\u6bd2\u6db2\u6548\u679c\u6709\u9650\"}, {\"step\": 6, \"action_zh\": \"\u5118\u901f\u9001\u91ab\u6216\u547c\u53eb\u6551\u63f4\", \"reason_zh\": \"\u6297\u86c7\u6bd2\u8840\u6e05\u662f\u552f\u4e00\u6709\u6548\u7684\u6cbb\u7642\u65b9\u5f0f\"}], \"do_not\": [\"\u274c \u52ff\u5207\u958b\u50b7\u53e3\u653e\u8840\", \"\u274c \u52ff\u7528\u5634\u5438\u6bd2\uff08\u53e3\u8154\u9ecf\u819c\u6703\u5438\u6536\u6bd2\u6db2\uff09\", \"\u274c \u52ff\u7d81\u6b62\u8840\u5e36\uff08\u53ef\u80fd\u5c0e\u81f4\u7d44\u7e54\u58de\u6b7b\uff09\", \"\u274c \u52ff\u51b0\u6577\uff08\u52a0\u901f\u7d44\u7e54\u640d\u50b7\uff09\", \"\u274c \u52ff\u98f2\u9152\uff08\u52a0\u901f\u8840\u6db2\u5faa\u74b0\uff09\", \"\u274c \u52ff\u81ea\u884c\u670d\u85e5\", \"\u274c \u52ff\u5617\u8a66\u6355\u86c7\uff08\u53ef\u80fd\u9020\u6210\u4e8c\u6b21\u54ac\u50b7\uff09\"], \"by_venom_type\": {\"hemorrhagic\": {\"name_zh\": \"\u51fa\u8840\u6027\u6bd2\uff08\u8d64\u5c3e\u9752\u7af9\u7d72\u3001\u9f9c\u6bbc\u82b1\u3001\u767e\u6b65\u86c7\uff09\", \"symptoms_zh\": \"\u50b7\u53e3\u5287\u75db\u3001\u8fc5\u901f\u816b\u8139\u3001\u51fa\u8840\u4e0d\u6b62\u3001\u7600\u9752\", \"key_action_zh\": \"\u6e1b\u5c11\u6d3b\u52d5\uff0c\u52ff\u52a0\u58d3\u5305\u7d2e\uff08\u53ef\u80fd\u52a0\u91cd\u5c40\u90e8\u51fa\u8840\uff09\", \"antivenom_zh\": \"\u51fa\u8840\u6027\u86c7\u6bd2\u8840\u6e05\"}, \"neurotoxic\": {\"name_zh\": \"\u795e\u7d93\u6027\u6bd2\uff08\u96e8\u5098\u7bc0\uff09\", \"symptoms_zh\": \"\u26a0\ufe0f \u521d\u671f\u53ef\u80fd\u4e0d\u75db\uff01\u4e4b\u5f8c\u773c\u77bc\u4e0b\u5782\u3001\u541e\u56a5\u56f0\u96e3\u3001\u547c\u5438\u56f0\u96e3\", \"key_action_zh\": \"\u5373\u4f7f\u4e0d\u75db\u4e5f\u5fc5\u9808\u5c31\u91ab\uff01\u53ef\u7528\u58d3\u529b\u56fa\u5b9a\u6cd5\u3002\u6ce8\u610f\u547c\u5438\uff0c\u6e96\u5099\u4eba\u5de5\u547c\u5438\u3002\", \"antivenom_zh\": \"\u795e\u7d93\u6027\u86c7\u6bd2\u8840\u6e05\uff08\u8d8a\u65e9\u65bd\u6253\u8d8a\u597d\uff09\"}, \"mixed\": {\"name_zh\": \"\u6df7\u5408\u6bd2\uff08\u773c\u93e1\u86c7\uff09\", \"symptoms_zh\": \"\u5c40\u90e8\u75bc\u75db\u816b\u8139 + \u795e\u7d93\u75c7\u72c0\", \"key_action_zh\": \"\u6309\u51fa\u8840\u6027\u8655\u7406\uff0c\u540c\u6642\u6ce8\u610f\u547c\u5438\u72c0\u6cc1\", \"antivenom_zh\": \"\u6df7\u5408\u86c7\u6bd2\u8840\u6e05\"}}}")

_KB_PLANT_POISONING = json.loads("{\"scenario\": \"plant_poisoning\", \"title_zh\": \"\u690d\u7269\u4e2d\u6bd2\u7dca\u6025\u8655\u7406 SOP\", \"general_steps\": [{\"step\": 1, \"action_zh\": \"\u7acb\u5373\u505c\u6b62\u98df\u7528\uff0c\u5c07\u53e3\u4e2d\u6b98\u9918\u7269\u5410\u51fa\", \"reason_zh\": \"\u6e1b\u5c11\u6bd2\u7d20\u651d\u5165\u91cf\"}, {\"step\": 2, \"action_zh\": \"\u7528\u5927\u91cf\u6e05\u6c34\u6f31\u53e3\uff08\u52ff\u541e\u56a5\uff09\", \"reason_zh\": \"\u6c96\u6d17\u53e3\u8154\u4e2d\u7684\u6bd2\u7d20\"}, {\"step\": 3, \"action_zh\": \"\u4fdd\u7559\u690d\u7269\u6a23\u672c\u6216\u62cd\u7167\u8a18\u9304\", \"reason_zh\": \"\u91ab\u5e2b\u9700\u8981\u8fa8\u8b58\u690d\u7269\u7a2e\u985e\u4ee5\u6c7a\u5b9a\u6cbb\u7642\u65b9\u6848\"}, {\"step\": 4, \"action_zh\": \"\u82e5\u610f\u8b58\u6e05\u9192\u4e14\u5728\u98df\u5f8c 1 \u5c0f\u6642\u5167\uff0c\u53ef\u8003\u616e\u50ac\u5410\uff08\u7528\u624b\u6307\u89f8\u78b0\u5589\u56a8\u5f8c\u65b9\uff09\", \"reason_zh\": \"\u6e1b\u5c11\u6bd2\u7d20\u5438\u6536\u3002\u4f46\u67d0\u4e9b\u60c5\u6cc1\u4e0d\u9069\u7528\uff0c\u898b\u4e0b\u65b9\u7981\u5fcc\u3002\"}, {\"step\": 5, \"action_zh\": \"\u88dc\u5145\u98f2\u6c34\uff08\u5c0f\u53e3\u6162\u559d\u6e05\u6c34\uff09\", \"reason_zh\": \"\u7a00\u91cb\u80c3\u4e2d\u6bd2\u7d20\"}, {\"step\": 6, \"action_zh\": \"\u5118\u901f\u9001\u91ab\u6216\u547c\u53eb\u6551\u63f4\", \"reason_zh\": \"\u90e8\u5206\u690d\u7269\u6bd2\u7d20\u6709\u5ef6\u9072\u6027\uff08\u5982\u9d5d\u818f\u6bd2\u7d20\uff09\uff0c\u521d\u671f\u7121\u75c7\u72c0\u4e0d\u4ee3\u8868\u5b89\u5168\"}], \"do_not_induce_vomiting_when\": [\"\u60a3\u8005\u610f\u8b58\u4e0d\u6e05\u6216\u660f\u8ff7\", \"\u60a3\u8005\u6709\u62bd\u6410\u73fe\u8c61\", \"\u98df\u5165\u8150\u8755\u6027\u690d\u7269\u6c41\u6db2\uff08\u5982\u59d1\u5a46\u828b\u2014\u2014\u50ac\u5410\u6703\u4e8c\u6b21\u707c\u50b7\u98df\u9053\uff09\", \"\u98df\u5f8c\u5df2\u8d85\u904e 2 \u5c0f\u6642\uff08\u6bd2\u7d20\u5df2\u9032\u5165\u8178\u9053\uff09\"], \"by_toxin_type\": {\"oxalate_crystal\": {\"name_zh\": \"\u8349\u9178\u9223\u91dd\u6676\u4e2d\u6bd2\uff08\u59d1\u5a46\u828b\u7b49\u5929\u5357\u661f\u79d1\uff09\", \"symptoms_zh\": \"\u53e3\u8154\u707c\u75db\u3001\u820c\u982d\u9ebb\u6728\u816b\u8139\u3001\u6d41\u6d8e\u3001\u5589\u56a8\u816b\u8139\u3001\u541e\u56a5\u56f0\u96e3\", \"first_aid_zh\": [\"\u52ff\u50ac\u5410\uff08\u907f\u514d\u4e8c\u6b21\u707c\u50b7\u98df\u9053\uff09\", \"\u5927\u91cf\u6e05\u6c34\u6f31\u53e3\", \"\u53ef\u542b\u51b0\u584a\u6216\u559d\u51b0\u725b\u5976\u7de9\u89e3\u53e3\u8154\u75bc\u75db\", \"\u6ce8\u610f\u547c\u5438\u9053\u2014\u2014\u5982\u5589\u56a8\u56b4\u91cd\u816b\u8139\u53ef\u80fd\u5f71\u97ff\u547c\u5438\", \"\u5118\u901f\u5c31\u91ab\"]}, \"cardiac_glycoside\": {\"name_zh\": \"\u5f37\u5fc3\u82f7\u4e2d\u6bd2\uff08\u593e\u7af9\u6843\u3001\u6d77\u6aac\u679c\uff09\", \"symptoms_zh\": \"\u5641\u5fc3\u5614\u5410\u3001\u5fc3\u5f8b\u4e0d\u6574\u3001\u5fc3\u8df3\u904e\u6162\u3001\u8996\u89ba\u7570\u5e38\uff08\u770b\u5230\u9ec3\u8272\u5149\u6688\uff09\", \"first_aid_zh\": [\"\u7acb\u5373\u50ac\u5410\uff08\u82e5\u610f\u8b58\u6e05\u9192\u4e14\u5728 1 \u5c0f\u6642\u5167\uff09\", \"\u5118\u901f\u9001\u91ab\u2014\u2014\u6b64\u70ba\u5fc3\u81df\u6025\u75c7\", \"\u9700\u6301\u7e8c\u5fc3\u96fb\u5716\u76e3\u6e2c\", \"\u544a\u77e5\u91ab\u5e2b\u7591\u4f3c\u5f37\u5fc3\u82f7\u4e2d\u6bd2\"]}, \"tropane_alkaloid\": {\"name_zh\": \"\u83a8\u83ea\u9e7c\u4e2d\u6bd2\uff08\u66fc\u9640\u7f85\uff09\", \"symptoms_zh\": \"\u77b3\u5b54\u653e\u5927\u3001\u53e3\u4e7e\u3001\u76ae\u819a\u6f6e\u7d05\u3001\u5fc3\u8df3\u52a0\u901f\u3001\u5e7b\u89ba\u3001\u8b6b\u5984\", \"first_aid_zh\": [\"\u50ac\u5410\uff08\u82e5\u610f\u8b58\u6e05\u9192\u4e14\u5728 1 \u5c0f\u6642\u5167\uff09\", \"\u4fdd\u6301\u60a3\u8005\u5728\u5b89\u5168\u74b0\u5883\u4e2d\uff08\u5e7b\u89ba\u53ef\u80fd\u5c0e\u81f4\u5371\u96aa\u884c\u70ba\uff09\", \"\u7dad\u6301\u547c\u5438\u9053\u66a2\u901a\", \"\u5118\u901f\u9001\u91ab\"]}, \"amatoxin\": {\"name_zh\": \"\u9d5d\u818f\u6bd2\u7d20\u4e2d\u6bd2\uff08\u9c57\u67c4\u767d\u6bd2\u9d5d\u818f\u7b49\uff09\", \"symptoms_zh\": \"\u26a0\ufe0f \u6f5b\u4f0f\u671f 6-24 \u5c0f\u6642\uff01\u521d\u671f\u5614\u5410\u8179\u7009 \u2192 \u5047\u6027\u6062\u5fa9 \u2192 \u809d\u814e\u8870\u7aed\", \"first_aid_zh\": [\"\u26a0\ufe0f \u98df\u7528\u4e0d\u660e\u83c7\u985e\u5f8c\u5373\u4f7f\u7121\u75c7\u72c0\u4e5f\u5fc5\u9808\u5c31\u91ab\", \"\u50ac\u5410\uff08\u82e5\u5728\u98df\u5f8c\u77ed\u6642\u9593\u5167\uff09\", \"\u4fdd\u7559\u83c7\u9ad4\u6a23\u672c\u9001\u91ab\", \"\u544a\u77e5\u91ab\u5e2b\u7591\u4f3c\u9d5d\u818f\u6bd2\u7d20\u4e2d\u6bd2\", \"\u6b64\u6bd2\u7d20\u7121\u7279\u6548\u89e3\u6bd2\u5291\uff0c\u9700\u7dca\u6025\u652f\u6301\u6027\u6cbb\u7642\", \"\u6642\u9593\u662f\u95dc\u9375\u2014\u2014\u8d8a\u65e9\u6cbb\u7642\u5b58\u6d3b\u7387\u8d8a\u9ad8\"]}}, \"skin_contact\": {\"title_zh\": \"\u76ae\u819a\u63a5\u89f8\u6709\u6bd2\u690d\u7269\u8655\u7406\", \"steps\": [\"\u7acb\u5373\u7528\u5927\u91cf\u6e05\u6c34\u6c96\u6d17\u63a5\u89f8\u90e8\u4f4d\uff0815 \u5206\u9418\u4ee5\u4e0a\uff09\", \"\u5982\u6709\u80a5\u7682\uff0c\u7528\u80a5\u7682\u6c34\u6e05\u6d17\uff08\u5c0d\u6f06\u6a39\u985e\u7279\u5225\u6709\u6548\uff0c\u9700\u5728 15 \u5206\u9418\u5167\uff09\", \"\u88ab\u712e\u523a\u6bdb\u523a\u50b7\uff08\u54ac\u4eba\u8c93/\u54ac\u4eba\u72d7\uff09\uff1a\u7528\u81a0\u5e36\u53cd\u8986\u9ecf\u9664\u76ae\u819a\u4e0a\u7684\u523a\u6bdb\", \"\u52ff\u6414\u6293\u60a3\u8655\", \"\u53ef\u5857\u62b9\u542b\u985e\u56fa\u9187\u7684\u6b62\u7662\u85e5\u818f\", \"\u56b4\u91cd\u816b\u8139\u6216\u904e\u654f\u53cd\u61c9\uff08\u547c\u5438\u56f0\u96e3\u3001\u5168\u8eab\u8541\u9ebb\u75b9\uff09\uff1a\u7acb\u5373\u5c31\u91ab\"]}}")

_KB_WOUND_CARE = json.loads("{\"scenario\": \"wound_care\", \"title_zh\": \"\u91ce\u5916\u50b7\u53e3\u8655\u7406 SOP\", \"wound_types\": {\"cut_abrasion\": {\"name_zh\": \"\u5207\u5272\u50b7 / \u64e6\u50b7\", \"steps\": [\"\u7528\u6e05\u6c34\uff08\u98f2\u7528\u6c34\uff09\u6c96\u6d17\u50b7\u53e3\uff0c\u53bb\u9664\u6ce5\u571f\u548c\u788e\u5c51\", \"\u5982\u6709\u51fa\u8840\uff0c\u7528\u4e7e\u6de8\u5e03\u6599\u52a0\u58d3\u6b62\u8840 10-15 \u5206\u9418\", \"\u50b7\u53e3\u4e7e\u6de8\u5f8c\uff0c\u53ef\u7528\u8eca\u524d\u8349\u8449\u6417\u788e\u6577\u5728\u50b7\u53e3\u4e0a\uff08\u5929\u7136\u6b62\u8840\u6d88\u708e\uff09\", \"\u7528\u4e7e\u6de8\u5e03\u6599\u6216\u5927\u578b\u8449\u7247\u5305\u7d2e\u50b7\u53e3\", \"\u6bcf 4-6 \u5c0f\u6642\u66f4\u63db\u6577\u6599\"], \"natural_remedies\": [{\"plant_id\": \"plantago_major\", \"name_zh\": \"\u8eca\u524d\u8349\", \"usage_zh\": \"\u6417\u788e\u65b0\u9bae\u8449\u7247\uff0c\u6577\u5728\u50b7\u53e3\u4e0a\uff0c\u6709\u6b62\u8840\u3001\u6d88\u708e\u3001\u6297\u83cc\u6548\u679c\", \"dosage_zh\": \"5-6 \u7247\u65b0\u9bae\u8449\u7247\u6417\u788e\u6210\u6ce5\uff0c\u6bcf 4-6 \u5c0f\u6642\u66f4\u63db\"}]}, \"burn\": {\"name_zh\": \"\u71d2\u71d9\u50b7\", \"steps\": [\"\u7acb\u5373\u7528\u5927\u91cf\u6e05\u6c34\u6c96\u6d17\u60a3\u8655\u81f3\u5c11 20 \u5206\u9418\", \"\u79fb\u9664\u60a3\u8655\u7684\u8863\u7269\u548c\u98fe\u54c1\uff08\u5982\u5c1a\u672a\u9ecf\u4f4f\u76ae\u819a\uff09\", \"\u52ff\u523a\u7834\u6c34\u6ce1\", \"\u7528\u4e7e\u6de8\u6fd5\u5e03\u8986\u84cb\u60a3\u8655\", \"\u56b4\u91cd\u71d2\u50b7\uff08\u5927\u9762\u7a4d\u3001\u4e09\u5ea6\u71d2\u50b7\uff09\u9700\u5118\u901f\u5c31\u91ab\"], \"natural_remedies\": []}, \"sprain\": {\"name_zh\": \"\u626d\u50b7\", \"steps\": [\"\u505c\u6b62\u6d3b\u52d5\uff0c\u4f11\u606f\uff08Rest\uff09\", \"\u5982\u6709\u6e05\u6f54\u51b7\u6c34\uff0c\u53ef\u51b7\u6577\u60a3\u8655 15-20 \u5206\u9418\uff08Ice\uff09\", \"\u7528\u5e03\u689d\u6216\u5f48\u6027\u6750\u6599\u5305\u7d2e\u56fa\u5b9a\uff08Compression\uff09\", \"\u76e1\u91cf\u62ac\u9ad8\u60a3\u80a2\uff08Elevation\uff09\", \"R.I.C.E. \u539f\u5247\"], \"natural_remedies\": []}, \"insect_sting\": {\"name_zh\": \"\u8702\u87ab / \u87f2\u54ac\", \"steps\": [\"\u9060\u96e2\u8702\u5de2\u5340\u57df\", \"\u5982\u6709\u87ab\u91dd\u6b98\u7559\uff0c\u7528\u786c\u7269\uff08\u5982\u4fe1\u7528\u5361\u908a\u7de3\uff09\u522e\u9664\uff0c\u52ff\u7528\u624b\u6307\u64e0\u58d3\", \"\u7528\u6e05\u6c34\u6c96\u6d17\u60a3\u8655\", \"\u51b7\u6577\u6e1b\u7de9\u816b\u8139\", \"\u26a0\ufe0f \u6ce8\u610f\u904e\u654f\u53cd\u61c9\uff1a\u547c\u5438\u56f0\u96e3\u3001\u5168\u8eab\u8541\u9ebb\u75b9\u3001\u982d\u6688 \u2192 \u53ef\u80fd\u662f\u904e\u654f\u6027\u4f11\u514b\uff0c\u7acb\u5373\u5c31\u91ab\"], \"natural_remedies\": [{\"plant_id\": \"plantago_major\", \"name_zh\": \"\u8eca\u524d\u8349\", \"usage_zh\": \"\u6417\u788e\u8449\u7247\u6577\u5728\u87f2\u54ac\u8655\uff0c\u53ef\u6e1b\u8f15\u816b\u8139\u548c\u6414\u7662\", \"dosage_zh\": \"\u6578\u7247\u8449\u7247\u6417\u788e\u6577\u4e0a\"}]}, \"dehydration\": {\"name_zh\": \"\u812b\u6c34\", \"steps\": [\"\u5c0b\u627e\u6c34\u6e90\uff08\u6eaa\u6c34\u3001\u96e8\u6c34\u3001\u690d\u7269\u5132\u6c34\uff09\", \"\u5982\u6709\u706b\u6e90\uff0c\u5c07\u6c34\u716e\u6cb8\u81f3\u5c11 1 \u5206\u9418\u518d\u98f2\u7528\", \"\u7121\u6cd5\u716e\u6cb8\u6642\uff0c\u512a\u5148\u9078\u64c7\u6d41\u52d5\u7684\u6eaa\u6c34\u800c\u975e\u975c\u6b62\u7684\u6c34\u5858\", \"\u814e\u8568\u7684\u5730\u4e0b\u5132\u6c34\u584a\u8396\u53ef\u63d0\u4f9b\u5c11\u91cf\u6c34\u5206\", \"\u5c0f\u53e3\u6162\u98f2\uff0c\u907f\u514d\u4e00\u6b21\u5927\u91cf\u98f2\u6c34\u5c0e\u81f4\u5614\u5410\"], \"natural_remedies\": [{\"plant_id\": \"nephrolepis_cordifolia\", \"name_zh\": \"\u814e\u8568\uff08\u5c71\u9cf3\u68a8\uff09\", \"usage_zh\": \"\u6316\u51fa\u5730\u4e0b\u7403\u5f62\u5132\u6c34\u584a\u8396\uff0c\u525d\u76ae\u5f8c\u53ef\u76f4\u63a5\u98df\u7528\uff0c\u542b\u6709\u6c34\u5206\", \"dosage_zh\": \"\u591a\u9846\u584a\u8396\u53ef\u63d0\u4f9b\u5c11\u91cf\u6c34\u5206\u61c9\u6025\"}]}}}")

_KB_BEE_STING = json.loads("{\"scenario\": \"bee_wasp_sting\", \"title_zh\": \"\u8702\u87ab\u7dca\u6025\u8655\u7406 SOP\", \"steps\": [{\"step\": 1, \"action_zh\": \"\u7acb\u5373\u9060\u96e2\u8702\u7fa4\u7bc4\u570d\uff08\u81f3\u5c11 50 \u516c\u5c3a\u4ee5\u4e0a\uff09\uff0c\u4e0d\u8981\u63ee\u624b\u9a45\u8d95\u3001\u4e0d\u8981\u5954\u8dd1\uff08\u7de9\u6b65\u96e2\u958b\uff09\", \"reason_zh\": \"\u8702\u985e\u6703\u91cb\u653e\u8b66\u5831\u8cbb\u6d1b\u8499\uff0c\u63ee\u624b\u548c\u5954\u8dd1\u6703\u5f15\u4f86\u66f4\u591a\u8702\u7fa4\u653b\u64ca\"}, {\"step\": 2, \"action_zh\": \"\u5982\u6709\u87ab\u91dd\u6b98\u7559\uff0c\u7528\u786c\u7269\uff08\u4fe1\u7528\u5361\u908a\u7de3\u3001\u6307\u7532\uff09\u5f9e\u5074\u9762\u522e\u9664\uff0c\u52ff\u7528\u624b\u6307\u6216\u9477\u5b50\u64e0\u58d3\", \"reason_zh\": \"\u871c\u8702\u87ab\u91dd\u5e36\u6709\u6bd2\u56ca\uff0c\u64e0\u58d3\u6703\u6ce8\u5165\u66f4\u591a\u6bd2\u6db2\u3002\u864e\u982d\u8702\u87ab\u91dd\u4e0d\u6b98\u7559\uff0c\u7121\u9700\u6b64\u6b65\u9a5f\u3002\"}, {\"step\": 3, \"action_zh\": \"\u7528\u6e05\u6c34\u6c96\u6d17\u50b7\u53e3\", \"reason_zh\": \"\u6e05\u6f54\u50b7\u53e3\u6e1b\u5c11\u611f\u67d3\u98a8\u96aa\"}, {\"step\": 4, \"action_zh\": \"\u51b0\u6577\u60a3\u8655 15-20 \u5206\u9418\uff0c\u6e1b\u7de9\u816b\u8139\u548c\u75bc\u75db\", \"reason_zh\": \"\u51b7\u6577\u53ef\u6e1b\u7de9\u6bd2\u6db2\u64f4\u6563\u548c\u816b\u8139\"}, {\"step\": 5, \"action_zh\": \"\u5bc6\u5207\u89c0\u5bdf\u662f\u5426\u51fa\u73fe\u904e\u654f\u53cd\u61c9\uff0830\u5206\u9418\u5167\uff09\uff1a\u5168\u8eab\u8541\u9ebb\u75b9\u3001\u547c\u5438\u56f0\u96e3\u3001\u982d\u6688\u3001\u81c9\u90e8\u6216\u5589\u56a8\u816b\u8139\", \"reason_zh\": \"\u904e\u654f\u6027\u4f11\u514b\uff08Anaphylaxis\uff09\u662f\u8702\u87ab\u6700\u81f4\u547d\u7684\u5a01\u8105\uff0c\u53ef\u5728\u6578\u5206\u9418\u5167\u81f4\u6b7b\"}, {\"step\": 6, \"action_zh\": \"\u5982\u6709\u904e\u654f\u53cd\u61c9\u6216\u88ab\u87ab\u8d85\u904e 10 \u8655\u4ee5\u4e0a\uff0c\u7acb\u5373\u547c\u53eb\u6551\u63f4\u6216\u9001\u91ab\", \"reason_zh\": \"\u591a\u8655\u87ab\u50b7\u5373\u4f7f\u7121\u904e\u654f\u4e5f\u53ef\u80fd\u56e0\u6bd2\u6db2\u7d2f\u7a4d\u5c0e\u81f4\u5668\u5b98\u8870\u7aed\"}], \"do_not\": [\"\u274c \u52ff\u7528\u624b\u6307\u64e0\u58d3\u87ab\u91dd\uff08\u6703\u6ce8\u5165\u66f4\u591a\u6bd2\u6db2\uff09\", \"\u274c \u52ff\u63ee\u624b\u9a45\u8d95\u8702\u7fa4\uff08\u6703\u5f15\u4f86\u66f4\u591a\u653b\u64ca\uff09\", \"\u274c \u52ff\u5728\u8702\u5de2\u9644\u8fd1\u5954\u8dd1\uff08\u8702\u985e\u6703\u8ffd\u9010\u79fb\u52d5\u76ee\u6a19\uff09\", \"\u274c \u52ff\u5857\u62b9\u963f\u6469\u5c3c\u4e9e\u3001\u5c3f\u6db2\u7b49\u504f\u65b9\", \"\u274c \u52ff\u5ffd\u8996\u904e\u654f\u53cd\u61c9\uff08\u5373\u4f7f\u662f\u55ae\u4e00\u87ab\u50b7\u4e5f\u53ef\u80fd\u81f4\u547d\uff09\"], \"by_species\": {\"honeybee\": {\"name_zh\": \"\u871c\u8702\u87ab\u50b7\", \"key_features_zh\": \"\u87ab\u91dd\u6703\u7559\u5728\u76ae\u819a\u4e2d\uff08\u5e36\u6bd2\u56ca\uff09\uff0c\u871c\u8702\u87ab\u5f8c\u6703\u6b7b\u4ea1\", \"special_action_zh\": \"\u7acb\u5373\u7528\u522e\u9664\u6cd5\u79fb\u9664\u87ab\u91dd\uff0c\u8d8a\u5feb\u8d8a\u597d\uff08\u6bcf\u79d2\u90fd\u5728\u6ce8\u5165\u6bd2\u6db2\uff09\"}, \"hornet\": {\"name_zh\": \"\u864e\u982d\u8702\u87ab\u50b7\", \"key_features_zh\": \"\u87ab\u91dd\u4e0d\u6b98\u7559\uff08\u53ef\u91cd\u8907\u87ab\u523a\uff09\uff0c\u6bd2\u6db2\u91cf\u9060\u5927\u65bc\u871c\u8702\", \"special_action_zh\": \"\u864e\u982d\u8702\u55ae\u6b21\u87ab\u50b7\u7684\u6bd2\u6db2\u91cf\u662f\u871c\u8702\u7684 5-10 \u500d\uff0c\u88ab\u87ab 3 \u8655\u4ee5\u4e0a\u5efa\u8b70\u5c31\u91ab\uff0c10 \u8655\u4ee5\u4e0a\u70ba\u6025\u91cd\u75c7\"}, \"paper_wasp\": {\"name_zh\": \"\u9577\u8173\u8702\u87ab\u50b7\", \"key_features_zh\": \"\u87ab\u91dd\u4e0d\u6b98\u7559\uff0c\u6bd2\u6027\u4ecb\u65bc\u871c\u8702\u548c\u864e\u982d\u8702\u4e4b\u9593\", \"special_action_zh\": \"\u901a\u5e38\u55ae\u7368\u6216\u5c11\u6578\u87ab\u50b7\uff0c\u51b0\u6577\u5373\u53ef\u3002\u4f46\u4ecd\u9700\u89c0\u5bdf\u904e\u654f\u53cd\u61c9\u3002\"}}, \"anaphylaxis_signs\": [\"\u5168\u8eab\u6027\u8541\u9ebb\u75b9\uff08\u4e0d\u53ea\u87ab\u50b7\u8655\uff0c\u5168\u8eab\u8d77\u7d05\u75b9\uff09\", \"\u81c9\u90e8\u3001\u5634\u5507\u3001\u820c\u982d\u6216\u5589\u56a8\u816b\u8139\", \"\u547c\u5438\u56f0\u96e3\u3001\u5598\u9cf4\u8072\", \"\u982d\u6688\u3001\u610f\u8b58\u6a21\u7cca\", \"\u8840\u58d3\u4e0b\u964d\u3001\u8108\u640f\u5fae\u5f31\", \"\u5641\u5fc3\u5614\u5410\u3001\u8179\u75db\"], \"anaphylaxis_action_zh\": \"\u26a0\ufe0f \u51fa\u73fe\u4e0a\u8ff0\u4efb\u4f55\u75c7\u72c0\uff0c\u9019\u662f\u904e\u654f\u6027\u4f11\u514b\uff0c\u9700\u7acb\u5373\u5c31\u91ab\uff01\u5982\u6709\u814e\u4e0a\u817a\u7d20\u81ea\u52d5\u6ce8\u5c04\u5668\uff08EpiPen\uff09\uff0c\u7acb\u5373\u4f7f\u7528\u3002\u8b93\u60a3\u8005\u5e73\u8eba\u3001\u62ac\u9ad8\u96d9\u817f\u3002\u4fdd\u6301\u547c\u5438\u9053\u66a2\u901a\u3002\", \"prevention\": [\"\u7a7f\u8457\u6dfa\u8272\u8863\u7269\uff08\u6df1\u8272\u548c\u82b1\u8272\u6703\u5438\u5f15\u8702\u985e\uff09\", \"\u907f\u514d\u4f7f\u7528\u9999\u6c34\u6216\u82b3\u9999\u7522\u54c1\", \"\u767c\u73fe\u8702\u5de2\u6642\u7e5e\u9053\u800c\u884c\uff0c\u4fdd\u6301\u81f3\u5c11 10 \u516c\u5c3a\u8ddd\u96e2\", \"\u98df\u7269\u548c\u542b\u7cd6\u98f2\u6599\u8981\u52a0\u84cb\", \"\u9047\u5230\u864e\u982d\u8702\u5de1\u908f\u8702\u6642\uff0c\u7de9\u6162\u8e72\u4e0b\u6216\u5f8c\u9000\uff0c\u52ff\u63ee\u624b\"]}")

_KB_ARTHROPOD_BITE = json.loads("{\"scenario\": \"arthropod_bite\", \"title_zh\": \"\u8708\u86a3\u3001\u8718\u86db\u53ca\u5176\u4ed6\u7bc0\u80a2\u52d5\u7269\u54ac\u50b7\uff0f\u53ee\u54ac\u8655\u7406 SOP\", \"categories\": {\"centipede\": {\"name_zh\": \"\u8708\u86a3\u54ac\u50b7\", \"steps\": [\"\u7528\u6e05\u6c34\u6c96\u6d17\u50b7\u53e3\", \"\u53ef\u7528\u5f31\u9e7c\u6027\u6eb6\u6db2\uff08\u5982\u80a5\u7682\u6c34\uff09\u6e05\u6d17\uff08\u8708\u86a3\u6bd2\u6db2\u504f\u9178\u6027\uff09\", \"\u51b0\u6577\u60a3\u8655 15-20 \u5206\u9418\", \"\u53ef\u670d\u7528\u4e00\u822c\u6b62\u75db\u85e5\u7de9\u89e3\u75bc\u75db\", \"\u89c0\u5bdf\u662f\u5426\u51fa\u73fe\u56b4\u91cd\u816b\u8139\u3001\u767c\u71d2\u3001\u6dcb\u5df4\u7d50\u816b\u5927\", \"\u75c7\u72c0\u6301\u7e8c\u8d85\u904e 24 \u5c0f\u6642\u6216\u52a0\u91cd\uff0c\u61c9\u5c31\u91ab\"], \"do_not\": [\"\u52ff\u7528\u5634\u5438\u6bd2\u6db2\", \"\u52ff\u5207\u958b\u50b7\u53e3\", \"\u52ff\u71b1\u6577\uff08\u6703\u52a0\u901f\u6bd2\u6db2\u64f4\u6563\uff09\"], \"symptoms_zh\": \"\u5287\u70c8\u75bc\u75db\uff08\u5982\u706b\u707c\u611f\uff09\u3001\u5c40\u90e8\u7d05\u816b\u3001\u53ef\u80fd\u6709\u5169\u500b\u5c0f\u7d05\u9ede\uff08\u54ac\u75d5\uff09\u3001\u56b4\u91cd\u8005\u767c\u71d2\u3001\u6dcb\u5df4\u7d50\u816b\u5927\"}, \"fire_ant\": {\"name_zh\": \"\u7d05\u706b\u87fb\u53ee\u54ac\", \"steps\": [\"\u96e2\u958b\u87fb\u4e18\u5340\u57df\uff0c\u6296\u843d\u8eab\u4e0a\u7684\u879e\u87fb\", \"\u7528\u6e05\u6c34\u6c96\u6d17\u53ee\u54ac\u8655\", \"\u51b0\u6577\u6e1b\u7de9\u816b\u8139\", \"\u53ee\u54ac\u8655 24 \u5c0f\u6642\u5f8c\u6703\u5f62\u6210\u767d\u8272\u81bf\u5305\uff08\u6b63\u5e38\u53cd\u61c9\uff09\", \"\u52ff\u64e0\u7834\u81bf\u5305\uff08\u5bb9\u6613\u611f\u67d3\uff09\", \"\u4fdd\u6301\u6e05\u6f54\u3001\u81ea\u7136\u7652\u5408\", \"\u591a\u8655\u53ee\u54ac\u6216\u6709\u904e\u654f\u53cd\u61c9\u8005\u61c9\u7acb\u5373\u5c31\u91ab\"], \"do_not\": [\"\u52ff\u64e0\u7834\u81bf\u5305\uff08\u611f\u67d3\u98a8\u96aa\uff09\", \"\u52ff\u6414\u6293\", \"\u52ff\u5ffd\u8996\u904e\u654f\u53cd\u61c9\"], \"symptoms_zh\": \"\u53ee\u54ac\u6642\u707c\u71b1\u75bc\u75db\u3001\u7d05\u816b\u30014-8 \u5c0f\u6642\u5f8c\u5f62\u6210\u6c34\u6ce1\u300124 \u5c0f\u6642\u5f8c\u8b8a\u6210\u767d\u8272\u81bf\u5305\u3001\u56b4\u91cd\u904e\u654f\u8005\u53ef\u80fd\u51fa\u73fe\u904e\u654f\u6027\u4f11\u514b\"}, \"tick\": {\"name_zh\": \"\u8731\u87f2\uff08\u58c1\u8768\uff09\u53ee\u54ac\", \"steps\": [\"\u767c\u73fe\u8731\u87f2\u6642\u4e0d\u8981\u614c\u5f35\uff0c\u8731\u87f2\u9700\u53ee\u54ac 24-48 \u5c0f\u6642\u624d\u6703\u50b3\u64ad\u75be\u75c5\", \"\u7528\u5c16\u982d\u9477\u5b50\u76e1\u91cf\u8cbc\u8fd1\u76ae\u819a\u593e\u4f4f\u8731\u87f2\u982d\u90e8\", \"\u7a69\u5b9a\u5782\u76f4\u5411\u4e0a\u62c9\u51fa\uff0c\u4e0d\u8981\u626d\u8f49\u6216\u6416\u6643\", \"\u5982\u982d\u90e8\u65b7\u5728\u76ae\u819a\u4e2d\uff0c\u7528\u6d88\u6bd2\u91dd\u6311\u51fa\u6216\u5c31\u91ab\u8655\u7406\", \"\u7528\u9152\u7cbe\u6216\u7898\u9152\u6d88\u6bd2\u50b7\u53e3\", \"\u62cd\u7167\u8a18\u9304\u8731\u87f2\u5916\u89c0\uff08\u4f9b\u91ab\u5e2b\u8fa8\u8b58\uff09\", \"\u63a5\u4e0b\u4f86 30 \u5929\u5167\u89c0\u5bdf\u662f\u5426\u51fa\u73fe\u74b0\u5f62\u7d05\u6591\uff08\u840a\u59c6\u75c5\u5fb5\u5146\uff09\u6216\u767c\u71d2\"], \"do_not\": [\"\u274c \u52ff\u7528\u624b\u6307\u76f4\u63a5\u62d4\uff08\u53ef\u80fd\u64e0\u51fa\u66f4\u591a\u75c5\u539f\u9ad4\uff09\", \"\u274c \u52ff\u7528\u51e1\u58eb\u6797\u3001\u9152\u7cbe\u3001\u706b\u70e4\u7b49\u65b9\u5f0f\u8a66\u5716\u903c\u8731\u87f2\u9b06\u53e3\", \"\u274c \u52ff\u626d\u8f49\u8731\u87f2\uff08\u982d\u90e8\u6703\u65b7\u88c2\u7559\u5728\u76ae\u819a\u4e2d\uff09\", \"\u274c \u52ff\u5ffd\u8996\u5f8c\u7e8c\u75c7\u72c0\uff08\u840a\u59c6\u75c5\u521d\u671f\u6613\u6cbb\u7642\u4f46\u5ef6\u8aa4\u53ef\u81f4\u6162\u6027\u75c5\uff09\"], \"symptoms_zh\": \"\u53ee\u54ac\u672c\u8eab\u901a\u5e38\u4e0d\u75db\uff08\u8731\u87f2\u5206\u6ccc\u9ebb\u9189\u7269\u8cea\uff09\u3002\u840a\u59c6\u75c5\u5fb5\u5146\uff1a\u53ee\u54ac\u8655\u51fa\u73fe\u9010\u6f38\u64f4\u5927\u7684\u74b0\u5f62\u7d05\u6591\uff08bull's eye rash\uff09\u3001\u767c\u71d2\u3001\u75b2\u5026\u3001\u95dc\u7bc0\u75db\u3002\"}, \"chigger_mite\": {\"name_zh\": \"\u6059\u87f2\uff08\u6059\u87ce\uff09\u53ee\u54ac\", \"steps\": [\"\u6059\u87f2\u53ee\u54ac\u901a\u5e38\u4e0d\u6703\u7acb\u5373\u767c\u73fe\uff08\u6975\u5c0f\u4e14\u4e0d\u75db\uff09\", \"\u5982\u767c\u73fe\u76ae\u819a\u4e0a\u6709\u7126\u75c2\uff08\u9ed1\u8272\u5c0f\u5713\u5f62\u6f70\u760d\uff09\u2192 \u9ad8\u5ea6\u7591\u4f3c\u6059\u87f2\u75c5\", \"\u7126\u75c2\u5e38\u51fa\u73fe\u5728\u814b\u4e0b\u3001\u8170\u90e8\u3001\u9f20\u8e4a\u90e8\u3001\u8033\u5f8c\u7b49\u76ae\u819a\u67d4\u8edf\u8655\", \"\u5982\u51fa\u73fe\u7126\u75c2 + \u767c\u9ad8\u71d2\uff087-14\u5929\u5f8c\uff09\uff0c\u7acb\u5373\u5c31\u91ab\", \"\u544a\u77e5\u91ab\u5e2b\u7591\u4f3c\u6059\u87f2\u75c5\uff0c\u9700\u4f7f\u7528\u56db\u74b0\u9ef4\u7d20\u985e\u6297\u751f\u7d20\u6cbb\u7642\", \"\u65e9\u671f\u6cbb\u7642\u6548\u679c\u6975\u597d\uff0c\u5ef6\u8aa4\u6cbb\u7642\u6b7b\u4ea1\u7387\u53ef\u9054 60%\"], \"do_not\": [\"\u274c \u52ff\u81ea\u884c\u670d\u7528\u6d88\u708e\u85e5\u800c\u5ef6\u8aa4\u5c31\u91ab\", \"\u274c \u52ff\u5ffd\u8996\u7126\u75c2\uff08\u9019\u662f\u6059\u87f2\u75c5\u6700\u95dc\u9375\u7684\u8a3a\u65b7\u7dda\u7d22\uff09\"], \"symptoms_zh\": \"\u6f5b\u4f0f\u671f 7-14 \u5929\u3002\u53ee\u54ac\u8655\u5f62\u6210\u7126\u75c2\uff08\u9ed1\u8272\u5713\u5f62\u6f70\u760d\uff0c\u76f4\u5f91 2-5mm\uff09\uff0c\u96a8\u5f8c\u9ad8\u71d2\u3001\u982d\u75db\u3001\u808c\u8089\u75db\u3001\u6dcb\u5df4\u7d50\u816b\u5927\u3002\", \"medical_urgency\": \"\u26a0\ufe0f \u6059\u87f2\u75c5\u662f\u6cd5\u5b9a\u50b3\u67d3\u75c5\uff0c\u672a\u6cbb\u7642\u6b7b\u4ea1\u7387\u53ef\u9054 60%\uff01\u767c\u73fe\u7126\u75c2+\u767c\u71d2\u5fc5\u9808\u7acb\u5373\u5c31\u91ab\u3002\"}, \"leech\": {\"name_zh\": \"\u6c34\u86ed\uff08\u879e\u8757\uff09\u5438\u8840\", \"steps\": [\"\u767c\u73fe\u6c34\u86ed\u5438\u9644\u6642\uff0c\u4e0d\u8981\u786c\u62d4\uff08\u6703\u6495\u88c2\u50b7\u53e3\u6216\u7559\u4e0b\u53e3\u5668\uff09\", \"\u5728\u6c34\u86ed\u5468\u570d\u6492\u9e7d\u3001\u6216\u7528\u6253\u706b\u6a5f\u9760\u8fd1\uff08\u4e0d\u662f\u76f4\u63a5\u71d2\uff09\u4f7f\u5176\u81ea\u884c\u812b\u843d\", \"\u4e5f\u53ef\u4ee5\u7528\u6307\u7532\u5f9e\u6c34\u86ed\u7684\u53e3\u7aef\uff08\u8f03\u7a84\u7684\u4e00\u7aef\uff09\u5283\u904e\u4f7f\u5176\u9b06\u53e3\", \"\u6c34\u86ed\u812b\u843d\u5f8c\uff0c\u6e05\u6c34\u6c96\u6d17\u50b7\u53e3\", \"\u52a0\u58d3\u6b62\u8840\uff08\u6c34\u86ed\u5206\u6ccc\u7684\u6297\u51dd\u8840\u7269\u8cea\u6703\u8b93\u50b7\u53e3\u6301\u7e8c\u6d41\u8840 30-60 \u5206\u9418\uff09\", \"\u4fdd\u6301\u50b7\u53e3\u6e05\u6f54\uff0c\u89c0\u5bdf\u662f\u5426\u611f\u67d3\"], \"do_not\": [\"\u274c \u52ff\u786c\u62d4\u6c34\u86ed\uff08\u6703\u6495\u88c2\u50b7\u53e3\uff09\", \"\u274c \u52ff\u7528\u706b\u76f4\u63a5\u71d2\u6c34\u86ed\uff08\u53ef\u80fd\u71d9\u50b7\u81ea\u5df1\uff09\", \"\u274c \u52ff\u6050\u614c\uff08\u6c34\u86ed\u5438\u8840\u672c\u8eab\u4e0d\u5371\u96aa\uff0c\u4e0d\u50b3\u64ad\u75be\u75c5\uff09\"], \"symptoms_zh\": \"\u901a\u5e38\u4e0d\u75db\uff08\u6c34\u86ed\u5206\u6ccc\u9ebb\u9189\u7269\u8cea\uff09\u3002\u50b7\u53e3\u6301\u7e8c\u6d41\u8840\uff08\u6297\u51dd\u8840\u7269\u8cea\u4f5c\u7528\uff09\u3002\u8f15\u5fae\u6414\u7662\u3002\u6975\u5c11\u6578\u4eba\u53ef\u80fd\u904e\u654f\u3002\"}}, \"prevention_general\": [\"\u767b\u5c71\u7a7f\u9577\u8932\u3001\u9577\u8896\uff0c\u8932\u7ba1\u7d2e\u5165\u896a\u5b50\u4e2d\", \"\u4f7f\u7528\u542b DEET \u6216 Picaridin \u7684\u9a45\u87f2\u5291\", \"\u907f\u514d\u5728\u8349\u53e2\u4e2d\u4e45\u5750\u6216\u8eba\u81e5\", \"\u884c\u9032\u4e2d\u8d70\u6b65\u9053\u4e2d\u592e\uff0c\u907f\u514d\u89f8\u78b0\u5169\u65c1\u8349\u53e2\", \"\u56de\u5bb6\u5f8c\u5168\u8eab\u6aa2\u67e5\u662f\u5426\u6709\u8731\u87f2\u9644\u8457\", \"\u96e8\u5f8c\u6216\u6f6e\u6fd5\u5929\u6c23\u7279\u5225\u6ce8\u610f\u6c34\u86ed\"]}")

print(f"✅ 知識庫載入完成：")
print(f"   有毒植物：{len(_KB_TOXIC_PLANTS)} 種")
print(f"   可食植物：{len(_KB_EDIBLE_PLANTS)} 種")
print(f"   危險動物：{len(_KB_DANGEROUS_ANIMALS)} 種")
print(f"   混淆物種對：{len(_KB_CONFUSION_PAIRS)} 組")
print(f"   急救 SOP：5 份")



## 🧠 核心辨識模組



In [ ]:
"""
環境上下文引擎 — 根據使用者的位置/環境資訊載入對應的知識庫。
"""

from dataclasses import dataclass, field
from datetime import datetime



@dataclass
class EnvironmentContext:
    """使用者當前的環境上下文"""
    region_id: str = DEFAULT_REGION
    country: str = "台灣"
    climate_zone: str = "亞熱帶"
    altitude: int = 500
    vegetation_zone: str = "低海拔闊葉林"
    season: str = ""
    month: int = 0

    def __post_init__(self):
        if not self.month:
            self.month = datetime.now().month
        if not self.season:
            self.season = self._month_to_season(self.month)

    @staticmethod
    def _month_to_season(month: int) -> str:
        if month in (3, 4, 5):
            return "春季"
        elif month in (6, 7, 8):
            return "夏季"
        elif month in (9, 10, 11):
            return "秋季"
        else:
            return "冬季"

    def to_prompt_header(self) -> str:
        return (
            f"【環境上下文】\n"
            f"地點：{self.country}\n"
            f"氣候帶：{self.climate_zone}\n"
            f"海拔：約 {self.altitude}m\n"
            f"植被帶：{self.vegetation_zone}\n"
            f"季節：{self.season}"
        )


@dataclass
class KnowledgeBase:
    """載入的區域知識庫"""
    toxic_plants: list = field(default_factory=list)
    confusion_pairs: list = field(default_factory=list)
    edible_plants: list = field(default_factory=list)
    dangerous_animals: list = field(default_factory=list)
    snakebite_first_aid: dict = field(default_factory=dict)
    plant_poisoning_first_aid: dict = field(default_factory=dict)
    wound_care: dict = field(default_factory=dict)


def load_knowledge_base(region_id: str = DEFAULT_REGION) -> KnowledgeBase:
    """直接從嵌入的知識庫資料載入"""
    return KnowledgeBase(
        toxic_plants=_KB_TOXIC_PLANTS,
        confusion_pairs=_KB_CONFUSION_PAIRS,
        edible_plants=_KB_EDIBLE_PLANTS,
        dangerous_animals=_KB_DANGEROUS_ANIMALS,
        snakebite_first_aid=_KB_SNAKEBITE_FIRST_AID,
        plant_poisoning_first_aid=_KB_PLANT_POISONING,
        wound_care=_KB_WOUND_CARE,
    )


def create_context(
    country: str = "台灣",
    altitude: int = 500,
    climate_zone: str = "亞熱帶",
    vegetation_zone: str = "低海拔闊葉林",
    region_id: str = DEFAULT_REGION,
) -> EnvironmentContext:
    """建立環境上下文"""
    return EnvironmentContext(
        region_id=region_id,
        country=country,
        climate_zone=climate_zone,
        altitude=altitude,
        vegetation_zone=vegetation_zone,
    )


def get_vegetation_zone(altitude: int) -> str:
    """根據海拔推算植被帶（台灣）"""
    if altitude < 500:
        return "低海拔闊葉林"
    elif altitude < 1500:
        return "中海拔闊葉林"
    elif altitude < 2500:
        return "中高海拔針闊混合林"
    elif altitude < 3500:
        return "高海拔針葉林"
    else:
        return "高山草原"



In [ ]:
"""
動態 Prompt 組裝器 — 兩階段辨識架構。

Stage 1: 危險物種比對 → 取信心度最高的 3 個
Stage 2: 可食用/藥用物種比對 → 取信心度最高的 3 個

信心度 = 絕對相似度（多少項形態特徵吻合），不是排名。
"""



# ═══════════════════════════════════════════════════════════════
# 內部格式化工具
# ═══════════════════════════════════════════════════════════════

def _format_toxic_plant(plant: dict) -> str:
    morph = plant.get("morphology", {})
    lines = [f"{plant['common_names']['zh-TW']} ({plant['scientific_name']})"]
    lines.append(f"   毒性：{plant['toxicity']}")

    field_map = {
        "leaf_shape": "葉形", "leaf_surface": "葉面", "leaf_venation": "葉脈",
        "leaf_tip": "葉尖", "petiole": "葉柄", "stem": "莖部",
        "flower": "花", "fruit": "果實",
        "leaf_arrangement": "葉排列",
        "cap": "傘蓋", "gills": "菌褶", "stipe": "菌柄",
        "spore_print": "孢子印", "ring": "菌環", "flesh": "菌肉",
    }
    for key, label in field_map.items():
        if key in morph:
            lines.append(f"   {label}：{morph[key]}")

    lines.append(f"   生長環境：{plant.get('habitat', 'N/A')}")

    if plant.get("confusion_with"):
        names = ", ".join(plant["confusion_with"])
        lines.append(f"   易混淆：{names}")

    return "\n".join(lines)


def _format_dangerous_animal(animal: dict) -> str:
    morph = animal.get("morphology", {})
    lines = [f"{animal['common_names']['zh-TW']} ({animal['scientific_name']})"]
    lines.append(f"   毒性：{animal.get('venom_type', 'N/A')}")

    field_map = {
        "body_color": "體色", "tail": "尾部", "head_shape": "頭形",
        "pupil": "瞳孔", "pit_organ": "頰窩", "body_size": "體型",
        "scales": "鱗片", "hood": "頸部", "dorsal_scales": "背鱗",
    }
    for key, label in field_map.items():
        if key in morph:
            lines.append(f"   {label}：{morph[key]}")

    lines.append(f"   棲地：{animal.get('habitat', 'N/A')}")
    lines.append(f"   行為：{animal.get('behavior', 'N/A')}")

    return "\n".join(lines)


def _format_edible_plant(plant: dict) -> str:
    morph = plant.get("morphology", {})
    lines = [f"{plant['common_names']['zh-TW']} ({plant['scientific_name']})"]
    lines.append(f"   食用部位：{', '.join(plant.get('edible_parts', []))}")

    for key, label in [
        ("leaf_shape", "葉形"), ("leaf_arrangement", "葉排列"),
        ("leaf_surface", "葉面"), ("flower", "花"), ("stem", "莖"),
        ("midrib", "中肋"), ("growth_pattern", "生長方式"),
        ("young_shoot", "嫩芽"), ("shape", "外形"), ("fruit", "果實"),
        ("underground", "地下部"), ("smell", "氣味"),
    ]:
        if key in morph:
            lines.append(f"   {label}：{morph[key]}")

    lines.append(f"   生長環境：{plant.get('habitat', 'N/A')}")

    harvesting = plant.get("harvesting", {})
    if harvesting:
        lines.append(f"   採集方式：{harvesting.get('method', 'N/A')}")

    prep = plant.get("preparation", {})
    if prep:
        lines.append(f"   食用方式：{prep.get('method', 'N/A')}")

    if plant.get("caution"):
        lines.append(f"   ⚠️ 注意：{plant['caution']}")

    if plant.get("medicinal_uses"):
        lines.append(f"   💊 藥用：{plant['medicinal_uses']}")

    return "\n".join(lines)


# ═══════════════════════════════════════════════════════════════
# Stage 1：危險物種比對 — 輸出 top 3
# ═══════════════════════════════════════════════════════════════

_CONFIDENCE_INSTRUCTION = """
IMPORTANT — How to calculate "confidence":
Confidence is an ABSOLUTE measure of visual similarity, NOT a ranking.
Count how many morphological features from the reference data match
what you observe in the photo. The more features that match, the higher
the confidence (0-100).
- 0%: No features match at all
- 30%: Only general shape/color roughly matches
- 50%: ~Half of listed features match
- 70%: Most key features match, minor differences remain
- 85%: Nearly all features match, very strong identification
- 95-100%: Every listed feature matches perfectly
If a feature cannot be verified from the photo, do NOT count it as matching.
"""


def build_danger_prompt(
    context: EnvironmentContext,
    kb: KnowledgeBase,
    target_type: str = "plant",
) -> str:
    """Stage 1：比對危險物種，輸出信心度最高的 3 個候選。"""
    header = context.to_prompt_header()

    if target_type == "plant":
        role = "a toxicology expert specializing in field identification of dangerous plants and fungi"
        target_word = "plant/fungus"
        species_list = "\n\n".join(
            f"{i+1}. {_format_toxic_plant(p)}"
            for i, p in enumerate(kb.toxic_plants)
        )
    else:
        role = "a herpetology and zoology expert specializing in identification of dangerous animals"
        target_word = "animal/snake"
        species_list = "\n\n".join(
            f"{i+1}. {_format_dangerous_animal(a)}"
            for i, a in enumerate(kb.dangerous_animals)
        )

    prompt = f"""{header}

You are {role}.
Analyze the {target_word} in the provided photo(s).

{_CONFIDENCE_INSTRUCTION}

Follow these steps:

Step 1: Carefully observe the photo(s). Describe ALL morphological features you can see.

Step 2: Compare your observations against EACH species in the list below.
For each species, count how many listed morphological features match the photo.

Step 3: Rank all species by confidence (absolute feature match count → percentage).
Select the TOP 3 with the highest confidence.

Step 4: For features you cannot determine from the photo, explicitly list them.

=== DANGEROUS SPECIES DATABASE ===

{species_list}

After completing Steps 1-4, output a JSON summary:

{JSON_START_MARKER}
{{
  "reasoning_summary": "用繁體中文簡述分析過程（描述你觀察到什麼、與哪些特徵吻合）",
  "observed_features": ["觀察到的特徵1", "觀察到的特徵2", "..."],
  "candidates": [
    {{
      "rank": 1,
      "common_name_zh": "中文名",
      "common_name_en": "English name",
      "scientific_name": "學名",
      "confidence": 85,
      "category": "dangerous",
      "key_matching_features": ["吻合特徵1", "吻合特徵2"],
      "danger_info": {{
        "toxicity": "毒性描述",
        "symptoms": ["中毒症狀"],
        "first_aid": "急救方式"
      }}
    }},
    {{
      "rank": 2,
      "common_name_zh": "...",
      "common_name_en": "...",
      "scientific_name": "...",
      "confidence": 45,
      "category": "dangerous",
      "key_matching_features": ["..."],
      "danger_info": {{ "toxicity": "...", "symptoms": [], "first_aid": "..." }}
    }},
    {{
      "rank": 3,
      "common_name_zh": "...",
      "common_name_en": "...",
      "scientific_name": "...",
      "confidence": 20,
      "category": "dangerous",
      "key_matching_features": ["..."],
      "danger_info": {{ "toxicity": "...", "symptoms": [], "first_aid": "..." }}
    }}
  ]
}}
{JSON_END_MARKER}

Rules:
- "confidence": integer 0-100, absolute feature-matching score
- "category": always "dangerous" in this stage
- Output exactly 3 candidates, even if confidence is low
- All Chinese text in Traditional Chinese (繁體中文)

First write your full analysis in Traditional Chinese, THEN output the JSON.
請用繁體中文回覆。"""

    return prompt


# ═══════════════════════════════════════════════════════════════
# Stage 2：可食用/藥用物種比對 — 輸出 top 3
# ═══════════════════════════════════════════════════════════════

def build_edible_prompt(
    context: EnvironmentContext,
    kb: KnowledgeBase,
) -> str:
    """Stage 2：比對可食用/藥用物種，輸出信心度最高的 3 個候選。"""
    header = context.to_prompt_header()

    edible_list = "\n\n".join(
        f"{i+1}. {_format_edible_plant(p)}"
        for i, p in enumerate(kb.edible_plants)
    )

    prompt = f"""{header}

You are a field survival botanist helping identify edible and medicinal plants.
Analyze the plant in the provided photo(s).

{_CONFIDENCE_INSTRUCTION}

Follow these steps:

Step 1: Carefully observe the photo(s). Describe ALL morphological features you can see.

Step 2: Compare your observations against EACH edible/medicinal species in the list below.
For each species, count how many listed morphological features match the photo.

Step 3: Rank all species by confidence (absolute feature match count → percentage).
Select the TOP 3 with the highest confidence.

Step 4: For features you cannot determine from the photo, explicitly list them.

=== EDIBLE / MEDICINAL SPECIES DATABASE ===

{edible_list}

After completing Steps 1-4, output a JSON summary:

{JSON_START_MARKER}
{{
  "reasoning_summary": "用繁體中文簡述分析過程",
  "observed_features": ["觀察到的特徵1", "觀察到的特徵2", "..."],
  "candidates": [
    {{
      "rank": 1,
      "common_name_zh": "中文名",
      "common_name_en": "English name",
      "scientific_name": "學名",
      "confidence": 85,
      "category": "edible",
      "key_matching_features": ["吻合特徵1", "吻合特徵2"],
      "danger_info": null
    }},
    {{
      "rank": 2,
      "common_name_zh": "...",
      "common_name_en": "...",
      "scientific_name": "...",
      "confidence": 45,
      "category": "edible",
      "key_matching_features": ["..."],
      "danger_info": null
    }},
    {{
      "rank": 3,
      "common_name_zh": "...",
      "common_name_en": "...",
      "scientific_name": "...",
      "confidence": 20,
      "category": "edible",
      "key_matching_features": ["..."],
      "danger_info": null
    }}
  ]
}}
{JSON_END_MARKER}

Rules:
- "confidence": integer 0-100, absolute feature-matching score
- "category": "edible" or "medicinal" (choose based on primary use)
- Output exactly 3 candidates, even if confidence is low
- All Chinese text in Traditional Chinese (繁體中文)

First write your full analysis in Traditional Chinese, THEN output the JSON.
請用繁體中文回覆。"""

    return prompt


# ═══════════════════════════════════════════════════════════════
# 混淆物種鑑別（保留原有邏輯）
# ═══════════════════════════════════════════════════════════════

def build_confusion_pairs_prompt(
    context: EnvironmentContext,
    pair: dict,
) -> str:
    """建構混淆物種專項鑑別 Prompt。"""
    header = context.to_prompt_header()
    threshold = THRESHOLDS.get("confusion_pairs", 80)

    safe = pair["safe_species"]
    danger = pair["dangerous_species"]

    safe_features = "\n".join(
        f"- {k}：{v}" for k, v in safe["key_features"].items()
    )
    danger_features = "\n".join(
        f"- {k}：{v}" for k, v in danger["key_features"].items()
    )

    comparison_rows = ""
    for row in pair.get("comparison_table", []):
        comparison_rows += f"| {row['feature']} | {row['safe']} | {row['danger']} |\n"

    prompt = f"""{header}

你是一位植物學家，專精於野外求生中的危險植物鑑別。
這張照片中的植物可能是以下兩種之一，它們外觀非常相似，
但一個可食用，一個有毒。誤判可能致命。

【可食用】{safe['common_name_zh']} ({safe['scientific_name']})
{safe_features}

【有毒】{danger['common_name_zh']} ({danger['scientific_name']})
{danger_features}

【關鍵鑑別對照表】
| 特徵 | {safe['common_name_zh']}（可食） | {danger['common_name_zh']}（有毒） |
|------|------|------|
{comparison_rows}

請根據照片中可辨識的特徵，逐項比對上表，判斷：
1. 你觀察到哪些特徵？逐一描述
2. 每個特徵更符合{safe['common_name_zh']}還是{danger['common_name_zh']}？
3. 有哪些特徵無法從照片中判斷？
4. 最終判斷：更可能是哪一種？信心度（0-100%）？

⚠️ 重要規則：
- 如果信心度 < {threshold}%，請標注「無法確定，視為有毒，禁止食用」
- 如果任何關鍵鑑別特徵無法從照片確認，也請標注「無法確定，視為有毒」
- 在求生情境中，誤判比不判更危險

After your analysis, output a JSON summary:

{JSON_START_MARKER}
{{
  "confusion_pair_id": "{pair['id']}",
  "judgment": "safe" or "dangerous" or "uncertain",
  "confidence": 0,
  "safe_species": "{safe['common_name_zh']}",
  "dangerous_species": "{danger['common_name_zh']}",
  "feature_comparison": [
    {{
      "feature": "特徵名",
      "observation": "照片中觀察到的情況",
      "matches": "safe" or "dangerous" or "uncertain"
    }}
  ],
  "unobservable_features": ["無法判斷的特徵"],
  "requires_interactive_test": true,
  "recommended_tests": ["建議的互動測試"],
  "final_message_zh": "給使用者的最終建議（繁體中文）"
}}
{JSON_END_MARKER}

請用繁體中文回覆。"""

    return prompt


def build_interactive_test_guidance(pair: dict) -> str:
    """混淆鑑別無法確定時，引導使用者做實地測試。"""
    safe = pair["safe_species"]
    danger = pair["dangerous_species"]
    tests = pair.get("interactive_tests", [])

    if not tests:
        return f"⚠️ 此植物可能是{safe['common_name_zh']}或{danger['common_name_zh']}，無法確定，請勿食用。"

    lines = [
        f"⚠️ 此植物可能是{safe['common_name_zh']}（可食）或{danger['common_name_zh']}（有毒），",
        "僅靠照片無法區分。請執行以下測試：",
        "",
    ]

    for test in sorted(tests, key=lambda t: t.get("priority", 99)):
        critical = "（最關鍵！）" if test.get("is_critical") else ""
        lines.append(f"🔬 {test['test_name']}{critical}：")
        lines.append(f"   {test['instruction_zh']}")
        lines.append(f"   → {test['safe_result']}")
        lines.append(f"   → {test['danger_result']}")
        lines.append("")

    lines.extend([
        "📷 完成測試後，請拍攝測試結果照片上傳，我將結合新資訊重新判斷。",
        f"⚠️ 在完成測試前，請勿食用此植物。",
    ])

    return "\n".join(lines)


# ═══════════════════════════════════════════════════════════════
# 多照片包裝器
# ═══════════════════════════════════════════════════════════════

def build_multi_photo_wrapper(base_prompt: str, num_photos: int) -> str:
    """將任何 Prompt 包裝成多照片版本（多角度可提升信心度）。"""
    photo_instructions = (
        f"I am providing {num_photos} photos of the SAME subject taken from different angles.\n"
        "Analyze ALL photos together to improve identification accuracy.\n"
        "For Step 1, describe features observed in EACH photo separately,\n"
        "then note which features are consistently observed across photos (higher reliability).\n\n"
    )

    return base_prompt.replace(
        "Step 1: Carefully observe the photo(s).",
        photo_instructions + "Step 1: For EACH photo, describe the morphological features you observe.",
    )



In [ ]:
"""
回應解析器 — 解析 AI 模型的 JSON 回覆，提取結構化辨識結果。

支援：
- <JSON_START>/<JSON_END> 標記擷取
- ```json codeblock 擷取（含不完整）
- 備用 { } 擷取
- 截斷修復（補齊括號）
"""

import json
import re
from dataclasses import dataclass, field
from typing import Optional



@dataclass
class CandidateMatch:
    """單一候選物種"""
    rank: int
    common_name_zh: str
    common_name_en: str
    scientific_name: str
    confidence: int
    category: str  # "dangerous", "edible", "medicinal"
    key_matching_features: list[str] = field(default_factory=list)
    danger_info: Optional[dict] = None


@dataclass
class UnifiedResult:
    """單次 AI 呼叫的解析結果"""
    raw_response: str
    json_data: Optional[dict]
    parse_success: bool
    parse_error: Optional[str] = None
    candidates: list[CandidateMatch] = field(default_factory=list)
    observed_features: list[str] = field(default_factory=list)


@dataclass
class ConfusionResult:
    """混淆鑑別結果"""
    raw_response: str
    json_data: Optional[dict]
    parse_success: bool
    parse_error: Optional[str] = None

    @property
    def is_safe(self) -> bool:
        if not self.json_data:
            return False
        return self.json_data.get("judgment") == "safe"

    @property
    def confidence(self) -> int:
        if not self.json_data:
            return 0
        return self.json_data.get("confidence", 0)

    @property
    def final_message(self) -> str:
        if not self.json_data:
            return ""
        return self.json_data.get("final_message_zh", "")


# ═══════════════════════════════════════════════════════════════
# JSON 擷取與修復
# ═══════════════════════════════════════════════════════════════

def _extract_json_from_markers(text: str) -> Optional[str]:
    if JSON_START_MARKER in text and JSON_END_MARKER in text:
        return text.split(JSON_START_MARKER, 1)[1].split(JSON_END_MARKER, 1)[0].strip()
    if JSON_START_MARKER in text:
        return text.split(JSON_START_MARKER, 1)[1].strip()
    return None


def _extract_json_from_codeblock(text: str) -> Optional[str]:
    m = re.search(r'```(?:json)?\s*\n?([\s\S]*?)```', text)
    if m:
        return m.group(1).strip()
    m = re.search(r'```(?:json)?\s*\n?([\s\S]+)', text)
    if m:
        return m.group(1).strip()
    return None


def _extract_json_fallback(text: str) -> Optional[str]:
    start = text.find('{')
    if start < 0:
        return None
    candidate = text[start:]
    end = candidate.rfind('}')
    if end >= 0:
        return candidate[:end + 1].strip()
    return candidate.strip()


def _repair_truncated_json(json_str: str) -> Optional[str]:
    last_complete = -1
    for m in re.finditer(r',\s*"[^"]+"\s*:', json_str):
        last_complete = m.start()

    if last_complete > 0:
        candidate = json_str[:last_complete]
        result = _close_json(candidate)
        if result:
            return result

    for marker in ['},', '}]', '}"', '],']:
        pos = json_str.rfind(marker)
        if pos > 0:
            candidate = json_str[:pos + 1]
            result = _close_json(candidate)
            if result:
                return result

    for ch in ['}', ']']:
        pos = json_str.rfind(ch)
        if pos > 0:
            candidate = json_str[:pos + 1]
            result = _close_json(candidate)
            if result:
                return result

    return _close_json(json_str)


def _close_json(candidate: str) -> Optional[str]:
    candidate = candidate.rstrip()
    while candidate.endswith(',') or candidate.endswith(':'):
        candidate = candidate[:-1].rstrip()

    opens = candidate.count('{') - candidate.count('}')
    brackets = candidate.count('[') - candidate.count(']')

    if opens <= 0 and brackets <= 0:
        return None

    candidate += ']' * max(0, brackets) + '}' * max(0, opens)
    return candidate


def _try_parse_json(json_str: str) -> tuple[Optional[dict], Optional[str]]:
    clean = json_str.strip()
    if clean.startswith("```"):
        clean = clean.split("\n", 1)[1] if "\n" in clean else clean[3:]
        clean = clean.rsplit("```", 1)[0]
    clean = clean.strip()

    try:
        return json.loads(clean), None
    except json.JSONDecodeError:
        pass

    repaired = _repair_truncated_json(clean)
    if repaired:
        try:
            return json.loads(repaired), None
        except json.JSONDecodeError:
            pass
        clean2 = re.sub(r',\s*([}\]])', r'\1', repaired)
        try:
            return json.loads(clean2), None
        except json.JSONDecodeError:
            pass

    clean3 = re.sub(r',\s*([}\]])', r'\1', clean)
    try:
        return json.loads(clean3), None
    except json.JSONDecodeError:
        pass

    for _ in range(5):
        truncated = _repair_truncated_json(clean)
        if truncated:
            truncated2 = re.sub(r',\s*([}\]])', r'\1', truncated)
            try:
                return json.loads(truncated2), None
            except json.JSONDecodeError:
                last_comma = truncated.rstrip('}]').rstrip().rfind(',')
                if last_comma > 0:
                    clean = truncated[:last_comma]
                else:
                    break
        else:
            break

    return None, "JSON 解析失敗"


# ═══════════════════════════════════════════════════════════════
# 主要解析函式
# ═══════════════════════════════════════════════════════════════

def parse_unified_response(response: str) -> UnifiedResult:
    """解析 AI 回覆，提取 candidates[] 列表。"""
    last_error = None
    for extractor in [_extract_json_from_markers, _extract_json_from_codeblock, _extract_json_fallback]:
        json_str = extractor(response)
        if json_str:
            data, error = _try_parse_json(json_str)
            if data is not None:
                candidates = _extract_candidates(data)
                return UnifiedResult(
                    raw_response=response,
                    json_data=data,
                    parse_success=True,
                    candidates=candidates,
                    observed_features=data.get("observed_features", []),
                )
            last_error = error

    return UnifiedResult(
        raw_response=response,
        json_data=None,
        parse_success=False,
        parse_error=last_error or "找不到可解析的 JSON",
    )


def _extract_candidates(data: dict) -> list[CandidateMatch]:
    raw_candidates = data.get("candidates", [])
    results = []
    for c in raw_candidates:
        if not isinstance(c, dict):
            continue
        results.append(CandidateMatch(
            rank=c.get("rank", len(results) + 1),
            common_name_zh=c.get("common_name_zh", "未知"),
            common_name_en=c.get("common_name_en", "Unknown"),
            scientific_name=c.get("scientific_name", ""),
            confidence=c.get("confidence", 0),
            category=c.get("category", "unknown"),
            key_matching_features=c.get("key_matching_features", []),
            danger_info=c.get("danger_info"),
        ))
    results.sort(key=lambda x: x.confidence, reverse=True)
    return results


def parse_confusion_response(response: str) -> ConfusionResult:
    """解析混淆物種鑑別回覆"""
    for extractor in [_extract_json_from_markers, _extract_json_from_codeblock, _extract_json_fallback]:
        json_str = extractor(response)
        if json_str:
            data, error = _try_parse_json(json_str)
            if data is not None:
                return ConfusionResult(
                    raw_response=response,
                    json_data=data,
                    parse_success=True,
                )

    return ConfusionResult(
        raw_response=response,
        json_data=None,
        parse_success=False,
        parse_error="找不到可解析的 JSON",
    )



In [ ]:
class GemmaModel:
    """Transformers 模式 — Kaggle 或有 GPU 的環境"""

    def __init__(self, model_id: str = MODEL_ID, device: str = "auto"):
        import torch
        from transformers import AutoProcessor, AutoModelForImageTextToText

        self.model_id = model_id
        self.processor = AutoProcessor.from_pretrained(model_id)

        device_map = device
        if device == "auto" and torch.cuda.device_count() > 1:
            device_map = {"": "cuda:0"}

        dtype = getattr(torch, DEFAULT_DTYPE, torch.bfloat16)
        self.model = AutoModelForImageTextToText.from_pretrained(
            model_id,
            torch_dtype=dtype,
            device_map=device_map,
        )
        self._device = self.model.device

    def generate(
        self,
        prompt: str,
        images: Optional[list[Image.Image]] = None,
        max_new_tokens: int = MAX_NEW_TOKENS,
    ) -> str:
        """
        執行多模態推理。

        Args:
            prompt: 文字 Prompt
            images: PIL Image 列表（可選）
            max_new_tokens: 最大生成 token 數

        Returns:
            模型的文字回覆
        """
        content = []

        if images:
            for img in images:
                content.append({"type": "image", "image": img})

        content.append({"type": "text", "text": prompt})

        messages = [{"role": "user", "content": content}]

        inputs = self.processor.apply_chat_template(
            messages,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            add_generation_prompt=True,
        ).to(self._device)

        input_len = inputs["input_ids"].shape[-1]
        outputs = self.model.generate(**inputs, max_new_tokens=max_new_tokens)
        response = self.processor.decode(
            outputs[0][input_len:], skip_special_tokens=True
        )

        return response



In [ ]:
"""
兩階段辨識 Pipeline — JungleSurvivor 核心邏輯。

流程：
  Stage 1: 危險物種比對 → 取信心度最高的 3 個
  Stage 2: 可食用/藥用物種比對 → 取信心度最高的 3 個
  Merge:   合併 6 個候選 → 依信心度排序 → 取前 3 名
  Confusion: 為前 3 名附上混淆物種對資訊 + 進一步觀察指引

信心度 = 絕對特徵吻合度（越多特徵符合，信心度越高）。
"""

from dataclasses import dataclass, field
from typing import Optional
from PIL import Image



@dataclass
class ConfusionInfo:
    """某候選物種的混淆警告"""
    candidate_name: str
    confused_with: str
    pair_id: str
    distinguishing_features: list[str] = field(default_factory=list)
    interactive_tests: list[str] = field(default_factory=list)


@dataclass
class PipelineResult:
    """Pipeline 最終輸出"""
    warning_level: str            # RED / YELLOW / GREEN / GRAY
    summary_zh: str               # 給使用者看的摘要
    candidates: list[CandidateMatch] = field(default_factory=list)
    observed_features: list[str] = field(default_factory=list)
    reasoning: str = ""
    confusion_warnings: list[ConfusionInfo] = field(default_factory=list)
    interactive_guidance: Optional[str] = None
    raw_responses: list[str] = field(default_factory=list)


class JungleSurvivorPipeline:
    """兩階段辨識 Pipeline"""

    def __init__(self, model):
        self.model = model

    def identify(
        self,
        images: list[Image.Image],
        context: Optional[EnvironmentContext] = None,
        mode: str = "auto",
        description: Optional[str] = None,
    ) -> PipelineResult:
        if context is None:
            context = create_context()

        kb = load_knowledge_base(context.region_id)

        target_type = "animal" if mode == "animal" else "plant"

        # ═══ Stage 1：危險物種比對 ═══
        danger_candidates, danger_obs, danger_reasoning, danger_raw = \
            self._run_stage(images, context, kb, "danger", target_type, description)

        # ═══ Stage 2：可食用/藥用物種比對 ═══
        if mode == "animal":
            edible_candidates, edible_obs, edible_reasoning, edible_raw = [], [], "", ""
        else:
            edible_candidates, edible_obs, edible_reasoning, edible_raw = \
                self._run_stage(images, context, kb, "edible", target_type, description)

        raw_responses = [r for r in [danger_raw, edible_raw] if r]

        # ═══ Merge：合併 6 → 取前 3 ═══
        all_six = danger_candidates + edible_candidates
        all_six.sort(key=lambda c: c.confidence, reverse=True)
        top3 = all_six[:3]

        for i, c in enumerate(top3, 1):
            c.rank = i

        all_obs = list(dict.fromkeys(danger_obs + edible_obs))

        reasoning_parts = []
        if danger_reasoning:
            reasoning_parts.append(f"【危險物種分析】\n{danger_reasoning}")
        if edible_reasoning:
            reasoning_parts.append(f"【可食用物種分析】\n{edible_reasoning}")
        combined_reasoning = "\n\n".join(reasoning_parts)

        # ═══ Confusion：為前 3 名檢查混淆物種 ═══
        confusion_warnings, guidance_text = self._check_confusion_pairs(
            images, context, kb, top3
        )

        # ═══ 決定警告等級與摘要 ═══
        warning_level, summary = self._build_summary(top3, confusion_warnings, guidance_text)

        return PipelineResult(
            warning_level=warning_level,
            summary_zh=summary,
            candidates=top3,
            observed_features=all_obs,
            reasoning=combined_reasoning,
            confusion_warnings=confusion_warnings,
            interactive_guidance=guidance_text,
            raw_responses=raw_responses,
        )

    # ───────────────────────────────────────────────────────────
    # 單階段執行
    # ───────────────────────────────────────────────────────────

    def _run_stage(
        self,
        images: list[Image.Image],
        context: EnvironmentContext,
        kb: KnowledgeBase,
        stage: str,
        target_type: str,
        description: Optional[str],
    ) -> tuple[list[CandidateMatch], list[str], str, str]:
        """
        執行單一階段的 AI 辨識。
        回傳 (candidates, observed_features, reasoning, raw_response)。
        """
        if stage == "danger":
            prompt = build_danger_prompt(context, kb, target_type)
        else:
            prompt = build_edible_prompt(context, kb)

        if len(images) > 1:
            prompt = build_multi_photo_wrapper(prompt, len(images))

        if description:
            prompt += f"\n\n使用者補充描述：{description}"

        response = self.model.generate(prompt, images)
        result = parse_unified_response(response)

        if not result.parse_success:
            return [], [], "", response

        reasoning = ""
        if result.json_data:
            reasoning = result.json_data.get("reasoning_summary", "")

        candidates = result.candidates[:3]
        return candidates, result.observed_features, reasoning, response

    # ───────────────────────────────────────────────────────────
    # 混淆物種偵測
    # ───────────────────────────────────────────────────────────

    def _check_confusion_pairs(
        self,
        images: list[Image.Image],
        context: EnvironmentContext,
        kb: KnowledgeBase,
        top3: list[CandidateMatch],
    ) -> tuple[list[ConfusionInfo], Optional[str]]:
        """
        對前 3 名候選檢查是否有已知的混淆物種對。
        如果有，附上區分特徵與互動測試指引。
        """
        confusion_infos: list[ConfusionInfo] = []
        guidance_parts: list[str] = []

        for candidate in top3:
            matched_pair = self._find_confusion_pair(kb, candidate)
            if not matched_pair:
                continue

            safe = matched_pair["safe_species"]
            danger = matched_pair["dangerous_species"]

            distinguishing = []
            for row in matched_pair.get("comparison_table", []):
                distinguishing.append(
                    f"{row['feature']}：安全→{row['safe']}，危險→{row['danger']}"
                )

            test_names = [
                t["test_name"]
                for t in matched_pair.get("interactive_tests", [])
            ]

            other = danger["common_name_zh"] \
                if candidate.common_name_zh == safe["common_name_zh"] \
                else safe["common_name_zh"]

            info = ConfusionInfo(
                candidate_name=candidate.common_name_zh,
                confused_with=other,
                pair_id=matched_pair["id"],
                distinguishing_features=distinguishing,
                interactive_tests=test_names,
            )
            confusion_infos.append(info)

            guidance = build_interactive_test_guidance(matched_pair)
            guidance_parts.append(
                f"⚠️ 「{candidate.common_name_zh}」與「{other}」容易混淆：\n{guidance}"
            )

        guidance_text = "\n\n".join(guidance_parts) if guidance_parts else None
        return confusion_infos, guidance_text

    def _find_confusion_pair(
        self, kb: KnowledgeBase, candidate: CandidateMatch
    ) -> Optional[dict]:
        """在知識庫的混淆物種對中尋找與此候選相關的配對。"""
        candidate_sci = candidate.scientific_name.lower().replace(" ", "_")
        candidate_zh = candidate.common_name_zh

        for pair in kb.confusion_pairs:
            safe = pair["safe_species"]
            danger = pair["dangerous_species"]

            safe_names = {
                safe.get("id", ""),
                safe.get("scientific_name", "").lower().replace(" ", "_"),
                safe.get("common_name_zh", ""),
            }
            danger_names = {
                danger.get("id", ""),
                danger.get("scientific_name", "").lower().replace(" ", "_"),
                danger.get("common_name_zh", ""),
            }

            if candidate_sci in safe_names or candidate_sci in danger_names:
                return pair
            if candidate_zh in safe_names or candidate_zh in danger_names:
                return pair

        return None

    # ───────────────────────────────────────────────────────────
    # 決定警告等級與摘要
    # ───────────────────────────────────────────────────────────

    def _build_summary(
        self,
        top3: list[CandidateMatch],
        confusion_warnings: list[ConfusionInfo],
        guidance_text: Optional[str],
    ) -> tuple[str, str]:
        """根據前 3 候選的類別與信心度決定警告等級和摘要文字。"""
        if not top3:
            return "GRAY", "⚪ 無法辨識此物種，建議不要接觸或食用。\n請嘗試拍攝更清楚的照片。"

        first = top3[0]
        has_danger_in_top = any(c.category == "dangerous" for c in top3)
        top_is_danger = first.category == "dangerous"
        has_confusion = len(confusion_warnings) > 0
        danger_threshold = THRESHOLDS.get("danger_screening", 60)

        # --- 1) 最高信心的是危險物種 ---
        if top_is_danger and first.confidence >= danger_threshold:
            lines = [
                f"🔴 **警告：最可能為危險物種！**",
                f"",
                f"最高匹配：**{first.common_name_zh}** (*{first.scientific_name}*)",
                f"信心度：{first.confidence}%（{len(first.key_matching_features)} 項特徵吻合）",
            ]
            if first.danger_info:
                tox = first.danger_info.get("toxicity", "")
                if tox:
                    lines.append(f"⚠️ 毒性：{tox}")
                fa = first.danger_info.get("first_aid", "")
                if fa:
                    lines.append(f"🩹 急救：{fa}")
            lines.append("")
            lines.append("📋 完整候選排名：")
            for c in top3:
                cat_label = "⚠️危險" if c.category == "dangerous" else "🍃可食" if c.category == "edible" else "💊藥用"
                lines.append(f"  #{c.rank} {c.common_name_zh}（{cat_label}）— {c.confidence}%")

            if has_confusion:
                lines.append("")
                lines.append("🔀 混淆物種警告：")
                for cw in confusion_warnings:
                    lines.append(f"  「{cw.candidate_name}」⟷「{cw.confused_with}」")
                lines.append("")
                lines.append("👉 請參考下方「互動測試指引」做進一步區分！")

            return "RED", "\n".join(lines)

        # --- 2) 前 3 中有危險物種但非最高 ---
        if has_danger_in_top:
            lines = [
                f"🟡 **注意：候選中包含危險物種，需進一步確認！**",
                f"",
                f"最高匹配：**{first.common_name_zh}** (*{first.scientific_name}*)",
                f"信心度：{first.confidence}%",
                "",
                "📋 完整候選排名：",
            ]
            for c in top3:
                cat_label = "⚠️危險" if c.category == "dangerous" else "🍃可食" if c.category == "edible" else "💊藥用"
                lines.append(f"  #{c.rank} {c.common_name_zh}（{cat_label}）— {c.confidence}%")

            lines.extend(self._build_verification_guidance(top3, confusion_warnings))
            return "YELLOW", "\n".join(lines)

        # --- 3) 前 3 都是可食/藥用但有混淆 ---
        if has_confusion:
            lines = [
                f"🟡 **辨識為可食用物種，但存在混淆風險！**",
                f"",
                f"最高匹配：**{first.common_name_zh}** (*{first.scientific_name}*)",
                f"信心度：{first.confidence}%",
                "",
                "📋 完整候選排名：",
            ]
            for c in top3:
                cat_label = "🍃可食" if c.category == "edible" else "💊藥用"
                lines.append(f"  #{c.rank} {c.common_name_zh}（{cat_label}）— {c.confidence}%")

            lines.append("")
            lines.append("🔀 混淆物種警告：")
            for cw in confusion_warnings:
                lines.append(f"  「{cw.candidate_name}」與「{cw.confused_with}」外觀極相似")

            lines.extend(self._build_verification_guidance(top3, confusion_warnings))
            return "YELLOW", "\n".join(lines)

        # --- 4) 前 3 都是安全物種，無混淆 ---
        if first.confidence >= THRESHOLDS.get("useful_resources", 70):
            lines = [
                f"🟢 **辨識為可利用資源**",
                f"",
                f"最高匹配：**{first.common_name_zh}** (*{first.scientific_name}*)",
                f"信心度：{first.confidence}%（{len(first.key_matching_features)} 項特徵吻合）",
                "",
                "📋 完整候選排名：",
            ]
            for c in top3:
                cat_label = "🍃可食" if c.category == "edible" else "💊藥用"
                lines.append(f"  #{c.rank} {c.common_name_zh}（{cat_label}）— {c.confidence}%")

            lines.extend(self._build_verification_guidance(top3, confusion_warnings))
            return "GREEN", "\n".join(lines)

        # --- 5) 信心度都偏低 ---
        lines = [
            f"⚪ **信心度不足，無法確定物種。**",
            f"",
            f"最高匹配：{first.common_name_zh}（{first.confidence}%）",
            "",
            "📋 候選排名（信心度均偏低）：",
        ]
        for c in top3:
            cat_label = {"dangerous": "⚠️危險", "edible": "🍃可食", "medicinal": "💊藥用"}.get(c.category, "❓")
            lines.append(f"  #{c.rank} {c.common_name_zh}（{cat_label}）— {c.confidence}%")

        lines.append("")
        lines.append("💡 建議：")
        lines.append("  - 拍攝更多角度的照片（葉面、葉背、花、莖、根部）")
        lines.append("  - 在信心度不足的情況下，切勿食用或觸碰")
        return "GRAY", "\n".join(lines)

    def _build_verification_guidance(
        self,
        top3: list[CandidateMatch],
        confusion_warnings: list[ConfusionInfo],
    ) -> list[str]:
        """產出「如何進一步確認」的指引。"""
        lines = [
            "",
            "━━━━━━━━━━━━━━━━━━━━━━━━━━",
            "🔍 **如何進一步確認？**",
            "",
        ]

        lines.append("📸 **多角度觀察（提升信心度）**：")
        lines.append("  1. 拍攝葉子的正面與背面（觀察葉脈、絨毛、質地）")
        lines.append("  2. 拍攝莖部橫切面（觀察是否中空、有乳汁）")
        lines.append("  3. 拍攝花和果實的近照")
        lines.append("  4. 拍攝整株植物的生長環境")

        if confusion_warnings:
            lines.append("")
            lines.append("🔬 **區分混淆物種的關鍵特徵**：")
            for cw in confusion_warnings:
                lines.append(f"")
                lines.append(f"  「{cw.candidate_name}」vs「{cw.confused_with}」：")
                for feat in cw.distinguishing_features[:5]:
                    lines.append(f"    • {feat}")
                if cw.interactive_tests:
                    lines.append(f"    🧪 實地測試：{'、'.join(cw.interactive_tests)}")

        lines.append("")
        lines.append("🧪 **通用可食性野外測試（不確定時務必執行）**：")
        lines.append("  1. 皮膚測試：取少量汁液塗在手腕內側，等待 15 分鐘觀察是否紅腫")
        lines.append("  2. 唇部測試：將少量放在嘴唇上，等待 5 分鐘觀察是否刺痛")
        lines.append("  3. 舌尖測試：少量放在舌尖上 15 分鐘，注意異常味道或麻痺感")
        lines.append("  4. 微量試食：咀嚼一小口後吐出，等待 8 小時觀察是否有不適")
        lines.append("  ⚠️ 每個步驟之間如有任何不適反應，立即停止並大量喝水！")

        return lines



## 🚀 載入模型

執行此 Cell 載入 Gemma 4 模型（約需 1-2 分鐘）。



In [ ]:
print("正在載入 Gemma 4 E2B 模型...")
_demo_model = GemmaModel()
print("✅ 模型載入完成！")
print(f"   模型：{_demo_model.model_id}")
print(f"   裝置：{_demo_model._device}")



## 🎮 Gradio 互動介面

執行下方 Cell 啟動互動介面。介面啟動後會產生一個公開 URL，可在瀏覽器中開啟。

**使用方式：**
1. 點擊「載入模型」（如果上方已載入則會直接使用）
2. 上傳照片（支援多張）
3. 選擇辨識模式和環境設定
4. 點擊「開始辨識」



In [ ]:
import gradio as gr
from datetime import datetime as _dt

_pipeline = None
_model_loaded = False

def _init_model():
    global _pipeline, _model_loaded
    if _model_loaded and _pipeline is not None:
        return "✅ 模型已載入，無需重複載入。"
    try:
        if '_demo_model' in dir():
            model = _demo_model
        elif '_demo_model' in globals():
            model = globals()['_demo_model']
        else:
            model = GemmaModel()
        _pipeline = JungleSurvivorPipeline(model)
        _model_loaded = True
        return "✅ 模型載入成功！可以開始辨識。"
    except Exception as e:
        _model_loaded = False
        return f"❌ 模型載入失敗：{e}"

_S = {
    "RED":    {"bg":"#FFF0F0","bd":"#FF0000","tx":"#CC0000","em":"🔴"},
    "YELLOW": {"bg":"#FFFBE6","bd":"#FFD700","tx":"#996600","em":"🟡"},
    "GREEN":  {"bg":"#F0FFF4","bd":"#00AA00","tx":"#006600","em":"🟢"},
    "GRAY":   {"bg":"#F5F5F5","bd":"#888888","tx":"#555555","em":"⚪"},
}
_WL = {"RED":"🔴 危險","YELLOW":"🟡 注意","GREEN":"🟢 安全","GRAY":"⚪ 不確定"}

def _result_html(r):
    s = _S.get(r.warning_level, _S["GRAY"])
    lbl = _WL.get(r.warning_level, "⚪ 不確定")
    sm = r.summary_zh.replace("\n", "<br>")
    return (
        f'<div style="border:3px solid {s["bd"]};border-radius:12px;padding:20px;'
        f'background:{s["bg"]};margin:8px 0;">'
        f'<h2 style="color:{s["tx"]};margin:0 0 12px 0;">{lbl}</h2>'
        f'<div style="font-size:15px;line-height:1.8;color:#333;white-space:pre-line;">{sm}</div></div>'
    )

def _detail_md(r):
    parts = []
    if r.reasoning:
        parts.append("### 🧠 AI 分析\n\n" + r.reasoning)
    if r.candidates:
        parts.append("### 📊 候選排名\n")
        for c in r.candidates:
            cat_icon = {"dangerous": "⚠️", "edible": "🍃", "medicinal": "💊"}.get(c.category, "❓")
            parts.append(f"**#{c.rank}** {cat_icon} {c.common_name_zh} (*{c.scientific_name}*) — {c.confidence}%")
            if c.key_matching_features:
                parts.append(f"   匹配特徵：{'、'.join(c.key_matching_features)}")
            if c.danger_info and c.danger_info.get("toxicity"):
                parts.append(f"   ⚠️ 毒性：{c.danger_info['toxicity']}")
    if r.observed_features:
        parts.append("\n### 🔍 觀察到的特徵\n\n" + "、".join(r.observed_features))
    if hasattr(r, 'confusion_warnings') and r.confusion_warnings:
        parts.append("\n### 🔀 混淆物種對照\n")
        for cw in r.confusion_warnings:
            parts.append(f"**「{cw.candidate_name}」⟷「{cw.confused_with}」**")
            if cw.distinguishing_features:
                parts.append("關鍵區別：")
                for feat in cw.distinguishing_features[:5]:
                    parts.append(f"- {feat}")
            if cw.interactive_tests:
                parts.append(f"🧪 建議測試：{'、'.join(cw.interactive_tests)}")
    return "\n\n".join(parts) if parts else "（無詳細分析）"

def _inter_html(r):
    if not r.interactive_guidance:
        return "<p style='color:#888;'>（不需要互動測試）</p>"
    g = r.interactive_guidance.replace("\n", "<br>")
    return (
        '<div style="border:3px solid #FF6600;border-radius:12px;padding:20px;'
        'background:#FFF8F0;margin:8px 0;">'
        '<h3 style="color:#CC5500;margin:0 0 10px 0;">🔬 互動測試指引</h3>'
        f'<div style="font-size:14px;line-height:1.8;color:#333;">{g}</div></div>'
    )

def _hist_html(h):
    if not h:
        return "<p style='color:#888;text-align:center;'>尚無辨識紀錄</p>"
    rows = ""
    for i, x in enumerate(reversed(h)):
        s = _S.get(x["level"], _S["GRAY"])
        rows += (
            f'<tr style="background:{s["bg"]};">'
            f'<td style="padding:8px;border-bottom:1px solid #ddd;">{len(h)-i}</td>'
            f'<td style="padding:8px;border-bottom:1px solid #ddd;">{x["time"]}</td>'
            f'<td style="padding:8px;border-bottom:1px solid #ddd;">{s["em"]} {x["species"]}</td>'
            f'<td style="padding:8px;border-bottom:1px solid #ddd;">{x["conf"]}%</td>'
            f'<td style="padding:8px;border-bottom:1px solid #ddd;">{x["mode"]}</td></tr>'
        )
    return (
        '<table style="width:100%;border-collapse:collapse;font-size:14px;">'
        '<thead><tr style="background:#f0f0f0;">'
        '<th style="padding:8px;text-align:left;">#</th>'
        '<th style="padding:8px;text-align:left;">時間</th>'
        '<th style="padding:8px;text-align:left;">辨識結果</th>'
        '<th style="padding:8px;text-align:left;">信心度</th>'
        '<th style="padding:8px;text-align:left;">模式</th>'
        f'</tr></thead><tbody>{rows}</tbody></table>'
    )

def _open_image(im):
    from PIL import Image as _Img
    if isinstance(im, _Img.Image):
        return im.convert("RGB")
    if isinstance(im, tuple):
        return _Img.open(im[0]).convert("RGB")
    if isinstance(im, dict):
        path = im.get("name") or im.get("path") or im.get("image", {}).get("path", "")
        if isinstance(path, dict):
            path = path.get("path", "")
        return _Img.open(path).convert("RGB")
    if isinstance(im, str):
        return _Img.open(im).convert("RGB")
    return _Img.open(str(im)).convert("RGB")

def _identify(images, desc, country, alt, veg, mode, hist):
    if not _model_loaded or _pipeline is None:
        return "<p style='color:red;'>❌ 請先載入模型</p>", "", "", _hist_html(hist), hist
    if images is None or len(images) == 0:
        return "<p style='color:red;'>❌ 請上傳至少一張照片</p>", "", "", _hist_html(hist), hist

    pil = []
    for im in images:
        try:
            pil.append(_open_image(im))
        except Exception as img_err:
            return (
                f"<p style='color:red;'>❌ 照片載入失敗：{img_err}<br>type={type(im)}, value={repr(im)[:200]}</p>",
                "", "", _hist_html(hist), hist,
            )

    ctx = create_context(country=country, altitude=int(alt), vegetation_zone=veg)
    _mm = {
        "自動（兩階段辨識）": "auto",
        "這個有毒嗎？（危險快篩）": "auto",
        "這個能吃嗎？（食物辨識）": "auto",
        "這是什麼動物？（動物辨識）": "animal",
    }
    pm = _mm.get(mode, "auto")
    d = desc.strip() if desc and desc.strip() else None

    try:
        res = _pipeline.identify(pil, ctx, mode=pm, description=d)
        sp = res.candidates[0].common_name_zh if res.candidates else "未知"
        conf = res.candidates[0].confidence if res.candidates else 0
        hist.append({
            "time": _dt.now().strftime("%H:%M:%S"),
            "species": sp, "conf": conf,
            "level": res.warning_level,
            "mode": mode.split("（")[0],
        })
        return _result_html(res), _detail_md(res), _inter_html(res), _hist_html(hist), hist
    except Exception as e:
        import traceback
        traceback.print_exc()
        return f"<p style='color:red;'>❌ 辨識過程發生錯誤：{e}</p>", "", "", _hist_html(hist), hist

with gr.Blocks(title="JungleSurvivor", theme=gr.themes.Soft()) as demo:
    hist_state = gr.State([])
    gr.HTML(
        '<div style="text-align:center;">'
        '<h1>🌿 JungleSurvivor — 叢林求生離線 AI 助手</h1>'
        '<p style="color:#666;">拍攝照片，AI 比對危險物種與可食物種，取信心度最高的前三名候選。<br>'
        '信心度 = 形態特徵吻合度（絕對值，越多特徵符合分數越高）<br>'
        '<b>安全第一</b> — 寧可誤報也不漏報。無法確定時一律視為危險。</p></div>'
    )

    with gr.Row():
        load_btn = gr.Button("🚀 載入模型", variant="primary", scale=1)
        load_st = gr.Textbox(label="載入狀態", interactive=False, scale=3)
    load_btn.click(fn=_init_model, inputs=[], outputs=[load_st])
    gr.HTML("<hr>")

    with gr.Row():
        with gr.Column(scale=1):
            img_in = gr.Gallery(label="📷 上傳照片（可多張，多角度可提升信心度）", type="filepath", columns=3, height=280)
            desc_in = gr.Textbox(label="📝 補充描述（選填）", placeholder="例如：葉子搓碎有辛辣味、莖切開有白色乳汁...", lines=2)
            mode_sel = gr.Dropdown(
                choices=["自動（兩階段辨識）", "這個有毒嗎？（危險快篩）",
                         "這個能吃嗎？（食物辨識）", "這是什麼動物？（動物辨識）"],
                value="自動（兩階段辨識）", label="辨識模式")
            gr.Markdown("#### 🌏 環境設定")
            country_in = gr.Textbox(value="台灣", label="地點")
            alt_in = gr.Slider(minimum=0, maximum=4000, value=500, step=100, label="海拔（公尺）")
            veg_in = gr.Dropdown(
                choices=["低海拔闊葉林", "中海拔闊葉林", "中高海拔針闊混合林",
                         "高海拔針葉林", "高山草原", "海岸地帶", "溪邊濕地"],
                value="低海拔闊葉林", label="植被帶")
            alt_in.change(fn=get_vegetation_zone, inputs=[alt_in], outputs=[veg_in])
            id_btn = gr.Button("🔍 開始辨識", variant="primary", size="lg")

        with gr.Column(scale=1):
            main_out = gr.HTML(label="辨識結果")
            with gr.Accordion("🔬 混淆物種 & 互動測試指引", open=True):
                inter_out = gr.HTML()
            with gr.Accordion("📋 詳細分析（Chain of Thought）", open=False):
                detail_out = gr.Markdown()

    gr.HTML("<hr>")
    with gr.Accordion("📜 歷史紀錄", open=True):
        hist_disp = gr.HTML(value="<p style='color:#888;text-align:center;'>尚無辨識紀錄</p>")

    id_btn.click(
        fn=_identify,
        inputs=[img_in, desc_in, country_in, alt_in, veg_in, mode_sel, hist_state],
        outputs=[main_out, detail_out, inter_out, hist_disp, hist_state],
    )

    gr.HTML(
        '<hr><div style="text-align:center;color:#888;font-size:13px;">'
        f'<b>辨識流程</b> | Stage 1：危險物種比對（Top 3）→ Stage 2：可食/藥用比對（Top 3）→ 合併取 Top 3<br>'
        f'<b>信心度</b> = 特徵吻合的絕對值（越多特徵符合分數越高，與排名無關）<br>'
        '<b>免責聲明</b>：本工具僅供參考，不能取代專業鑑定。在野外，寧可不吃也不誤食。</div>'
    )

demo.launch(share=True, debug=True)

